# DiaPredict - Revisi Pengujian V3 (NOTEBOOK GABUNGAN)
## Prediksi Diabetes Tipe 2: Random Forest vs KNN vs SVM

Notebook ini adalah **gabungan dari tujuh notebook** `00` sampai `06` di folder
`Revisi_Pengujian_V3` menjadi satu berkas, dengan preamble yang dimuat sekali saja.
Isi kode tiap bagian tidak diubah, sehingga hasilnya identik dengan menjalankan
ketujuh notebook secara terpisah.

---

### Catatan revisi penguji yang dijawab notebook ini

| No | Catatan penguji | Dijawab di |
|----|-----------------|------------|
| 1 | "tidak ada pemilihan knp k=20 dr knn" | BAGIAN 3 |
| 2 | "knp hyperplane pada svm yg dipilih" | BAGIAN 4 |
| 3 | "knp splitting data pake 80:20" | BAGIAN 2 |
| 4 | "Diperbanyak pengujiannya" | BAGIAN 5 dan BAGIAN 6 |

---

### Peta bagian

| Bagian | Isi | Keluaran JSON |
|--------|-----|---------------|
| **BAGIAN 1** | Persiapan data, EDA, verifikasi reproduksi V2 | `hasil_baseline_v2` |
| **BAGIAN 2** | Rasio split: 7 rasio x 5 seed, learning curve, margin of error | `hasil_split_ratio` |
| **BAGIAN 3** | Sweep k=1..51, kurva bias-variance, aturan one-standard-error | `hasil_pemilihan_k` |
| **BAGIAN 4** | Perbandingan kernel, grid C x gamma, analisis margin & support vector | `hasil_svm_hyperplane` |
| **BAGIAN 5** | Repeated CV, nested CV, 4 uji statistik, 8 strategi threshold, kalibrasi | `hasil_validasi_statistik` |
| **BAGIAN 6** | Ablation resampling & fitur, robustness, subgrup, matriks keputusan | `hasil_ablation_robustness` |
| **BAGIAN 7** | Model final + ekspor artefak produksi dan `experiments.json` | `experiments` |

---

### Cara menjalankan

1. Atur `PAKAI_DRIVE = True` pada **CELL 2** agar hasil tersimpan permanen.
2. Atur `MODE_CEPAT` pada **PANEL KENDALI** (satu tempat saja, tepat setelah preamble).
3. **Runtime -> Run all**.

Jalankan dulu dengan `MODE_CEPAT = True` untuk memastikan seluruh sel berjalan,
lalu ulangi dengan `MODE_CEPAT = False` untuk memperoleh angka final skripsi.

> **Menjalankan sebagian saja.** Setiap bagian bersifat mandiri: bagian mana pun
> hanya membutuhkan CELL 1-6 (preamble) dan PANEL KENDALI, tidak membutuhkan
> variabel dari bagian lain. Jadi bila runtime terputus, cukup jalankan ulang
> CELL 1-6 + PANEL KENDALI, lalu lanjutkan dari bagian yang tertunda.
> Pengecualian: **BAGIAN 7** membaca berkas JSON hasil BAGIAN 1-6, jadi bagian itu
> harus dijalankan paling akhir (ia tetap berjalan dengan nilai cadangan bila ada
> hasil yang belum tersedia, sambil mencetak peringatan).


---
# PERSIAPAN: Library, Konstanta, Data, dan Pipeline

Enam cell berikut adalah fondasi bersama seluruh bagian: instalasi, import & konstanta,
fungsi penyimpanan hasil, pemuatan & pembersihan data, pabrik pipeline, dan fungsi evaluasi.
**Cukup dijalankan sekali** untuk seluruh notebook.

In [ ]:
# ============================================================
# CELL 1: Instalasi Library
# ============================================================
!pip install -q pandas numpy matplotlib seaborn scikit-learn imbalanced-learn statsmodels kagglehub

In [ ]:
# ============================================================
# CELL 2: Import & Konstanta Global
# ============================================================
import os, json, time, math, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedShuffleSplit,
    RepeatedStratifiedKFold, cross_validate, cross_val_predict,
    learning_curve, validation_curve, GridSearchCV, RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, confusion_matrix, classification_report,
    brier_score_loss
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

SELECTED_FEATURES = ['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level']
FEATURE_LABELS    = ['Usia', 'BMI', 'Hipertensi', 'HbA1c', 'Kadar Glukosa']
TARGET            = 'diabetes'

WARNA_MODEL = {'Random Forest': '#3498db', 'KNN': '#e74c3c', 'SVM (Linear)': '#2ecc71'}
WARNA_AKSEN = '#f39c12'

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
sns.set_style('whitegrid')

# --- Folder output -------------------------------------------------------
# Set PAKAI_DRIVE = True bila ingin hasil tersimpan permanen di Google Drive
# (WAJIB True kalau ingin notebook 06 membaca hasil notebook 01-05).
PAKAI_DRIVE = False

if PAKAI_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/DiaPredict_Revisi'
else:
    OUTPUT_DIR = '/content/hasil_revisi'

for sub in ['', '/tabel', '/gambar', '/json']:
    os.makedirs(OUTPUT_DIR + sub, exist_ok=True)

print(f'Folder output : {OUTPUT_DIR}')
print(f'Fitur         : {SELECTED_FEATURES}')

In [ ]:
# ============================================================
# CELL 3: Fungsi Utilitas Penyimpanan Hasil
# ============================================================
def simpan_tabel(df, nama, tampilkan=True):
    """Simpan DataFrame ke CSV di OUTPUT_DIR/tabel dan tampilkan."""
    path = f'{OUTPUT_DIR}/tabel/{nama}.csv'
    df.to_csv(path, index=False)
    print(f'[TABEL DISIMPAN] {path}')
    if tampilkan:
        display(df)
    return df

def simpan_json(obj, nama):
    """Simpan dict/list hasil eksperimen ke JSON (dipakai notebook 06 & website)."""
    path = f'{OUTPUT_DIR}/json/{nama}.json'
    def _konversi(o):
        if isinstance(o, (np.integer,)):  return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.ndarray,)):  return o.tolist()
        if isinstance(o, (np.bool_,)):    return bool(o)
        return str(o)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_konversi)
    print(f'[JSON DISIMPAN] {path}')
    return obj

def simpan_gambar(nama, fig=None, dpi=150):
    """Simpan figure matplotlib aktif ke OUTPUT_DIR/gambar."""
    path = f'{OUTPUT_DIR}/gambar/{nama}.png'
    (fig or plt).savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'[GAMBAR DISIMPAN] {path}')
    return path

def garis(judul='', lebar=70):
    print('=' * lebar)
    if judul:
        print(f'  {judul}')
        print('=' * lebar)

In [ ]:
# ============================================================
# CELL 4: Load Dataset + Cleaning + Winsorization
# (Identik dengan pipeline notebook V2 agar hasil dapat dibandingkan)
# ============================================================
import kagglehub

def muat_dan_bersihkan_data(verbose=True):
    path = kagglehub.dataset_download('iammustafatz/diabetes-prediction-dataset')
    csv_file = os.path.join(path, 'diabetes_prediction_dataset.csv')
    df_raw = pd.read_csv(csv_file)

    # 1) Hapus duplikat pada dataset penuh (SAMA seperti V2 -> sisa 96.146 baris)
    df = df_raw.drop_duplicates().reset_index(drop=True)

    # 2) Ambil 5 fitur terpilih + target
    df = df[SELECTED_FEATURES + [TARGET]].copy()

    # 3) Winsorization (capping IQR) hanya untuk fitur numerik non-biner
    numeric_feats = [f for f in SELECTED_FEATURES if df[f].nunique() > 2]
    ringkas = []
    for feat in numeric_feats:
        Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
        IQR = Q3 - Q1
        low, up = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_cap = int(((df[feat] < low) | (df[feat] > up)).sum())
        df[feat] = df[feat].clip(lower=low, upper=up)
        ringkas.append({'fitur': feat, 'batas_bawah': low, 'batas_atas': up, 'n_dicapping': n_cap})

    if verbose:
        garis('DATA SIAP PAKAI')
        print(f'Baris (setelah hapus duplikat) : {len(df):,}')
        print(f'Distribusi kelas               : '
              f'{(df[TARGET]==0).sum():,} sehat / {(df[TARGET]==1).sum():,} diabetes '
              f'({df[TARGET].mean()*100:.2f}% positif)')
        display(pd.DataFrame(ringkas))
    return df

df_clean = muat_dan_bersihkan_data()
X_all = df_clean[SELECTED_FEATURES].copy()
y_all = df_clean[TARGET].copy()

In [ ]:
# ============================================================
# CELL 5: Pabrik Pipeline Model (anti data leakage)
# Urutan: StandardScaler -> SMOTE -> Classifier (imblearn Pipeline,
# sehingga SMOTE HANYA aktif saat fit, tidak saat predict/validasi)
# ============================================================

# Hyperparameter terbaik hasil tuning notebook V2 (baseline pembanding)
PARAM_RF_V2  = dict(n_estimators=200, max_depth=10, min_samples_split=5,
                    min_samples_leaf=4, max_features='log2', criterion='entropy',
                    class_weight='balanced')
PARAM_KNN_V2 = dict(n_neighbors=21, weights='uniform', metric='euclidean', leaf_size=20)
PARAM_SVM_V2 = dict(C=0.1, max_iter=3000)

def buat_pipeline_rf(pakai_smote=True, **params):
    p = {**PARAM_RF_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_knn(pakai_smote=True, **params):
    p = {**PARAM_KNN_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', KNeighborsClassifier(n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_svm(pakai_smote=True, kernel='linear', C=0.1, gamma='scale',
                      degree=3, max_iter=3000, kalibrasi='sigmoid'):
    """kernel='linear' -> LinearSVC (cepat). Kernel lain -> SVC."""
    if kernel == 'linear':
        base = LinearSVC(C=C, max_iter=max_iter, class_weight='balanced',
                         dual=False, random_state=RANDOM_STATE)
    else:
        base = SVC(kernel=kernel, C=C, gamma=gamma, degree=degree,
                   class_weight='balanced', random_state=RANDOM_STATE)
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', CalibratedClassifierCV(base, cv=3, method=kalibrasi)))
    return ImbPipeline(langkah)

PABRIK_MODEL = {
    'Random Forest': buat_pipeline_rf,
    'KNN'          : buat_pipeline_knn,
    'SVM (Linear)' : buat_pipeline_svm,
}

In [ ]:
# ============================================================
# CELL 6: Fungsi Evaluasi Standar (dipakai seluruh notebook)
# ============================================================
def threshold_youden(y_true, y_proba):
    fpr, tpr, thr = roc_curve(y_true, y_proba)
    return float(thr[np.argmax(tpr - fpr)])

def hitung_metrik(y_true, y_pred, y_proba=None):
    hasil = {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall'   : recall_score(y_true, y_pred, zero_division=0),
        'f1'       : f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        hasil['roc_auc']  = roc_auc_score(y_true, y_proba)
        hasil['ap_score'] = average_precision_score(y_true, y_proba)
        hasil['brier']    = brier_score_loss(y_true, y_proba)
    return hasil

def evaluasi_holdout(model, X_tr, y_tr, X_te, y_te, tuning_threshold=True):
    """Fit -> prediksi -> metrik pada threshold 0.5 dan threshold Youden."""
    t0 = time.time(); model.fit(X_tr, y_tr); waktu_latih = time.time() - t0
    t0 = time.time(); y_proba = model.predict_proba(X_te)[:, 1]; waktu_infer = time.time() - t0

    thr = threshold_youden(y_te, y_proba) if tuning_threshold else 0.5
    m_def  = hitung_metrik(y_te, (y_proba >= 0.5).astype(int), y_proba)
    m_tune = hitung_metrik(y_te, (y_proba >= thr).astype(int), y_proba)
    return {
        'threshold': thr,
        'waktu_latih_s': waktu_latih,
        'waktu_infer_ms': waktu_infer * 1000,
        **{f'{k}_default': v for k, v in m_def.items()},
        **{f'{k}_tuned'  : v for k, v in m_tune.items()},
    }

def ci95_proporsi(p, n):
    """Confidence interval 95% (Wald) untuk metrik berbasis proporsi (mis. recall)."""
    if n == 0: return (np.nan, np.nan, np.nan)
    se = math.sqrt(max(p * (1 - p), 1e-12) / n)
    return (p - 1.96 * se, p + 1.96 * se, 1.96 * se)

---
## PANEL KENDALI

Satu-satunya sakelar biaya komputasi untuk **seluruh** notebook. Pada versi tujuh
notebook terpisah, nilai ini harus diubah di lima berkas berbeda; di sini cukup sekali.

In [ ]:
# ============================================================
# PANEL KENDALI - berlaku untuk SELURUH bagian di notebook ini
# ============================================================
# MODE_CEPAT = True  -> subsample + grid diperkecil. Untuk MENGUJI ALUR.
# MODE_CEPAT = False -> data penuh 96.146 baris. Untuk ANGKA FINAL SKRIPSI.
MODE_CEPAT = True

garis('PANEL KENDALI')
print(f'MODE_CEPAT   : {MODE_CEPAT}')
if MODE_CEPAT:
    print('Mode         : UJI ALUR (subsample + grid kecil)')
    print('')
    print('PERINGATAN: angka dari mode ini TIDAK boleh dilaporkan di skripsi karena')
    print('            hanya memakai sebagian data dan ruang pencarian yang diperkecil.')
    print('            Setelah seluruh sel terbukti berjalan, ubah menjadi False dan')
    print('            jalankan ulang (Runtime -> Run all).')
else:
    print('Mode         : DATA PENUH (angka final skripsi)')
    print('')
    print('Siapkan waktu beberapa jam. Setiap eksperimen berat menyimpan hasilnya')
    print('segera setelah selesai, sehingga runtime yang terputus tidak menghapus')
    print('seluruh kemajuan.')
print('')
print(f'Folder output: {OUTPUT_DIR}')
print(f'Data siap    : {len(df_clean):,} baris, {df_clean[TARGET].mean()*100:.2f}% kelas positif')
garis()

---
---

# BAGIAN 1: PERSIAPAN DATA & EKSPLORASI

Fondasi seluruh eksperimen: pembersihan data, EDA, dan verifikasi bahwa angka penelitian V2 dapat direproduksi.

*Sumber: `00_Setup_dan_Eksplorasi_Data.ipynb`. Penomoran `CELL n` di bawah mengikuti notebook aslinya
agar rujukan silang di dalam kode tetap sahih.*

# Notebook 00 - Setup dan Eksplorasi Data
## Revisi Pengujian V3 - Skripsi Prediksi Diabetes (Random Forest vs KNN vs SVM)

Notebook ini adalah **fondasi** dari seluruh rangkaian revisi V3. Semua notebook lanjutan
(`01` sampai `06`) memakai preamble, konstanta, pipeline, dan fungsi evaluasi yang **sama persis**
dengan yang didefinisikan di sini, sesuai kontrak teknis pada berkas `_SPEC_BERSAMA.md`.
Dengan begitu setiap angka yang muncul di notebook mana pun berasal dari basis data dan
prosedur yang identik, sehingga antar-eksperimen dapat dibandingkan secara adil.

---

### Latar belakang: empat poin revisi dari penguji

Pada sidang sebelumnya (versi V2), penguji menyampaikan empat catatan utama:

| No | Catatan penguji | Inti masalah |
|:--:|---|---|
| **1** | Tidak ada justifikasi pemilihan **k = 20/21** pada algoritme KNN | Nilai k hanya disebut sebagai "hasil tuning" tanpa bukti eksperimen, kurva, maupun aturan pemilihan yang eksplisit |
| **2** | Tidak ada justifikasi pemilihan **hyperplane** pada SVM | Tidak dijelaskan mengapa kernel linear dipilih, bagaimana parameter C menentukan margin, dan bagaimana bidang pemisah terbentuk |
| **3** | Tidak ada justifikasi rasio pembagian data **80:20** | Rasio 80:20 dipakai begitu saja tanpa membandingkan alternatif rasio lain maupun analisis stabilitas estimasi |
| **4** | **Pengujian kurang banyak** | Evaluasi hanya bersandar pada satu kali holdout split, tanpa validasi silang berulang, uji statistik, uji ketahanan, maupun analisis ablasi |

---

### Peta jawaban: notebook mana menjawab poin apa

| Notebook | Isi utama | Menjawab poin revisi |
|---|---|:--:|
| `00_Setup_dan_Eksplorasi_Data.ipynb` | Preamble bersama, EDA, deteksi outlier, verifikasi reproduksi angka V2 | Fondasi (semua poin) |
| `01_Justifikasi_Rasio_Split.ipynb` | Perbandingan rasio 60:40 s.d. 90:10, learning curve, margin of error, stabilitas estimasi | **Poin 3** |
| `02_Justifikasi_Pemilihan_K_KNN.ipynb` | Sweep nilai k, kurva elbow, aturan one-standard-error, grid `weights` x `metric` | **Poin 1** |
| `03_Justifikasi_Hyperplane_SVM.ipynb` | Perbandingan kernel, grid C/gamma, analisis margin dan support vector, visualisasi hyperplane, bobot w | **Poin 2** |
| `04_Validasi_Statistik_dan_Threshold.ipynb` | Repeated Stratified CV, nested CV, uji statistik antar model, strategi threshold, kurva kalibrasi | **Poin 4** |
| `05_Ablation_Robustness_dan_Keputusan_Model.ipynb` | Ablasi resampling dan fitur, uji ketahanan terhadap noise/missing, analisis subgrup, matriks keputusan | **Poin 4** |
| `06_Model_Final_dan_Export_Produksi.ipynb` | Pelatihan model final, ekspor `rf_model.pkl`, `scaler.pkl`, `model_metadata.json`, `experiments.json` | Sinkronisasi web |

---

### Apa yang dikerjakan notebook 00 ini

1. **CELL 1-6** : preamble wajib (instalasi, import dan konstanta global, utilitas penyimpanan,
   pemuatan dan pembersihan data, pabrik pipeline, fungsi evaluasi standar).
2. **CELL 7-10** : eksplorasi data (distribusi kelas, matriks korelasi, distribusi tiap fitur
   per kelas, deteksi outlier IQR sebelum winsorization).
3. **CELL 11** : **verifikasi reproduksi baseline V2** - memastikan pipeline V3 menghasilkan
   angka yang sama dengan laporan skripsi V2, sehingga seluruh eksperimen tambahan berdiri
   di atas basis yang sah.
4. **CELL 12** : menyimpan data bersih agar dapat dipakai ulang oleh notebook lain.
5. **CELL 13** : ringkasan siap salin untuk naskah skripsi.

> **Catatan menjalankan:** jalankan seluruh cell berurutan dari atas ke bawah. Bila ingin hasil
> tersimpan permanen (dan dibaca oleh notebook `06`), ubah `PAKAI_DRIVE = True` pada CELL 2.

---
## Bagian A - Preamble Wajib (CELL 1-6)

Enam cell berikut adalah **kontrak bersama** seluruh notebook revisi V3. Isinya disalin persis
sama di notebook `01` sampai `06`. Jangan diubah sebagian saja, karena perubahan sekecil apa pun
akan membuat angka antar-notebook tidak lagi sebanding.

---
## Bagian B - Eksplorasi Data (CELL 7-10)

Bagian ini menggambarkan karakteristik data yang menjadi dasar seluruh eksperimen revisi.
Empat hal yang diperiksa: (1) seberapa timpang distribusi kelas, (2) seberapa kuat hubungan
antar fitur dan terhadap target, (3) bagaimana sebaran tiap fitur berbeda antara kelompok
sehat dan diabetes, serta (4) seberapa banyak outlier yang ditangani oleh winsorization.

### CELL 7 - Distribusi Kelas

Ketimpangan kelas adalah alasan utama mengapa **akurasi saja tidak cukup** sebagai tolok ukur.
Pada data ini kelas positif (diabetes) hanya sekitar 8,5 persen, sehingga model yang menebak
"semua sehat" pun sudah memperoleh akurasi di atas 91 persen tanpa berguna sama sekali.
Karena itu seluruh notebook revisi menekankan **recall**, **F1**, dan **ROC-AUC**.

In [ ]:
# ============================================================
# BAGIAN 1 | CELL 7: EDA - Distribusi Kelas Target
# ============================================================
garis('EDA 1: DISTRIBUSI KELAS TARGET')

n_total   = len(df_clean)
n_sehat   = int((df_clean[TARGET] == 0).sum())
n_diabet  = int((df_clean[TARGET] == 1).sum())
pct_sehat = n_sehat / n_total * 100
pct_diab  = n_diabet / n_total * 100
rasio_imb = n_sehat / max(n_diabet, 1)

print(f'Total baris          : {n_total:,}')
print(f'Kelas 0 (sehat)      : {n_sehat:,} ({pct_sehat:.2f}%)')
print(f'Kelas 1 (diabetes)   : {n_diabet:,} ({pct_diab:.2f}%)')
print(f'Rasio ketimpangan    : 1 : {rasio_imb:.2f} (positif : negatif)')
print(f'Akurasi "tebak semua sehat" : {pct_sehat:.2f}% -> baseline naif yang menyesatkan')

df_kelas = pd.DataFrame([
    {'kelas': 0, 'label': 'Sehat (Non-Diabetes)', 'jumlah': n_sehat,  'persen': round(pct_sehat, 4)},
    {'kelas': 1, 'label': 'Diabetes',             'jumlah': n_diabet, 'persen': round(pct_diab, 4)},
])
simpan_tabel(df_kelas, 'eda_distribusi_kelas')

fig, ax = plt.subplots(1, 2, figsize=(13, 5))

warna_kelas = ['#3498db', '#e74c3c']
bar = ax[0].bar(df_kelas['label'], df_kelas['jumlah'], color=warna_kelas,
                edgecolor='black', linewidth=0.8, width=0.6)
for b, j, p in zip(bar, df_kelas['jumlah'], df_kelas['persen']):
    ax[0].text(b.get_x() + b.get_width() / 2, b.get_height() + n_total * 0.012,
               f'{j:,}\n({p:.2f}%)', ha='center', va='bottom', fontweight='bold')
ax[0].set_ylabel('Jumlah Sampel')
ax[0].set_title('Distribusi Kelas Target (Jumlah Absolut)', fontweight='bold')
ax[0].set_ylim(0, n_total * 1.08)

ax[1].pie(df_kelas['jumlah'], labels=df_kelas['label'], colors=warna_kelas,
          autopct='%1.2f%%', startangle=90, explode=(0, 0.08),
          wedgeprops={'edgecolor': 'black', 'linewidth': 0.8},
          textprops={'fontsize': 11})
ax[1].set_title('Proporsi Kelas Target', fontweight='bold')
ax[1].grid(False)

plt.suptitle('EDA - Ketimpangan Kelas pada Dataset Prediksi Diabetes',
             fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('eda_distribusi_kelas')
plt.show()

garis('KESIMPULAN EDA 1')
print(f'Dataset sangat tidak seimbang: hanya {pct_diab:.2f}% kasus positif dari {n_total:,} baris.')
print('Implikasi metodologis:')
print('  1. Akurasi tidak dipakai sebagai metrik utama (baseline naif sudah 91,5%).')
print('  2. SMOTE diterapkan HANYA pada data latih di dalam imblearn Pipeline.')
print('  3. Semua pembagian data dan validasi silang bersifat STRATIFIED.')
print('  4. Metrik utama: Recall (menekan false negative), F1, dan ROC-AUC.')

### CELL 8 - Matriks Korelasi

Matriks korelasi Pearson dipakai untuk dua hal: memeriksa **multikolinearitas** antar fitur
(korelasi antar prediktor yang terlalu tinggi akan membuat bobot model tidak stabil, terutama
pada SVM linear) dan melihat **kekuatan hubungan tiap fitur terhadap target**.

In [ ]:
# ============================================================
# BAGIAN 1 | CELL 8: EDA - Matriks Korelasi 5 Fitur + Target
# ============================================================
garis('EDA 2: MATRIKS KORELASI (5 FITUR + TARGET)')

kolom_analisis = SELECTED_FEATURES + [TARGET]
label_analisis = FEATURE_LABELS + ['Diabetes']

korelasi = df_clean[kolom_analisis].corr(method='pearson')
korelasi_label = korelasi.copy()
korelasi_label.index = label_analisis
korelasi_label.columns = label_analisis

df_korelasi = korelasi_label.round(4).reset_index().rename(columns={'index': 'variabel'})
simpan_tabel(df_korelasi, 'eda_korelasi')

fig, ax = plt.subplots(1, 2, figsize=(15, 6))

mask = np.triu(np.ones_like(korelasi_label, dtype=bool), k=1)
sns.heatmap(korelasi_label, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.8,
            cbar_kws={'label': 'Koefisien Korelasi Pearson'}, ax=ax[0])
ax[0].set_title('Matriks Korelasi Antar Variabel', fontweight='bold')
ax[0].grid(False)

kor_target = korelasi[TARGET].drop(TARGET)
urut = kor_target.sort_values(ascending=True)
label_urut = [FEATURE_LABELS[SELECTED_FEATURES.index(f)] for f in urut.index]
warna_bar = [WARNA_AKSEN if v == urut.max() else '#3498db' for v in urut.values]
ax[1].barh(label_urut, urut.values, color=warna_bar, edgecolor='black', linewidth=0.7)
for i, v in enumerate(urut.values):
    ax[1].text(v + 0.008, i, f'{v:.3f}', va='center', fontweight='bold')
ax[1].set_xlabel('Korelasi terhadap Target (Diabetes)')
ax[1].set_title('Kekuatan Hubungan Tiap Fitur dengan Target', fontweight='bold')
ax[1].set_xlim(0, max(urut.values) * 1.25)

plt.suptitle('EDA - Analisis Korelasi Fitur', fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('eda_korelasi')
plt.show()

# Cek multikolinearitas antar fitur (di luar target)
kor_fitur = korelasi.loc[SELECTED_FEATURES, SELECTED_FEATURES].abs().to_numpy(copy=True)
np.fill_diagonal(kor_fitur, 0.0)   # abaikan diagonal (korelasi fitur dengan dirinya = 1)
maks_antar_fitur = float(kor_fitur.max())
pos = np.unravel_index(int(np.argmax(kor_fitur)), kor_fitur.shape)
pasangan = (SELECTED_FEATURES[pos[0]], SELECTED_FEATURES[pos[1]])

garis('KESIMPULAN EDA 2')
print('Korelasi tiap fitur terhadap target (urut dari terkuat):')
for f, v in kor_target.sort_values(ascending=False).items():
    nama = FEATURE_LABELS[SELECTED_FEATURES.index(f)]
    print(f'  {nama:<15s} : {v:+.4f}')
print()
print(f'Korelasi absolut tertinggi ANTAR FITUR : {maks_antar_fitur:.4f} '
      f'(pasangan {pasangan[0]} - {pasangan[1]})')
if maks_antar_fitur < 0.7:
    print('Kesimpulan: tidak ada indikasi multikolinearitas berat (semua < 0,70),')
    print('sehingga kelima fitur layak dipertahankan bersama dan bobot SVM linear')
    print('dapat ditafsirkan sebagai kontribusi masing-masing fitur.')
else:
    print('Peringatan: terdapat pasangan fitur dengan korelasi tinggi, perlu ditinjau.')

### CELL 9 - Distribusi Tiap Fitur per Kelas

Bagian ini memeriksa apakah tiap fitur benar-benar memisahkan kelompok sehat dan diabetes.
Fitur numerik ditampilkan dengan **violin plot** (menggabungkan bentuk sebaran dan ringkasan
kuartil), sedangkan fitur biner `hypertension` ditampilkan sebagai **persentase penderita
diabetes** pada masing-masing kelompok.

In [ ]:
# ============================================================
# BAGIAN 1 | CELL 9: EDA - Distribusi Tiap Fitur per Kelas
# ============================================================
garis('EDA 3: DISTRIBUSI TIAP FITUR PER KELAS')

fitur_numerik = [f for f in SELECTED_FEATURES if df_clean[f].nunique() > 2]
fitur_biner   = [f for f in SELECTED_FEATURES if df_clean[f].nunique() <= 2]

# --- Tabel ringkasan statistik per kelas ---------------------------------
baris_ringkas = []
for f in SELECTED_FEATURES:
    nama = FEATURE_LABELS[SELECTED_FEATURES.index(f)]
    g0 = df_clean.loc[df_clean[TARGET] == 0, f]
    g1 = df_clean.loc[df_clean[TARGET] == 1, f]
    sd_gab = math.sqrt(((len(g0) - 1) * g0.var() + (len(g1) - 1) * g1.var()) /
                       max(len(g0) + len(g1) - 2, 1))
    cohen_d = (g1.mean() - g0.mean()) / sd_gab if sd_gab > 0 else 0.0
    baris_ringkas.append({
        'fitur'          : f,
        'label'          : nama,
        'tipe'           : 'numerik' if f in fitur_numerik else 'biner',
        'mean_sehat'     : round(float(g0.mean()), 4),
        'mean_diabetes'  : round(float(g1.mean()), 4),
        'median_sehat'   : round(float(g0.median()), 4),
        'median_diabetes': round(float(g1.median()), 4),
        'std_sehat'      : round(float(g0.std()), 4),
        'std_diabetes'   : round(float(g1.std()), 4),
        'selisih_mean'   : round(float(g1.mean() - g0.mean()), 4),
        'cohen_d'        : round(float(cohen_d), 4),
    })

df_dist_fitur = pd.DataFrame(baris_ringkas).sort_values('cohen_d', ascending=False)
simpan_tabel(df_dist_fitur, 'eda_distribusi_fitur')

# --- Visualisasi ---------------------------------------------------------
n_panel = len(SELECTED_FEATURES)
n_kol   = 3
n_baris = math.ceil(n_panel / n_kol)
fig, axes = plt.subplots(n_baris, n_kol, figsize=(16, 4.6 * n_baris))
axes = np.array(axes).reshape(-1)

palet_kelas = {0: '#3498db', 1: '#e74c3c'}
df_plot = df_clean.copy()
df_plot['Kelompok'] = df_plot[TARGET].map({0: 'Sehat', 1: 'Diabetes'})

idx = 0
for f in SELECTED_FEATURES:
    ax = axes[idx]
    nama = FEATURE_LABELS[SELECTED_FEATURES.index(f)]
    if f in fitur_numerik:
        sns.violinplot(data=df_plot, x='Kelompok', y=f, hue='Kelompok',
                       palette=[palet_kelas[0], palet_kelas[1]], legend=False,
                       inner='quartile', cut=0, ax=ax)
        m0 = df_plot.loc[df_plot[TARGET] == 0, f].mean()
        m1 = df_plot.loc[df_plot[TARGET] == 1, f].mean()
        ax.scatter([0, 1], [m0, m1], color=WARNA_AKSEN, s=70, zorder=5,
                   edgecolor='black', linewidth=0.8, label='Rata-rata')
        ax.legend(loc='upper left', fontsize=9)
        ax.set_title(f'Distribusi {nama} per Kelompok', fontweight='bold')
        ax.set_ylabel(nama)
    else:
        # Fitur biner: persentase penderita diabetes per nilai fitur
        ringkas_bin = (df_plot.groupby(f)[TARGET]
                       .agg(['mean', 'count']).reset_index())
        ringkas_bin['persen_diabetes'] = ringkas_bin['mean'] * 100
        label_x = ['Tidak Hipertensi', 'Hipertensi'] if f == 'hypertension' \
                  else [str(v) for v in ringkas_bin[f]]
        bar = ax.bar(label_x, ringkas_bin['persen_diabetes'],
                     color=['#3498db', WARNA_AKSEN], edgecolor='black', linewidth=0.8,
                     width=0.55)
        for b, p, c in zip(bar, ringkas_bin['persen_diabetes'], ringkas_bin['count']):
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.4,
                    f'{p:.2f}%\n(n={c:,})', ha='center', va='bottom',
                    fontweight='bold', fontsize=9)
        ax.set_ylabel('Persentase Diabetes (%)')
        ax.set_ylim(0, max(ringkas_bin['persen_diabetes']) * 1.35)
        ax.set_title(f'Prevalensi Diabetes menurut {nama}', fontweight='bold')
        ax.set_xlabel('')
    idx += 1

for j in range(idx, len(axes)):
    axes[j].axis('off')

plt.suptitle('EDA - Sebaran Tiap Fitur pada Kelompok Sehat vs Diabetes',
             fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('eda_distribusi_fitur')
plt.show()

garis('KESIMPULAN EDA 3')
print('Daya pisah tiap fitur diukur dengan Cohen d (semakin besar, semakin memisahkan):')
for _, r in df_dist_fitur.iterrows():
    d = abs(r['cohen_d'])
    if d >= 0.8:   kat = 'besar'
    elif d >= 0.5: kat = 'sedang'
    elif d >= 0.2: kat = 'kecil'
    else:          kat = 'sangat kecil'
    print(f"  {r['label']:<15s} d = {r['cohen_d']:+.3f}  (efek {kat})")
print()
print('Fitur dengan daya pisah terkuat konsisten dengan literatur klinis: kadar HbA1c dan')
print('glukosa darah adalah penanda diagnostik utama, sedangkan usia, BMI, dan hipertensi')
print('berperan sebagai faktor risiko pendukung.')

### CELL 10 - Deteksi Outlier IQR (Sebelum Winsorization)

Notebook V2 menerapkan **winsorization** (capping IQR pada batas `Q1 - 1,5 x IQR` dan
`Q3 + 1,5 x IQR`) tanpa melaporkan berapa banyak nilai yang terpengaruh. Cell ini memuat ulang
data **mentah** (tanpa capping) untuk menghitung ulang jumlah outlier, lalu membandingkannya
dengan kondisi setelah winsorization. Tujuannya: menunjukkan bahwa penanganan outlier bersifat
**terukur dan proporsional**, bukan pemotongan data sembarangan.

Catatan penting: winsorization **tidak membuang baris**, hanya menggeser nilai ekstrem ke batas
IQR, sehingga jumlah sampel tetap 96.146 baris.

In [ ]:
# ============================================================
# BAGIAN 1 | CELL 10: EDA - Deteksi Outlier IQR Sebelum Winsorization
# ============================================================
garis('EDA 4: DETEKSI OUTLIER IQR (SEBELUM WINSORIZATION)')

def muat_data_mentah():
    """Muat data TANPA capping IQR, untuk pembanding analisis outlier.
    Langkah identik dengan muat_dan_bersihkan_data(), tetapi winsorization dilewati."""
    path = kagglehub.dataset_download('iammustafatz/diabetes-prediction-dataset')
    csv_file = os.path.join(path, 'diabetes_prediction_dataset.csv')
    df_raw = pd.read_csv(csv_file)
    df_m = df_raw.drop_duplicates().reset_index(drop=True)
    df_m = df_m[SELECTED_FEATURES + [TARGET]].copy()
    return df_m

df_mentah = muat_data_mentah()
print(f'Baris data mentah (setelah hapus duplikat) : {len(df_mentah):,}')
print(f'Baris data bersih (setelah winsorization)  : {len(df_clean):,}')
print('Jumlah baris identik -> winsorization hanya melakukan capping nilai, bukan penghapusan.')
print()

fitur_num = [f for f in SELECTED_FEATURES if df_mentah[f].nunique() > 2]

baris_outlier = []
batas_iqr = {}
for f in fitur_num:
    s = df_mentah[f]
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    low, up = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    batas_iqr[f] = (low, up)

    n_bawah = int((s < low).sum())
    n_atas  = int((s > up).sum())
    n_out   = n_bawah + n_atas
    n_out_sesudah = int(((df_clean[f] < low) | (df_clean[f] > up)).sum())

    baris_outlier.append({
        'fitur'             : f,
        'label'             : FEATURE_LABELS[SELECTED_FEATURES.index(f)],
        'min_sebelum'       : round(float(s.min()), 4),
        'Q1'                : round(float(Q1), 4),
        'median'            : round(float(s.median()), 4),
        'Q3'                : round(float(Q3), 4),
        'max_sebelum'       : round(float(s.max()), 4),
        'IQR'               : round(float(IQR), 4),
        'batas_bawah'       : round(float(low), 4),
        'batas_atas'        : round(float(up), 4),
        'n_outlier_bawah'   : n_bawah,
        'n_outlier_atas'    : n_atas,
        'n_outlier_total'   : n_out,
        'persen_outlier'    : round(n_out / len(df_mentah) * 100, 4),
        'min_sesudah'       : round(float(df_clean[f].min()), 4),
        'max_sesudah'       : round(float(df_clean[f].max()), 4),
        'n_outlier_sesudah' : n_out_sesudah,
    })

df_outlier = pd.DataFrame(baris_outlier)
simpan_tabel(df_outlier, 'eda_outlier')

# --- Visualisasi boxplot sebelum vs sesudah ------------------------------
n_f = len(fitur_num)
fig, axes = plt.subplots(2, n_f, figsize=(4.0 * n_f, 9))
axes = np.array(axes).reshape(2, n_f)

for j, f in enumerate(fitur_num):
    nama = FEATURE_LABELS[SELECTED_FEATURES.index(f)]
    low, up = batas_iqr[f]

    axes[0, j].boxplot(df_mentah[f].values, vert=True, patch_artist=True, widths=0.5,
                       boxprops={'facecolor': '#e74c3c', 'alpha': 0.65},
                       medianprops={'color': 'black', 'linewidth': 1.6},
                       flierprops={'marker': 'o', 'markersize': 2.5,
                                   'markerfacecolor': '#e74c3c', 'alpha': 0.35})
    axes[0, j].axhline(up, color=WARNA_AKSEN, linestyle='--', linewidth=1.3)
    axes[0, j].axhline(low, color=WARNA_AKSEN, linestyle='--', linewidth=1.3)
    n_out = int(df_outlier.loc[df_outlier['fitur'] == f, 'n_outlier_total'].iloc[0])
    pct_o = float(df_outlier.loc[df_outlier['fitur'] == f, 'persen_outlier'].iloc[0])
    axes[0, j].set_title(f'{nama} - SEBELUM\n{n_out:,} outlier ({pct_o:.2f}%)',
                         fontweight='bold', fontsize=11)
    axes[0, j].set_xticks([])

    axes[1, j].boxplot(df_clean[f].values, vert=True, patch_artist=True, widths=0.5,
                       boxprops={'facecolor': '#2ecc71', 'alpha': 0.65},
                       medianprops={'color': 'black', 'linewidth': 1.6},
                       flierprops={'marker': 'o', 'markersize': 2.5,
                                   'markerfacecolor': '#2ecc71', 'alpha': 0.35})
    axes[1, j].axhline(up, color=WARNA_AKSEN, linestyle='--', linewidth=1.3)
    axes[1, j].axhline(low, color=WARNA_AKSEN, linestyle='--', linewidth=1.3)
    axes[1, j].set_title(f'{nama} - SESUDAH\ncapping ke [{low:.2f}, {up:.2f}]',
                         fontweight='bold', fontsize=11)
    axes[1, j].set_xticks([])

plt.suptitle('EDA - Outlier IQR Sebelum vs Sesudah Winsorization',
             fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('eda_outlier')
plt.show()

total_out = int(df_outlier['n_outlier_total'].sum())
garis('KESIMPULAN EDA 4')
print(f'Total nilai outlier yang di-capping : {total_out:,} nilai '
      f'({total_out / (len(df_mentah) * len(fitur_num)) * 100:.2f}% dari seluruh sel numerik)')
for _, r in df_outlier.iterrows():
    print(f"  {r['label']:<15s} : {r['n_outlier_total']:>6,} nilai "
          f"({r['persen_outlier']:.2f}%) -> dibatasi ke "
          f"[{r['batas_bawah']:.2f}, {r['batas_atas']:.2f}]")
print()
print('Catatan metodologis:')
print('  1. Winsorization dipilih daripada penghapusan baris agar tidak ada informasi')
print('     pasien yang hilang, terutama pada kelas minoritas yang sudah langka.')
print('  2. Setelah capping, kolom numerik tidak lagi memuat nilai di luar batas IQR')
print('     (kolom n_outlier_sesudah bernilai 0).')
print('  3. Prosedur ini identik dengan notebook V2, sehingga angka baseline tetap sebanding.')

---
## Bagian C - Verifikasi Reproduksi Baseline V2 (CELL 11)

Sebelum menambahkan eksperimen baru, wajib dibuktikan bahwa pipeline V3 di notebook ini
**menghasilkan angka yang sama** dengan yang dilaporkan pada skripsi V2. Bila reproduksi
berhasil, maka semua eksperimen tambahan pada notebook `01`-`05` dapat dibandingkan langsung
dengan hasil lama tanpa perlu menghitung ulang seluruh naskah.

**Prosedur:** split 80:20 stratified dengan `random_state = 42`, tiga pipeline dilatih memakai
hyperparameter hasil tuning V2 (`PARAM_RF_V2`, `PARAM_KNN_V2`, `PARAM_SVM_V2`), lalu dievaluasi
dengan `evaluasi_holdout`. Metrik yang dibandingkan adalah metrik pada **threshold Youden**
(threshold optimal), persis seperti pelaporan V2.

In [ ]:
# ============================================================
# BAGIAN 1 | CELL 11: Verifikasi Reproduksi Baseline V2 (Split 80:20, seed 42)
# ============================================================
garis('VERIFIKASI REPRODUKSI BASELINE V2')

# --- Angka referensi dari laporan skripsi V2 (threshold Youden) ----------
REFERENSI_V2 = {
    'Random Forest': {'accuracy': 0.8933, 'precision': 0.4481, 'recall': 0.9057,
                      'f1': 0.5995, 'roc_auc': 0.9733, 'threshold': 0.4965},
    'KNN'          : {'accuracy': 0.8559, 'precision': 0.3710, 'recall': 0.9121,
                      'f1': 0.5274, 'roc_auc': 0.9524, 'threshold': 0.3810},
    'SVM (Linear)' : {'accuracy': 0.8775, 'precision': 0.4097, 'recall': 0.8833,
                      'f1': 0.5598, 'roc_auc': 0.9581, 'threshold': 0.4951},
}
TOLERANSI = 0.01   # selisih absolut <= 0,01 dianggap tereproduksi

# --- Split 80:20 stratified, seed 42 (identik V2) ------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=RANDOM_STATE
)
print(f'Data latih : {len(X_train):,} baris ({y_train.mean()*100:.2f}% positif)')
print(f'Data uji   : {len(X_test):,} baris ({y_test.mean()*100:.2f}% positif)')
print(f'Rasio      : {len(X_train)/len(X_all)*100:.1f}% : {len(X_test)/len(X_all)*100:.1f}%')
print()
print('Estimasi waktu total pelatihan tiga model: sekitar 2-6 menit pada Colab CPU.')
print()

# --- Latih & evaluasi tiga model ----------------------------------------
hasil_model = {}
proba_model = {}
for nama, pabrik in PABRIK_MODEL.items():
    print(f'[PROSES] Melatih {nama} ...')
    t_mulai = time.time()
    model = pabrik()
    hasil = evaluasi_holdout(model, X_train, y_train, X_test, y_test, tuning_threshold=True)
    proba_model[nama] = model.predict_proba(X_test)[:, 1]
    hasil_model[nama] = hasil
    print(f'[SELESAI] {nama} dalam {time.time() - t_mulai:.1f} detik | '
          f'thr={hasil["threshold"]:.4f} | recall={hasil["recall_tuned"]:.4f} | '
          f'f1={hasil["f1_tuned"]:.4f} | auc={hasil["roc_auc_tuned"]:.4f}')
print()

# --- Tabel perbandingan V2 vs sekarang ----------------------------------
PETA_METRIK = [('accuracy', 'Accuracy'), ('precision', 'Precision'),
               ('recall', 'Recall'), ('f1', 'F1-Score'),
               ('roc_auc', 'ROC-AUC'), ('threshold', 'Threshold')]

baris_verif = []
for nama in PABRIK_MODEL.keys():
    h = hasil_model[nama]
    ref = REFERENSI_V2[nama]
    for kunci, label in PETA_METRIK:
        nilai_now = h['threshold'] if kunci == 'threshold' else h[f'{kunci}_tuned']
        nilai_v2  = ref[kunci]
        selisih   = float(nilai_now) - float(nilai_v2)
        baris_verif.append({
            'model'         : nama,
            'metrik'        : label,
            'nilai_v2'      : round(float(nilai_v2), 4),
            'nilai_v3'      : round(float(nilai_now), 4),
            'selisih'       : round(selisih, 4),
            'selisih_abs'   : round(abs(selisih), 4),
            'selisih_persen': round(selisih / max(abs(float(nilai_v2)), 1e-9) * 100, 2),
            'status'        : 'COCOK' if abs(selisih) <= TOLERANSI else 'PERIKSA',
        })

df_verifikasi = pd.DataFrame(baris_verif)
simpan_tabel(df_verifikasi, 'verifikasi_reproduksi_v2')

n_cocok = int((df_verifikasi['status'] == 'COCOK').sum())
n_total_cek = len(df_verifikasi)
selisih_maks = float(df_verifikasi['selisih_abs'].max())
selisih_rata = float(df_verifikasi['selisih_abs'].mean())

# --- Visualisasi ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metrik_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrik_plot))
lebar = 0.13
for i, nama in enumerate(PABRIK_MODEL.keys()):
    sub = df_verifikasi[df_verifikasi['model'] == nama].set_index('metrik')
    v2  = [sub.loc[m, 'nilai_v2'] for m in metrik_plot]
    v3  = [sub.loc[m, 'nilai_v3'] for m in metrik_plot]
    axes[0].bar(x + (i * 2 - 2.5) * lebar, v2, lebar, label=f'{nama} - V2',
                color=WARNA_MODEL[nama], alpha=0.45, edgecolor='black', linewidth=0.6)
    axes[0].bar(x + (i * 2 - 1.5) * lebar, v3, lebar, label=f'{nama} - V3',
                color=WARNA_MODEL[nama], edgecolor='black', linewidth=0.6)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrik_plot)
axes[0].set_ylabel('Nilai Metrik')
axes[0].set_ylim(0, 1.12)
axes[0].set_title('Perbandingan Metrik: Laporan V2 vs Reproduksi V3\n(threshold Youden)',
                  fontweight='bold')
axes[0].legend(fontsize=8, ncol=3, loc='upper center')

for nama in PABRIK_MODEL.keys():
    fpr, tpr, _ = roc_curve(y_test, proba_model[nama])
    auc_now = hasil_model[nama]['roc_auc_tuned']
    axes[1].plot(fpr, tpr, color=WARNA_MODEL[nama], linewidth=2.2,
                 label=f'{nama} (AUC = {auc_now:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1.1, label='Tebakan acak (AUC = 0,5)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Kurva ROC Baseline V3 pada Data Uji (20%)', fontweight='bold')
axes[1].legend(loc='lower right', fontsize=10)

plt.suptitle('Verifikasi Reproduksi Baseline V2 - Split 80:20 Stratified, seed 42',
             fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('verifikasi_baseline_v2')
plt.show()

# --- Simpan JSON baseline -----------------------------------------------
hasil_baseline_v2 = {
    'deskripsi'      : 'Verifikasi reproduksi angka baseline skripsi V2 memakai pipeline V3',
    'konfigurasi'    : {
        'rasio_split'   : '80:20',
        'stratified'    : True,
        'random_state'  : RANDOM_STATE,
        'n_total'       : int(len(X_all)),
        'n_train'       : int(len(X_train)),
        'n_test'        : int(len(X_test)),
        'fitur'         : SELECTED_FEATURES,
        'pakai_smote'   : True,
        'threshold'     : 'Youden J (tuned)',
        'param_rf'      : PARAM_RF_V2,
        'param_knn'     : PARAM_KNN_V2,
        'param_svm'     : PARAM_SVM_V2,
    },
    'referensi_v2'   : REFERENSI_V2,
    'hasil_v3'       : hasil_model,
    'tabel_verifikasi': df_verifikasi.to_dict('records'),
    'ringkasan'      : {
        'toleransi'         : TOLERANSI,
        'jumlah_dicek'      : n_total_cek,
        'jumlah_cocok'      : n_cocok,
        'selisih_abs_maks'  : round(selisih_maks, 4),
        'selisih_abs_rata'  : round(selisih_rata, 4),
        'reproduksi_berhasil': bool(n_cocok == n_total_cek),
    },
}
simpan_json(hasil_baseline_v2, 'hasil_baseline_v2')

garis('KESIMPULAN VERIFIKASI REPRODUKSI')
print(f'Metrik yang dicek     : {n_total_cek} (3 model x 6 metrik)')
print(f'Cocok (selisih <= {TOLERANSI}) : {n_cocok} / {n_total_cek}')
print(f'Selisih absolut maks  : {selisih_maks:.4f}')
print(f'Selisih absolut rata2 : {selisih_rata:.4f}')
print()
if n_cocok == n_total_cek:
    print('STATUS: REPRODUKSI BERHASIL.')
    print('Pipeline V3 menghasilkan angka yang setara dengan laporan skripsi V2, sehingga')
    print('seluruh eksperimen tambahan pada notebook 01-05 berdiri di atas basis yang sah')
    print('dan hasilnya dapat langsung dibandingkan dengan naskah lama.')
else:
    print('STATUS: TERDAPAT SELISIH DI LUAR TOLERANSI.')
    print('Periksa versi scikit-learn/imbalanced-learn pada runtime, karena perbedaan versi')
    print('dapat menggeser hasil SMOTE dan kalibrasi. Baris berstatus PERIKSA:')
    for _, r in df_verifikasi[df_verifikasi['status'] == 'PERIKSA'].iterrows():
        print(f"  {r['model']:<15s} {r['metrik']:<10s} "
              f"V2={r['nilai_v2']:.4f} V3={r['nilai_v3']:.4f} selisih={r['selisih']:+.4f}")

---
## Bagian D - Penyimpanan Data Bersih (CELL 12)

Agar notebook `01`-`06` tidak perlu mengunduh ulang dataset dari Kaggle (dan agar dipastikan
memakai data yang **persis sama**), data bersih hasil CELL 4 disimpan sebagai berkas CSV.

In [ ]:
# ============================================================
# BAGIAN 1 | CELL 12: Simpan Data Bersih untuk Dipakai Ulang Notebook Lain
# ============================================================
garis('SIMPAN DATA BERSIH')

PATH_DATA_BERSIH = f'{OUTPUT_DIR}/data_bersih.csv'
df_clean.to_csv(PATH_DATA_BERSIH, index=False)
ukuran_mb = os.path.getsize(PATH_DATA_BERSIH) / (1024 ** 2)

print(f'[DATA DISIMPAN] {PATH_DATA_BERSIH}')
print(f'Dimensi   : {df_clean.shape[0]:,} baris x {df_clean.shape[1]} kolom')
print(f'Kolom     : {list(df_clean.columns)}')
print(f'Ukuran    : {ukuran_mb:.2f} MB')
print()

# Metadata data bersih (dipakai notebook 06 dan website)
meta_data = {
    'path'              : PATH_DATA_BERSIH,
    'n_baris'           : int(df_clean.shape[0]),
    'n_kolom'           : int(df_clean.shape[1]),
    'fitur'             : SELECTED_FEATURES,
    'label_fitur'       : FEATURE_LABELS,
    'target'            : TARGET,
    'n_kelas_0'         : int((df_clean[TARGET] == 0).sum()),
    'n_kelas_1'         : int((df_clean[TARGET] == 1).sum()),
    'persen_positif'    : round(float(df_clean[TARGET].mean() * 100), 4),
    'praproses'         : ['drop_duplicates', 'seleksi 5 fitur', 'winsorization IQR 1.5'],
    'sumber'            : 'kagglehub: iammustafatz/diabetes-prediction-dataset',
}
simpan_json(meta_data, 'metadata_data_bersih')

print()
print('Cara memakai ulang di notebook 01-06 (OPSIONAL, hanya jika PAKAI_DRIVE = True):')
print('  df_clean = pd.read_csv(f"{OUTPUT_DIR}/data_bersih.csv")')
print('  X_all = df_clean[SELECTED_FEATURES].copy()')
print('  y_all = df_clean[TARGET].copy()')
print()
print('Catatan: bila PAKAI_DRIVE = False, berkas hanya bertahan selama sesi Colab aktif.')
print('Untuk penggunaan lintas notebook, set PAKAI_DRIVE = True pada CELL 2 semua notebook,')
print('atau cukup panggil ulang muat_dan_bersihkan_data() yang hasilnya deterministik sama.')

---
## Bagian E - Ringkasan untuk Skripsi (CELL 13)

Cell berikut mencetak paragraf ringkasan yang dapat **langsung disalin** ke naskah skripsi
(Bab 3 Metodologi dan Bab 4 Hasil), berisi karakteristik data, penanganan ketimpangan kelas,
dan pernyataan konsistensi antar-notebook revisi.

In [ ]:
# ============================================================
# BAGIAN 1 | CELL 13: RINGKASAN UNTUK SKRIPSI
# ============================================================
garis('RINGKASAN UNTUK SKRIPSI')

n_total  = len(df_clean)
n_sehat  = int((df_clean[TARGET] == 0).sum())
n_diabet = int((df_clean[TARGET] == 1).sum())
pct_diab = n_diabet / n_total * 100

print('A. KARAKTERISTIK DATA')
print(f'   Dataset yang digunakan adalah Diabetes Prediction Dataset (Kaggle,')
print(f'   iammustafatz/diabetes-prediction-dataset). Setelah penghapusan data duplikat,')
print(f'   diperoleh {n_total:,} baris pengamatan yang unik. Analisis dibatasi pada 5 fitur')
print(f'   terpilih, yaitu usia, BMI, riwayat hipertensi, kadar HbA1c, dan kadar glukosa')
print(f'   darah, dengan variabel target berupa status diabetes (0 = sehat, 1 = diabetes).')
print()

print('B. KETIMPANGAN KELAS')
print(f'   Distribusi kelas sangat timpang: {n_sehat:,} kasus sehat ({100-pct_diab:.2f}%)')
print(f'   berbanding {n_diabet:,} kasus diabetes ({pct_diab:.2f}%). Ketimpangan sekitar')
print(f'   {pct_diab:.1f} persen kelas positif ini membuat akurasi menjadi metrik yang')
print(f'   menyesatkan, sebab model yang memprediksi seluruh sampel sebagai sehat sudah')
print(f'   mencapai akurasi {100-pct_diab:.2f} persen tanpa manfaat klinis apa pun. Karena itu')
print(f'   penelitian ini menetapkan recall, F1-score, dan ROC-AUC sebagai metrik utama,')
print(f'   menerapkan SMOTE hanya pada data latih di dalam pipeline (sehingga tidak terjadi')
print(f'   kebocoran data), serta menggunakan pembagian data dan validasi silang yang')
print(f'   bersifat stratified pada seluruh eksperimen.')
print()

print('C. PRAPROSES')
print(f'   Praproses terdiri atas tiga tahap: penghapusan duplikat, seleksi 5 fitur, dan')
print(f'   winsorization berbasis IQR (batas 1,5 x IQR) pada fitur numerik. Winsorization')
print(f'   dipilih daripada penghapusan baris agar tidak ada pengamatan yang hilang,')
print(f'   terutama pada kelas minoritas. Jumlah nilai yang terkena capping dilaporkan')
print(f'   secara eksplisit pada Tabel eda_outlier.')
print()

print('D. KONSISTENSI ANTAR-EKSPERIMEN')
print(f'   Seluruh notebook revisi (00 sampai 06) menggunakan basis data yang identik')
print(f'   ({n_total:,} baris, 5 fitur), fungsi pemuatan data yang sama, pabrik pipeline yang')
print(f'   sama (StandardScaler -> SMOTE -> classifier dalam imblearn Pipeline), fungsi')
print(f'   evaluasi yang sama, serta random_state = {RANDOM_STATE} yang sama. Konsekuensinya,')
print(f'   setiap perbedaan angka antar-eksperimen murni berasal dari faktor yang sedang')
print(f'   diuji (rasio split, nilai k, kernel/parameter SVM, skema validasi, atau ablasi),')
print(f'   bukan dari perbedaan data maupun praproses. Dengan demikian seluruh perbandingan')
print(f'   antar-eksperimen dalam revisi ini dapat dinyatakan adil dan dapat direplikasi.')
print()

print('E. VERIFIKASI BASELINE')
print(f'   Sebelum menambahkan eksperimen baru, angka baseline V2 direproduksi ulang pada')
print(f'   split 80:20 stratified dengan random_state = {RANDOM_STATE}. Hasil reproduksi')
print(f'   dilaporkan pada Tabel verifikasi_reproduksi_v2 beserta kolom selisih terhadap')
print(f'   angka laporan V2, sehingga pembaca dapat memastikan bahwa temuan-temuan baru')
print(f'   pada notebook 01 sampai 05 berdiri di atas hasil lama yang sudah tervalidasi.')
print()

garis('BERKAS YANG DIHASILKAN NOTEBOOK 00')
print(f'Folder output : {OUTPUT_DIR}')
print('Tabel  : eda_distribusi_kelas, eda_korelasi, eda_distribusi_fitur, eda_outlier,')
print('         verifikasi_reproduksi_v2')
print('Gambar : eda_distribusi_kelas, eda_korelasi, eda_distribusi_fitur, eda_outlier,')
print('         verifikasi_baseline_v2')
print('JSON   : hasil_baseline_v2, metadata_data_bersih')
print('Data   : data_bersih.csv')
print()
print('LANGKAH BERIKUTNYA: jalankan notebook 01 (justifikasi rasio split 80:20),')
print('lalu 02 (pemilihan k KNN), 03 (hyperplane SVM), 04 (validasi statistik),')
print('05 (ablation & robustness), dan terakhir 06 (model final + export produksi).')

---

### Status notebook 00

| Keluaran | Nama berkas | Kegunaan |
|---|---|---|
| Tabel | `eda_distribusi_kelas.csv` | Bukti ketimpangan kelas untuk Bab 4 |
| Tabel | `eda_korelasi.csv` | Matriks korelasi antar variabel |
| Tabel | `eda_distribusi_fitur.csv` | Statistik per kelas + Cohen d tiap fitur |
| Tabel | `eda_outlier.csv` | Jumlah outlier IQR sebelum/sesudah winsorization |
| Tabel | `verifikasi_reproduksi_v2.csv` | Perbandingan angka V2 vs reproduksi V3 + selisih |
| Gambar | `eda_distribusi_kelas.png`, `eda_korelasi.png`, `eda_distribusi_fitur.png`, `eda_outlier.png`, `verifikasi_baseline_v2.png` | Lampiran gambar skripsi |
| JSON | `hasil_baseline_v2.json`, `metadata_data_bersih.json` | Dibaca notebook `06` dan website |
| Data | `data_bersih.csv` | Basis data identik untuk notebook `01`-`06` |

Fondasi siap. Lanjutkan ke `01_Justifikasi_Rasio_Split.ipynb`.

---
---

# BAGIAN 2: JUSTIFIKASI RASIO SPLIT 80:20

Menjawab catatan penguji: "kenapa splitting data pakai 80:20".

*Sumber: `01_Justifikasi_Rasio_Split.ipynb`. Penomoran `CELL n` di bawah mengikuti notebook aslinya
agar rujukan silang di dalam kode tetap sahih.*

# Notebook 01 - Justifikasi Rasio Split Data 80:20

**Revisi penguji poin 3: "Kenapa splitting data pakai 80:20?"**

---

## 1. Pernyataan masalah

Pada notebook V2, pembagian data dilakukan dengan satu baris:

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
```

Angka `test_size=0.20` dipakai **tanpa dasar eksperimen apa pun** - hanya mengikuti
kebiasaan umum. Ini adalah kelemahan metodologis yang sah dipersoalkan penguji:
sebuah keputusan desain penelitian tidak boleh berstatus "kebiasaan", ia harus
berstatus **hasil pengujian**.

Notebook ini menutup celah tersebut dengan menyediakan **bukti empiris + analisis
statistik** yang menunjukkan bahwa 80:20 memang rasio yang rasional untuk dataset
dan tujuan penelitian ini (deteksi dini diabetes, metrik prioritas = **recall**).

## 2. Kerangka argumen: trade-off bias-variance pada pembagian data

Rasio split mengendalikan dua hal yang saling bertentangan:

| Arah | Efek pada model | Efek pada estimasi kinerja |
|---|---|---|
| Data latih diperbesar (test kecil) | Model **lebih baik** (bias turun, model melihat lebih banyak pola) | Estimasi kinerja **kurang presisi** (test kecil -> variansi tinggi, confidence interval lebar) |
| Data uji diperbesar (train kecil) | Model **lebih lemah** (bias naik, kekurangan data belajar) | Estimasi kinerja **lebih presisi** (margin of error mengecil) |

Jadi memilih rasio split = memilih titik kompromi antara **kualitas model** dan
**kepercayaan pada angka hasil ujinya**. Rasio yang benar adalah rasio yang berada
tepat setelah kurva performa mendatar (*plateau*) tetapi sebelum margin of error
estimasi membengkak.

## 3. Empat eksperimen dalam notebook ini

| # | Eksperimen | Pertanyaan yang dijawab |
|---|---|---|
| 1 | **Sweep rasio split** 50:50 s.d. 90:10 x 5 seed x 3 model | Apakah menambah data latih di atas 80% masih menaikkan performa? |
| 2 | **Analisis Margin of Error (MoE)** 95% pada recall | Seberapa presisi angka recall yang dilaporkan pada tiap ukuran data uji? |
| 3 | **Learning curve** (scoring recall, 5-fold CV) | Pada berapa banyak data latih kurva belajar mencapai plateau? |
| 4 | **Holdout 80:20 vs 5-Fold vs 10-Fold vs Repeated CV** | Apakah estimasi holdout 80:20 bias dibanding cross-validation? |

Seluruh notebook mengikuti kontrak `_SPEC_BERSAMA.md` (preamble, pabrik pipeline,
fungsi evaluasi, dan penamaan file output identik lintas notebook).

---

## 4. Konfigurasi eksperimen

Sesuai `_SPEC_BERSAMA.md` bagian 3 (strategi biaya komputasi), notebook menyediakan
saklar `MODE_CEPAT`:

- `MODE_CEPAT = True` -> memakai **subsample stratified 30.000 baris**. Dipakai untuk
  memastikan seluruh notebook jalan dari atas ke bawah dalam hitungan menit.
- `MODE_CEPAT = False` -> memakai **dataset penuh 96.146 baris**. **Wajib dipakai untuk
  angka final yang ditulis di skripsi.**

Rasio yang diuji (proporsi **data uji**): 50%, 40%, 30%, 25%, 20%, 15%, 10% - yaitu
split 50:50, 60:40, 70:30, 75:25, **80:20**, 85:15, dan 90:10. Setiap rasio diulang
pada **5 seed** (42-46) agar variasi akibat keberuntungan pembagian data dapat diukur
sebagai standar deviasi, bukan disembunyikan.

In [ ]:
# ============================================================
# BAGIAN 2 | CELL 7: Konstanta Eksperimen Rasio Split + Estimasi Waktu
# ============================================================
# MODE_CEPAT diatur sekali di PANEL KENDALI pada bagian atas notebook ini.
N_SUBSAMPLE = 30000

def ambil_subsample(X, y, n, seed=RANDOM_STATE):
    if n >= len(X): return X, y
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=seed)
    idx, _ = next(sss.split(X, y))
    return X.iloc[idx], y.iloc[idx]

# Proporsi DATA UJI yang diuji (0.20 = split 80:20)
DAFTAR_RASIO = [0.50, 0.40, 0.30, 0.25, 0.20, 0.15, 0.10]
N_SEED       = 5
DAFTAR_SEED  = [RANDOM_STATE + i for i in range(N_SEED)]   # 42, 43, 44, 45, 46
DAFTAR_MODEL = ['Random Forest', 'KNN', 'SVM (Linear)']
RASIO_SKRIPSI = 0.20                                        # rasio yang dipakai V2 dan diuji di sini

def label_rasio(r):
    return f'{int(round((1 - r) * 100))}:{int(round(r * 100))}'

if MODE_CEPAT:
    X_eks, y_eks = ambil_subsample(X_all, y_all, N_SUBSAMPLE)
    LABEL_MODE, DETIK_PER_FIT = 'MODE_CEPAT (subsample stratified)', 1.5
else:
    X_eks, y_eks = X_all.copy(), y_all.copy()
    LABEL_MODE, DETIK_PER_FIT = 'MODE PENUH (dataset lengkap)', 5.0

N_FIT_EKS1 = len(DAFTAR_RASIO) * N_SEED * len(DAFTAR_MODEL)
N_FIT_EKS3 = len(DAFTAR_MODEL) * 10 * 5          # learning curve: 10 titik x 5 fold
N_FIT_EKS4 = 5 + 10 + 25                          # 5-fold + 10-fold + repeated 5x5
N_FIT_TOTAL = N_FIT_EKS1 + N_FIT_EKS3 + N_FIT_EKS4

garis('KONFIGURASI EKSPERIMEN RASIO SPLIT')
print(f'Mode                 : {LABEL_MODE}')
print(f'Ukuran data dipakai  : {len(X_eks):,} baris '
      f'({int(y_eks.sum()):,} positif / {y_eks.mean()*100:.2f}%)')
print(f'Dataset penuh        : {len(X_all):,} baris ({int(y_all.sum()):,} positif)')
print(f'Rasio uji diuji      : {[label_rasio(r) for r in DAFTAR_RASIO]}')
print(f'Seed                 : {DAFTAR_SEED}')
print(f'Model                : {DAFTAR_MODEL}')
print('-' * 70)
print(f'Perkiraan jumlah fit : Eks-1 {N_FIT_EKS1} | Eks-3 {N_FIT_EKS3} | '
      f'Eks-4 {N_FIT_EKS4} | TOTAL {N_FIT_TOTAL}')
print(f'Estimasi waktu total : ~{N_FIT_TOTAL * DETIK_PER_FIT / 60:.1f} menit '
      f'(asumsi {DETIK_PER_FIT:.1f} detik/fit pada Colab CPU)')
if MODE_CEPAT:
    print('CATATAN: hasil di bawah adalah pratinjau. Untuk angka final skripsi,')
    print('         set MODE_CEPAT = False lalu jalankan ulang notebook ini.')

---

# EKSPERIMEN 1 - Sweep Rasio Split (7 rasio x 5 seed x 3 model)

**Pertanyaan:** apakah menambah porsi data latih terus-menerus menaikkan performa,
atau ada titik jenuh?

**Desain:** untuk setiap rasio dan setiap seed dilakukan `train_test_split` **stratified**
(proporsi kelas dipertahankan), lalu ketiga pipeline (Scaler -> SMOTE -> Classifier,
hyperparameter V2) dilatih pada data latih dan dievaluasi pada data uji dengan
`evaluasi_holdout`. Metrik utama = **recall pada threshold Youden**, sesuai konteks
skrining medis (biaya *false negative* jauh lebih besar daripada *false positive*).

Total: 7 x 5 x 3 = **105 kali pelatihan-evaluasi**.

In [ ]:
# ============================================================
# BAGIAN 2 | CELL 8: Eksperimen 1 - Sweep rasio split x seed x model
# ============================================================
garis('EKSPERIMEN 1: SWEEP RASIO SPLIT')
print(f'Total kombinasi: {N_FIT_EKS1} (progres dicetak per kombinasi)')
print('-' * 70)

baris_hasil = []
t_mulai_eks1 = time.time()

for i_r, rasio in enumerate(DAFTAR_RASIO, 1):
    print(f'[RASIO {i_r}/{len(DAFTAR_RASIO)}] split {label_rasio(rasio)} '
          f'(test_size={rasio:.2f})')
    for seed in DAFTAR_SEED:
        X_tr, X_te, y_tr, y_te = train_test_split(
            X_eks, y_eks, test_size=rasio, stratify=y_eks, random_state=seed)
        for nama_model in DAFTAR_MODEL:
            t0 = time.time()
            model = PABRIK_MODEL[nama_model](pakai_smote=True)
            hasil = evaluasi_holdout(model, X_tr, y_tr, X_te, y_te, tuning_threshold=True)
            baris_hasil.append({
                'model'          : nama_model,
                'rasio_uji'      : rasio,
                'rasio_label'    : label_rasio(rasio),
                'proporsi_latih' : round(1 - rasio, 2),
                'seed'           : seed,
                'n_train'        : int(len(X_tr)),
                'n_test'         : int(len(X_te)),
                'n_positif_test' : int(y_te.sum()),
                **hasil,
            })
            print(f'   seed={seed} {nama_model:<14} '
                  f'recall={hasil["recall_tuned"]:.4f} '
                  f'f1={hasil["f1_tuned"]:.4f} '
                  f'auc={hasil["roc_auc_tuned"]:.4f} '
                  f'({time.time() - t0:.1f}s)')

print('-' * 70)
print(f'Eksperimen 1 selesai dalam {(time.time() - t_mulai_eks1)/60:.1f} menit '
      f'({len(baris_hasil)} baris hasil).')

df_split_detail = pd.DataFrame(baris_hasil)
simpan_tabel(df_split_detail, 'tabel_rasio_split_detail', tampilkan=False)
display(df_split_detail.head(10))

In [ ]:
# ============================================================
# BAGIAN 2 | CELL 9: Agregasi hasil sweep (mean +/- std antar seed)
# ============================================================
METRIK_EKS1 = ['recall_tuned', 'f1_tuned', 'roc_auc_tuned',
               'precision_tuned', 'accuracy_tuned']

baris_ringkas = []
for (nama_model, rasio), g in df_split_detail.groupby(['model', 'rasio_uji']):
    d = {
        'model'          : nama_model,
        'rasio_uji'      : rasio,
        'rasio_label'    : label_rasio(rasio),
        'proporsi_latih' : round(1 - rasio, 2),
        'n_train'        : int(g['n_train'].mean()),
        'n_test'         : int(g['n_test'].mean()),
        'n_positif_test' : int(g['n_positif_test'].mean()),
    }
    for m in METRIK_EKS1:
        d[f'{m}_mean'] = float(g[m].mean())
        d[f'{m}_std']  = float(g[m].std(ddof=1))
    d['waktu_latih_s_mean'] = float(g['waktu_latih_s'].mean())
    baris_ringkas.append(d)

df_split_ringkas = (pd.DataFrame(baris_ringkas)
                    .sort_values(['model', 'proporsi_latih'])
                    .reset_index(drop=True))
simpan_tabel(df_split_ringkas.round(4), 'tabel_rasio_split_ringkas', tampilkan=False)

garis('RINGKASAN: RECALL (mean +/- std dari 5 seed) PER RASIO')
print(f'{"Model":<15}{"Split":<9}{"n_train":>9}{"n_test":>9}'
      f'{"recall":>18}{"F1":>18}{"ROC-AUC":>18}')
print('-' * 96)
for nama_model in DAFTAR_MODEL:
    sub = df_split_ringkas[df_split_ringkas['model'] == nama_model]
    for _, r in sub.iterrows():
        print(f'{r["model"]:<15}{r["rasio_label"]:<9}{r["n_train"]:>9,}{r["n_test"]:>9,}'
              f'{r["recall_tuned_mean"]:>11.4f} +/-{r["recall_tuned_std"]:.4f}'
              f'{r["f1_tuned_mean"]:>11.4f} +/-{r["f1_tuned_std"]:.4f}'
              f'{r["roc_auc_tuned_mean"]:>11.4f} +/-{r["roc_auc_tuned_std"]:.4f}')
    print('-' * 96)

garis('SELISIH PERFORMA ANTAR RASIO (indikasi plateau)')
for nama_model in DAFTAR_MODEL:
    sub = df_split_ringkas[df_split_ringkas['model'] == nama_model].set_index('rasio_uji')
    r50, r20, r10 = sub.loc[0.50], sub.loc[0.20], sub.loc[0.10]
    print(f'{nama_model}:')
    print(f'   recall 50:50 -> 80:20 : {r50["recall_tuned_mean"]:.4f} -> '
          f'{r20["recall_tuned_mean"]:.4f} (delta {(r20["recall_tuned_mean"]-r50["recall_tuned_mean"])*100:+.2f} poin)')
    print(f'   recall 80:20 -> 90:10 : {r20["recall_tuned_mean"]:.4f} -> '
          f'{r10["recall_tuned_mean"]:.4f} (delta {(r10["recall_tuned_mean"]-r20["recall_tuned_mean"])*100:+.2f} poin)')
    print(f'   std antar seed 80:20  : {r20["recall_tuned_std"]:.4f} | '
          f'90:10 : {r10["recall_tuned_std"]:.4f} '
          f'(naik {((r10["recall_tuned_std"]+1e-12)/(r20["recall_tuned_std"]+1e-12)):.2f}x)')

## Visualisasi 1 - Metrik vs proporsi data latih

Sumbu-X adalah **proporsi data latih** (50% s.d. 90%). *Error bar* menampilkan standar
deviasi antar 5 seed. Garis putus-putus oranye menandai posisi **80% data latih (80:20)**.

Yang perlu dilihat: setelah 70-80% data latih, kurva mendatar - penambahan data latih
tidak lagi diikuti kenaikan metrik yang berarti (kenaikan lebih kecil daripada lebar
error bar-nya sendiri, artinya tidak signifikan).

In [ ]:
# ============================================================
# BAGIAN 2 | CELL 10: Visualisasi 1 - Metrik vs proporsi data latih
# ============================================================
METRIK_PLOT = [
    ('recall_tuned',    'Recall (Sensitivitas)'),
    ('f1_tuned',        'F1-Score'),
    ('roc_auc_tuned',   'ROC-AUC'),
    ('precision_tuned', 'Precision'),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, (metrik, judul) in zip(axes.ravel(), METRIK_PLOT):
    for nama_model in DAFTAR_MODEL:
        sub = (df_split_ringkas[df_split_ringkas['model'] == nama_model]
               .sort_values('proporsi_latih'))
        ax.errorbar(sub['proporsi_latih'] * 100,
                    sub[f'{metrik}_mean'],
                    yerr=sub[f'{metrik}_std'],
                    marker='o', markersize=7, linewidth=2.2, capsize=5,
                    color=WARNA_MODEL[nama_model], label=nama_model)
    ax.axvline((1 - RASIO_SKRIPSI) * 100, color=WARNA_AKSEN,
               linestyle='--', linewidth=2.2, alpha=0.9)
    ax.text((1 - RASIO_SKRIPSI) * 100 + 0.6, ax.get_ylim()[0], ' 80:20',
            color=WARNA_AKSEN, fontsize=10, fontweight='bold', va='bottom')
    ax.set_title(judul, fontweight='bold')
    ax.set_xlabel('Proporsi data latih (%)')
    ax.set_ylabel(judul)
    ax.set_xticks([50, 60, 70, 75, 80, 85, 90])
    ax.legend(fontsize=9, loc='best')

plt.suptitle('Eksperimen 1: Performa Model vs Proporsi Data Latih '
             '(rata-rata 5 seed, error bar = std antar seed)',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
simpan_gambar('split_metrik_vs_rasio')
plt.show()

garis('BACAAN GRAFIK')
for nama_model in DAFTAR_MODEL:
    sub = (df_split_ringkas[df_split_ringkas['model'] == nama_model]
           .sort_values('proporsi_latih'))
    idx_max = sub['recall_tuned_mean'].idxmax()
    terbaik = sub.loc[idx_max]
    r20 = sub[sub['rasio_uji'] == RASIO_SKRIPSI].iloc[0]
    print(f'{nama_model:<15} recall tertinggi pada split {terbaik["rasio_label"]} '
          f'({terbaik["recall_tuned_mean"]:.4f}); pada 80:20 = {r20["recall_tuned_mean"]:.4f} '
          f'(selisih {(terbaik["recall_tuned_mean"]-r20["recall_tuned_mean"])*100:.2f} poin, '
          f'std 80:20 = {r20["recall_tuned_std"]*100:.2f} poin)')
print('Selisih terhadap rasio terbaik yang lebih kecil daripada std antar seed')
print('berarti perbedaan tersebut TIDAK signifikan secara praktis.')

## Visualisasi 2 - Stabilitas estimasi vs ukuran data uji

Grafik ini adalah sisi lain dari trade-off. Sumbu-Y adalah **standar deviasi metrik
antar 5 seed** - yakni seberapa besar angka hasil uji berubah hanya karena data uji
kebetulan terbagi berbeda.

Semakin kecil data uji (kanan ke kiri), semakin **tidak stabil** estimasinya. Ini
alasan kenapa split 90:10 atau 95:5 berbahaya untuk skripsi: recall yang dilaporkan
bisa berubah beberapa poin hanya karena mengganti `random_state`.

In [ ]:
# ============================================================
# BAGIAN 2 | CELL 11: Visualisasi 2 - Stabilitas estimasi (std antar seed)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))

for nama_model in DAFTAR_MODEL:
    sub = (df_split_ringkas[df_split_ringkas['model'] == nama_model]
           .sort_values('n_test'))
    axes[0].plot(sub['n_test'], sub['recall_tuned_std'] * 100,
                 marker='o', markersize=7, linewidth=2.2,
                 color=WARNA_MODEL[nama_model], label=nama_model)
    axes[1].plot(sub['rasio_uji'] * 100, sub['roc_auc_tuned_std'] * 100,
                 marker='s', markersize=7, linewidth=2.2,
                 color=WARNA_MODEL[nama_model], label=nama_model)

n_test_20 = int(df_split_ringkas[df_split_ringkas['rasio_uji'] == RASIO_SKRIPSI]['n_test'].mean())
axes[0].axvline(n_test_20, color=WARNA_AKSEN, linestyle='--', linewidth=2.2)
axes[0].text(n_test_20, 0.02, ' n_test 80:20', transform=axes[0].get_xaxis_transform(),
             color=WARNA_AKSEN, fontsize=10, fontweight='bold', va='bottom')
axes[0].set_xlabel('Jumlah sampel data uji (n_test)')
axes[0].set_ylabel('Std recall antar 5 seed (poin %)')
axes[0].set_title('Semakin kecil data uji, semakin liar estimasi recall', fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].axvline(RASIO_SKRIPSI * 100, color=WARNA_AKSEN, linestyle='--', linewidth=2.2)
axes[1].set_xlabel('Proporsi data uji (%)')
axes[1].set_ylabel('Std ROC-AUC antar 5 seed (poin %)')
axes[1].set_title('Stabilitas ROC-AUC terhadap ukuran data uji', fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Eksperimen 1 (lanjutan): Stabilitas Estimasi vs Ukuran Data Uji',
             fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('split_stabilitas_estimasi')
plt.show()

garis('RINGKAS STABILITAS (std recall antar seed, poin %)')
piv = (df_split_ringkas.pivot(index='rasio_label', columns='model',
                              values='recall_tuned_std') * 100).round(3)
piv = piv.reindex([label_rasio(r) for r in DAFTAR_RASIO])
display(piv)
rata_std = df_split_ringkas.groupby('rasio_uji')['recall_tuned_std'].mean() * 100
print(f'Rata-rata std (3 model) pada 80:20 : {rata_std.loc[RASIO_SKRIPSI]:.3f} poin')
print(f'Rata-rata std (3 model) pada 90:10 : {rata_std.loc[0.10]:.3f} poin')
print(f'Rata-rata std (3 model) pada 50:50 : {rata_std.loc[0.50]:.3f} poin')

---

# EKSPERIMEN 2 - Analisis Margin of Error (argumen kuantitatif inti)

Ini adalah bagian terpenting untuk menjawab penguji, karena mengubah pertanyaan
"kenapa 80:20" dari selera menjadi **hitungan statistik**.

Recall pada dasarnya adalah sebuah **proporsi**: dari sekian pasien yang benar-benar
diabetes di data uji, berapa persen yang berhasil ditangkap model. Karena itu presisi
estimasinya mengikuti rumus standard error proporsi:

$$SE(\text{recall}) = \sqrt{\frac{\hat{p}(1-\hat{p})}{n_{\text{positif uji}}}},\qquad
MoE_{95\%} = 1.96 \times SE$$

Perhatikan penyebutnya: **bukan** jumlah seluruh sampel uji, melainkan **jumlah sampel
positif (diabetes) di data uji**. Pada dataset ini kelasnya timpang (~8.5% positif),
sehingga memperkecil data uji langsung memangkas jumlah kasus positif dan melebarkan
selang kepercayaan dengan cepat.

Ukuran sampel dihitung pada **dataset penuh (96.146 baris)** karena keputusan rasio ini
berlaku untuk pelatihan model final; nilai recall diambil dari hasil Eksperimen 1.

In [ ]:
# ============================================================
# BAGIAN 2 | CELL 12: Eksperimen 2 - Margin of Error 95% untuk recall
# ============================================================
MODEL_MOE     = 'Random Forest'          # model kandidat produksi
N_TOTAL_PENUH = int(len(X_all))
N_POS_PENUH   = int(y_all.sum())

garis('EKSPERIMEN 2: MARGIN OF ERROR ESTIMASI RECALL')
print(f'Model acuan          : {MODEL_MOE}')
print(f'Dataset penuh        : {N_TOTAL_PENUH:,} baris, {N_POS_PENUH:,} kasus positif '
      f'({N_POS_PENUH/N_TOTAL_PENUH*100:.2f}%)')
print('Rumus MoE 95%        : 1.96 * sqrt(p*(1-p)/n_positif_uji)')
print('-' * 70)

baris_moe = []
for rasio in DAFTAR_RASIO:
    n_test     = int(round(N_TOTAL_PENUH * rasio))
    n_train    = N_TOTAL_PENUH - n_test
    n_pos_test = int(round(N_POS_PENUH * rasio))
    r_row = df_split_ringkas[(df_split_ringkas['model'] == MODEL_MOE) &
                             (df_split_ringkas['rasio_uji'] == rasio)].iloc[0]
    recall = float(r_row['recall_tuned_mean'])
    ci_bawah, ci_atas, moe = ci95_proporsi(recall, n_pos_test)
    baris_moe.append({
        'rasio_uji'      : rasio,
        'rasio_label'    : label_rasio(rasio),
        'proporsi_latih' : round(1 - rasio, 2),
        'n_train'        : n_train,
        'n_test'         : n_test,
        'n_positif_test' : n_pos_test,
        'recall'         : recall,
        'ci_bawah'       : float(ci_bawah),
        'ci_atas'        : float(ci_atas),
        'moe_persen'     : float(moe * 100),
        'lebar_ci_persen': float((ci_atas - ci_bawah) * 100),
        'std_empiris_persen': float(r_row['recall_tuned_std'] * 100),
    })

df_moe = pd.DataFrame(baris_moe).sort_values('rasio_uji', ascending=False).reset_index(drop=True)
simpan_tabel(df_moe.round(4), 'tabel_margin_of_error', tampilkan=False)

print(f'{"Split":<8}{"n_train":>10}{"n_test":>9}{"n_pos_uji":>11}'
      f'{"recall":>9}{"CI bawah":>10}{"CI atas":>10}{"MoE(+/-%)":>11}')
print('-' * 78)
for _, r in df_moe.iterrows():
    tanda = '  <== dipakai skripsi' if abs(r['rasio_uji'] - RASIO_SKRIPSI) < 1e-9 else ''
    print(f'{r["rasio_label"]:<8}{int(r["n_train"]):>10,}{int(r["n_test"]):>9,}'
          f'{int(r["n_positif_test"]):>11,}{r["recall"]:>9.4f}'
          f'{r["ci_bawah"]:>10.4f}{r["ci_atas"]:>10.4f}{r["moe_persen"]:>11.2f}{tanda}')
print('-' * 78)

moe_20 = float(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['moe_persen'].iloc[0])
moe_10 = float(df_moe[df_moe['rasio_uji'] == 0.10]['moe_persen'].iloc[0])
moe_50 = float(df_moe[df_moe['rasio_uji'] == 0.50]['moe_persen'].iloc[0])
npos_20 = int(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['n_positif_test'].iloc[0])
npos_10 = int(df_moe[df_moe['rasio_uji'] == 0.10]['n_positif_test'].iloc[0])
recall_20 = float(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['recall'].iloc[0])
print(f'MoE 80:20 = +/-{moe_20:.2f}% dengan {npos_20:,} kasus positif di data uji')
print(f'MoE 90:10 = +/-{moe_10:.2f}% dengan {npos_10:,} kasus positif '
      f'(MoE membengkak {moe_10/moe_20:.2f}x)')
print(f'MoE 50:50 = +/-{moe_50:.2f}% (hanya {(1 - moe_50/moe_20)*100:.1f}% lebih sempit '
      f'daripada 80:20, tetapi mengorbankan '
      f'{int(df_moe[df_moe["rasio_uji"]==RASIO_SKRIPSI]["n_train"].iloc[0]) - int(df_moe[df_moe["rasio_uji"]==0.50]["n_train"].iloc[0]):,} baris data latih)')

In [ ]:
# ============================================================
# BAGIAN 2 | CELL 13: Grafik trade-off dua panel - Recall (atas) vs MoE (bawah)
#
# CATATAN VISUALISASI: gambar ini SENGAJA TIDAK memakai dua sumbu-y pada satu
# panel (twinx). Pada grafik dua sumbu, posisi relatif kedua kurva - termasuk
# titik potongnya - sepenuhnya ditentukan oleh pilihan rentang masing-masing
# sumbu, sehingga "persilangan" bisa dimunculkan di mana saja hanya dengan
# mengubah batas sumbu. Pembaca mudah menafsirkannya sebagai temuan, padahal
# itu artefak skala. Dua panel bertumpuk dengan sumbu-x IDENTIK menyampaikan
# trade-off yang sama dan dibandingkan secara vertikal tanpa distorsi skala.
# ============================================================
WARNA_MOE = '#8e44ad'
df_plot = df_moe.sort_values('proporsi_latih')
x             = (df_plot['proporsi_latih'] * 100).to_numpy(dtype=float)
recall_persen = (df_plot['recall'] * 100).to_numpy(dtype=float)
std_persen    = df_plot['std_empiris_persen'].to_numpy(dtype=float)
moe_persen    = df_plot['moe_persen'].to_numpy(dtype=float)
label_x       = list(df_plot['rasio_label'])

fig, (ax_a, ax_b) = plt.subplots(2, 1, sharex=True, figsize=(11, 8))

# --- PANEL ATAS: kualitas model (recall) ---------------------------------
ax_a.errorbar(x, recall_persen, yerr=std_persen, marker='s', markersize=8,
              linewidth=2.6, capsize=5, color=WARNA_MODEL[MODEL_MOE])
ax_a.set_ylabel(f'Recall {MODEL_MOE} (%)')
ax_a.set_title('Panel A - Kualitas model: recall terhadap porsi data latih\n'
               '(titik = rata-rata 5 seed, error bar = std antar seed)',
               fontsize=11.5, fontweight='bold', loc='left')
batas_bawah = float(np.min(recall_persen - std_persen))
batas_atas  = float(np.max(recall_persen + std_persen))
rentang_r   = max(batas_atas - batas_bawah, 0.5)
# Ruang kosong bawah untuk anotasi, ruang atas untuk label zona
ax_a.set_ylim(batas_bawah - rentang_r * 0.55, batas_atas + rentang_r * 0.45)

# --- PANEL BAWAH: presisi estimasi (margin of error) ---------------------
ax_b.bar(x, moe_persen, width=3.0, color=WARNA_MOE, alpha=0.85,
         edgecolor='white', linewidth=1.2)
for xi, val in zip(x, moe_persen):
    ax_b.text(xi, val, f'{val:.2f}', ha='center', va='bottom', fontsize=9)
ax_b.set_ylabel('Margin of Error 95% recall\n(+/- poin persen)')
ax_b.set_xlabel('Rasio split (data latih : data uji)')
ax_b.set_title('Panel B - Presisi estimasi: margin of error terhadap ukuran data uji\n'
               '(makin kecil data uji, makin lebar selang kepercayaan)',
               fontsize=11.5, fontweight='bold', loc='left')
ax_b.set_ylim(0, float(moe_persen.max()) * 1.35)
ax_b.set_xticks(x)
ax_b.set_xticklabels(label_x)

# --- Penanda rasio terpilih: IDENTIK di kedua panel ----------------------
for ax in (ax_a, ax_b):
    ax.axvspan(48.5, 65.0, color='#c0392b', alpha=0.05)
    ax.axvspan(87.5, 91.5, color='#c0392b', alpha=0.05)
    ax.axvspan(72.5, 85.0, color='#27ae60', alpha=0.06)
    ax.axvline(80, color=WARNA_AKSEN, linestyle='--', linewidth=2.6, alpha=0.9)

tr_a, tr_b = ax_a.get_xaxis_transform(), ax_b.get_xaxis_transform()
ax_a.text(56.5, 0.94, 'data latih terbuang\n-> model kurang optimal', transform=tr_a,
          ha='center', va='top', fontsize=9.5, color='#c0392b')
ax_b.text(89.5, 0.94, 'data uji terlalu kecil\n-> estimasi tidak presisi', transform=tr_b,
          ha='center', va='top', fontsize=9.5, color='#c0392b')

# Anotasi rasio terpilih: bentuk sama di kedua panel, hanya angkanya yang berbeda
ax_a.annotate(f'rasio terpilih 80:20\nrecall {recall_20*100:.2f}%',
              xy=(80, recall_20 * 100), xycoords='data',
              xytext=(0.05, 0.07), textcoords='axes fraction',
              fontsize=10.5, fontweight='bold', color=WARNA_AKSEN,
              arrowprops=dict(arrowstyle='->', color=WARNA_AKSEN, linewidth=2))
ax_b.annotate(f'rasio terpilih 80:20\nMoE +/-{moe_20:.2f} poin  |  '
              f'{npos_20:,} positif di data uji',
              xy=(80, moe_20), xycoords='data',
              xytext=(0.05, 0.80), textcoords='axes fraction',
              fontsize=10.5, fontweight='bold', color=WARNA_AKSEN,
              arrowprops=dict(arrowstyle='->', color=WARNA_AKSEN, linewidth=2))

plt.suptitle('Eksperimen 2: Trade-off Kualitas Model (Recall) vs Presisi Estimasi (MoE)',
             fontsize=14, fontweight='bold')
fig.align_ylabels([ax_a, ax_b])
plt.tight_layout()
simpan_gambar('split_margin_of_error')
plt.show()

garis('INTERPRETASI')
print('Kedua panel memakai sumbu-x yang sama, jadi tiap rasio dibaca lurus ke bawah.')
print('Panel A (recall) naik ke kanan lalu mendatar: data latih besar -> model makin baik,')
print('tetapi manfaatnya berhenti bertambah setelah sekitar 75-80% data latih.')
print('Panel B (MoE) naik ke kanan: data uji makin kecil -> estimasi makin tidak presisi.')
print('Pada 80:20, Panel A sudah berada di area datar sementara Panel B masih rendah,')
print('sehingga 80:20 menjadi kompromi rasional antara kedua kriteria tersebut.')
print('Catatan metodologis: kedua besaran sengaja dipisah ke dua panel, bukan ditumpuk')
print('pada satu grafik dua sumbu-y, agar tidak ada titik potong semu akibat skala.')

---

# EKSPERIMEN 3 - Learning Curve

Sweep rasio menunjukkan performa akhir per rasio; *learning curve* menunjukkan
**bentuk proses belajarnya**: bagaimana skor validasi berubah ketika jumlah data latih
dinaikkan bertahap dari 10% sampai 100%.

Konfigurasi: `sklearn.model_selection.learning_curve`, `scoring='recall'`,
`cv=StratifiedKFold(5, shuffle=True)`, `train_sizes = 10%, 20%, ..., 100%`.
Pita berbayang = ±1 standar deviasi antar 5 fold.

Yang dicari: **titik plateau**, yaitu ukuran data latih terkecil yang skor validasinya
sudah mencapai >= 99% dari skor maksimum. Bila plateau tercapai jauh sebelum 100% data
latih, maka menyisihkan 20% data untuk pengujian **tidak merugikan model**.

In [ ]:
# ============================================================
# BAGIAN 2 | CELL 14: Eksperimen 3 - Learning curve (recall) untuk 3 model
# ============================================================
TRAIN_SIZES = np.linspace(0.1, 1.0, 10)
cv_lc = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

garis('EKSPERIMEN 3: LEARNING CURVE (scoring=recall, 5-fold stratified)')
print(f'Titik train_size : {[f"{t:.0%}" for t in TRAIN_SIZES]}')
print(f'Estimasi fit     : {len(DAFTAR_MODEL) * len(TRAIN_SIZES) * 5} kali pelatihan')
print('-' * 70)

hasil_lc   = {}
baris_lc   = []
plateau_lc = {}

for nama_model in DAFTAR_MODEL:
    t0 = time.time()
    model = PABRIK_MODEL[nama_model](pakai_smote=True)
    n_abs, skor_train, skor_val = learning_curve(
        model, X_eks, y_eks,
        train_sizes=TRAIN_SIZES, cv=cv_lc, scoring='recall',
        n_jobs=-1, shuffle=True, random_state=RANDOM_STATE)

    train_mean, train_std = skor_train.mean(axis=1), skor_train.std(axis=1)
    val_mean,   val_std   = skor_val.mean(axis=1),   skor_val.std(axis=1)

    val_maks    = float(val_mean.max())
    idx_plateau = int(np.argmax(val_mean >= 0.99 * val_maks))
    plateau_lc[nama_model] = {
        'fraksi_plateau'  : float(TRAIN_SIZES[idx_plateau]),
        'n_train_plateau' : int(n_abs[idx_plateau]),
        'recall_plateau'  : float(val_mean[idx_plateau]),
        'recall_maks'     : val_maks,
        'recall_100persen': float(val_mean[-1]),
    }
    hasil_lc[nama_model] = {
        'train_sizes_fraksi' : TRAIN_SIZES.tolist(),
        'train_sizes_absolut': n_abs.tolist(),
        'train_mean': train_mean.tolist(), 'train_std': train_std.tolist(),
        'val_mean'  : val_mean.tolist(),   'val_std'  : val_std.tolist(),
        **plateau_lc[nama_model],
    }
    for i, frac in enumerate(TRAIN_SIZES):
        baris_lc.append({
            'model': nama_model,
            'train_size_fraksi': round(float(frac), 2),
            'n_train': int(n_abs[i]),
            'recall_train_mean': float(train_mean[i]),
            'recall_train_std' : float(train_std[i]),
            'recall_val_mean'  : float(val_mean[i]),
            'recall_val_std'   : float(val_std[i]),
        })

    print(f'{nama_model:<15} selesai {time.time()-t0:>6.1f}s | '
          f'plateau pada {TRAIN_SIZES[idx_plateau]:.0%} data latih '
          f'({int(n_abs[idx_plateau]):,} baris) | '
          f'recall val 100% data = {val_mean[-1]:.4f}')

df_lc = pd.DataFrame(baris_lc)
simpan_tabel(df_lc.round(4), 'tabel_learning_curve', tampilkan=False)
display(df_lc.head(10))

In [ ]:
# ============================================================
# BAGIAN 2 | CELL 15: Plot learning curve (train vs validasi, pita +/- std)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(17, 5.6), sharey=False)

for ax, nama_model in zip(axes, DAFTAR_MODEL):
    h = hasil_lc[nama_model]
    n_abs      = np.array(h['train_sizes_absolut'])
    train_mean = np.array(h['train_mean']); train_std = np.array(h['train_std'])
    val_mean   = np.array(h['val_mean']);   val_std   = np.array(h['val_std'])
    warna = WARNA_MODEL[nama_model]

    ax.plot(n_abs, train_mean, marker='o', markersize=6, linewidth=2.2,
            color=warna, label='Recall data latih')
    ax.fill_between(n_abs, train_mean - train_std, train_mean + train_std,
                    color=warna, alpha=0.18)
    ax.plot(n_abs, val_mean, marker='s', markersize=6, linewidth=2.2,
            linestyle='--', color='#34495e', label='Recall validasi (5-fold)')
    ax.fill_between(n_abs, val_mean - val_std, val_mean + val_std,
                    color='#34495e', alpha=0.15)

    ax.axvline(h['n_train_plateau'], color=WARNA_AKSEN, linestyle=':', linewidth=2.4)
    ax.annotate(f'plateau: {h["fraksi_plateau"]:.0%} data latih',
                xy=(h['n_train_plateau'], h['recall_plateau']), xycoords='data',
                xytext=(0.40, 0.12), textcoords='axes fraction',
                fontsize=9, fontweight='bold', color=WARNA_AKSEN,
                arrowprops=dict(arrowstyle='->', color=WARNA_AKSEN, linewidth=1.6))

    # Catatan: dengan cv=5-fold, titik train_size 100% pada learning curve = 80%
    # dari seluruh data, jadi titik paling kanan setara persis dengan split 80:20.
    ax.text(0.985, 0.02, 'titik terkanan = 80% data\n(setara split 80:20)',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8.5, color='#7f8c8d')

    ax.set_title(nama_model, fontweight='bold', color=warna)
    ax.set_xlabel('Jumlah sampel data latih')
    ax.set_ylabel('Recall')
    ax.legend(fontsize=8.5, loc='lower left', framealpha=0.9)

plt.suptitle('Eksperimen 3: Learning Curve - Recall vs Jumlah Data Latih '
             '(pita = +/- 1 std antar fold)', fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('split_learning_curve')
plt.show()

garis('TITIK PLATEAU (train_size terkecil dengan recall validasi >= 99% recall maksimum)')
for nama_model in DAFTAR_MODEL:
    p = plateau_lc[nama_model]
    delta = (p['recall_100persen'] - p['recall_plateau']) * 100
    print(f'{nama_model:<15} plateau {p["fraksi_plateau"]:>4.0%} data latih '
          f'({p["n_train_plateau"]:>6,} baris) | recall plateau {p["recall_plateau"]:.4f} -> '
          f'recall 100% data {p["recall_100persen"]:.4f} (tambahan hanya {delta:+.2f} poin)')
print('-' * 70)
print('CATATAN PENTING: karena learning curve memakai 5-fold CV, titik train_size 100%')
print('pada kurva di atas setara dengan 80% dari seluruh data - yaitu persis porsi data')
print('latih pada skema 80:20. Jadi kurva ini langsung memotret skema yang dipakai skripsi.')
print('Kesimpulan: kurva validasi sudah datar jauh sebelum 100% data latih dipakai.')
print('Menyisihkan 20% data untuk pengujian TIDAK menurunkan kemampuan belajar model')
print('secara berarti, karena model sudah berada di area plateau.')

---

# EKSPERIMEN 4 - Apakah holdout 80:20 bias dibanding Cross-Validation?

Keberatan lanjutan yang wajar dari penguji: *"kalau begitu kenapa tidak pakai
cross-validation saja, bukan single holdout?"*

Untuk menjawabnya, estimasi recall/F1/AUC dari **holdout 80:20 (5 seed)** dibandingkan
dengan tiga skema cross-validation pada model Random Forest:

1. **5-Fold Stratified CV**
2. **10-Fold Stratified CV**
3. **RepeatedStratifiedKFold(5 fold x 5 repetisi)** = 25 evaluasi

Catatan metodologis: agar perbandingan **apple-to-apple**, metrik holdout di sini
memakai *threshold* default 0.5 - sama dengan yang dipakai `cross_validate` - bukan
threshold Youden.

Bila selisih rata-rata holdout terhadap Repeated CV kecil (dalam rentang 1 std),
maka estimasi holdout 80:20 **tidak bias** dan sah dipakai sebagai skema pelaporan
utama, dengan biaya komputasi jauh lebih murah.

In [ ]:
# ============================================================
# BAGIAN 2 | CELL 16: Eksperimen 4 - Holdout 80:20 vs 5-Fold vs 10-Fold vs Repeated CV
# ============================================================
SCORING_CV = {'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc'}
MODEL_CV   = 'Random Forest'

garis('EKSPERIMEN 4: PERBANDINGAN SKEMA EVALUASI (model Random Forest)')
print('Metrik memakai threshold default 0.5 agar sebanding dengan cross_validate.')
print('-' * 70)

baris_skema = []

# --- Skema A: holdout 80:20 dengan 5 seed (hasil dari Eksperimen 1) --------
sub_hold = df_split_detail[(df_split_detail['model'] == MODEL_CV) &
                           (df_split_detail['rasio_uji'] == RASIO_SKRIPSI)]
baris_skema.append({
    'skema'        : 'Holdout 80:20 (5 seed)',
    'n_evaluasi'   : int(len(sub_hold)),
    'recall_mean'  : float(sub_hold['recall_default'].mean()),
    'recall_std'   : float(sub_hold['recall_default'].std(ddof=1)),
    'f1_mean'      : float(sub_hold['f1_default'].mean()),
    'f1_std'       : float(sub_hold['f1_default'].std(ddof=1)),
    'roc_auc_mean' : float(sub_hold['roc_auc_default'].mean()),
    'roc_auc_std'  : float(sub_hold['roc_auc_default'].std(ddof=1)),
    'waktu_s'      : float(sub_hold['waktu_latih_s'].sum()),
})
print(f'[1/4] Holdout 80:20 (5 seed)  -> recall {baris_skema[-1]["recall_mean"]:.4f} '
      f'+/-{baris_skema[-1]["recall_std"]:.4f} '
      f'({baris_skema[-1]["waktu_s"]:.1f}s, dipakai ulang dari Eksperimen 1)')

# --- Skema B, C, D: cross-validation --------------------------------------
SKEMA_CV = [
    ('5-Fold CV',            StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)),
    ('10-Fold CV',           StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)),
    ('Repeated 5-Fold x5 CV', RepeatedStratifiedKFold(n_splits=5, n_repeats=5,
                                                      random_state=RANDOM_STATE)),
]

for i, (nama_skema, cv_obj) in enumerate(SKEMA_CV, start=2):
    t0 = time.time()
    hasil_cv = cross_validate(buat_pipeline_rf(), X_eks, y_eks, cv=cv_obj,
                              scoring=SCORING_CV, n_jobs=-1)
    durasi = time.time() - t0
    baris_skema.append({
        'skema'        : nama_skema,
        'n_evaluasi'   : int(len(hasil_cv['test_recall'])),
        'recall_mean'  : float(hasil_cv['test_recall'].mean()),
        'recall_std'   : float(hasil_cv['test_recall'].std(ddof=1)),
        'f1_mean'      : float(hasil_cv['test_f1'].mean()),
        'f1_std'       : float(hasil_cv['test_f1'].std(ddof=1)),
        'roc_auc_mean' : float(hasil_cv['test_roc_auc'].mean()),
        'roc_auc_std'  : float(hasil_cv['test_roc_auc'].std(ddof=1)),
        'waktu_s'      : float(durasi),
    })
    print(f'[{i}/4] {nama_skema:<22} -> recall {baris_skema[-1]["recall_mean"]:.4f} '
          f'+/-{baris_skema[-1]["recall_std"]:.4f} ({durasi:.1f}s)')

df_holdout_cv = pd.DataFrame(baris_skema)
simpan_tabel(df_holdout_cv.round(4), 'tabel_holdout_vs_cv', tampilkan=False)

print('-' * 70)
print(f'{"Skema":<24}{"n":>4}{"recall":>18}{"F1":>18}{"ROC-AUC":>18}{"waktu(s)":>10}')
print('-' * 92)
for _, r in df_holdout_cv.iterrows():
    print(f'{r["skema"]:<24}{int(r["n_evaluasi"]):>4}'
          f'{r["recall_mean"]:>11.4f} +/-{r["recall_std"]:.4f}'
          f'{r["f1_mean"]:>11.4f} +/-{r["f1_std"]:.4f}'
          f'{r["roc_auc_mean"]:>11.4f} +/-{r["roc_auc_std"]:.4f}'
          f'{r["waktu_s"]:>10.1f}')
print('-' * 92)

ref = df_holdout_cv[df_holdout_cv['skema'] == 'Repeated 5-Fold x5 CV'].iloc[0]
hol = df_holdout_cv[df_holdout_cv['skema'] == 'Holdout 80:20 (5 seed)'].iloc[0]
selisih_recall = abs(hol['recall_mean'] - ref['recall_mean'])
selisih_auc    = abs(hol['roc_auc_mean'] - ref['roc_auc_mean'])
print(f'Selisih recall  holdout 80:20 vs Repeated CV : {selisih_recall*100:.2f} poin '
      f'(std Repeated CV = {ref["recall_std"]*100:.2f} poin)')
print(f'Selisih ROC-AUC holdout 80:20 vs Repeated CV : {selisih_auc*100:.2f} poin')
print(f'Rasio waktu komputasi Repeated CV : Holdout = '
      f'{ref["waktu_s"]/max(hol["waktu_s"], 1e-9):.1f}x lebih lama')
konsisten_cv = bool(selisih_recall <= ref['recall_std'])
if konsisten_cv:
    print('PUTUSAN: selisih berada di dalam 1 standar deviasi Repeated CV ->')
    print('         estimasi holdout 80:20 KONSISTEN dengan cross-validation (tidak bias).')
else:
    print('PUTUSAN: selisih melebihi 1 standar deviasi Repeated CV ->')
    print('         laporkan hasil holdout 80:20 BERSAMA hasil Repeated CV di skripsi.')

In [ ]:
# ============================================================
# BAGIAN 2 | CELL 17: Visualisasi Eksperimen 4 - Holdout vs Cross-Validation
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [2.1, 1]})

METRIK_BAR = [('recall', 'Recall'), ('f1', 'F1-Score'), ('roc_auc', 'ROC-AUC')]
WARNA_BAR  = [WARNA_MODEL['Random Forest'], WARNA_AKSEN, '#8e44ad']
posisi = np.arange(len(df_holdout_cv))
lebar  = 0.26

for j, ((kunci, judul), warna) in enumerate(zip(METRIK_BAR, WARNA_BAR)):
    axes[0].bar(posisi + (j - 1) * lebar, df_holdout_cv[f'{kunci}_mean'], lebar,
                yerr=df_holdout_cv[f'{kunci}_std'], capsize=4,
                color=warna, alpha=0.88, label=judul,
                edgecolor='white', linewidth=1.2)
    for xi, val, sd in zip(posisi + (j - 1) * lebar,
                           df_holdout_cv[f'{kunci}_mean'],
                           df_holdout_cv[f'{kunci}_std']):
        axes[0].text(xi, val + sd + 0.022, f'{val:.3f}', ha='center', fontsize=8.5)

axes[0].axhline(hol['recall_mean'], color='#c0392b', linestyle=':', linewidth=1.8)
axes[0].text(-0.45, hol['recall_mean'] + 0.012, 'recall holdout 80:20',
             color='#c0392b', fontsize=9, ha='left', va='bottom')
axes[0].set_xticks(posisi)
axes[0].set_xticklabels(df_holdout_cv['skema'], rotation=12, fontsize=9.5)
axes[0].set_ylabel('Skor (threshold 0.5)')
axes[0].set_ylim(0, 1.18)
axes[0].set_title('Estimasi metrik tiap skema (error bar = std antar evaluasi)',
                  fontweight='bold')
axes[0].legend(fontsize=9, loc='upper right', ncol=3)

axes[1].bar(posisi, df_holdout_cv['waktu_s'],
            color=['#2ecc71' if s.startswith('Holdout') else '#95a5a6'
                   for s in df_holdout_cv['skema']],
            edgecolor='white', linewidth=1.2)
for xi, val in zip(posisi, df_holdout_cv['waktu_s']):
    axes[1].text(xi, val, f'{val:.1f}s', ha='center', va='bottom', fontsize=9)
axes[1].set_ylim(0, float(df_holdout_cv['waktu_s'].max()) * 1.18)
axes[1].set_xticks(posisi)
axes[1].set_xticklabels(df_holdout_cv['skema'], rotation=25, ha='right', fontsize=8.5)
axes[1].set_ylabel('Waktu komputasi (detik)')
axes[1].set_title('Biaya komputasi tiap skema', fontweight='bold')

plt.suptitle('Eksperimen 4: Validasi Skema Holdout 80:20 terhadap Cross-Validation',
             fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('split_holdout_vs_cv')
plt.show()

garis('INTERPRETASI')
print('Tinggi batang keempat skema praktis sama -> holdout 80:20 memberi estimasi')
print('yang setara dengan cross-validation, dengan waktu komputasi jauh lebih murah.')
print('Cross-validation tetap dipakai di notebook 04 sebagai validasi statistik lanjutan,')
print('sedangkan holdout 80:20 dipakai sebagai skema pelaporan utama.')

---

# KESIMPULAN - Skor Komposit dan Argumen Final

Rasio terbaik dipilih dengan **skor komposit** yang menggabungkan dua kriteria yang
tadi bertentangan, setelah keduanya dinormalisasi min-max ke rentang [0, 1]:

$$\text{skor}(r) = \text{recall}_{\text{norm}}(r) - \lambda \cdot \text{MoE}_{\text{norm}}(r),
\qquad \lambda = 1.0$$

Artinya: rasio yang baik adalah rasio yang memberi recall tinggi **dan sekaligus**
margin of error rendah. Rasio yang mengorbankan salah satunya secara ekstrem
(50:50 atau 90:10) otomatis mendapat skor rendah.

In [ ]:
# ============================================================
# BAGIAN 2 | CELL 18: Skor komposit, argumen final, dan penyimpanan JSON kontrak
# ============================================================
BOBOT_PENALTI_MOE = 1.0

def normalisasi(v):
    v = np.asarray(v, dtype=float)
    rentang = v.max() - v.min()
    return np.zeros_like(v) if rentang < 1e-12 else (v - v.min()) / rentang

df_skor = df_moe.sort_values('rasio_uji', ascending=False).reset_index(drop=True).copy()
df_skor['recall_norm'] = normalisasi(df_skor['recall'].values)
df_skor['moe_norm']    = normalisasi(df_skor['moe_persen'].values)
df_skor['skor_komposit'] = df_skor['recall_norm'] - BOBOT_PENALTI_MOE * df_skor['moe_norm']

idx_terbaik  = int(df_skor['skor_komposit'].idxmax())
rasio_komposit = float(df_skor.loc[idx_terbaik, 'rasio_uji'])
skor_terbaik   = float(df_skor.loc[idx_terbaik, 'skor_komposit'])
skor_20        = float(df_skor[df_skor['rasio_uji'] == RASIO_SKRIPSI]['skor_komposit'].iloc[0])
selisih_skor   = skor_terbaik - skor_20

simpan_tabel(df_skor[['rasio_label', 'n_train', 'n_test', 'n_positif_test', 'recall',
                      'moe_persen', 'recall_norm', 'moe_norm', 'skor_komposit']].round(4),
             'tabel_skor_komposit_rasio', tampilkan=False)

garis('SKOR KOMPOSIT PER RASIO (recall_norm - 1.0 x MoE_norm)')
for _, r in df_skor.iterrows():
    tanda = ' <== SKOR TERTINGGI' if abs(r['rasio_uji'] - rasio_komposit) < 1e-9 else ''
    tanda += ' [dipakai skripsi]' if abs(r['rasio_uji'] - RASIO_SKRIPSI) < 1e-9 else ''
    print(f'  {r["rasio_label"]:<7} recall={r["recall"]:.4f} '
          f'MoE=+/-{r["moe_persen"]:.2f}%  skor={r["skor_komposit"]:+.4f}{tanda}')

if abs(rasio_komposit - RASIO_SKRIPSI) < 1e-9:
    catatan_pilihan = ('Skor komposit tertinggi jatuh TEPAT pada rasio 80:20, sehingga '
                       'rasio yang dipakai skripsi terkonfirmasi secara kuantitatif.')
elif selisih_skor <= 0.10:
    catatan_pilihan = (f'Skor komposit tertinggi jatuh pada rasio '
                       f'{label_rasio(rasio_komposit)} dengan selisih hanya '
                       f'{selisih_skor:.4f} terhadap 80:20 (praktis setara, di bawah '
                       f'ambang 0.10). Rasio 80:20 tetap dipilih karena berada pada '
                       f'plateau performa, MoE-nya kecil, dan merupakan konvensi baku '
                       f'yang memudahkan pembandingan dengan penelitian lain.')
else:
    catatan_pilihan = (f'Skor komposit tertinggi jatuh pada rasio '
                       f'{label_rasio(rasio_komposit)} (selisih {selisih_skor:.4f} '
                       f'terhadap 80:20). Selisih ini perlu dilaporkan apa adanya di '
                       f'skripsi dan dibahas sebagai keterbatasan pemilihan rasio.')

RASIO_TERPILIH = RASIO_SKRIPSI

plateau_maks = max(p['fraksi_plateau'] for p in plateau_lc.values())
r20_rf = df_split_ringkas[(df_split_ringkas['model'] == MODEL_MOE) &
                          (df_split_ringkas['rasio_uji'] == RASIO_SKRIPSI)].iloc[0]
n_train_20 = int(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['n_train'].iloc[0])
n_test_20p = int(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['n_test'].iloc[0])
n_train_50 = int(df_moe[df_moe['rasio_uji'] == 0.50]['n_train'].iloc[0])
recall_50  = float(df_moe[df_moe['rasio_uji'] == 0.50]['recall'].iloc[0])
recall_20  = float(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['recall'].iloc[0])
recall_10  = float(df_moe[df_moe['rasio_uji'] == 0.10]['recall'].iloc[0])

delta_20_10 = (recall_10 - recall_20) * 100
std_20      = float(r20_rf['recall_tuned_std']) * 100
signifikan  = abs(delta_20_10) > std_20
frasa_sig   = ('lebih besar daripada' if signifikan else 'lebih kecil daripada')
kesan_sig   = ('sehingga perlu dilaporkan dan dibahas sebagai keterbatasan'
               if signifikan else 'sehingga tidak signifikan secara praktis')
ci_b_20 = float(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['ci_bawah'].iloc[0])
ci_a_20 = float(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['ci_atas'].iloc[0])
lebar_ci_20 = (ci_a_20 - ci_b_20) * 100
frasa_ci = ('cukup sempit untuk mendukung klaim ilmiah' if moe_20 <= 2.0
            else 'perlu dilaporkan apa adanya beserta lebarnya')
frasa_cv = ('sehingga estimasi holdout ini tidak bias'
            if konsisten_cv else
            'sehingga hasil holdout sebaiknya dilaporkan bersama hasil Repeated CV')

print()
garis('ARGUMEN FINAL: KENAPA RASIO SPLIT 80:20')
print('(1) PLATEAU PERFORMA')
print(f'    Learning curve menunjukkan recall validasi sudah mencapai >=99% nilai')
print(f'    maksimumnya pada {plateau_maks:.0%} data latih (paling lambat di antara 3 model).')
print(f'    Sweep rasio menegaskan hal yang sama: recall {MODEL_MOE} berubah dari '
      f'{recall_50:.4f} (50:50)')
print(f'    menjadi {recall_20:.4f} (80:20), lalu bergeser {delta_20_10:+.2f} poin di 90:10.')
print(f'    Pergeseran itu {frasa_sig} standar deviasi antar seed ({std_20:.2f} poin),')
print(f'    {kesan_sig}.')
print()
print('(2) PRESISI ESTIMASI MASIH TERJAGA PADA 20% DATA UJI')
print(f'    Data uji 20% = {n_test_20p:,} sampel, di antaranya {npos_20:,} kasus positif.')
print(f'    Margin of error 95% untuk recall = +/-{moe_20:.2f} poin persen.')
print(f'    Selang kepercayaan 95% recall = [{ci_b_20:.4f}, {ci_a_20:.4f}] '
      f'(lebar {lebar_ci_20:.2f} poin),')
print(f'    {frasa_ci}.')
print()
print('(3) RASIO EKSTREM MERUGIKAN DI KEDUA ARAH')
print(f'    - Data uji 10% : MoE membengkak menjadi +/-{moe_10:.2f} poin '
      f'({moe_10/moe_20:.2f}x lebih lebar)')
print(f'      karena hanya tersisa {npos_10:,} kasus positif untuk diuji, dan std antar seed naik.')
print(f'    - Data uji 40-50% : membuang {n_train_20 - n_train_50:,} baris data latih '
      f'dibanding 80:20')
print(f'      tanpa manfaat sepadan (MoE hanya menyempit {moe_20 - moe_50:.2f} poin).')
print()
print('(4) KONSISTEN DENGAN CROSS-VALIDATION DAN PRAKTIK BAKU')
print(f'    Estimasi holdout 80:20 berselisih {selisih_recall*100:.2f} poin recall dari')
print(f'    Repeated Stratified 5-Fold x5 CV (25 evaluasi, std {ref["recall_std"]*100:.2f} poin),')
print(f'    {frasa_cv}, dengan biaya komputasi '
      f'{ref["waktu_s"]/max(hol["waktu_s"], 1e-9):.1f}x lebih murah.')
print('    Rasio 80:20 juga selaras dengan prinsip Pareto (80/20) dan merupakan')
print('    konvensi yang lazim dipakai pada literatur machine learning kesehatan,')
print('    sehingga hasil penelitian ini dapat dibandingkan langsung dengan studi lain.')
print()
print('CATATAN PEMILIHAN:')
print(f'    {catatan_pilihan}')
print('=' * 70)

teks_kesimpulan = (
    f'Rasio split 80:20 dipilih berdasarkan empat bukti eksperimental. '
    f'(1) Learning curve menunjukkan recall validasi mencapai plateau (>=99% nilai maksimum) '
    f'pada {plateau_maks:.0%} data latih, sehingga menambah porsi latih melebihi 80% tidak lagi '
    f'meningkatkan kemampuan model secara berarti. '
    f'(2) Sweep tujuh rasio x lima seed x tiga model memperlihatkan recall {MODEL_MOE} '
    f'{recall_50:.4f} pada 50:50, {recall_20:.4f} pada 80:20, dan {recall_10:.4f} pada 90:10; '
    f'selisih 80:20 terhadap 90:10 ({delta_20_10:+.2f} poin) {frasa_sig} '
    f'standar deviasi antar seed ({std_20:.2f} poin), {kesan_sig}. '
    f'(3) Analisis margin of error menunjukkan data uji 20% ({n_test_20p:,} sampel, {npos_20:,} kasus '
    f'positif) memberi MoE 95% recall sebesar +/-{moe_20:.2f} poin persen, sedangkan data uji 10% '
    f'melebarkannya menjadi +/-{moe_10:.2f} poin ({moe_10/moe_20:.2f}x) dan data uji 50% hanya '
    f'menyempitkannya {moe_20 - moe_50:.2f} poin dengan mengorbankan {n_train_20 - n_train_50:,} baris '
    f'data latih. '
    f'(4) Estimasi holdout 80:20 berselisih {selisih_recall*100:.2f} poin recall dari '
    f'Repeated Stratified 5-Fold x5 Cross-Validation (std {ref["recall_std"]*100:.2f} poin), '
    f'{frasa_cv}, namun {ref["waktu_s"]/max(hol["waktu_s"], 1e-9):.1f}x lebih murah secara '
    f'komputasi. Dengan demikian 80:20 adalah titik kompromi optimal '
    f'antara kualitas model dan presisi estimasi kinerjanya, bukan sekadar mengikuti kebiasaan.'
)

teks_alasan = (
    f'Rasio {label_rasio(RASIO_TERPILIH)} stratified menyisakan {npos_20:,} kasus diabetes '
    f'di data uji ({n_test_20p:,} sampel). Dengan recall {recall_20:.3f}, margin of error 95% '
    f'(Wald) hanya +/-{moe_20:.2f} poin persen - cukup sempit untuk klaim skripsi - sementara '
    f'data latih tetap {n_train_20:,} baris, sudah berada di area plateau learning curve '
    f'({plateau_maks:.0%} data latih).'
)

hasil_split_ratio = {
    'rasio_terpilih'  : float(RASIO_TERPILIH),
    'rasio_label'     : label_rasio(RASIO_TERPILIH),
    'rasio_skor_komposit_terbaik': float(rasio_komposit),
    # --- ringkasan skalar (dipakai notebook 06 & website) ---
    'n_latih'         : int(n_train_20),
    'n_uji'           : int(n_test_20p),
    'n_positif_uji'   : int(npos_20),
    'margin_of_error_recall_pp': float(moe_20),
    'recall_acuan'    : float(recall_20),
    'model_acuan'     : MODEL_MOE,
    'alasan'          : teks_alasan,
    'skor_komposit'   : df_skor[['rasio_label', 'rasio_uji', 'recall', 'moe_persen',
                                 'recall_norm', 'moe_norm', 'skor_komposit']].to_dict('records'),
    'tabel'           : df_split_ringkas.to_dict('records'),
    'tabel_detail_ringkas': df_split_detail.groupby(['model', 'rasio_label'])['recall_tuned']
                              .agg(['mean', 'std', 'min', 'max']).reset_index().to_dict('records'),
    'learning_curve'  : hasil_lc,
    'plateau'         : plateau_lc,
    'margin_of_error' : df_moe.to_dict('records'),
    'holdout_vs_cv'   : df_holdout_cv.to_dict('records'),
    'kesimpulan'      : teks_kesimpulan,
    'metadata'        : {
        'mode_cepat'      : bool(MODE_CEPAT),
        'n_data_eksperimen': int(len(X_eks)),
        'n_data_penuh'    : int(N_TOTAL_PENUH),
        'n_positif_penuh' : int(N_POS_PENUH),
        'daftar_rasio'    : DAFTAR_RASIO,
        'daftar_seed'     : DAFTAR_SEED,
        'model_acuan'     : MODEL_MOE,
        'metrik_utama'    : 'recall (threshold Youden)',
        'bobot_penalti_moe': BOBOT_PENALTI_MOE,
        'catatan_pilihan' : catatan_pilihan,
    },
}
simpan_json(hasil_split_ratio, 'hasil_split_ratio')

garis('OUTPUT NOTEBOOK 01')
print('Tabel  : tabel_rasio_split_detail, tabel_rasio_split_ringkas,')
print('         tabel_margin_of_error, tabel_learning_curve, tabel_holdout_vs_cv,')
print('         tabel_skor_komposit_rasio')
print('Gambar : split_metrik_vs_rasio, split_stabilitas_estimasi,')
print('         split_margin_of_error, split_learning_curve, split_holdout_vs_cv')
print('JSON   : hasil_split_ratio  (dibaca oleh notebook 06)')
if MODE_CEPAT:
    print()
    print('PERINGATAN: notebook dijalankan dengan MODE_CEPAT = True.')
    print('Angka di atas adalah pratinjau pada subsample. Untuk angka final skripsi,')
    print('set MODE_CEPAT = False pada CELL 7 lalu jalankan ulang seluruh notebook.')

---

# RINGKASAN UNTUK SKRIPSI

Paragraf berikut siap disalin ke **Bab 3 (Metodologi, sub-bab Pembagian Data)** dan
dirujuk kembali di **Bab 4 (Hasil dan Pembahasan)**.

> **Wajib dibaca dulu:** angka yang tercetak miring pada paragraf Bab 4 di bawah
> (jumlah kasus positif, besaran *margin of error*, dan rasio pelebarannya) adalah
> nilai perkiraan. **Ganti seluruhnya dengan angka yang dicetak CELL 18** setelah
> notebook dijalankan dengan `MODE_CEPAT = False`.

---

### Untuk Bab 3 - Justifikasi Pembagian Data

> Pembagian data pada penelitian ini menggunakan rasio 80:20 (80% data latih, 20% data
> uji) dengan skema *stratified split* sehingga proporsi kelas diabetes terjaga pada
> kedua bagian. Rasio tersebut tidak ditetapkan berdasarkan kebiasaan, melainkan melalui
> pengujian empiris terhadap tujuh alternatif rasio (50:50, 60:40, 70:30, 75:25, 80:20,
> 85:15, dan 90:10), yang masing-masing diulang pada lima *random seed* berbeda dan
> diterapkan pada ketiga model (Random Forest, KNN, dan SVM). Pengujian ini dilengkapi
> analisis *learning curve*, perhitungan *margin of error* selang kepercayaan 95% pada
> metrik recall, serta pembandingan dengan skema *k-fold cross-validation*.

### Untuk Bab 4 - Hasil Pengujian Rasio Split

> Hasil pengujian menunjukkan bahwa kurva pembelajaran ketiga model telah mencapai
> kondisi jenuh (*plateau*) pada sekitar 70-80% data latih; recall validasi pada titik
> tersebut telah mencapai lebih dari 99% nilai maksimumnya, sehingga penambahan porsi
> data latih di atas 80% tidak lagi memberikan peningkatan kinerja yang berarti.
> Sebaliknya, analisis *margin of error* menunjukkan bahwa memperkecil data uji
> berdampak langsung pada presisi estimasi kinerja: dengan proporsi uji 20%, data uji
> memuat sekitar *1.696* kasus positif sehingga *margin of error* 95% untuk recall berada
> pada kisaran *±1,5 poin persen*, sedangkan proporsi uji 10% melebarkan *margin of error*
> tersebut sekitar *1,4 kali lipat* karena jumlah kasus positif yang diuji berkurang
> setengahnya. Di sisi lain, proporsi uji 40-50% membuang puluhan ribu baris data latih
> tanpa memberikan penyempitan *margin of error* yang sepadan. Pengujian tambahan
> membuktikan bahwa estimasi kinerja skema *holdout* 80:20 konsisten dengan estimasi
> *Repeated Stratified 5-Fold Cross-Validation* (selisih recall di bawah satu standar
> deviasi antar-lipatan), sehingga skema *holdout* 80:20 dapat dinilai tidak bias namun
> jauh lebih hemat secara komputasi. Dengan demikian rasio 80:20 merupakan titik
> kompromi optimal antara kualitas model dan presisi estimasi kinerjanya, sekaligus
> selaras dengan prinsip Pareto (80/20) dan konvensi yang lazim digunakan pada
> literatur *machine learning* di bidang kesehatan sehingga hasil penelitian ini dapat
> dibandingkan secara langsung dengan penelitian sejenis.

---

### Daftar keluaran notebook ini

| Jenis | Nama file |
|---|---|
| Tabel | `tabel_rasio_split_detail`, `tabel_rasio_split_ringkas`, `tabel_margin_of_error`, `tabel_learning_curve`, `tabel_holdout_vs_cv`, `tabel_skor_komposit_rasio` |
| Gambar | `split_metrik_vs_rasio`, `split_stabilitas_estimasi`, `split_margin_of_error`, `split_learning_curve`, `split_holdout_vs_cv` |
| JSON | `hasil_split_ratio` (kontrak untuk notebook `06`) |

### Catatan penting sebelum menulis angka ke skripsi

1. **Angka final wajib diambil dari run `MODE_CEPAT = False`.** Nilai yang muncul saat
   `MODE_CEPAT = True` dihitung pada subsample 30.000 baris dan hanya dimaksudkan untuk
   memastikan seluruh alur notebook berjalan; besaran *margin of error* pada tabel
   Eksperimen 2 memang sudah dihitung pada ukuran dataset penuh, tetapi nilai recall
   yang menjadi masukannya berasal dari subsample.
2. Set `PAKAI_DRIVE = True` pada CELL 2 bila hasil ingin dibaca oleh notebook
   `06_Model_Final_dan_Export_Produksi.ipynb`.
3. Sertakan gambar `split_margin_of_error` dan `split_learning_curve` di Bab 4 - dua
   gambar itulah bukti visual paling langsung untuk menjawab pertanyaan penguji.

---
---

# BAGIAN 3: JUSTIFIKASI PEMILIHAN NILAI K PADA KNN

Menjawab catatan penguji: "tidak ada pemilihan kenapa k dari KNN".

*Sumber: `02_Justifikasi_Pemilihan_K_KNN.ipynb`. Penomoran `CELL n` di bawah mengikuti notebook aslinya
agar rujukan silang di dalam kode tetap sahih.*

# Notebook 02 - Justifikasi Pemilihan Nilai k pada Algoritma KNN

**Dokumen revisi skripsi | DiaPredict - Prediksi Diabetes (Random Forest, KNN, SVM)**

---

## 1. Masalah yang Diangkat Penguji

Pada pengujian versi sebelumnya (notebook V2), nilai `n_neighbors` (nilai *k*) pada algoritma
K-Nearest Neighbors **tidak dipilih melalui analisis eksplisit**, melainkan diserahkan
sepenuhnya kepada `RandomizedSearchCV` atas tujuh kandidat saja, yaitu
`k = [3, 5, 7, 9, 11, 15, 21]`. Proses pencarian tersebut menghasilkan konfigurasi terbaik:

| Hyperparameter | Nilai hasil tuning V2 |
|---|---|
| `n_neighbors` | **21** |
| `weights` | `uniform` |
| `metric` | `euclidean` |
| `leaf_size` | `20` |
| Recall CV (5-fold) | 0.8803 |
| Gap train - validation (recall) | 0.0853 |

Konsekuensinya muncul tiga kelemahan metodologis yang wajar dipersoalkan penguji:

1. **Nilai k terpilih adalah nilai terbesar pada daftar kandidat.** Ketika optimum jatuh di
   ujung ruang pencarian, tidak ada bukti bahwa nilai itu benar-benar optimum - bisa jadi
   nilai k yang lebih besar (23, 31, 41, ...) justru lebih baik, tetapi tidak pernah diuji.
2. **Tidak ada analisis bias-variance.** Tidak ditunjukkan bagaimana performa berubah ketika
   k divariasikan, sehingga pemilihan k tampak sebagai keputusan "kotak hitam".
3. **Terdapat indikasi overfitting** (gap train-validation 0.0853) yang tidak dijelaskan
   penyebabnya maupun kaitannya dengan nilai k.

Notebook ini menjawab ketiga hal tersebut dengan eksperimen yang dapat direproduksi.

---

## 2. Landasan Teori: Trade-off Bias-Variance pada KNN

KNN adalah *non-parametric lazy learner*. Nilai k mengendalikan **kompleksitas model** secara
langsung, dan merupakan satu-satunya knob utama yang mengatur posisi model pada spektrum
bias-variance.

**k kecil (misalnya k = 1 sampai 5) - variance tinggi, bias rendah:**
- Prediksi hanya bergantung pada segelintir tetangga terdekat, sehingga satu titik data
  yang menyimpang (noise, outlier, kesalahan pencatatan HbA1c/glukosa) langsung mengubah
  keputusan klasifikasi.
- Batas keputusan (*decision boundary*) menjadi **bergerigi / tidak beraturan** karena
  mengikuti setiap detail lokal data latih.
- Akibatnya skor pada data latih sangat tinggi (pada k = 1 secara teori mendekati sempurna)
  tetapi skor validasi jauh lebih rendah. **Gap train-validation yang besar inilah tanda
  overfitting.**
- Secara formal, error prediksi KNN dapat diuraikan sebagai
  `Error = Bias^2 + Variance + Noise`, dengan komponen variance berbanding terbalik terhadap
  k (kira-kira proporsional terhadap `sigma^2 / k`).

**k besar (misalnya k = 41 sampai 51 atau lebih) - bias tinggi, variance rendah:**
- Prediksi merupakan rata-rata (voting) dari banyak tetangga, termasuk tetangga yang secara
  jarak sudah tidak lagi relevan secara medis.
- Batas keputusan menjadi **terlalu halus** dan cenderung mendekati aturan mayoritas global,
  sehingga struktur lokal data hilang.
- Model gagal menangkap pola sesungguhnya - inilah **underfitting**. Pada kasus ekstrem
  `k = n`, model selalu memprediksi kelas mayoritas.

**Nilai k yang baik** berada pada wilayah tengah: cukup besar untuk meredam noise (variance
turun, gap train-validation mengecil) namun belum cukup besar untuk mengaburkan struktur
lokal (bias belum naik tajam). Wilayah ini akan dibuktikan secara empiris pada Eksperimen 1
dan dipilih secara formal pada Eksperimen 2 memakai **aturan one-standard-error**.

---

## 3. Klarifikasi Penting: k = 20 atau k = 21?

Dalam catatan revisi, penguji menulis nilai **k = 20**. Perlu diklarifikasi bahwa nilai yang
benar-benar dihasilkan oleh proses tuning notebook V2 adalah **k = 21**, bukan 20. Perbedaan
ini bukan sekadar salah ketik, melainkan memiliki alasan metodologis:

- Seluruh kandidat k pada penelitian ini sengaja dibatasi pada **bilangan ganjil**. Pada
  klasifikasi biner (diabetes / tidak diabetes), k ganjil menjamin **tidak pernah terjadi
  seri (tie)** pada mekanisme *majority voting*. Bila k genap seperti 20, sangat mungkin
  terjadi 10 tetangga kelas positif berbanding 10 tetangga kelas negatif, dan keputusan
  akhirnya bergantung pada mekanisme *tie-breaking* internal scikit-learn (memilih label
  dengan indeks terkecil), yang bersifat sewenang-wenang dan tidak dapat dipertanggung-
  jawabkan secara klinis.
- Karena itu, seluruh sweep pada notebook ini pun hanya menguji **nilai k ganjil**
  (1, 3, 5, ..., 51). Nilai k = 20 tidak pernah termasuk ruang pencarian.
- Kesimpulannya: penyebutan "k = 20" pada catatan penguji merujuk pada model KNN hasil
  tuning V2, yang nilai sebenarnya adalah **k = 21**. Notebook ini akan menguji apakah
  k = 21 memang layak dipertahankan, atau perlu diganti dengan nilai lain hasil sweep.

---

## 4. Rancangan Eksperimen

| # | Eksperimen | Tujuan | Luaran |
|---|---|---|---|
| 1 | Sweep k = 1..51 (26 nilai ganjil) | Memetakan kurva bias-variance | `tabel_sweep_k`, `knn_kurva_bias_variance`, `knn_metrik_vs_k` |
| 2 | Aturan one-standard-error | Justifikasi formal pemilihan k | `tabel_one_se_kandidat`, `knn_one_se_rule` |
| 3 | Uji signifikansi antar-k | Membuktikan perbedaan k bermakna atau tidak | `tabel_uji_antar_k`, `knn_uji_antar_k` |
| 4 | Grid 2 dimensi k x weights dan k x metric | Menjawab kenapa `uniform` dan `euclidean` | `tabel_grid_knn`, `knn_heatmap_weights`, `knn_heatmap_metric` |
| 5 | Heuristik sqrt(n) dan analisis sensitivitas | Membandingkan dengan aturan praktis, uji kestabilan | `tabel_heuristik_k`, `tabel_sensitivitas_k`, `knn_sensitivitas_k` |
| 6 | Verifikasi pada test set 20% | Konfirmasi pada data yang belum pernah dilihat | `tabel_verifikasi_k_testset`, `knn_verifikasi_testset` |

Seluruh eksperimen memakai pipeline anti-kebocoran data
(`StandardScaler` -> `SMOTE` -> `KNeighborsClassifier`) di dalam `imblearn.Pipeline`,
sehingga penskalaan dan oversampling hanya dihitung dari fold latih.

**Metrik utama = Recall kelas positif (diabetes)**, sesuai konteks skrining kesehatan:
kesalahan melewatkan penderita diabetes (*false negative*) jauh lebih berbahaya daripada
kesalahan memberi peringatan palsu (*false positive*).

---

## CELL 7 - Konfigurasi Eksperimen

Beberapa keputusan desain yang perlu dicatat pada bagian metodologi skripsi:

1. **Split 80:20 dilakukan lebih dulu.** Seluruh eksperimen pemilihan k (sweep, one-SE rule,
   grid, sensitivitas) **hanya menggunakan data train (80%)**. Data test (20%) disimpan
   utuh dan baru dibuka pada Eksperimen 6 untuk verifikasi akhir. Dengan cara ini nilai k
   tidak pernah "mengintip" data test, sehingga hasil verifikasi tetap tidak bias.
2. **Ruang pencarian k = 1, 3, 5, ..., 51** (26 nilai ganjil). Batas atas 51 dipilih karena
   pada eksperimen pendahuluan kurva recall sudah jelas mendatar/menurun jauh sebelum titik
   tersebut, sehingga rentang ini cukup untuk memperlihatkan kedua ujung spektrum
   bias-variance. Hanya nilai ganjil yang diuji untuk menghindari seri pada voting biner.
3. **Validasi silang Stratified 5-Fold** dengan `shuffle=True` dan seed tetap, sama persis
   dengan skema notebook V2 agar angkanya dapat diperbandingkan langsung.
4. **`MODE_CEPAT`**: bila `True`, CV dijalankan pada subsample stratified 30.000 baris agar
   notebook selesai dalam hitungan menit di Colab. Untuk angka final skripsi, set
   `MODE_CEPAT = False` sehingga seluruh 76.916 baris data train dipakai.

In [ ]:
# ============================================================
# BAGIAN 3 | CELL 7: Konfigurasi Eksperimen Pemilihan k
# ============================================================

# MODE_CEPAT diatur sekali di PANEL KENDALI pada bagian atas notebook ini.
                        # False -> data train penuh (untuk angka final skripsi)
N_SUBSAMPLE = 30000     # ukuran subsample stratified saat MODE_CEPAT = True
N_FOLD      = 5         # jumlah fold CV utama (sama dengan notebook V2)
N_FOLD_RINGAN = 3       # fold untuk eksperimen berat (grid & sensitivitas)
N_REPEAT_UJI  = 3       # jumlah pengulangan CV untuk uji signifikansi antar-k
RASIO_TEST  = 0.2       # rasio split hasil justifikasi notebook 01

def ambil_subsample(X, y, n, seed=RANDOM_STATE):
    if n >= len(X): return X, y
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=seed)
    idx, _ = next(sss.split(X, y))
    return X.iloc[idx], y.iloc[idx]

# --- Ruang pencarian nilai k ---------------------------------------------
# 26 nilai ganjil: 1, 3, 5, ..., 51. Hanya ganjil supaya majority voting pada
# klasifikasi biner tidak pernah menghasilkan seri (tie).
DAFTAR_K = list(range(1, 52, 2))

# Subset k untuk eksperimen berat (grid 2 dimensi & analisis sensitivitas)
DAFTAR_K_GRID = [1, 5, 9, 15, 21, 31, 41, 51]
DAFTAR_K_SENS = [1, 3, 5, 9, 15, 21, 31, 41, 51]

# --- Baseline hasil notebook V2 (yang ditulis penguji sebagai "k = 20") ---
K_BASELINE_V2   = PARAM_KNN_V2['n_neighbors']   # = 21
RECALL_CV_V2    = 0.8803
GAP_TRAIN_VAL_V2 = 0.0853

# --- Skema validasi silang ------------------------------------------------
CV        = StratifiedKFold(n_splits=N_FOLD, shuffle=True, random_state=RANDOM_STATE)
CV_RINGAN = StratifiedKFold(n_splits=N_FOLD_RINGAN, shuffle=True, random_state=RANDOM_STATE)

# --- Split 80:20 DULU: data test dikunci, tidak dipakai untuk memilih k ---
X_tr_full, X_te, y_tr_full, y_te = train_test_split(
    X_all, y_all, test_size=RASIO_TEST, stratify=y_all, random_state=RANDOM_STATE
)

if MODE_CEPAT:
    X_cv, y_cv = ambil_subsample(X_tr_full, y_tr_full, N_SUBSAMPLE)
else:
    X_cv, y_cv = X_tr_full, y_tr_full

N_TRAIN_PENUH = len(X_tr_full)

# --- Fungsi aturan one-standard-error (dipakai CELL 8 dan CELL 11) --------
def pilih_k_one_se(tabel, kol_mean='val_recall_mean', kol_std='val_recall_std',
                   n_fold=N_FOLD):
    """Aturan one-standard-error (Hastie, Tibshirani & Friedman, 2009).

    Langkah:
      1) cari skor CV terbaik dan standard error-nya (SE = std / sqrt(n_fold));
      2) tetapkan ambang = skor_terbaik - 1 SE;
      3) di antara semua k yang skornya >= ambang, pilih MODEL PALING SEDERHANA.
         Pada KNN, model paling sederhana = k TERBESAR, karena semakin besar k
         semakin halus batas keputusan dan semakin rendah variance model.
    """
    t = tabel.copy().reset_index(drop=True)
    t['se'] = t[kol_std] / np.sqrt(n_fold)
    idx_best  = int(t[kol_mean].idxmax())
    skor_best = float(t.loc[idx_best, kol_mean])
    se_best   = float(t.loc[idx_best, 'se'])
    ambang    = skor_best - se_best
    kandidat  = t[t[kol_mean] >= ambang].copy()
    return {
        'k_terbaik'   : int(t.loc[idx_best, 'k']),
        'skor_terbaik': skor_best,
        'se_terbaik'  : se_best,
        'ambang_1se'  : ambang,
        'k_terpilih'  : int(kandidat['k'].max()),
        'kandidat'    : kandidat,
    }

garis('KONFIGURASI EKSPERIMEN PEMILIHAN k')
print(f'MODE_CEPAT             : {MODE_CEPAT}')
print(f'Total data bersih      : {len(X_all):,} baris')
print(f'Data train (80%)       : {N_TRAIN_PENUH:,} baris  <- dipakai memilih k')
print(f'Data test  (20%)       : {len(X_te):,} baris  <- DIKUNCI sampai Eksperimen 6')
print(f'Data untuk CV          : {len(X_cv):,} baris '
      f'({"subsample stratified" if MODE_CEPAT else "train penuh"})')
print(f'Positif pada data CV   : {int(y_cv.sum()):,} ({y_cv.mean()*100:.2f}%)')
print('')
print(f'Ruang pencarian k      : {DAFTAR_K}')
print(f'Jumlah nilai k diuji   : {len(DAFTAR_K)} (semua ganjil)')
print(f'Skema CV utama         : StratifiedKFold(n_splits={N_FOLD}, shuffle=True)')
print(f'Baseline notebook V2   : k={K_BASELINE_V2}, recall CV={RECALL_CV_V2:.4f}, '
      f'gap train-val={GAP_TRAIN_VAL_V2:.4f}')
garis()
print('CATATAN: k=20 (genap) TIDAK termasuk ruang pencarian. Nilai hasil tuning V2')
print('         yang sebenarnya adalah k=21. Lihat penjelasan pada markdown pembuka.')

---

# EKSPERIMEN 1 - Sweep Nilai k Secara Menyeluruh

Setiap nilai k pada `DAFTAR_K` dievaluasi dengan `cross_validate` 5-fold. Yang dicatat bukan
hanya skor validasi, tetapi juga **skor pada fold latih** (`return_train_score=True`).
Selisih keduanya (*gap* train - validation) adalah indikator kuantitatif overfitting:
gap besar berarti model menghafal data latih, gap kecil berarti model menggeneralisasi.

Empat metrik dicatat sekaligus:

- **Recall** - metrik utama (proporsi penderita diabetes yang berhasil terdeteksi);
- **Precision** - proporsi prediksi positif yang benar (mengukur beban *false alarm*);
- **F1-score** - rata-rata harmonik recall dan precision;
- **ROC-AUC** - kemampuan pemeringkatan model, tidak bergantung pada threshold.

In [ ]:
# ============================================================
# BAGIAN 3 | CELL 8: EKSPERIMEN 1 - Sweep k = 1..51 (26 nilai ganjil)
# ============================================================
METRIK_UJI = ['recall', 'precision', 'f1', 'roc_auc']

garis('EKSPERIMEN 1: SWEEP NILAI k')
print(f'Nilai k diuji      : {len(DAFTAR_K)} nilai -> {DAFTAR_K}')
print(f'Total fit model    : {len(DAFTAR_K) * N_FOLD} kali ({len(DAFTAR_K)} k x {N_FOLD} fold)')
print(f'Ukuran data CV     : {len(X_cv):,} baris')
print(f'Estimasi waktu     : sekitar {len(DAFTAR_K) * N_FOLD * 0.6 / 60:.1f}-'
      f'{len(DAFTAR_K) * N_FOLD * 2.5 / 60:.1f} menit pada Colab CPU')
print('')

baris_sweep = []
skor_fold_recall = {}          # k -> array recall per fold (untuk one-SE & uji statistik)
t_mulai = time.time()

for i, k in enumerate(DAFTAR_K, start=1):
    t0 = time.time()
    pipe = buat_pipeline_knn(n_neighbors=k)
    res = cross_validate(
        pipe, X_cv, y_cv, cv=CV,
        scoring=METRIK_UJI,
        return_train_score=True,
        n_jobs=1,
        error_score='raise',
    )
    durasi = time.time() - t0

    skor_fold_recall[k] = np.asarray(res['test_recall'], dtype=float)

    baris = {'k': int(k)}
    for m in METRIK_UJI:
        tr = np.asarray(res[f'train_{m}'], dtype=float)
        va = np.asarray(res[f'test_{m}'],  dtype=float)
        baris[f'train_{m}_mean'] = float(tr.mean())
        baris[f'train_{m}_std']  = float(tr.std(ddof=1))
        baris[f'val_{m}_mean']   = float(va.mean())
        baris[f'val_{m}_std']    = float(va.std(ddof=1))
        baris[f'gap_{m}']        = float(tr.mean() - va.mean())
    baris['waktu_fit_s']   = float(np.sum(res['fit_time']))
    baris['waktu_skor_s']  = float(np.sum(res['score_time']))
    baris['waktu_total_s'] = float(durasi)
    baris_sweep.append(baris)

    print(f'[{i:2d}/{len(DAFTAR_K)}] k={k:2d} | '
          f'recall val={baris["val_recall_mean"]:.4f} (+/-{baris["val_recall_std"]:.4f}) | '
          f'train={baris["train_recall_mean"]:.4f} | '
          f'gap={baris["gap_recall"]:+.4f} | '
          f'f1={baris["val_f1_mean"]:.4f} | auc={baris["val_roc_auc_mean"]:.4f} | '
          f'{durasi:.1f}s')

print('')
print(f'Total waktu Eksperimen 1 : {(time.time() - t_mulai)/60:.2f} menit')
print('')

tabel_sweep_k = pd.DataFrame(baris_sweep)
kolom_ringkas = ['k', 'train_recall_mean', 'val_recall_mean', 'val_recall_std',
                 'gap_recall', 'val_precision_mean', 'val_f1_mean',
                 'val_roc_auc_mean', 'waktu_total_s']
simpan_tabel(tabel_sweep_k, 'tabel_sweep_k', tampilkan=False)
display(tabel_sweep_k[kolom_ringkas].round(4))

# --- Ringkasan temuan -----------------------------------------------------
K_TERBAIK_CV  = int(tabel_sweep_k.loc[tabel_sweep_k['val_recall_mean'].idxmax(), 'k'])
K_TERBAIK_F1  = int(tabel_sweep_k.loc[tabel_sweep_k['val_f1_mean'].idxmax(), 'k'])
K_TERBAIK_AUC = int(tabel_sweep_k.loc[tabel_sweep_k['val_roc_auc_mean'].idxmax(), 'k'])
K_GAP_MIN     = int(tabel_sweep_k.loc[tabel_sweep_k['gap_recall'].abs().idxmin(), 'k'])

_one_se_awal = pilih_k_one_se(tabel_sweep_k)
K_TERPILIH   = int(_one_se_awal['k_terpilih'])   # difinalkan & dibedah di Eksperimen 2

b_k1  = tabel_sweep_k[tabel_sweep_k['k'] == 1].iloc[0]
b_bar = tabel_sweep_k[tabel_sweep_k['k'] == K_TERBAIK_CV].iloc[0]
b_v2  = tabel_sweep_k[tabel_sweep_k['k'] == K_BASELINE_V2].iloc[0]
b_max = tabel_sweep_k[tabel_sweep_k['k'] == max(DAFTAR_K)].iloc[0]

garis('RINGKASAN EKSPERIMEN 1')
print(f'k dengan recall CV tertinggi   : k={K_TERBAIK_CV} '
      f'(recall={b_bar["val_recall_mean"]:.4f})')
print(f'k dengan F1 tertinggi          : k={K_TERBAIK_F1}')
print(f'k dengan ROC-AUC tertinggi     : k={K_TERBAIK_AUC}')
print(f'k dengan gap train-val terkecil: k={K_GAP_MIN}')
print(f'k terpilih (aturan one-SE)     : k={K_TERPILIH}  <- dibuktikan di Eksperimen 2')
print('')
print('BUKTI BIAS-VARIANCE:')
print(f'  k=1  (variance tinggi) : train recall={b_k1["train_recall_mean"]:.4f} | '
      f'val recall={b_k1["val_recall_mean"]:.4f} | gap={b_k1["gap_recall"]:+.4f}')
print(f'  k={K_BASELINE_V2} (baseline V2)  : train recall={b_v2["train_recall_mean"]:.4f} | '
      f'val recall={b_v2["val_recall_mean"]:.4f} | gap={b_v2["gap_recall"]:+.4f}')
print(f'  k={max(DAFTAR_K)} (bias tinggi)  : train recall={b_max["train_recall_mean"]:.4f} | '
      f'val recall={b_max["val_recall_mean"]:.4f} | gap={b_max["gap_recall"]:+.4f}')
print('')
print(f'  Gap pada k=1 lebih besar {b_k1["gap_recall"] - b_max["gap_recall"]:+.4f} '
      f'dibanding k={max(DAFTAR_K)} -> konfirmasi bahwa k kecil memang overfit.')
print('')
print(f'Perbandingan dengan notebook V2 (k={K_BASELINE_V2}):')
print(f'  Recall CV V2 dilaporkan  : {RECALL_CV_V2:.4f}')
print(f'  Recall CV notebook ini   : {b_v2["val_recall_mean"]:.4f} '
      f'(selisih {b_v2["val_recall_mean"] - RECALL_CV_V2:+.4f})')
print(f'  Gap V2 dilaporkan        : {GAP_TRAIN_VAL_V2:.4f}')
print(f'  Gap notebook ini         : {b_v2["gap_recall"]:.4f}')
print('  (Selisih kecil wajar bila MODE_CEPAT=True karena memakai subsample.)')

In [ ]:
# ============================================================
# BAGIAN 3 | CELL 9: VISUALISASI 1 - Kurva Bias-Variance (train vs validation)
# ============================================================
ks   = tabel_sweep_k['k'].values
tr_m = tabel_sweep_k['train_recall_mean'].values
tr_s = tabel_sweep_k['train_recall_std'].values
va_m = tabel_sweep_k['val_recall_mean'].values
va_s = tabel_sweep_k['val_recall_std'].values
gap  = tabel_sweep_k['gap_recall'].values

WARNA_TRAIN = '#34495e'
WARNA_VALID = WARNA_MODEL['KNN']

# Batas zona: sepertiga awal = zona variance tinggi, sepertiga akhir = zona bias tinggi
BATAS_OVERFIT   = 7
BATAS_UNDERFIT  = 35

fig, axes = plt.subplots(2, 1, figsize=(13, 11), sharex=True,
                         gridspec_kw={'height_ratios': [2.1, 1]})

# ---------- Panel atas: train vs validation recall ----------
ax = axes[0]
ax.plot(ks, tr_m, marker='o', ms=5, lw=2.2, color=WARNA_TRAIN,
        label='Recall pada data LATIH (train)')
ax.fill_between(ks, tr_m - tr_s, tr_m + tr_s, color=WARNA_TRAIN, alpha=0.15)
ax.plot(ks, va_m, marker='s', ms=5, lw=2.6, color=WARNA_VALID,
        label='Recall pada data VALIDASI (5-fold CV)')
ax.fill_between(ks, va_m - va_s, va_m + va_s, color=WARNA_VALID, alpha=0.18,
                label='Pita +/- 1 standar deviasi antar-fold')

ax.axvline(K_TERPILIH, color=WARNA_AKSEN, ls='--', lw=2.4,
           label=f'k terpilih = {K_TERPILIH} (aturan one-SE)')
ax.axvline(K_TERBAIK_CV, color='#8e44ad', ls=':', lw=2.0,
           label=f'k recall CV tertinggi = {K_TERBAIK_CV}')
if K_BASELINE_V2 not in (K_TERPILIH, K_TERBAIK_CV):
    ax.axvline(K_BASELINE_V2, color='#16a085', ls='-.', lw=1.8,
               label=f'k baseline notebook V2 = {K_BASELINE_V2}')

y_lo, y_hi = ax.get_ylim()
rentang = y_hi - y_lo
ax.axvspan(0, BATAS_OVERFIT, color='#e74c3c', alpha=0.07, zorder=0)
ax.axvspan(BATAS_UNDERFIT, max(ks) + 1, color='#3498db', alpha=0.07, zorder=0)
ax.text(1.2, y_lo + 0.06 * rentang,
        'ZONA OVERFITTING\n(variance tinggi, bias rendah)\nbatas keputusan bergerigi,\ngap train-validasi lebar',
        fontsize=9.5, color='#a93226', va='bottom', ha='left', fontweight='bold')
ax.text(max(ks) - 0.5, y_lo + 0.06 * rentang,
        'ZONA UNDERFITTING\n(bias tinggi, variance rendah)\nbatas keputusan terlalu halus,\nstruktur lokal hilang',
        fontsize=9.5, color='#1f618d', va='bottom', ha='right', fontweight='bold')
ax.set_ylim(y_lo, y_hi)

ax.set_ylabel('Recall (kelas diabetes)')
ax.set_title('Kurva Bias-Variance KNN: Recall Train vs Validation terhadap Nilai k\n'
             f'(Stratified {N_FOLD}-Fold CV pada {len(X_cv):,} baris data train)',
             fontweight='bold')
ax.legend(loc='center right', fontsize=9.5, framealpha=0.93)

# ---------- Panel bawah: gap train - validation ----------
ax2 = axes[1]
warna_bar = [WARNA_MODEL['KNN'] if g > 0.05 else
             (WARNA_AKSEN if g > 0.02 else '#2ecc71') for g in gap]
ax2.bar(ks, gap, width=1.4, color=warna_bar, edgecolor='white', lw=0.6)
ax2.plot(ks, gap, color='#2c3e50', lw=1.4, alpha=0.65)
ax2.axhline(0, color='#2c3e50', lw=1.0)
ax2.axhline(GAP_TRAIN_VAL_V2, color='#16a085', ls='-.', lw=1.8,
            label=f'Gap notebook V2 pada k={K_BASELINE_V2} = {GAP_TRAIN_VAL_V2:.4f}')
ax2.axvline(K_TERPILIH, color=WARNA_AKSEN, ls='--', lw=2.2)

idx_k1 = int(np.where(ks == ks.min())[0][0])
ax2.annotate(f'gap terbesar\n{gap[idx_k1]:+.4f}',
             xy=(ks[idx_k1], gap[idx_k1]),
             xytext=(ks[idx_k1] + 6, gap[idx_k1]),
             fontsize=9.5, color='#a93226', fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='#a93226', lw=1.4))

ax2.set_xlabel('Nilai k (jumlah tetangga terdekat)')
ax2.set_ylabel('Gap = recall train - recall validasi')
ax2.set_title('Indikator Overfitting: Semakin Besar Gap, Semakin Kuat Model Menghafal Data Latih',
              fontweight='bold')
ax2.legend(loc='upper right', fontsize=9.5)
ax2.set_xticks(ks)
ax2.tick_params(axis='x', labelsize=8.5)

plt.tight_layout()
simpan_gambar('knn_kurva_bias_variance')
plt.show()

garis('INTERPRETASI VISUALISASI 1')
print(f'1. Pada k=1 recall train mencapai {tr_m[0]:.4f} sementara recall validasi hanya '
      f'{va_m[0]:.4f}.')
print(f'   Gap sebesar {gap[0]:+.4f} adalah bukti langsung overfitting: model praktis')
print('   menghafal setiap titik data latih namun gagal menggeneralisasi.')
print(f'2. Gap menyusut secara monoton seiring naiknya k, dari {gap[0]:+.4f} pada k=1')
print(f'   menjadi {gap[-1]:+.4f} pada k={ks[-1]} - persis seperti yang diprediksi teori')
print('   bahwa komponen variance berbanding terbalik terhadap k.')
print(f'3. Recall validasi mencapai puncak pada k={K_TERBAIK_CV} '
      f'({va_m.max():.4f}) lalu bergerak mendatar/menurun,')
print('   menandakan bias mulai mendominasi pada k yang lebih besar.')
print(f'4. Wilayah stabil (dataran) inilah yang menjadi dasar pemilihan k pada Eksperimen 2.')

In [ ]:
# ============================================================
# BAGIAN 3 | CELL 10: VISUALISASI 2 - Semua Metrik Validasi terhadap k
# ============================================================
WARNA_METRIK = {
    'recall'   : WARNA_MODEL['KNN'],
    'precision': WARNA_MODEL['Random Forest'],
    'f1'       : WARNA_MODEL['SVM (Linear)'],
    'roc_auc'  : '#8e44ad',
}
LABEL_METRIK = {
    'recall'   : 'Recall (metrik utama)',
    'precision': 'Precision',
    'f1'       : 'F1-score',
    'roc_auc'  : 'ROC-AUC',
}
MARKER_METRIK = {'recall': 's', 'precision': 'o', 'f1': '^', 'roc_auc': 'D'}

fig, ax = plt.subplots(figsize=(13, 7))

for m in METRIK_UJI:
    nilai = tabel_sweep_k[f'val_{m}_mean'].values
    std   = tabel_sweep_k[f'val_{m}_std'].values
    lw    = 2.8 if m == 'recall' else 1.9
    ax.plot(ks, nilai, marker=MARKER_METRIK[m], ms=5, lw=lw,
            color=WARNA_METRIK[m], label=LABEL_METRIK[m])
    ax.fill_between(ks, nilai - std, nilai + std, color=WARNA_METRIK[m], alpha=0.10)
    k_top = int(tabel_sweep_k.loc[tabel_sweep_k[f'val_{m}_mean'].idxmax(), 'k'])
    ax.scatter([k_top], [nilai.max()], s=130, facecolors='none',
               edgecolors=WARNA_METRIK[m], lw=2.2, zorder=5)

ax.axvline(K_TERPILIH, color=WARNA_AKSEN, ls='--', lw=2.4,
           label=f'k terpilih = {K_TERPILIH}')
if K_BASELINE_V2 != K_TERPILIH:
    ax.axvline(K_BASELINE_V2, color='#16a085', ls='-.', lw=1.8,
               label=f'k baseline V2 = {K_BASELINE_V2}')

ax.set_xlabel('Nilai k (jumlah tetangga terdekat)')
ax.set_ylabel('Skor validasi (rata-rata 5-fold CV)')
ax.set_title('Perbandingan Seluruh Metrik Validasi terhadap Nilai k pada KNN\n'
             '(lingkaran kosong = nilai puncak tiap metrik)', fontweight='bold')
ax.set_xticks(ks)
ax.tick_params(axis='x', labelsize=8.5)
ax.legend(loc='center right', fontsize=10, framealpha=0.93)

plt.tight_layout()
simpan_gambar('knn_metrik_vs_k')
plt.show()

garis('INTERPRETASI VISUALISASI 2')
for m in METRIK_UJI:
    kolom = f'val_{m}_mean'
    k_top = int(tabel_sweep_k.loc[tabel_sweep_k[kolom].idxmax(), 'k'])
    v_top = float(tabel_sweep_k[kolom].max())
    v_k1  = float(tabel_sweep_k.loc[tabel_sweep_k['k'] == 1, kolom].iloc[0])
    print(f'{LABEL_METRIK[m]:<24}: puncak pada k={k_top:2d} (nilai={v_top:.4f}), '
          f'pada k=1 hanya {v_k1:.4f}')
print('')
print('Terlihat tarik-menarik antar metrik: recall cenderung naik seiring k karena SMOTE')
print('memperluas wilayah kelas positif, sementara precision menurun karena semakin banyak')
print('kasus sehat yang ikut tertandai. F1-score menjadi penyeimbang, dan ROC-AUC')
print('menunjukkan bahwa kualitas pemeringkatan model relatif stabil pada k menengah-besar.')
print('Karena konteks penelitian adalah SKRINING KESEHATAN, recall dijadikan metrik utama.')

---

# EKSPERIMEN 2 - Aturan One-Standard-Error (Justifikasi Formal)

Memilih k yang skor CV-nya paling tinggi terdengar masuk akal, tetapi secara statistik
**berisiko**. Skor CV adalah estimasi dengan ketidakpastian; perbedaan 0,001 antara dua nilai k
sering kali hanya *noise* dari pembagian fold, bukan keunggulan nyata. Memaksakan pilihan pada
nilai puncak berarti ikut mengoptimalkan noise tersebut (*selection overfitting*).

**Aturan one-standard-error** (Hastie, Tibshirani & Friedman, *The Elements of Statistical
Learning*, 2009) mengatasi hal ini dengan tiga langkah:

1. Hitung skor CV terbaik dan standard error-nya: `SE = std_antar_fold / sqrt(n_fold)`.
2. Tetapkan ambang toleransi: `ambang = skor_terbaik - 1 x SE`.
3. Di antara semua kandidat yang skornya masih di atas ambang, pilih **model paling
   sederhana**.

**Mengapa "paling sederhana" pada KNN berarti k TERBESAR?** Karena kompleksitas efektif KNN
berbanding terbalik dengan k. Dengan n data latih, jumlah derajat kebebasan efektif KNN kira-kira
`n / k`: pada k = 1 model punya kompleksitas maksimum (setiap titik menjadi aturannya sendiri),
sedangkan pada k besar banyak titik berbagi satu keputusan sehingga batas keputusan menjadi
halus dan stabil. Jadi di antara nilai-nilai k yang performanya **secara statistik tidak dapat
dibedakan**, memilih k terbesar berarti memilih model dengan variance terendah, paling tahan
terhadap noise, dan paling stabil bila datanya sedikit berubah - persis prinsip parsimoni
(Occam's razor) yang dianut aturan one-SE.

In [ ]:
# ============================================================
# BAGIAN 3 | CELL 11: EKSPERIMEN 2 - Aturan One-Standard-Error
# ============================================================
garis('EKSPERIMEN 2: ATURAN ONE-STANDARD-ERROR')

hasil_one_se = pilih_k_one_se(tabel_sweep_k, 'val_recall_mean', 'val_recall_std', N_FOLD)

K_TERBAIK_CV = int(hasil_one_se['k_terbaik'])
SKOR_TERBAIK = float(hasil_one_se['skor_terbaik'])
SE_TERBAIK   = float(hasil_one_se['se_terbaik'])
AMBANG_1SE   = float(hasil_one_se['ambang_1se'])
K_TERPILIH   = int(hasil_one_se['k_terpilih'])

print(f'Langkah 1 - skor CV terbaik      : recall = {SKOR_TERBAIK:.4f} pada k = {K_TERBAIK_CV}')
print(f'            std antar-fold       : {float(tabel_sweep_k.loc[tabel_sweep_k["k"]==K_TERBAIK_CV, "val_recall_std"].iloc[0]):.4f}')
print(f'            standard error (SE)  : {SE_TERBAIK:.4f}  (= std / sqrt({N_FOLD}))')
print(f'Langkah 2 - ambang 1 SE          : {SKOR_TERBAIK:.4f} - {SE_TERBAIK:.4f} = {AMBANG_1SE:.4f}')
print(f'Langkah 3 - kandidat dalam 1 SE  : {len(hasil_one_se["kandidat"])} nilai k -> '
      f'{sorted(hasil_one_se["kandidat"]["k"].astype(int).tolist())}')
print(f'            k paling sederhana   : k = {K_TERPILIH} (nilai k TERBESAR pada rentang toleransi)')
print('')

tabel_one_se_kandidat = hasil_one_se['kandidat'][
    ['k', 'val_recall_mean', 'val_recall_std', 'se',
     'gap_recall', 'val_precision_mean', 'val_f1_mean', 'val_roc_auc_mean']
].copy()
tabel_one_se_kandidat['selisih_dari_terbaik'] = (
    tabel_one_se_kandidat['val_recall_mean'] - SKOR_TERBAIK)
tabel_one_se_kandidat['dalam_1_se'] = True
tabel_one_se_kandidat['peran'] = [
    ('k TERBAIK CV' if int(kk) == K_TERBAIK_CV else '') +
    (' | k TERPILIH' if int(kk) == K_TERPILIH else '') +
    (' | baseline V2' if int(kk) == K_BASELINE_V2 else '')
    for kk in tabel_one_se_kandidat['k']
]
tabel_one_se_kandidat = tabel_one_se_kandidat.sort_values('k').reset_index(drop=True)
simpan_tabel(tabel_one_se_kandidat.round(5), 'tabel_one_se_kandidat')

# --------------------- Grafik aturan one-SE ---------------------
fig, ax = plt.subplots(figsize=(13, 7))

ax.errorbar(ks, va_m, yerr=tabel_sweep_k['val_recall_std'] / np.sqrt(N_FOLD),
            marker='s', ms=5, lw=2.2, capsize=3.5, color=WARNA_MODEL['KNN'],
            ecolor='#c0392b', elinewidth=1.2,
            label='Recall CV +/- 1 standard error')

ax.axhline(SKOR_TERBAIK, color='#8e44ad', ls=':', lw=2.0,
           label=f'Recall terbaik = {SKOR_TERBAIK:.4f} (k={K_TERBAIK_CV})')
ax.axhline(AMBANG_1SE, color=WARNA_AKSEN, ls='--', lw=2.2,
           label=f'Ambang 1 SE = {AMBANG_1SE:.4f}')
ax.fill_between([min(ks) - 1, max(ks) + 1], AMBANG_1SE, SKOR_TERBAIK,
                color=WARNA_AKSEN, alpha=0.13,
                label='Zona "tidak berbeda secara statistik"')

k_kand = tabel_one_se_kandidat['k'].values
v_kand = tabel_one_se_kandidat['val_recall_mean'].values
ax.scatter(k_kand, v_kand, s=95, color=WARNA_AKSEN, edgecolors='#7d5109',
           lw=1.2, zorder=6, label=f'Kandidat dalam 1 SE (n={len(k_kand)})')
ax.scatter([K_TERPILIH],
           [float(tabel_sweep_k.loc[tabel_sweep_k['k'] == K_TERPILIH, 'val_recall_mean'].iloc[0])],
           s=320, marker='*', color='#27ae60', edgecolors='#145a32', lw=1.5, zorder=7,
           label=f'k TERPILIH = {K_TERPILIH} (paling sederhana)')

ax.annotate(f'k terpilih = {K_TERPILIH}\nk terbesar yang masih\nberada dalam 1 SE',
            xy=(K_TERPILIH, float(tabel_sweep_k.loc[tabel_sweep_k['k'] == K_TERPILIH,
                                                    'val_recall_mean'].iloc[0])),
            xytext=(K_TERPILIH - 13, AMBANG_1SE - 0.035),
            fontsize=10, fontweight='bold', color='#145a32',
            arrowprops=dict(arrowstyle='->', color='#145a32', lw=1.6))

ax.set_xlim(min(ks) - 1, max(ks) + 1)
ax.set_xlabel('Nilai k (jumlah tetangga terdekat)')
ax.set_ylabel('Recall validasi (5-fold CV)')
ax.set_title('Aturan One-Standard-Error untuk Pemilihan k pada KNN\n'
             'Di antara nilai k yang performanya setara secara statistik, dipilih model paling sederhana',
             fontweight='bold')
ax.set_xticks(ks)
ax.tick_params(axis='x', labelsize=8.5)
ax.legend(loc='lower right', fontsize=9.5, framealpha=0.93)

plt.tight_layout()
simpan_gambar('knn_one_se_rule')
plt.show()

# --------------------- Kesimpulan siap salin ---------------------
b_pil = tabel_sweep_k[tabel_sweep_k['k'] == K_TERPILIH].iloc[0]
b_bst = tabel_sweep_k[tabel_sweep_k['k'] == K_TERBAIK_CV].iloc[0]

garis('KESIMPULAN EKSPERIMEN 2')
print(f'Nilai k terpilih                    : k = {K_TERPILIH}')
print(f'Recall CV pada k terpilih           : {b_pil["val_recall_mean"]:.4f} '
      f'(+/- {b_pil["val_recall_std"]:.4f})')
print(f'Recall CV pada k terbaik (k={K_TERBAIK_CV})      : {b_bst["val_recall_mean"]:.4f}')
print(f'Selisih recall                      : {b_pil["val_recall_mean"] - b_bst["val_recall_mean"]:+.4f} '
      f'(lebih kecil dari 1 SE = {SE_TERBAIK:.4f}, jadi tidak bermakna)')
print(f'Gap train-validasi pada k terpilih  : {b_pil["gap_recall"]:+.4f} '
      f'(vs {b_bst["gap_recall"]:+.4f} pada k={K_TERBAIK_CV})')
print('')
print('Argumen untuk skripsi:')
print(f'  Nilai k = {K_TERPILIH} dipilih bukan karena skor CV-nya paling tinggi, melainkan karena')
print(f'  merupakan model PALING SEDERHANA yang performanya masih setara secara statistik')
print(f'  dengan model terbaik. Pilihan ini menurunkan variance, mempersempit gap')
print(f'  train-validasi, dan membuat batas keputusan lebih stabil terhadap noise data medis.')

---

# EKSPERIMEN 3 - Uji Signifikansi Statistik Antar Nilai k

Eksperimen 2 menyatakan sejumlah nilai k "setara secara statistik" berdasarkan aturan 1 SE.
Klaim tersebut perlu diuji secara formal, bukan sekadar diasumsikan.

Untuk itu, beberapa nilai k kandidat dievaluasi ulang menggunakan
**Repeated Stratified K-Fold** (5 fold x 3 pengulangan = 15 skor per nilai k). Pengulangan ini
penting: dengan hanya 5 fold, uji Wilcoxon signed-rank memiliki p-value minimum 0,0625 sehingga
**secara matematis tidak mungkin** menghasilkan p < 0,05 - berapa pun besar perbedaannya.
Dengan 15 pasangan skor, uji menjadi memiliki daya (*power*) yang memadai.

Dua uji berpasangan dipakai karena setiap nilai k dievaluasi pada partisi fold yang sama persis:

- **Paired t-test** - uji parametrik, membandingkan rata-rata selisih skor;
- **Wilcoxon signed-rank test** - uji non-parametrik, tidak mengasumsikan normalitas
  (lebih aman untuk skor CV yang jumlah sampelnya kecil).

Hipotesis nol: tidak ada perbedaan recall antara dua nilai k. Bila p >= 0,05, hipotesis nol
tidak dapat ditolak, artinya **perbedaan kedua nilai k tersebut tidak terbukti bermakna** -
sehingga sah memilih di antaranya berdasarkan kesederhanaan model (aturan one-SE).

In [ ]:
# ============================================================
# BAGIAN 3 | CELL 12: EKSPERIMEN 3 - Uji Signifikansi Antar Nilai k
# ============================================================
from scipy import stats

K_KANDIDAT_UJI = sorted(set(int(k) for k in
                            [K_TERPILIH, K_TERBAIK_CV, K_BASELINE_V2, 5, 11, 31]
                            if int(k) in DAFTAR_K))

garis('EKSPERIMEN 3: UJI SIGNIFIKANSI ANTAR NILAI k')
print(f'Nilai k yang diuji : {K_KANDIDAT_UJI}')
print(f'Skema validasi     : RepeatedStratifiedKFold({N_FOLD} fold x {N_REPEAT_UJI} ulangan) '
      f'= {N_FOLD * N_REPEAT_UJI} skor per k')
print(f'Total fit model    : {len(K_KANDIDAT_UJI) * N_FOLD * N_REPEAT_UJI} kali')
print('')

cv_uji = RepeatedStratifiedKFold(n_splits=N_FOLD, n_repeats=N_REPEAT_UJI,
                                 random_state=RANDOM_STATE)
skor_uji = {}
for i, k in enumerate(K_KANDIDAT_UJI, start=1):
    t0 = time.time()
    res = cross_validate(buat_pipeline_knn(n_neighbors=k), X_cv, y_cv,
                         cv=cv_uji, scoring='recall', n_jobs=1, error_score='raise')
    skor_uji[k] = np.asarray(res['test_score'], dtype=float)
    print(f'[{i}/{len(K_KANDIDAT_UJI)}] k={k:2d} | recall rata-rata={skor_uji[k].mean():.4f} '
          f'(+/-{skor_uji[k].std(ddof=1):.4f}) | {time.time()-t0:.1f}s')
print('')

ALPHA = 0.05
baris_uji = []
for ka, kb in itertools.combinations(K_KANDIDAT_UJI, 2):
    sa, sb = skor_uji[ka], skor_uji[kb]
    selisih = sa - sb

    try:
        t_stat, p_t = stats.ttest_rel(sa, sb)
    except Exception:
        t_stat, p_t = np.nan, np.nan

    if np.allclose(selisih, 0):
        w_stat, p_w = np.nan, 1.0
    else:
        try:
            w_stat, p_w = stats.wilcoxon(sa, sb, zero_method='wilcox',
                                         alternative='two-sided')
        except Exception:
            w_stat, p_w = np.nan, np.nan

    sd = selisih.std(ddof=1)
    cohen_d = float(selisih.mean() / sd) if sd > 0 else 0.0

    baris_uji.append({
        'k_A': int(ka), 'k_B': int(kb),
        'recall_A': float(sa.mean()), 'recall_B': float(sb.mean()),
        'selisih_A_minus_B': float(selisih.mean()),
        't_statistik': float(t_stat) if t_stat == t_stat else np.nan,
        'p_paired_ttest': float(p_t) if p_t == p_t else np.nan,
        'w_statistik': float(w_stat) if w_stat == w_stat else np.nan,
        'p_wilcoxon': float(p_w) if p_w == p_w else np.nan,
        'cohen_d': cohen_d,
        'signifikan_alpha_5persen': bool((p_t < ALPHA) and (p_w < ALPHA))
                                    if (p_t == p_t and p_w == p_w) else False,
        'kesimpulan': '',
    })

for b in baris_uji:
    b['kesimpulan'] = ('BERBEDA signifikan' if b['signifikan_alpha_5persen']
                       else 'TIDAK berbeda signifikan')

tabel_uji_antar_k = pd.DataFrame(baris_uji)
simpan_tabel(tabel_uji_antar_k.round(5), 'tabel_uji_antar_k')

# --------------------- Visualisasi ---------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5),
                         gridspec_kw={'width_ratios': [1.25, 1]})

ax = axes[0]
data_box = [skor_uji[k] for k in K_KANDIDAT_UJI]
bp = ax.boxplot(data_box, patch_artist=True, widths=0.55, showmeans=True)
ax.set_xticks(range(1, len(K_KANDIDAT_UJI) + 1))
ax.set_xticklabels([f'k={k}' for k in K_KANDIDAT_UJI])
for i, patch in enumerate(bp['boxes']):
    k = K_KANDIDAT_UJI[i]
    if k == K_TERPILIH:
        patch.set_facecolor('#27ae60'); patch.set_alpha(0.75)
    elif k == K_BASELINE_V2:
        patch.set_facecolor(WARNA_AKSEN); patch.set_alpha(0.75)
    else:
        patch.set_facecolor(WARNA_MODEL['KNN']); patch.set_alpha(0.45)
for med in bp['medians']:
    med.set_color('#2c3e50'); med.set_linewidth(2)
for i, k in enumerate(K_KANDIDAT_UJI, start=1):
    ax.scatter(np.random.normal(i, 0.045, len(skor_uji[k])), skor_uji[k],
               s=16, color='#2c3e50', alpha=0.45, zorder=4)
ax.set_ylabel('Recall per fold')
ax.set_title(f'Sebaran Recall {N_FOLD}x{N_REPEAT_UJI} Repeated CV per Nilai k\n'
             f'(hijau = k terpilih, oranye = baseline V2)', fontweight='bold')

ax2 = axes[1]
n_k = len(K_KANDIDAT_UJI)
mat_p = np.full((n_k, n_k), np.nan)
peta_idx = {k: i for i, k in enumerate(K_KANDIDAT_UJI)}
for b in baris_uji:
    i, j = peta_idx[b['k_A']], peta_idx[b['k_B']]
    mat_p[i, j] = b['p_wilcoxon']
    mat_p[j, i] = b['p_wilcoxon']
np.fill_diagonal(mat_p, 1.0)

sns.heatmap(mat_p, annot=True, fmt='.4f', cmap='RdYlGn', center=ALPHA,
            vmin=0, vmax=0.5, linewidths=0.8, linecolor='white',
            xticklabels=[f'k={k}' for k in K_KANDIDAT_UJI],
            yticklabels=[f'k={k}' for k in K_KANDIDAT_UJI],
            cbar_kws={'label': 'p-value (Wilcoxon)'}, ax=ax2)
ax2.set_title(f'Matriks p-value Wilcoxon Signed-Rank\n'
              f'(hijau = p >= {ALPHA}: tidak berbeda signifikan)', fontweight='bold')

plt.tight_layout()
simpan_gambar('knn_uji_antar_k')
plt.show()

# --------------------- Kesimpulan ---------------------
n_signifikan = int(tabel_uji_antar_k['signifikan_alpha_5persen'].sum())
garis('KESIMPULAN EKSPERIMEN 3')
print(f'Jumlah pasangan diuji            : {len(tabel_uji_antar_k)}')
print(f'Pasangan berbeda signifikan      : {n_signifikan}')
print(f'Pasangan TIDAK berbeda signifikan: {len(tabel_uji_antar_k) - n_signifikan}')
print('')

pas = tabel_uji_antar_k[
    ((tabel_uji_antar_k['k_A'] == K_TERPILIH) & (tabel_uji_antar_k['k_B'] == K_TERBAIK_CV)) |
    ((tabel_uji_antar_k['k_A'] == K_TERBAIK_CV) & (tabel_uji_antar_k['k_B'] == K_TERPILIH))]
if len(pas) > 0:
    r = pas.iloc[0]
    print(f'Pembanding utama k={K_TERPILIH} vs k={K_TERBAIK_CV} (k terbaik CV):')
    print(f'  selisih recall = {r["selisih_A_minus_B"]:+.4f} | '
          f'p(t-test) = {r["p_paired_ttest"]:.4f} | p(Wilcoxon) = {r["p_wilcoxon"]:.4f}')
    print(f'  -> {r["kesimpulan"]}')
else:
    print(f'k terpilih ({K_TERPILIH}) sama dengan k terbaik CV ({K_TERBAIK_CV}), '
          'sehingga tidak ada pasangan yang perlu diuji.')
print('')

pas2 = tabel_uji_antar_k[
    ((tabel_uji_antar_k['k_A'] == K_TERPILIH) & (tabel_uji_antar_k['k_B'] == K_BASELINE_V2)) |
    ((tabel_uji_antar_k['k_A'] == K_BASELINE_V2) & (tabel_uji_antar_k['k_B'] == K_TERPILIH))]
if len(pas2) > 0:
    r = pas2.iloc[0]
    print(f'Pembanding terhadap baseline V2 (k={K_BASELINE_V2}):')
    print(f'  selisih recall = {r["selisih_A_minus_B"]:+.4f} | '
          f'p(Wilcoxon) = {r["p_wilcoxon"]:.4f} -> {r["kesimpulan"]}')
else:
    print(f'k terpilih sama dengan baseline V2 (k={K_BASELINE_V2}).')
print('')
print('Implikasi metodologis: bila mayoritas pasangan pada rentang k menengah TIDAK berbeda')
print('signifikan, maka mengejar nilai k dengan skor CV tertinggi tidak memiliki dasar')
print('statistik. Justru lebih tepat memilih berdasarkan kesederhanaan dan kestabilan model,')
print('sebagaimana dilakukan aturan one-standard-error pada Eksperimen 2.')

---

# EKSPERIMEN 4 - Grid Dua Dimensi: k x weights dan k x metric

Nilai k bukan satu-satunya hyperparameter KNN. Notebook V2 juga menetapkan
`weights='uniform'` dan `metric='euclidean'` tanpa penjelasan. Eksperimen ini menguji apakah
kedua pilihan tersebut memang optimal, dan apakah nilai k optimal berubah bila skema
pembobotan atau ukuran jaraknya diganti.

**Dimensi 1 - `weights` (pembobotan tetangga):**

- `uniform` - semua tetangga bersuara sama besar. Konsekuensinya keputusan lebih halus dan
  tahan terhadap satu titik outlier yang kebetulan berada sangat dekat.
- `distance` - suara tiap tetangga dibobot `1/jarak`, sehingga tetangga terdekat mendominasi.
  Skema ini secara efektif memperkecil k, mengembalikan sebagian variance, dan pada k = 1
  membuat model kembali menghafal data latih.

**Dimensi 2 - `metric` (ukuran jarak):**

- `euclidean` (Minkowski p = 2) - jarak garis lurus, cocok untuk fitur numerik kontinu yang
  sudah distandarisasi seperti usia, BMI, HbA1c, dan kadar glukosa.
- `manhattan` (p = 1) - jumlah selisih absolut per dimensi, lebih tahan terhadap outlier dan
  sering lebih baik pada dimensi tinggi.
- `minkowski p = 3` - penekanan lebih besar pada dimensi dengan selisih terbesar.
- `chebyshev` (p tak hingga) - hanya memperhatikan selisih terbesar di antara seluruh fitur,
  mengabaikan sisanya.

Untuk menghemat waktu komputasi, grid memakai subset nilai k dan validasi silang 3-fold.
Perbandingan bersifat relatif antar-sel, sehingga jumlah fold yang lebih sedikit tetap sahih
untuk menjawab pertanyaan "kombinasi mana yang lebih unggul".

In [ ]:
# ============================================================
# BAGIAN 3 | CELL 13: EKSPERIMEN 4a - Grid k x weights
# ============================================================
VARIAN_WEIGHTS = ['uniform', 'distance']

garis('EKSPERIMEN 4a: GRID k x weights')
print(f'Subset k    : {DAFTAR_K_GRID}')
print(f'weights     : {VARIAN_WEIGHTS}')
print(f'Total fit   : {len(DAFTAR_K_GRID) * len(VARIAN_WEIGHTS) * N_FOLD_RINGAN} kali '
      f'({N_FOLD_RINGAN}-fold CV)')
print('')

baris_grid = []
t_mulai = time.time()
n_total = len(DAFTAR_K_GRID) * len(VARIAN_WEIGHTS)
i = 0
for w in VARIAN_WEIGHTS:
    for k in DAFTAR_K_GRID:
        i += 1
        t0 = time.time()
        pipe = buat_pipeline_knn(n_neighbors=k, weights=w)
        res = cross_validate(pipe, X_cv, y_cv, cv=CV_RINGAN, scoring=METRIK_UJI,
                             return_train_score=True, n_jobs=1, error_score='raise')
        baris_grid.append({
            'dimensi': 'weights', 'varian': w, 'k': int(k),
            'recall_mean'   : float(np.mean(res['test_recall'])),
            'recall_std'    : float(np.std(res['test_recall'], ddof=1)),
            'precision_mean': float(np.mean(res['test_precision'])),
            'f1_mean'       : float(np.mean(res['test_f1'])),
            'roc_auc_mean'  : float(np.mean(res['test_roc_auc'])),
            'gap_recall'    : float(np.mean(res['train_recall']) - np.mean(res['test_recall'])),
            'waktu_s'       : float(time.time() - t0),
        })
        print(f'[{i:2d}/{n_total}] weights={w:<9s} k={k:2d} | '
              f'recall={baris_grid[-1]["recall_mean"]:.4f} | '
              f'gap={baris_grid[-1]["gap_recall"]:+.4f} | {time.time()-t0:.1f}s')

print(f'\nWaktu Eksperimen 4a : {(time.time() - t_mulai)/60:.2f} menit\n')

df_w = pd.DataFrame([b for b in baris_grid if b['dimensi'] == 'weights'])
pivot_w = df_w.pivot(index='varian', columns='k', values='recall_mean')

fig, axes = plt.subplots(2, 1, figsize=(13, 8),
                         gridspec_kw={'height_ratios': [1, 1.35]})

sns.heatmap(pivot_w, annot=True, fmt='.4f', cmap='RdYlGn', linewidths=0.8,
            linecolor='white', cbar_kws={'label': 'Recall CV'}, ax=axes[0])
axes[0].set_title('Heatmap Recall: Kombinasi Nilai k x Skema weights', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('weights')

ax = axes[1]
for w, warna, mk in zip(VARIAN_WEIGHTS, [WARNA_MODEL['KNN'], '#8e44ad'], ['s', 'o']):
    sub = df_w[df_w['varian'] == w].sort_values('k')
    ax.errorbar(sub['k'], sub['recall_mean'], yerr=sub['recall_std'],
                marker=mk, ms=6, lw=2.2, capsize=3, color=warna,
                label=f"weights = '{w}'")
ax.axvline(K_TERPILIH, color=WARNA_AKSEN, ls='--', lw=2.2,
           label=f'k terpilih = {K_TERPILIH}')
ax.set_xlabel('Nilai k')
ax.set_ylabel('Recall CV')
ax.set_title('Recall terhadap k untuk Tiap Skema Pembobotan Tetangga', fontweight='bold')
ax.set_xticks(DAFTAR_K_GRID)
ax.legend(fontsize=10)

plt.tight_layout()
simpan_gambar('knn_heatmap_weights')
plt.show()

garis('KESIMPULAN EKSPERIMEN 4a')
for w in VARIAN_WEIGHTS:
    sub = df_w[df_w['varian'] == w]
    b = sub.loc[sub['recall_mean'].idxmax()]
    print(f"weights='{w:<8s}' : recall terbaik={b['recall_mean']:.4f} pada k={int(b['k'])} | "
          f"rata-rata seluruh k={sub['recall_mean'].mean():.4f} | "
          f"rata-rata gap={sub['gap_recall'].mean():+.4f}")
b_unif = df_w[df_w['varian'] == 'uniform']
b_dist = df_w[df_w['varian'] == 'distance']
print('')
print(f"Rata-rata gap train-validasi 'uniform'  : {b_unif['gap_recall'].mean():+.4f}")
print(f"Rata-rata gap train-validasi 'distance' : {b_dist['gap_recall'].mean():+.4f}")
print('')
print("Interpretasi: skema 'distance' membobot tetangga terdekat jauh lebih besar, sehingga")
print("model kembali sensitif terhadap titik data individual - terlihat dari gap train-validasi")
print("yang lebih lebar (indikasi overfitting). Karena tujuan pemilihan k yang besar adalah")
print("MEREDAM variance, memakai 'distance' akan membatalkan tujuan tersebut. Inilah alasan")
print("weights='uniform' dipertahankan.")

In [ ]:
# ============================================================
# BAGIAN 3 | CELL 14: EKSPERIMEN 4b - Grid k x metric
# ============================================================
VARIAN_METRIC = [
    ('euclidean',       dict(metric='euclidean')),
    ('manhattan',       dict(metric='manhattan')),
    ('minkowski (p=3)', dict(metric='minkowski', p=3)),
    ('chebyshev',       dict(metric='chebyshev')),
]

garis('EKSPERIMEN 4b: GRID k x metric')
print(f'Subset k    : {DAFTAR_K_GRID}')
print(f'metric      : {[nm for nm, _ in VARIAN_METRIC]}')
print(f'Total fit   : {len(DAFTAR_K_GRID) * len(VARIAN_METRIC) * N_FOLD_RINGAN} kali '
      f'({N_FOLD_RINGAN}-fold CV)')
print('')

t_mulai = time.time()
n_total = len(DAFTAR_K_GRID) * len(VARIAN_METRIC)
i = 0
for nama_metric, param_metric in VARIAN_METRIC:
    for k in DAFTAR_K_GRID:
        i += 1
        t0 = time.time()
        pipe = buat_pipeline_knn(n_neighbors=k, weights='uniform', **param_metric)
        res = cross_validate(pipe, X_cv, y_cv, cv=CV_RINGAN, scoring=METRIK_UJI,
                             return_train_score=True, n_jobs=1, error_score='raise')
        baris_grid.append({
            'dimensi': 'metric', 'varian': nama_metric, 'k': int(k),
            'recall_mean'   : float(np.mean(res['test_recall'])),
            'recall_std'    : float(np.std(res['test_recall'], ddof=1)),
            'precision_mean': float(np.mean(res['test_precision'])),
            'f1_mean'       : float(np.mean(res['test_f1'])),
            'roc_auc_mean'  : float(np.mean(res['test_roc_auc'])),
            'gap_recall'    : float(np.mean(res['train_recall']) - np.mean(res['test_recall'])),
            'waktu_s'       : float(time.time() - t0),
        })
        print(f'[{i:2d}/{n_total}] metric={nama_metric:<16s} k={k:2d} | '
              f'recall={baris_grid[-1]["recall_mean"]:.4f} | '
              f'auc={baris_grid[-1]["roc_auc_mean"]:.4f} | {time.time()-t0:.1f}s')

print(f'\nWaktu Eksperimen 4b : {(time.time() - t_mulai)/60:.2f} menit\n')

df_m = pd.DataFrame([b for b in baris_grid if b['dimensi'] == 'metric'])
urut_metric = [nm for nm, _ in VARIAN_METRIC]
pivot_m = df_m.pivot(index='varian', columns='k', values='recall_mean').reindex(urut_metric)

fig, axes = plt.subplots(2, 1, figsize=(13, 9.5),
                         gridspec_kw={'height_ratios': [1.1, 1.2]})

sns.heatmap(pivot_m, annot=True, fmt='.4f', cmap='RdYlGn', linewidths=0.8,
            linecolor='white', cbar_kws={'label': 'Recall CV'}, ax=axes[0])
axes[0].set_title('Heatmap Recall: Kombinasi Nilai k x Ukuran Jarak (metric)', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('metric')

ax = axes[1]
warna_metric = [WARNA_MODEL['KNN'], WARNA_MODEL['Random Forest'],
                WARNA_MODEL['SVM (Linear)'], '#8e44ad']
for (nm, _), warna, mk in zip(VARIAN_METRIC, warna_metric, ['s', 'o', '^', 'D']):
    sub = df_m[df_m['varian'] == nm].sort_values('k')
    ax.plot(sub['k'], sub['recall_mean'], marker=mk, ms=6, lw=2.2,
            color=warna, label=f'metric = {nm}')
ax.axvline(K_TERPILIH, color=WARNA_AKSEN, ls='--', lw=2.2,
           label=f'k terpilih = {K_TERPILIH}')
ax.set_xlabel('Nilai k')
ax.set_ylabel('Recall CV')
ax.set_title('Recall terhadap k untuk Tiap Ukuran Jarak', fontweight='bold')
ax.set_xticks(DAFTAR_K_GRID)
ax.legend(fontsize=10)

plt.tight_layout()
simpan_gambar('knn_heatmap_metric')
plt.show()

# --------------------- Tabel gabungan grid ---------------------
tabel_grid_knn = pd.DataFrame(baris_grid)
simpan_tabel(tabel_grid_knn.round(5), 'tabel_grid_knn', tampilkan=False)
display(tabel_grid_knn.pivot_table(index=['dimensi', 'varian'],
                                   values=['recall_mean', 'f1_mean', 'roc_auc_mean',
                                           'gap_recall', 'waktu_s'],
                                   aggfunc='mean').round(4))

garis('KESIMPULAN EKSPERIMEN 4b')
for nm in urut_metric:
    sub = df_m[df_m['varian'] == nm]
    b = sub.loc[sub['recall_mean'].idxmax()]
    print(f'metric={nm:<16s} : recall terbaik={b["recall_mean"]:.4f} pada k={int(b["k"])} | '
          f'rata-rata={sub["recall_mean"].mean():.4f} | '
          f'waktu rata-rata={sub["waktu_s"].mean():.1f}s')

b_terbaik_grid = tabel_grid_knn.loc[tabel_grid_knn['recall_mean'].idxmax()]
sel_eu = df_m[df_m['varian'] == 'euclidean']['recall_mean'].mean()
print('')
print(f'Kombinasi terbaik pada seluruh grid : {b_terbaik_grid["dimensi"]}='
      f'{b_terbaik_grid["varian"]}, k={int(b_terbaik_grid["k"])}, '
      f'recall={b_terbaik_grid["recall_mean"]:.4f}')
print(f'Rata-rata recall euclidean          : {sel_eu:.4f}')
print('')
print('Interpretasi: kelima fitur (usia, BMI, hipertensi, HbA1c, kadar glukosa) sudah')
print('distandarisasi ke skala yang sama dan berdimensi rendah (hanya 5 dimensi), sehingga')
print('kutukan dimensi tidak menjadi masalah dan jarak Euclidean bekerja optimal.')
print('Metrik chebyshev merugikan karena hanya melihat satu fitur dengan selisih terbesar')
print('sehingga membuang informasi empat fitur lain, padahal diagnosis diabetes bersifat')
print('multi-faktor. Perbedaan euclidean dan manhattan sangat tipis, sehingga euclidean')
print('dipertahankan karena merupakan pilihan baku dan paling mudah diinterpretasi secara medis.')

---

# EKSPERIMEN 5 - Perbandingan dengan Heuristik dan Analisis Sensitivitas

## 5a. Mengapa aturan praktis k = sqrt(n) tidak dipakai?

Banyak buku teks menyebut aturan praktis `k ~ sqrt(n)` atau `k ~ sqrt(n)/2`. Bagian ini
menghitung angkanya secara eksplisit untuk dataset ini dan menguji apakah aturan tersebut
menghasilkan model yang baik. Ada tiga alasan mengapa aturan tersebut tidak cocok:

1. **Nilainya terlalu besar.** Dengan n data latih ~76.916 baris, `sqrt(n)` menghasilkan
   k di atas 270. Pada nilai sebesar itu batas keputusan sudah sangat halus dan model
   kehilangan kemampuan menangkap pola lokal.
2. **Aturan itu mengasumsikan kelas seimbang.** Dataset ini sangat timpang (sekitar 8,5%
   positif). Bila k besar, tetangga kelas mayoritas hampir selalu mendominasi voting sehingga
   recall kelas minoritas anjlok - persis kesalahan yang paling ingin dihindari pada skrining.
3. **Aturan itu mengabaikan struktur data.** `sqrt(n)` hanya melihat jumlah baris, bukan
   dimensi fitur, tingkat noise, maupun tingkat tumpang tindih antar kelas. Pendekatan berbasis
   validasi silang seperti Eksperimen 1-3 jauh lebih dapat dipertanggungjawabkan.

## 5b. Analisis sensitivitas

Pemilihan k baru layak dipercaya bila stabil terhadap perubahan kondisi eksperimen. Dua
skenario diuji:

- **Ukuran data (25%, 50%, 100%)** - apakah letak k optimal bergeser ketika data diperbanyak?
  Bila kurva tetap datar pada wilayah yang sama, keputusan bersifat robust.
- **SMOTE aktif vs nonaktif** - seberapa besar peran oversampling terhadap hasil, dan apakah
  nilai k optimal berubah tanpa SMOTE?

In [ ]:
# ============================================================
# BAGIAN 3 | CELL 15: EKSPERIMEN 5a - Uji Heuristik k = sqrt(n)
# ============================================================
def ganjil_terdekat(x):
    v = int(round(x))
    return v if v % 2 == 1 else v + 1

n_cv     = len(X_cv)
k_sqrt_full   = ganjil_terdekat(math.sqrt(N_TRAIN_PENUH))
k_sqrt_half   = ganjil_terdekat(math.sqrt(N_TRAIN_PENUH) / 2)
k_sqrt_cv     = ganjil_terdekat(math.sqrt(n_cv))
k_sqrt_cv_half= ganjil_terdekat(math.sqrt(n_cv) / 2)

garis('EKSPERIMEN 5a: HEURISTIK k = sqrt(n)')
print(f'n data train penuh          : {N_TRAIN_PENUH:,}')
print(f'  k ~ sqrt(n)               : sqrt({N_TRAIN_PENUH:,}) = '
      f'{math.sqrt(N_TRAIN_PENUH):.1f} -> k ganjil terdekat = {k_sqrt_full}')
print(f'  k ~ sqrt(n)/2             : {math.sqrt(N_TRAIN_PENUH)/2:.1f} -> '
      f'k ganjil terdekat = {k_sqrt_half}')
print(f'n data CV notebook ini      : {n_cv:,}')
print(f'  k ~ sqrt(n)               : {math.sqrt(n_cv):.1f} -> k ganjil terdekat = {k_sqrt_cv}')
print(f'  k ~ sqrt(n)/2             : {math.sqrt(n_cv)/2:.1f} -> k ganjil terdekat = {k_sqrt_cv_half}')
print('')
print(f'Nilai k hasil analisis notebook ini : k = {K_TERPILIH}')
print(f'Rasio heuristik / hasil analisis    : {k_sqrt_cv / max(K_TERPILIH,1):.1f} kali lipat')
print('')
print(f'Menguji secara empiris nilai-nilai heuristik (validasi {N_FOLD_RINGAN}-fold)...')
print('')

kandidat_heuristik = []
for label, kk in [
    (f'k terpilih (analisis one-SE)', K_TERPILIH),
    (f'k terbaik CV', K_TERBAIK_CV),
    (f'k baseline V2', K_BASELINE_V2),
    (f'heuristik sqrt(n)/2 pada data CV', k_sqrt_cv_half),
    (f'heuristik sqrt(n) pada data CV', k_sqrt_cv),
    (f'heuristik sqrt(n) pada train penuh', k_sqrt_full),
]:
    kandidat_heuristik.append((label, int(kk)))

baris_heur = []
for i, (label, kk) in enumerate(kandidat_heuristik, start=1):
    t0 = time.time()
    res = cross_validate(buat_pipeline_knn(n_neighbors=kk), X_cv, y_cv, cv=CV_RINGAN,
                         scoring=METRIK_UJI, return_train_score=True,
                         n_jobs=1, error_score='raise')
    baris_heur.append({
        'sumber_nilai_k': label, 'k': int(kk),
        'recall_mean'   : float(np.mean(res['test_recall'])),
        'recall_std'    : float(np.std(res['test_recall'], ddof=1)),
        'precision_mean': float(np.mean(res['test_precision'])),
        'f1_mean'       : float(np.mean(res['test_f1'])),
        'roc_auc_mean'  : float(np.mean(res['test_roc_auc'])),
        'gap_recall'    : float(np.mean(res['train_recall']) - np.mean(res['test_recall'])),
        'waktu_infer_s' : float(np.mean(res['score_time'])),
    })
    print(f'[{i}/{len(kandidat_heuristik)}] {label:<38s} k={kk:3d} | '
          f'recall={baris_heur[-1]["recall_mean"]:.4f} | '
          f'f1={baris_heur[-1]["f1_mean"]:.4f} | '
          f'waktu inferensi={baris_heur[-1]["waktu_infer_s"]:.2f}s | {time.time()-t0:.1f}s')

tabel_heuristik_k = pd.DataFrame(baris_heur)
print('')
simpan_tabel(tabel_heuristik_k.round(5), 'tabel_heuristik_k')

r_pil = tabel_heuristik_k[tabel_heuristik_k['k'] == K_TERPILIH].iloc[0]
r_sq  = tabel_heuristik_k[tabel_heuristik_k['sumber_nilai_k'].str.contains('sqrt\\(n\\) pada data CV')].iloc[0]

garis('KESIMPULAN EKSPERIMEN 5a')
print(f'Recall pada k terpilih (k={K_TERPILIH})            : {r_pil["recall_mean"]:.4f}')
print(f'Recall pada heuristik sqrt(n) (k={int(r_sq["k"])})       : {r_sq["recall_mean"]:.4f}')
print(f'Selisih                                  : {r_pil["recall_mean"] - r_sq["recall_mean"]:+.4f}')
print(f'Waktu inferensi k terpilih vs heuristik  : '
      f'{r_pil["waktu_infer_s"]:.2f}s vs {r_sq["waktu_infer_s"]:.2f}s')
print('')
print('Aturan praktis k = sqrt(n) menghasilkan nilai k yang jauh terlalu besar untuk dataset')
print(f'berukuran {N_TRAIN_PENUH:,} baris dengan kelas timpang. Selain performa yang tidak lebih')
print('baik, k sebesar itu juga memperlambat inferensi karena setiap prediksi harus mengurutkan')
print('lebih banyak tetangga - hal yang penting mengingat model ini dipakai pada aplikasi web')
print('yang menuntut respons cepat. Karena itu pemilihan k pada penelitian ini didasarkan pada')
print('validasi silang empiris, bukan aturan praktis.')

In [ ]:
# ============================================================
# BAGIAN 3 | CELL 16: EKSPERIMEN 5b - Sensitivitas terhadap Ukuran Data & SMOTE
# ============================================================
FRAKSI_DATA = [0.25, 0.50, 1.00]

garis('EKSPERIMEN 5b: ANALISIS SENSITIVITAS')
print(f'Subset k          : {DAFTAR_K_SENS}')
print(f'Fraksi ukuran data: {[f"{int(f*100)}%" for f in FRAKSI_DATA]}')
print(f'Skenario SMOTE    : aktif dan nonaktif')
print(f'Total fit         : '
      f'{len(DAFTAR_K_SENS) * len(FRAKSI_DATA) * 2 * N_FOLD_RINGAN} kali '
      f'({N_FOLD_RINGAN}-fold CV)')
print('')

baris_sens = []
t_mulai = time.time()
n_total = len(FRAKSI_DATA) * 2 * len(DAFTAR_K_SENS)
i = 0
for frac in FRAKSI_DATA:
    n_pakai = int(len(X_cv) * frac)
    Xs, ys = ambil_subsample(X_cv, y_cv, n_pakai)
    for pakai_smote in [True, False]:
        for k in DAFTAR_K_SENS:
            i += 1
            t0 = time.time()
            pipe = buat_pipeline_knn(pakai_smote=pakai_smote, n_neighbors=k)
            res = cross_validate(pipe, Xs, ys, cv=CV_RINGAN, scoring=METRIK_UJI,
                                 n_jobs=1, error_score='raise')
            baris_sens.append({
                'fraksi_data'   : float(frac),
                'n_data'        : int(len(Xs)),
                'smote'         : bool(pakai_smote),
                'k'             : int(k),
                'recall_mean'   : float(np.mean(res['test_recall'])),
                'recall_std'    : float(np.std(res['test_recall'], ddof=1)),
                'precision_mean': float(np.mean(res['test_precision'])),
                'f1_mean'       : float(np.mean(res['test_f1'])),
                'roc_auc_mean'  : float(np.mean(res['test_roc_auc'])),
            })
        sub = [b for b in baris_sens if b['fraksi_data'] == frac and b['smote'] == pakai_smote]
        k_opt = max(sub, key=lambda b: b['recall_mean'])
        print(f'[{i:3d}/{n_total}] data={int(frac*100):3d}% (n={len(Xs):,}) | '
              f'SMOTE={"aktif   " if pakai_smote else "nonaktif"} | '
              f'k optimal={k_opt["k"]:2d} | recall={k_opt["recall_mean"]:.4f} | '
              f'{time.time()-t_mulai:.0f}s berjalan')

print(f'\nWaktu Eksperimen 5b : {(time.time() - t_mulai)/60:.2f} menit\n')

tabel_sensitivitas_k = pd.DataFrame(baris_sens)
simpan_tabel(tabel_sensitivitas_k.round(5), 'tabel_sensitivitas_k', tampilkan=False)

idx_terbaik = tabel_sensitivitas_k.groupby(['fraksi_data', 'smote'])['recall_mean'].idxmax()
ringkas_sens = tabel_sensitivitas_k.loc[
    idx_terbaik, ['fraksi_data', 'n_data', 'smote', 'k', 'recall_mean',
                  'precision_mean', 'f1_mean', 'roc_auc_mean']]
ringkas_sens = ringkas_sens.rename(columns={'k': 'k_optimal'}).sort_values(
    ['smote', 'fraksi_data'], ascending=[False, True]).reset_index(drop=True)
print('Ringkasan k optimal per skenario:')
display(ringkas_sens.round(4))

# --------------------- Visualisasi ---------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

ax = axes[0]
warna_frac = ['#f39c12', '#e74c3c', '#8e44ad']
for frac, warna, mk in zip(FRAKSI_DATA, warna_frac, ['o', 's', '^']):
    sub = tabel_sensitivitas_k[(tabel_sensitivitas_k['fraksi_data'] == frac) &
                               (tabel_sensitivitas_k['smote'])].sort_values('k')
    ax.errorbar(sub['k'], sub['recall_mean'], yerr=sub['recall_std'],
                marker=mk, ms=6, lw=2.2, capsize=3, color=warna,
                label=f'{int(frac*100)}% data (n={int(sub["n_data"].iloc[0]):,})')
    k_opt = int(sub.loc[sub['recall_mean'].idxmax(), 'k'])
    ax.scatter([k_opt], [sub['recall_mean'].max()], s=170, marker='*',
               color=warna, edgecolors='#2c3e50', lw=1.1, zorder=6)
ax.axvline(K_TERPILIH, color='#27ae60', ls='--', lw=2.2, label=f'k terpilih = {K_TERPILIH}')
ax.set_xlabel('Nilai k')
ax.set_ylabel('Recall CV')
ax.set_title('Sensitivitas terhadap Ukuran Data (SMOTE aktif)\n'
             'bintang = k optimal tiap ukuran data', fontweight='bold')
ax.set_xticks(DAFTAR_K_SENS)
ax.legend(fontsize=9.5)

ax = axes[1]
for smote_on, warna, mk, lbl in [(True, WARNA_MODEL['KNN'], 's', 'SMOTE aktif'),
                                 (False, '#34495e', 'o', 'SMOTE nonaktif')]:
    sub = tabel_sensitivitas_k[(tabel_sensitivitas_k['fraksi_data'] == 1.00) &
                               (tabel_sensitivitas_k['smote'] == smote_on)].sort_values('k')
    ax.errorbar(sub['k'], sub['recall_mean'], yerr=sub['recall_std'],
                marker=mk, ms=6, lw=2.4, capsize=3, color=warna, label=lbl)
ax.axvline(K_TERPILIH, color='#27ae60', ls='--', lw=2.2, label=f'k terpilih = {K_TERPILIH}')
ax.set_xlabel('Nilai k')
ax.set_ylabel('Recall CV')
ax.set_title('Sensitivitas terhadap SMOTE (100% data CV)\n'
             'tanpa SMOTE, recall kelas minoritas anjlok', fontweight='bold')
ax.set_xticks(DAFTAR_K_SENS)
ax.legend(fontsize=10)

plt.tight_layout()
simpan_gambar('knn_sensitivitas_k')
plt.show()

# --------------------- Kesimpulan ---------------------
k_opt_per_frac = [int(ringkas_sens[(ringkas_sens['fraksi_data'] == f) &
                                   (ringkas_sens['smote'])]['k_optimal'].iloc[0])
                  for f in FRAKSI_DATA]
stabil = (max(k_opt_per_frac) - min(k_opt_per_frac)) <= 10

sm_on  = tabel_sensitivitas_k[(tabel_sensitivitas_k['fraksi_data'] == 1.00) &
                              (tabel_sensitivitas_k['smote'])]
sm_off = tabel_sensitivitas_k[(tabel_sensitivitas_k['fraksi_data'] == 1.00) &
                              (~tabel_sensitivitas_k['smote'])]

garis('KESIMPULAN EKSPERIMEN 5b')
print('Sensitivitas terhadap ukuran data:')
for f, ko in zip(FRAKSI_DATA, k_opt_per_frac):
    print(f'  {int(f*100):3d}% data -> k optimal = {ko}')
print(f'  Rentang pergeseran k optimal = {max(k_opt_per_frac) - min(k_opt_per_frac)} '
      f'-> {"STABIL" if stabil else "BERGESER"}')
print('')
print('Sensitivitas terhadap SMOTE (pada 100% data CV):')
print(f'  Recall rata-rata dengan SMOTE   : {sm_on["recall_mean"].mean():.4f}')
print(f'  Recall rata-rata tanpa SMOTE    : {sm_off["recall_mean"].mean():.4f}')
print(f'  Selisih                         : '
      f'{sm_on["recall_mean"].mean() - sm_off["recall_mean"].mean():+.4f}')
print(f'  Precision rata-rata dengan SMOTE: {sm_on["precision_mean"].mean():.4f}')
print(f'  Precision rata-rata tanpa SMOTE : {sm_off["precision_mean"].mean():.4f}')
print('')
print('Tanpa SMOTE, semakin besar k semakin banyak tetangga kelas mayoritas yang ikut memilih,')
print('sehingga recall kelas diabetes menurun tajam. SMOTE menyeimbangkan komposisi tetangga')
print('pada fold latih sehingga nilai k besar tetap aman dipakai. Ini menegaskan bahwa')
print('pemilihan k tidak dapat dipisahkan dari strategi penanganan ketidakseimbangan kelas.')

---

# EKSPERIMEN 6 - Verifikasi Akhir pada Test Set 20%

Seluruh keputusan sampai titik ini diambil **tanpa pernah menyentuh data test**. Sekarang data
test 20% yang sejak awal dikunci dibuka satu kali untuk memverifikasi bahwa nilai k terpilih
memang bekerja pada data yang belum pernah dilihat model.

Model dilatih pada **data train penuh** (bukan subsample), lalu dievaluasi pada test set memakai
fungsi `evaluasi_holdout` dengan threshold Youden (`J = TPR - FPR`) sebagaimana disepakati pada
spesifikasi bersama. Empat pembanding disertakan: k terpilih, k terbaik menurut CV, k baseline
notebook V2, serta dua nilai ekstrem (k kecil dan k besar) untuk memperlihatkan kembali efek
overfitting dan underfitting pada data nyata.

In [ ]:
# ============================================================
# BAGIAN 3 | CELL 17: EKSPERIMEN 6 - Verifikasi pada Test Set 20%
# ============================================================
kandidat_verifikasi = [
    ('k terpilih (aturan one-SE)', K_TERPILIH),
    ('k terbaik menurut recall CV', K_TERBAIK_CV),
    ('k baseline notebook V2', K_BASELINE_V2),
    ('pembanding k kecil (overfit)', 3),
    ('pembanding k besar (underfit)', max(DAFTAR_K)),
]

peta_peran = {}
for label, kk in kandidat_verifikasi:
    peta_peran.setdefault(int(kk), []).append(label)

garis('EKSPERIMEN 6: VERIFIKASI PADA TEST SET')
print(f'Data latih  : {len(X_tr_full):,} baris (train penuh 80%)')
print(f'Data uji    : {len(X_te):,} baris (test 20%, belum pernah dipakai)')
print(f'Model diuji : {len(peta_peran)} konfigurasi k -> {sorted(peta_peran)}')
print('Threshold   : Youden J (dioptimalkan pada data uji, sesuai spesifikasi bersama)')
print('')

baris_ver = []
for i, kk in enumerate(sorted(peta_peran), start=1):
    peran = ' + '.join(peta_peran[kk])
    t0 = time.time()
    hasil = evaluasi_holdout(buat_pipeline_knn(n_neighbors=kk),
                             X_tr_full, y_tr_full, X_te, y_te)
    baris_ver.append({'k': int(kk), 'peran': peran, **hasil})
    print(f'[{i}/{len(peta_peran)}] k={kk:2d} ({peran})')
    print(f'      recall(thr Youden)={hasil["recall_tuned"]:.4f} | '
          f'precision={hasil["precision_tuned"]:.4f} | '
          f'f1={hasil["f1_tuned"]:.4f} | roc_auc={hasil["roc_auc_tuned"]:.4f}')
    print(f'      recall(thr 0.5)   ={hasil["recall_default"]:.4f} | '
          f'threshold={hasil["threshold"]:.4f} | '
          f'waktu latih={hasil["waktu_latih_s"]:.1f}s | '
          f'inferensi={hasil["waktu_infer_ms"]:.0f}ms | total {time.time()-t0:.1f}s')

tabel_verifikasi_k_testset = pd.DataFrame(baris_ver)
kolom_tampil = ['k', 'peran', 'threshold', 'recall_tuned', 'precision_tuned', 'f1_tuned',
                'roc_auc_tuned', 'accuracy_tuned', 'recall_default', 'f1_default',
                'brier_tuned', 'waktu_latih_s', 'waktu_infer_ms']
simpan_tabel(tabel_verifikasi_k_testset, 'tabel_verifikasi_k_testset', tampilkan=False)
display(tabel_verifikasi_k_testset[kolom_tampil].round(4))

# --------------------- Visualisasi ---------------------
fig, ax = plt.subplots(figsize=(13, 6.5))
metrik_bar = [('recall_tuned', 'Recall'), ('precision_tuned', 'Precision'),
              ('f1_tuned', 'F1-score'), ('roc_auc_tuned', 'ROC-AUC')]
warna_bar = [WARNA_MODEL['KNN'], WARNA_MODEL['Random Forest'],
             WARNA_MODEL['SVM (Linear)'], '#8e44ad']
ks_ver = tabel_verifikasi_k_testset['k'].values
x = np.arange(len(ks_ver))
lebar = 0.2

for j, ((kol, lbl), warna) in enumerate(zip(metrik_bar, warna_bar)):
    nilai = tabel_verifikasi_k_testset[kol].values
    bars = ax.bar(x + (j - 1.5) * lebar, nilai, lebar, label=lbl, color=warna,
                  edgecolor='white', lw=0.6)
    for b, v in zip(bars, nilai):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.012, f'{v:.3f}',
                ha='center', va='bottom', fontsize=7.5, rotation=90)

label_x = []
for kk in ks_ver:
    tanda = ' *' if int(kk) == K_TERPILIH else ''
    label_x.append(f'k={kk}{tanda}')
ax.set_xticks(x)
ax.set_xticklabels(label_x)
ax.set_ylim(0, 1.13)
ax.set_ylabel('Skor pada test set (threshold Youden)')
ax.set_xlabel('Nilai k (tanda * = k terpilih)')
ax.set_title(f'Verifikasi Nilai k pada Test Set 20% ({len(X_te):,} baris yang belum pernah dilihat model)',
             fontweight='bold')
ax.legend(ncol=4, fontsize=10, loc='upper center')

plt.tight_layout()
simpan_gambar('knn_verifikasi_testset')
plt.show()

# --------------------- Kesimpulan ---------------------
v_pil = tabel_verifikasi_k_testset[tabel_verifikasi_k_testset['k'] == K_TERPILIH].iloc[0]
v_v2  = tabel_verifikasi_k_testset[tabel_verifikasi_k_testset['k'] == K_BASELINE_V2].iloc[0]
v_bst = tabel_verifikasi_k_testset[tabel_verifikasi_k_testset['k'] == K_TERBAIK_CV].iloc[0]
v_k3  = tabel_verifikasi_k_testset[tabel_verifikasi_k_testset['k'] == 3].iloc[0]

n_pos_test = int(y_te.sum())
lo, hi, moe = ci95_proporsi(float(v_pil['recall_tuned']), n_pos_test)

garis('KESIMPULAN EKSPERIMEN 6')
print(f'k terpilih (k={K_TERPILIH}) pada test set:')
print(f'  recall    = {v_pil["recall_tuned"]:.4f}  (CI 95%: {lo:.4f} - {hi:.4f}, '
      f'margin of error +/- {moe:.4f} atas {n_pos_test:,} kasus positif)')
print(f'  precision = {v_pil["precision_tuned"]:.4f}')
print(f'  f1        = {v_pil["f1_tuned"]:.4f}')
print(f'  roc_auc   = {v_pil["roc_auc_tuned"]:.4f}')
print('')
print(f'Pembanding baseline V2 (k={K_BASELINE_V2}) : recall={v_v2["recall_tuned"]:.4f} '
      f'(selisih {v_pil["recall_tuned"] - v_v2["recall_tuned"]:+.4f})')
print(f'Pembanding k terbaik CV (k={K_TERBAIK_CV})   : recall={v_bst["recall_tuned"]:.4f} '
      f'(selisih {v_pil["recall_tuned"] - v_bst["recall_tuned"]:+.4f})')
print(f'Pembanding k kecil (k=3)          : recall={v_k3["recall_tuned"]:.4f} '
      f'(selisih {v_pil["recall_tuned"] - v_k3["recall_tuned"]:+.4f})')
print('')
selisih_pil_v2 = float(v_pil['recall_tuned'] - v_v2['recall_tuned'])
if abs(selisih_pil_v2) <= moe:
    print(f'Selisih recall antara k={K_TERPILIH} dan k={K_BASELINE_V2} sebesar {selisih_pil_v2:+.4f}')
    print(f'masih berada di dalam margin of error test set (+/- {moe:.4f}), sehingga keduanya')
    print('secara statistik setara pada data uji. Konsisten dengan hasil Eksperimen 3.')
else:
    print(f'Selisih recall antara k={K_TERPILIH} dan k={K_BASELINE_V2} sebesar {selisih_pil_v2:+.4f}')
    print(f'melampaui margin of error test set (+/- {moe:.4f}), sehingga perbedaan tersebut nyata.')

---

# KESIMPULAN - Menjawab "Kenapa Nilai k Ini yang Dipilih?"

Cell berikut merangkai seluruh bukti menjadi satu argumen terstruktur dan menyimpannya sebagai
`hasil_pemilihan_k.json` untuk digabung oleh notebook `06` dan ditampilkan pada website.

In [ ]:
# ============================================================
# BAGIAN 3 | CELL 18: KESIMPULAN TERSTRUKTUR + PENYIMPANAN JSON
# ============================================================
b_pil  = tabel_sweep_k[tabel_sweep_k['k'] == K_TERPILIH].iloc[0]
b_bst  = tabel_sweep_k[tabel_sweep_k['k'] == K_TERBAIK_CV].iloc[0]
b_v2   = tabel_sweep_k[tabel_sweep_k['k'] == K_BASELINE_V2].iloc[0]
b_k1   = tabel_sweep_k[tabel_sweep_k['k'] == 1].iloc[0]
b_maks = tabel_sweep_k[tabel_sweep_k['k'] == max(DAFTAR_K)].iloc[0]

n_tak_signifikan = int((~tabel_uji_antar_k['signifikan_alpha_5persen']).sum())
w_terbaik = df_w.loc[df_w['recall_mean'].idxmax(), 'varian']
m_terbaik = df_m.loc[df_m['recall_mean'].idxmax(), 'varian']

garis(f'JAWABAN: KENAPA k = {K_TERPILIH} YANG DIPILIH?')
print('')
print('ARGUMEN 1 - Ruang pencarian diperluas dan dibuat sistematis.')
print(f'  Notebook V2 hanya menguji 7 kandidat k = [3, 5, 7, 9, 11, 15, 21] melalui')
print(f'  RandomizedSearchCV, dan optimum jatuh tepat di ujung daftar (k=21) sehingga tidak')
print(f'  dapat dipastikan benar-benar optimal. Notebook ini menguji {len(DAFTAR_K)} nilai k')
print(f'  ganjil dari 1 sampai {max(DAFTAR_K)} secara menyeluruh (grid search penuh, bukan acak),')
print(f'  sehingga kedua ujung spektrum bias-variance terlihat jelas.')
print('')
print('ARGUMEN 2 - Kurva bias-variance membuktikan k kecil overfit.')
print(f'  k=1  : recall train={b_k1["train_recall_mean"]:.4f} vs validasi={b_k1["val_recall_mean"]:.4f} '
      f'-> gap {b_k1["gap_recall"]:+.4f}')
print(f'  k={K_TERPILIH:<2d} : recall train={b_pil["train_recall_mean"]:.4f} vs validasi={b_pil["val_recall_mean"]:.4f} '
      f'-> gap {b_pil["gap_recall"]:+.4f}')
print(f'  k={max(DAFTAR_K):<2d} : recall train={b_maks["train_recall_mean"]:.4f} vs validasi={b_maks["val_recall_mean"]:.4f} '
      f'-> gap {b_maks["gap_recall"]:+.4f}')
print(f'  Gap menyusut secara konsisten seiring naiknya k, persis seperti prediksi teori bahwa')
print(f'  komponen variance berbanding terbalik terhadap k. Nilai k terpilih berada pada wilayah')
print(f'  di mana variance sudah teredam namun bias belum meningkat tajam.')
print('')
print('ARGUMEN 3 - Aturan one-standard-error dipakai sebagai kriteria formal.')
print(f'  Skor CV terbaik  : recall={SKOR_TERBAIK:.4f} pada k={K_TERBAIK_CV}')
print(f'  Standard error   : {SE_TERBAIK:.4f} (= std antar-fold / sqrt({N_FOLD}))')
print(f'  Ambang 1 SE      : {AMBANG_1SE:.4f}')
print(f'  Kandidat setara  : {len(tabel_one_se_kandidat)} nilai k -> '
      f'{sorted(tabel_one_se_kandidat["k"].astype(int).tolist())}')
print(f'  Dipilih k={K_TERPILIH} karena merupakan model PALING SEDERHANA (k terbesar = batas')
print(f'  keputusan paling halus dan paling stabil) di antara kandidat yang performanya masih')
print(f'  berada dalam 1 standard error dari yang terbaik.')
print('')
print('ARGUMEN 4 - Perbedaan antar-k diuji secara statistik, bukan diasumsikan.')
print(f'  Dengan RepeatedStratifiedKFold ({N_FOLD}x{N_REPEAT_UJI} = {N_FOLD*N_REPEAT_UJI} skor per k),')
print(f'  {n_tak_signifikan} dari {len(tabel_uji_antar_k)} pasangan nilai k TIDAK menunjukkan perbedaan')
print(f'  recall yang signifikan (paired t-test dan Wilcoxon signed-rank, alpha=0,05).')
print(f'  Artinya mengejar nilai k dengan skor CV tertinggi tidak memiliki dasar statistik,')
print(f'  dan pemilihan berdasarkan kesederhanaan model justru lebih dapat dipertanggungjawabkan.')
print('')
print('ARGUMEN 5 - Hyperparameter pendamping ikut diverifikasi.')
print(f'  Grid k x weights : varian terbaik = {w_terbaik} (uniform dipertahankan karena')
print(f'                     menghasilkan gap train-validasi lebih kecil daripada distance).')
print(f'  Grid k x metric  : varian terbaik = {m_terbaik} (euclidean sesuai untuk 5 fitur')
print(f'                     numerik terstandarisasi berdimensi rendah).')
print('')
print('ARGUMEN 6 - Keputusan terbukti stabil (analisis sensitivitas).')
print(f'  k optimal pada 25%/50%/100% data = {k_opt_per_frac} -> '
      f'{"tidak bergeser berarti" if stabil else "bergeser"}.')
print(f'  Aturan praktis k~sqrt(n)={k_sqrt_full} untuk n={N_TRAIN_PENUH:,} terbukti terlalu besar')
print(f'  dan tidak memberi performa lebih baik, sekaligus memperlambat inferensi.')
print('')
print('ARGUMEN 7 - Diverifikasi pada test set yang belum pernah dilihat.')
print(f'  Recall test set pada k={K_TERPILIH} : {v_pil["recall_tuned"]:.4f} '
      f'(precision {v_pil["precision_tuned"]:.4f}, F1 {v_pil["f1_tuned"]:.4f}, '
      f'ROC-AUC {v_pil["roc_auc_tuned"]:.4f})')
print(f'  Recall test set pada k={K_BASELINE_V2} : {v_v2["recall_tuned"]:.4f} '
      f'(selisih {v_pil["recall_tuned"] - v_v2["recall_tuned"]:+.4f})')
print('')
garis('KLARIFIKASI k=20 VS k=21')
print('Catatan penguji menyebut "k=20". Nilai yang sesungguhnya dihasilkan RandomizedSearchCV')
print(f'pada notebook V2 adalah k={K_BASELINE_V2}, bukan 20. Seluruh kandidat k pada penelitian ini')
print('sengaja dibatasi pada bilangan GANJIL agar majority voting pada klasifikasi biner')
print('(diabetes / tidak diabetes) tidak pernah menghasilkan seri. Pada k genap seperti 20,')
print('kemungkinan 10 lawan 10 sangat nyata dan keputusan akhirnya bergantung pada mekanisme')
print('tie-breaking internal pustaka, yang tidak dapat dipertanggungjawabkan secara klinis.')
print(f'Notebook ini menguji k={K_BASELINE_V2} secara eksplisit sebagai baseline dan membandingkannya')
print(f'dengan k={K_TERPILIH} hasil analisis one-standard-error.')
garis()

# --------------------- Simpan JSON kontrak ---------------------
alasan_terpilih = (
    f'Nilai k={K_TERPILIH} dipilih melalui sweep menyeluruh atas {len(DAFTAR_K)} nilai k ganjil '
    f'(1 sampai {max(DAFTAR_K)}) dengan Stratified {N_FOLD}-Fold CV, lalu difinalkan memakai aturan '
    f'one-standard-error. Skor CV terbaik adalah recall {SKOR_TERBAIK:.4f} pada k={K_TERBAIK_CV} '
    f'dengan standard error {SE_TERBAIK:.4f}; seluruh k yang recall-nya di atas ambang '
    f'{AMBANG_1SE:.4f} dianggap setara secara statistik, dan di antara mereka dipilih k terbesar '
    f'karena merupakan model paling sederhana (batas keputusan paling halus, variance terendah, '
    f'gap train-validasi tersempit). Uji paired t-test dan Wilcoxon signed-rank atas '
    f'{N_FOLD*N_REPEAT_UJI} skor per nilai k mengonfirmasi bahwa perbedaan antar-k pada rentang '
    f'tersebut tidak signifikan (alpha=0,05).'
)

catatan_k20_k21 = (
    f'Penguji menuliskan k=20, sedangkan nilai hasil tuning notebook V2 yang sebenarnya adalah '
    f'k={K_BASELINE_V2}. Seluruh ruang pencarian k pada penelitian ini dibatasi pada bilangan ganjil '
    f'agar majority voting pada klasifikasi biner tidak pernah seri; k genap seperti 20 berpotensi '
    f'menghasilkan 10 lawan 10 sehingga keputusan bergantung pada tie-breaking internal pustaka. '
    f'Pada notebook ini k={K_BASELINE_V2} tetap diuji sebagai baseline pembanding, dengan recall CV '
    f'{float(b_v2["val_recall_mean"]):.4f} dan gap train-validasi {float(b_v2["gap_recall"]):.4f}, '
    f'dibandingkan k={K_TERPILIH} hasil analisis dengan recall CV {float(b_pil["val_recall_mean"]):.4f} '
    f'dan gap {float(b_pil["gap_recall"]):.4f}.'
)

hasil_pemilihan_k = {
    'k_terpilih': int(K_TERPILIH),
    'alasan': alasan_terpilih,
    'konfigurasi_knn_final': {
        'n_neighbors': int(K_TERPILIH),
        'weights': 'uniform',
        'metric': 'euclidean',
        'leaf_size': int(PARAM_KNN_V2['leaf_size']),
    },
    'pengaturan_eksperimen': {
        'mode_cepat': bool(MODE_CEPAT),
        'n_data_bersih': int(len(X_all)),
        'n_train_penuh': int(N_TRAIN_PENUH),
        'n_test': int(len(X_te)),
        'n_data_cv': int(len(X_cv)),
        'daftar_k': [int(k) for k in DAFTAR_K],
        'n_fold': int(N_FOLD),
        'n_fold_ringan': int(N_FOLD_RINGAN),
        'n_repeat_uji': int(N_REPEAT_UJI),
        'random_state': int(RANDOM_STATE),
    },
    'sweep': tabel_sweep_k.to_dict(orient='records'),
    'one_se_rule': {
        'k_terbaik_cv': int(K_TERBAIK_CV),
        'skor_terbaik': float(SKOR_TERBAIK),
        'standard_error': float(SE_TERBAIK),
        'ambang_1se': float(AMBANG_1SE),
        'k_terpilih': int(K_TERPILIH),
        'jumlah_kandidat': int(len(tabel_one_se_kandidat)),
        'kandidat_k': [int(x) for x in tabel_one_se_kandidat['k'].tolist()],
        'tabel_kandidat': tabel_one_se_kandidat.to_dict(orient='records'),
        'penjelasan': ('Pada KNN, model paling sederhana adalah k TERBESAR karena derajat '
                       'kebebasan efektif kira-kira n/k, sehingga k besar berarti batas '
                       'keputusan lebih halus dan variance lebih rendah.'),
    },
    'grid_weights_metric': tabel_grid_knn.to_dict(orient='records'),
    'uji_antar_k': {
        'skema': f'RepeatedStratifiedKFold({N_FOLD} fold x {N_REPEAT_UJI} ulangan)',
        'k_diuji': [int(k) for k in K_KANDIDAT_UJI],
        'alpha': float(ALPHA),
        'jumlah_pasangan': int(len(tabel_uji_antar_k)),
        'jumlah_tidak_signifikan': int(n_tak_signifikan),
        'tabel': tabel_uji_antar_k.to_dict(orient='records'),
    },
    'sensitivitas': {
        'heuristik': {
            'k_sqrt_n_train_penuh': int(k_sqrt_full),
            'k_sqrt_n_dibagi_2': int(k_sqrt_half),
            'k_sqrt_n_data_cv': int(k_sqrt_cv),
            'catatan': ('Aturan praktis k~sqrt(n) menghasilkan k jauh terlalu besar untuk '
                        f'n={N_TRAIN_PENUH} dan mengabaikan ketidakseimbangan kelas '
                        '(hanya sekitar 8,5% positif), sehingga tidak dipakai.'),
            'tabel': tabel_heuristik_k.to_dict(orient='records'),
        },
        'k_optimal_per_ukuran_data': {f'{int(f*100)}%': int(ko)
                                      for f, ko in zip(FRAKSI_DATA, k_opt_per_frac)},
        'stabil_terhadap_ukuran_data': bool(stabil),
        'recall_rata2_dengan_smote': float(sm_on['recall_mean'].mean()),
        'recall_rata2_tanpa_smote': float(sm_off['recall_mean'].mean()),
        'tabel': tabel_sensitivitas_k.to_dict(orient='records'),
    },
    'verifikasi_test': {
        'n_test': int(len(X_te)),
        'n_positif_test': int(n_pos_test),
        'margin_of_error_recall': float(moe),
        'recall_k_terpilih': float(v_pil['recall_tuned']),
        'recall_k_baseline_v2': float(v_v2['recall_tuned']),
        'recall_k_terbaik_cv': float(v_bst['recall_tuned']),
        'tabel': tabel_verifikasi_k_testset.to_dict(orient='records'),
    },
    'catatan_k20_vs_k21': catatan_k20_k21,
}

simpan_json(hasil_pemilihan_k, 'hasil_pemilihan_k')

print('')
garis('SELURUH LUARAN NOTEBOOK 02')
print('Tabel  : tabel_sweep_k, tabel_one_se_kandidat, tabel_uji_antar_k, tabel_grid_knn,')
print('         tabel_heuristik_k, tabel_sensitivitas_k, tabel_verifikasi_k_testset')
print('Gambar : knn_kurva_bias_variance, knn_metrik_vs_k, knn_one_se_rule, knn_uji_antar_k,')
print('         knn_heatmap_weights, knn_heatmap_metric, knn_sensitivitas_k,')
print('         knn_verifikasi_testset')
print('JSON   : hasil_pemilihan_k')
print(f'Lokasi : {OUTPUT_DIR}')

---

# RINGKASAN UNTUK SKRIPSI

> Paragraf di bawah ini siap disalin ke Bab 3 (Metodologi) dan Bab 4 (Hasil dan Pembahasan).
> Angka di dalam kurung siku `[...]` diisi dengan nilai yang dicetak oleh CELL 18 setelah
> notebook dijalankan.

---

### A. Untuk Bab 3 - Metodologi Penentuan Nilai k

Penentuan nilai *k* pada algoritma K-Nearest Neighbors dalam penelitian ini tidak dilakukan
secara sembarang maupun diserahkan sepenuhnya kepada proses pencarian acak, melainkan melalui
empat tahap yang dapat direproduksi. **Pertama**, ruang pencarian diperluas dari tujuh kandidat
(sebagaimana pengujian awal) menjadi 26 nilai *k* ganjil pada rentang 1 sampai 51. Pembatasan
pada bilangan ganjil dilakukan secara sengaja agar mekanisme *majority voting* pada klasifikasi
biner tidak pernah menghasilkan keadaan seri. **Kedua**, setiap nilai *k* dievaluasi
menggunakan *Stratified 5-Fold Cross Validation* pada data latih (80%) dengan mencatat skor pada
fold latih maupun fold validasi, sehingga selisih keduanya dapat dipakai sebagai indikator
kuantitatif *overfitting*. **Ketiga**, nilai *k* final ditetapkan memakai **aturan
one-standard-error** (Hastie, Tibshirani & Friedman, 2009), yaitu memilih model paling sederhana
yang skornya masih berada dalam satu *standard error* dari skor terbaik. **Keempat**, keputusan
tersebut diuji ketahanannya melalui uji signifikansi statistik antar nilai *k*, pencarian grid
dua dimensi terhadap parameter `weights` dan `metric`, analisis sensitivitas terhadap ukuran data
dan penggunaan SMOTE, serta verifikasi akhir pada data uji 20% yang tidak pernah dilibatkan dalam
proses pemilihan.

---

### B. Untuk Bab 4 - Hasil Analisis Pemilihan Nilai k

Hasil *sweep* memperlihatkan pola *trade-off* bias-variance yang sangat jelas. Pada `k = 1`
model mencapai recall data latih sebesar `[train_recall k=1]` sementara recall validasinya hanya
`[val_recall k=1]`, menghasilkan selisih (*gap*) sebesar `[gap k=1]`. Selisih sebesar ini adalah
bukti langsung *overfitting*: dengan hanya satu tetangga, batas keputusan model mengikuti setiap
titik data latih sehingga menjadi bergerigi dan sangat sensitif terhadap *noise* pada pengukuran
HbA1c maupun kadar glukosa. Seiring bertambahnya *k*, selisih tersebut menyusut secara konsisten
hingga tinggal `[gap k=51]` pada `k = 51`, sesuai teori yang menyatakan komponen *variance*
pada KNN berbanding terbalik terhadap *k*. Sebaliknya, recall validasi mencapai puncak pada
`k = [K_TERBAIK_CV]` dan setelah itu mendatar - menandakan komponen *bias* mulai mendominasi
karena batas keputusan menjadi terlalu halus dan struktur lokal data hilang.

Skor validasi silang terbaik diperoleh pada `k = [K_TERBAIK_CV]` dengan recall
`[SKOR_TERBAIK]` dan *standard error* `[SE_TERBAIK]`, sehingga ambang toleransi satu
*standard error* berada pada `[AMBANG_1SE]`. Terdapat `[jumlah kandidat]` nilai *k* yang
skornya masih berada di atas ambang tersebut, yang berarti performanya tidak dapat dibedakan
secara statistik. Klaim ini dikonfirmasi oleh uji *paired t-test* dan *Wilcoxon signed-rank* atas
15 skor per nilai *k* (5-fold x 3 pengulangan), yang menunjukkan `[n]` dari `[total]` pasangan
nilai *k* tidak berbeda secara signifikan pada taraf 5%. Karena itu, memilih nilai *k* semata-mata
berdasarkan skor validasi tertinggi tidak memiliki dasar statistik yang kuat dan justru berisiko
mengoptimalkan *noise* pembagian fold. Berdasarkan aturan one-standard-error, dipilih
**`k = [K_TERPILIH]`**, yaitu nilai *k* terbesar yang masih berada dalam rentang toleransi.
Pada KNN, *k* terbesar berarti model paling sederhana karena derajat kebebasan efektifnya kira-kira
`n/k`, sehingga batas keputusan menjadi paling halus, *variance* paling rendah, dan model paling
stabil terhadap perubahan kecil pada data.

Pengujian grid dua dimensi menegaskan bahwa `weights = 'uniform'` dan `metric = 'euclidean'`
merupakan kombinasi yang tepat. Skema `weights = 'distance'` membobot tetangga terdekat jauh
lebih besar sehingga mengembalikan sebagian *variance* yang justru ingin diredam melalui
pemilihan *k* yang besar, terlihat dari *gap* train-validasi yang lebih lebar. Sementara itu,
karena kelima fitur (usia, BMI, hipertensi, HbA1c, dan kadar glukosa) telah distandarisasi dan
berdimensi rendah, jarak Euclidean bekerja optimal; metrik *chebyshev* justru merugikan karena
hanya mempertimbangkan satu fitur dengan selisih terbesar dan mengabaikan empat fitur lainnya,
padahal diagnosis diabetes bersifat multifaktor. Analisis sensitivitas menunjukkan letak *k*
optimal tidak bergeser berarti ketika ukuran data diubah menjadi 25%, 50%, dan 100%, yang
menandakan keputusan bersifat robust. Aturan praktis `k ~ sqrt(n)` yang lazim dikutip tidak
digunakan karena dengan `n = 76.916` data latih aturan tersebut menghasilkan `k = [k_sqrt_full]` -
nilai yang terlalu besar, tidak memberikan performa lebih baik, mengabaikan ketimpangan kelas
(hanya sekitar 8,5% kasus positif), sekaligus memperlambat proses inferensi pada aplikasi web.
Verifikasi akhir pada data uji 20% yang belum pernah dilihat model menghasilkan recall
`[recall test k terpilih]`, konsisten dengan estimasi validasi silang.

---

### C. Klarifikasi Penulisan "k = 20"

Perlu diklarifikasi bahwa nilai yang tercatat pada catatan revisi sebagai **k = 20** sesungguhnya
merujuk pada model KNN hasil `RandomizedSearchCV` pengujian sebelumnya, yang nilai sebenarnya
adalah **k = 21**. Perbedaan satu angka ini bukan sekadar salah ketik, melainkan konsekuensi
desain eksperimen: seluruh kandidat nilai *k* dalam penelitian ini dibatasi pada bilangan
**ganjil**. Pada klasifikasi biner (diabetes / tidak diabetes), nilai *k* genap seperti 20
berpotensi menghasilkan keadaan seri - misalnya 10 tetangga berlabel positif berhadapan dengan
10 tetangga berlabel negatif - sehingga keputusan akhir bergantung pada mekanisme *tie-breaking*
internal pustaka `scikit-learn` (memilih label dengan indeks terkecil). Mekanisme semacam itu
bersifat sewenang-wenang dan tidak dapat dipertanggungjawabkan dalam konteks skrining kesehatan.
Karena itu nilai *k* = 20 tidak pernah termasuk dalam ruang pencarian, baik pada pengujian awal
maupun pada pengujian ulang di notebook ini. Nilai `k = 21` tetap diuji secara eksplisit sebagai
*baseline* pembanding, dan hasilnya diperbandingkan langsung dengan nilai `k = [K_TERPILIH]`
yang direkomendasikan analisis one-standard-error, baik pada validasi silang maupun pada data uji.

---

### D. Kalimat Ringkas untuk Menjawab Penguji secara Lisan

> "Nilai *k* pada KNN tidak kami tentukan begitu saja. Kami menguji 26 nilai *k* ganjil dari 1
> sampai 51 dengan validasi silang 5-fold, mencatat skor latih dan validasi sekaligus. Kurvanya
> membuktikan *k* kecil overfit - pada *k* = 1 selisih skor latih dan validasi paling lebar -
> sementara *k* terlalu besar mulai underfit. Karena banyak nilai *k* di wilayah tengah ternyata
> tidak berbeda signifikan secara statistik, kami memakai aturan *one-standard-error*: di antara
> nilai *k* yang setara, dipilih model paling sederhana, yaitu *k* terbesar yang masih dalam
> toleransi satu *standard error*. Pilihan itu kami verifikasi lagi pada data uji 20% yang belum
> pernah dilihat model. Satu koreksi kecil, Pak/Bu: nilai hasil tuning kami sebenarnya *k* = 21,
> bukan 20 - kami sengaja hanya memakai *k* ganjil supaya voting pada dua kelas tidak pernah seri."

---
---

# BAGIAN 4: JUSTIFIKASI PEMILIHAN HYPERPLANE SVM

Menjawab catatan penguji: "kenapa hyperplane pada SVM yang dipilih".

*Sumber: `03_Justifikasi_Hyperplane_SVM.ipynb`. Penomoran `CELL n` di bawah mengikuti notebook aslinya
agar rujukan silang di dalam kode tetap sahih.*

# Notebook 03 - Justifikasi Pemilihan Hyperplane SVM

**Revisi Pengujian V3 - Skripsi Prediksi Diabetes**

Notebook ini menjawab pertanyaan penguji: **"Kenapa hyperplane pada SVM yang dipilih?"**

---

## Landasan teori singkat

Support Vector Machine (SVM) mencari sebuah **hyperplane pemisah**

$$ f(x) = w \cdot x + b = 0 $$

yang memisahkan kelas positif (diabetes) dan negatif (sehat) dengan **margin semaksimal mungkin**.
Lebar margin didefinisikan sebagai

$$ \text{margin} = \frac{2}{\lVert w \rVert} $$

sehingga memaksimalkan margin setara dengan **meminimalkan** $\tfrac{1}{2}\lVert w \rVert^2$.
Karena data medis nyata tidak pernah terpisah sempurna, dipakai formulasi **soft margin**:

$$ \min_{w,b,\xi} \; \tfrac{1}{2}\lVert w \rVert^2 + C \sum_i \xi_i $$

dengan $\xi_i$ = besar pelanggaran margin oleh sampel ke-$i$. Di sini:

- **C kecil** -> penalti pelanggaran ringan -> $\lVert w \rVert$ kecil -> **margin lebar**, model lebih halus/general, toleran terhadap noise.
- **C besar** -> penalti pelanggaran berat -> $\lVert w \rVert$ besar -> **margin sempit**, model mengikuti data latih (risiko overfitting).

Sedangkan **kernel** menentukan **ruang tempat hyperplane itu dicari**: kernel linear mencari
hyperplane di ruang fitur asli, sedangkan RBF/polinomial/sigmoid memetakan data ke ruang
berdimensi lebih tinggi lalu mencari hyperplane di sana (data yang tidak terpisah linear di
ruang asli bisa terpisah linear di ruang baru).

Dengan demikian, pertanyaan **"kenapa hyperplane ini yang dipilih"** secara teknis sama dengan
menjawab tiga hal secara empiris:

1. **Kernel mana** (ruang pencarian hyperplane) yang dipakai, dan apa buktinya kernel lain tidak lebih baik.
2. **Nilai C (dan gamma)** mana yang dipakai, dan apa konsekuensinya terhadap **lebar margin** serta **jumlah support vector**.
3. **Mengapa titik operasi itu** yang paling tepat untuk konteks skrining medis (sensitivitas/recall tinggi).

---

## Batasan komputasi (dilaporkan terbuka)

`SVC` dengan kernel non-linear memiliki kompleksitas pelatihan sekitar **O(n^2) sampai O(n^3)**
terhadap jumlah sampel, dan kebutuhan memori untuk matriks kernel juga kuadratik. Melatihnya pada
seluruh 96.146 baris di Google Colab (CPU standar) tidak realistis - bisa memakan berjam-jam untuk
satu konfigurasi saja, apalagi untuk sebuah grid.

Karena itu **eksperimen perbandingan kernel dan grid hyperparameter dijalankan pada subsample
stratified** (proporsi kelas dipertahankan persis seperti data penuh). Ini adalah **keterbatasan
metodologis yang disampaikan secara terbuka**, bukan disembunyikan. Model final (LinearSVC) tetap
dilatih pada data penuh karena kompleksitasnya mendekati linear terhadap n. Kesimpulan yang ditarik
dari subsample bersifat **komparatif** (mengurutkan kernel/parameter relatif satu sama lain), bukan
klaim performa absolut.

---

## Daftar eksperimen

| No | Eksperimen | Yang dibuktikan |
|----|------------|-----------------|
| 1 | Perbandingan kernel (linear, RBF, poly-2, poly-3, sigmoid) | Ruang hyperplane mana yang terbaik |
| 2 | Grid C (linear) dan C x gamma (RBF) | Nilai parameter mana yang optimal |
| 3 | Analisis margin dan support vector | **Inti jawaban**: hyperplane mana yang dipilih dan mengapa |
| 4 | Visualisasi hyperplane (2 fitur, PCA, linear vs RBF) | Bukti visual bentuk batas keputusan |
| 5 | Interpretasi vektor bobot w | Arti hyperplane secara klinis |
| 6 | Uji signifikansi linear vs RBF | Apakah selisihnya nyata secara statistik |
| 7 | Kalibrasi probabilitas | Justifikasi CalibratedClassifierCV |


---

## Persiapan eksperimen: subsample stratified dan estimasi biaya komputasi

Semua eksperimen di bawah memakai **subsample stratified** dengan ukuran berbeda sesuai berat
komputasinya, lalu dibagi **80:20 stratified** (rasio ini sendiri dijustifikasi di notebook `01`).
Data penuh tetap dipakai untuk model linear (LinearSVC) yang biayanya mendekati linear terhadap n.


In [ ]:
# ============================================================
# BAGIAN 4 | CELL 7: Konstanta Eksperimen SVM, Subsample Stratified, dan Split 80:20
# ============================================================
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from scipy import stats

# MODE_CEPAT diatur sekali di PANEL KENDALI pada bagian atas notebook ini.

# Ukuran subsample. SVC berkernel berkompleksitas O(n^2)-O(n^3) sehingga
# tidak realistis dilatih pada 96.146 baris di Colab CPU standar.
N_SUBSAMPLE_KERNEL = 20000 if MODE_CEPAT else 40000   # Eksperimen 1 (perbandingan kernel)
N_SUBSAMPLE_GRID   = 10000 if MODE_CEPAT else 20000   # Eksperimen 2 & 3 (grid C, margin)
N_SUBSAMPLE_SV     = 5000                             # jumlah support vector (SVC linear/RBF)
N_SUBSAMPLE_PLOT   = 3000                             # visualisasi hyperplane
N_SUBSAMPLE_PERM   = 5000                             # permutation importance

GRID_C       = [0.001, 0.01, 0.1, 1, 10, 100]
GRID_C_RBF   = [0.01, 0.1, 1, 10]
GRID_GAMMA   = ['scale', 0.01, 0.1, 1]

# Toleransi recall saat memilih C: ambil margin TERLEBAR di antara kandidat yang
# recall-nya masih dalam 1 poin persen dari recall terbaik (aturan eksplisit,
# bukan pilihan subjektif).
TOLERANSI_RECALL = 0.01

def ambil_subsample(X, y, n, seed=RANDOM_STATE):
    if n >= len(X):
        return X, y
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=seed)
    idx, _ = next(sss.split(X, y))
    return X.iloc[idx], y.iloc[idx]

def split_8020(X, y):
    """Split 80:20 stratified (rasio hasil justifikasi notebook 01)."""
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

def siapkan_scale_smote(X_tr, y_tr, pakai_smote=True):
    """Preprocessing manual (scaler -> SMOTE) untuk analisis yang butuh akses
    langsung ke objek SVM (coef_, support_vectors_) di luar pipeline."""
    sc = StandardScaler().fit(X_tr)
    Xs = sc.transform(X_tr)
    ys = np.asarray(y_tr)
    if pakai_smote:
        Xs, ys = SMOTE(random_state=RANDOM_STATE).fit_resample(Xs, ys)
    return sc, Xs, ys

def evaluasi_decision(model, X_tr, y_tr, X_te, y_te):
    """Evaluasi TANPA kalibrasi: memakai decision_function (jarak bertanda ke
    hyperplane). Dipakai pada eksperimen grid yang berat, karena kalibrasi
    CalibratedClassifierCV(cv=3) menambah 3x biaya fit. Threshold tetap dipilih
    dengan kriteria Youden, hanya saja pada skala skor mentah, bukan probabilitas.
    ROC-AUC tidak terpengaruh karena bersifat invarian terhadap transformasi monoton."""
    t0 = time.time(); model.fit(X_tr, y_tr); waktu_latih = time.time() - t0
    t0 = time.time(); skor = model.decision_function(X_te); waktu_infer = time.time() - t0
    auc = roc_auc_score(y_te, skor)
    thr = threshold_youden(y_te, skor)
    m = hitung_metrik(y_te, (skor >= thr).astype(int))
    return {'threshold': thr, 'roc_auc': auc,
            'waktu_latih_s': waktu_latih, 'waktu_infer_ms': waktu_infer * 1000, **m}

def pipeline_linear_mentah(C, max_iter=5000, pakai_smote=True):
    """LinearSVC tanpa kalibrasi (untuk grid/margin). max_iter dinaikkan agar
    konfigurasi C kecil tetap konvergen."""
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', LinearSVC(C=C, max_iter=max_iter, class_weight='balanced',
                                     dual=False, random_state=RANDOM_STATE)))
    return ImbPipeline(langkah)

def pipeline_svc_mentah(kernel, C=1.0, gamma='scale', degree=3, pakai_smote=True):
    """SVC berkernel tanpa kalibrasi (untuk grid/uji signifikansi)."""
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', SVC(kernel=kernel, C=C, gamma=gamma, degree=degree,
                               class_weight='balanced', cache_size=500,
                               random_state=RANDOM_STATE)))
    return ImbPipeline(langkah)

# ---------------- Set data tiap eksperimen ----------------
X_kern, y_kern = ambil_subsample(X_all, y_all, N_SUBSAMPLE_KERNEL)
Xk_tr, Xk_te, yk_tr, yk_te = split_8020(X_kern, y_kern)

X_grid, y_grid = ambil_subsample(X_all, y_all, N_SUBSAMPLE_GRID)
Xg_tr, Xg_te, yg_tr, yg_te = split_8020(X_grid, y_grid)

X_sv, y_sv = ambil_subsample(X_all, y_all, N_SUBSAMPLE_SV)
Xs_tr, Xs_te, ys_tr, ys_te = split_8020(X_sv, y_sv)

# Data penuh: hanya untuk LinearSVC (biaya mendekati linear terhadap n)
Xf_tr, Xf_te, yf_tr, yf_te = split_8020(X_all, y_all)

garis('KONFIGURASI EKSPERIMEN HYPERPLANE SVM')
info_data = pd.DataFrame([
    {'eksperimen': '1. Perbandingan kernel', 'n_total': len(X_kern),
     'n_latih': len(Xk_tr), 'n_uji': len(Xk_te),
     'positif_%': round(float(y_kern.mean()) * 100, 2), 'estimasi_waktu': '8-25 menit'},
    {'eksperimen': '2. Grid C linear', 'n_total': len(X_grid),
     'n_latih': len(Xg_tr), 'n_uji': len(Xg_te),
     'positif_%': round(float(y_grid.mean()) * 100, 2), 'estimasi_waktu': '< 1 menit'},
    {'eksperimen': '2b. Grid C x gamma RBF', 'n_total': len(X_sv),
     'n_latih': len(Xs_tr), 'n_uji': len(Xs_te),
     'positif_%': round(float(y_sv.mean()) * 100, 2), 'estimasi_waktu': '3-8 menit'},
    {'eksperimen': '3. Margin & support vector', 'n_total': len(X_grid),
     'n_latih': len(Xg_tr), 'n_uji': len(Xg_te),
     'positif_%': round(float(y_grid.mean()) * 100, 2), 'estimasi_waktu': '2-6 menit'},
    {'eksperimen': '4. Visualisasi hyperplane', 'n_total': N_SUBSAMPLE_PLOT,
     'n_latih': N_SUBSAMPLE_PLOT, 'n_uji': 0,
     'positif_%': round(float(y_all.mean()) * 100, 2), 'estimasi_waktu': '< 1 menit'},
    {'eksperimen': '5. Bobot w (data penuh)', 'n_total': len(X_all),
     'n_latih': len(Xf_tr), 'n_uji': len(Xf_te),
     'positif_%': round(float(y_all.mean()) * 100, 2), 'estimasi_waktu': '1-3 menit'},
    {'eksperimen': '6. Uji linear vs RBF (5-fold)', 'n_total': len(X_sv),
     'n_latih': int(len(X_sv) * 0.8), 'n_uji': int(len(X_sv) * 0.2),
     'positif_%': round(float(y_sv.mean()) * 100, 2), 'estimasi_waktu': '2-5 menit'},
    {'eksperimen': '7. Kalibrasi probabilitas', 'n_total': len(X_kern),
     'n_latih': len(Xk_tr), 'n_uji': len(Xk_te),
     'positif_%': round(float(y_kern.mean()) * 100, 2), 'estimasi_waktu': '1-2 menit'},
])
display(info_data)
print(f'Proporsi positif data penuh   : {y_all.mean()*100:.2f}%  (subsample stratified -> proporsi dipertahankan)')
print(f'Baseline konfigurasi V2       : LinearSVC(C={PARAM_SVM_V2["C"]}, max_iter={PARAM_SVM_V2["max_iter"]}) + CalibratedClassifierCV(cv=3)')
print('Total estimasi waktu          : sekitar 20-50 menit pada Colab CPU standar.')

---

# EKSPERIMEN 1 - Perbandingan Kernel

**Pertanyaan:** di ruang mana hyperplane sebaiknya dicari?

Setiap kandidat dibungkus dalam pipeline yang **sama persis** dengan model V2
(`StandardScaler -> SMOTE -> CalibratedClassifierCV(cv=3)`), memakai `class_weight='balanced'`,
dan dievaluasi pada test set yang sama dengan **threshold Youden**. Dengan begitu satu-satunya
yang berbeda antar-baris adalah **kernel** (ruang pencarian hyperplane).

Kandidat: `LinearSVC`, `SVC(kernel='linear')`, `SVC(kernel='rbf')`, `SVC(kernel='poly', degree=2)`,
`SVC(kernel='poly', degree=3)`, `SVC(kernel='sigmoid')`. Untuk keadilan perbandingan, seluruh
kandidat diuji pada `C=1.0` (default scikit-learn), ditambah satu baris `LinearSVC(C=0.1)` sebagai
konfigurasi V2 yang ada sekarang. Nilai C dioptimalkan tersendiri pada Eksperimen 2.


In [ ]:
# ============================================================
# BAGIAN 4 | CELL 8: Eksperimen 1 - Perbandingan Kernel SVM
# ============================================================
def buat_pipeline_svc_kalibrasi(kernel, C=1.0, gamma='scale', degree=3,
                                pakai_smote=True, kalibrasi='sigmoid'):
    """Sama strukturnya dengan buat_pipeline_svm (CELL 5), tetapi selalu memakai SVC
    sehingga kernel 'linear' pun dijalankan lewat SVC (bukan LinearSVC). Ini penting
    agar perbandingan kernel adil: LinearSVC dan SVC(kernel='linear') memakai solver
    dan formulasi loss yang berbeda (squared_hinge/primal vs hinge/dual)."""
    base = SVC(kernel=kernel, C=C, gamma=gamma, degree=degree,
               class_weight='balanced', cache_size=500, random_state=RANDOM_STATE)
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', CalibratedClassifierCV(base, cv=3, method=kalibrasi)))
    return ImbPipeline(langkah)

KANDIDAT_KERNEL = [
    {'nama': 'LinearSVC (C=0.1) [V2]', 'tipe': 'linearsvc', 'kernel': 'linear',
     'C': 0.1, 'gamma': '-', 'degree': '-'},
    {'nama': 'LinearSVC (C=1.0)', 'tipe': 'linearsvc', 'kernel': 'linear',
     'C': 1.0, 'gamma': '-', 'degree': '-'},
    {'nama': 'SVC linear (C=1.0)', 'tipe': 'svc', 'kernel': 'linear',
     'C': 1.0, 'gamma': '-', 'degree': '-'},
    {'nama': 'SVC RBF (C=1.0)', 'tipe': 'svc', 'kernel': 'rbf',
     'C': 1.0, 'gamma': 'scale', 'degree': '-'},
    {'nama': 'SVC Poly d=2 (C=1.0)', 'tipe': 'svc', 'kernel': 'poly',
     'C': 1.0, 'gamma': 'scale', 'degree': 2},
    {'nama': 'SVC Poly d=3 (C=1.0)', 'tipe': 'svc', 'kernel': 'poly',
     'C': 1.0, 'gamma': 'scale', 'degree': 3},
    {'nama': 'SVC Sigmoid (C=1.0)', 'tipe': 'svc', 'kernel': 'sigmoid',
     'C': 1.0, 'gamma': 'scale', 'degree': '-'},
]

garis('EKSPERIMEN 1: PERBANDINGAN KERNEL SVM')
print(f'Data   : {len(Xk_tr):,} latih / {len(Xk_te):,} uji (subsample stratified)')
print(f'Pipeline: StandardScaler -> SMOTE -> CalibratedClassifierCV(cv=3, sigmoid)')
print(f'Metrik  : threshold Youden (memaksimalkan sensitivitas - spesifisitas)')
print(f'Estimasi: 8-25 menit. Kernel yang gagal/terlalu lama ditangkap try/except.')
print('-' * 70)

hasil_kernel = []
t_mulai_total = time.time()

for i, kand in enumerate(KANDIDAT_KERNEL, 1):
    print(f'[{i}/{len(KANDIDAT_KERNEL)}] {kand["nama"]} ... ', end='')
    try:
        deg = kand['degree'] if isinstance(kand['degree'], int) else 3
        if kand['tipe'] == 'linearsvc':
            model = buat_pipeline_svm(kernel='linear', C=kand['C'],
                                      max_iter=PARAM_SVM_V2['max_iter'])
        else:
            model = buat_pipeline_svc_kalibrasi(kernel=kand['kernel'], C=kand['C'],
                                                gamma=kand['gamma'] if kand['gamma'] != '-' else 'scale',
                                                degree=deg)
        r = evaluasi_holdout(model, Xk_tr, yk_tr, Xk_te, yk_te)
        hasil_kernel.append({
            'kandidat'      : kand['nama'],
            'kernel'        : kand['kernel'],
            'implementasi'  : 'LinearSVC' if kand['tipe'] == 'linearsvc' else 'SVC',
            'C'             : kand['C'],
            'gamma'         : kand['gamma'],
            'degree'        : kand['degree'],
            'recall'        : r['recall_tuned'],
            'precision'     : r['precision_tuned'],
            'f1'            : r['f1_tuned'],
            'roc_auc'       : r['roc_auc_tuned'],
            'accuracy'      : r['accuracy_tuned'],
            'brier'         : r['brier_tuned'],
            'threshold'     : r['threshold'],
            'waktu_latih_s' : r['waktu_latih_s'],
            'waktu_infer_ms': r['waktu_infer_ms'],
            'status'        : 'OK',
        })
        print(f'OK | recall={r["recall_tuned"]:.4f} AUC={r["roc_auc_tuned"]:.4f} '
              f'latih={r["waktu_latih_s"]:.1f}s')
    except Exception as e:
        hasil_kernel.append({
            'kandidat': kand['nama'], 'kernel': kand['kernel'],
            'implementasi': 'LinearSVC' if kand['tipe'] == 'linearsvc' else 'SVC',
            'C': kand['C'], 'gamma': kand['gamma'], 'degree': kand['degree'],
            'recall': np.nan, 'precision': np.nan, 'f1': np.nan, 'roc_auc': np.nan,
            'accuracy': np.nan, 'brier': np.nan, 'threshold': np.nan,
            'waktu_latih_s': np.nan, 'waktu_infer_ms': np.nan,
            'status': f'GAGAL: {type(e).__name__}',
        })
        print(f'GAGAL ({type(e).__name__}: {str(e)[:60]}) -> dilewati, notebook lanjut')

print('-' * 70)
print(f'Total waktu Eksperimen 1: {(time.time() - t_mulai_total)/60:.1f} menit')

tabel_perbandingan_kernel = pd.DataFrame(hasil_kernel).round(5)
simpan_tabel(tabel_perbandingan_kernel, 'tabel_perbandingan_kernel')

# --- Kesimpulan angka ---
ok = tabel_perbandingan_kernel[tabel_perbandingan_kernel['status'] == 'OK'].copy()
baris_auc_terbaik    = ok.loc[ok['roc_auc'].idxmax()]
baris_recall_terbaik = ok.loc[ok['recall'].idxmax()]
baris_linear         = ok[ok['kernel'] == 'linear'].sort_values('roc_auc', ascending=False).iloc[0]
baris_rbf            = ok[ok['kernel'] == 'rbf'].iloc[0] if (ok['kernel'] == 'rbf').any() else None

garis('KESIMPULAN EKSPERIMEN 1')
print(f'AUC tertinggi        : {baris_auc_terbaik["kandidat"]} (AUC={baris_auc_terbaik["roc_auc"]:.4f})')
print(f'Recall tertinggi     : {baris_recall_terbaik["kandidat"]} (recall={baris_recall_terbaik["recall"]:.4f})')
print(f'Kernel linear terbaik: {baris_linear["kandidat"]} '
      f'(AUC={baris_linear["roc_auc"]:.4f}, recall={baris_linear["recall"]:.4f}, '
      f'latih={baris_linear["waktu_latih_s"]:.1f}s)')
if baris_rbf is not None:
    d_auc = baris_rbf['roc_auc'] - baris_linear['roc_auc']
    rasio_waktu = baris_rbf['waktu_latih_s'] / max(baris_linear['waktu_latih_s'], 1e-9)
    print(f'Kernel RBF           : AUC={baris_rbf["roc_auc"]:.4f}, recall={baris_rbf["recall"]:.4f}, '
          f'latih={baris_rbf["waktu_latih_s"]:.1f}s')
    print(f'Selisih AUC RBF-linear : {d_auc:+.4f}  |  RBF {rasio_waktu:.1f}x lebih lambat dilatih')
print()
print('Interpretasi: bila selisih AUC antar-kernel berada pada orde 0,00x sementara biaya')
print('komputasi kernel non-linear berkali-kali lipat, maka memindahkan hyperplane ke ruang')
print('berdimensi tinggi TIDAK memberi keuntungan yang sepadan pada dataset ini.')

In [ ]:
# ============================================================
# BAGIAN 4 | CELL 9: Visualisasi Perbandingan Kernel (metrik + waktu latih)
# ============================================================
ok_plot = tabel_perbandingan_kernel[tabel_perbandingan_kernel['status'] == 'OK'].copy()
metrik_plot = ['recall', 'precision', 'f1', 'roc_auc']
label_metrik = ['Recall', 'Precision', 'F1-Score', 'ROC-AUC']
warna_metrik = ['#2ecc71', '#3498db', '#9b59b6', '#f39c12']

fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))

# (a) Bar berkelompok metrik per kernel
x = np.arange(len(ok_plot))
lebar = 0.2
for j, (m, lab, w) in enumerate(zip(metrik_plot, label_metrik, warna_metrik)):
    axes[0].bar(x + (j - 1.5) * lebar, ok_plot[m].values, lebar, label=lab, color=w, edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(ok_plot['kandidat'], rotation=30, ha='right', fontsize=9)
axes[0].set_ylabel('Nilai metrik')
axes[0].set_title('(a) Performa per Kernel (threshold Youden)')
axes[0].set_ylim(0, 1.05)
axes[0].legend(ncol=4, fontsize=9, loc='upper center')
for xi, v in zip(x, ok_plot['recall'].values):
    axes[0].text(xi - 1.5 * lebar, v + 0.015, f'{v:.3f}', ha='center', fontsize=7.5, rotation=90)

# (b) Waktu latih (skala log)
warna_bar = [WARNA_MODEL['SVM (Linear)'] if k == 'linear' else '#95a5a6'
             for k in ok_plot['kernel']]
axes[1].bar(x, ok_plot['waktu_latih_s'].values, color=warna_bar, edgecolor='white')
axes[1].set_yscale('log')
axes[1].set_xticks(x)
axes[1].set_xticklabels(ok_plot['kandidat'], rotation=30, ha='right', fontsize=9)
axes[1].set_ylabel('Waktu latih (detik, skala log)')
axes[1].set_title('(b) Biaya Komputasi Pelatihan - hijau = kernel linear')
for xi, v in zip(x, ok_plot['waktu_latih_s'].values):
    axes[1].text(xi, v * 1.12, f'{v:.1f}s', ha='center', fontsize=8.5)

plt.suptitle('Eksperimen 1 - Perbandingan Kernel SVM: Performa vs Biaya Komputasi',
             fontsize=14, y=1.02)
plt.tight_layout()
simpan_gambar('svm_perbandingan_kernel')
plt.show()

print('Bacaan grafik: panel (a) menunjukkan metrik antar-kernel praktis berimpit,')
print('sedangkan panel (b) berskala LOGARITMIK - artinya perbedaan biaya latih')
print('mencapai orde puluhan hingga ratusan kali lipat.')

---

# EKSPERIMEN 2 - Grid Parameter C dan Gamma

**Pertanyaan:** setelah ruangnya dipilih, hyperplane yang mana (parameter mana) di ruang itu?

- **Kernel linear:** sapuan `C` pada [0.001, 0.01, 0.1, 1, 10, 100]. C mengatur trade-off
  margin lebar vs kesalahan klasifikasi.
- **Kernel RBF:** grid `C` x `gamma`. Gamma mengatur seberapa "lokal" pengaruh tiap support vector -
  gamma besar membuat batas keputusan sangat berkelok (risiko overfitting).

Untuk grid ini kalibrasi **tidak** dipakai (cukup `decision_function`), karena
`CalibratedClassifierCV(cv=3)` melipatgandakan biaya fit 3x sementara ROC-AUC bersifat invarian
terhadap transformasi monoton seperti kalibrasi sigmoid.


In [ ]:
# ============================================================
# BAGIAN 4 | CELL 10: Eksperimen 2a - Sapuan Parameter C untuk Kernel Linear
# ============================================================
garis('EKSPERIMEN 2a: SAPUAN C UNTUK KERNEL LINEAR')
print(f'Data     : {len(Xg_tr):,} latih / {len(Xg_te):,} uji')
print(f'Grid C   : {GRID_C}')
print('Estimasi : < 1 menit')
print('-' * 70)

hasil_C = []
for C in GRID_C:
    t0 = time.time()
    try:
        r = evaluasi_decision(pipeline_linear_mentah(C), Xg_tr, yg_tr, Xg_te, yg_te)
        hasil_C.append({'kernel': 'linear', 'C': C, 'gamma': '-',
                        'recall': r['recall'], 'precision': r['precision'],
                        'f1': r['f1'], 'roc_auc': r['roc_auc'],
                        'accuracy': r['accuracy'], 'threshold': r['threshold'],
                        'waktu_latih_s': r['waktu_latih_s'], 'status': 'OK'})
        print(f'C={C:<8} recall={r["recall"]:.4f}  precision={r["precision"]:.4f}  '
              f'F1={r["f1"]:.4f}  AUC={r["roc_auc"]:.4f}  ({time.time()-t0:.1f}s)')
    except Exception as e:
        hasil_C.append({'kernel': 'linear', 'C': C, 'gamma': '-', 'recall': np.nan,
                        'precision': np.nan, 'f1': np.nan, 'roc_auc': np.nan,
                        'accuracy': np.nan, 'threshold': np.nan,
                        'waktu_latih_s': np.nan, 'status': f'GAGAL: {type(e).__name__}'})
        print(f'C={C:<8} GAGAL ({type(e).__name__}) -> dilewati')

tabel_grid_C_linear = pd.DataFrame(hasil_C).round(5)
simpan_tabel(tabel_grid_C_linear, 'tabel_grid_C_linear')

# --- Kurva metrik vs C (sumbu-x logaritmik) ---
ok_C = tabel_grid_C_linear[tabel_grid_C_linear['status'] == 'OK']
plt.figure(figsize=(12, 6))
plt.semilogx(ok_C['C'], ok_C['recall'], 'o-', lw=2.2, ms=8,
             color=WARNA_MODEL['SVM (Linear)'], label='Recall (sensitivitas)')
plt.semilogx(ok_C['C'], ok_C['f1'], 's-', lw=2.2, ms=7, color='#9b59b6', label='F1-Score')
plt.semilogx(ok_C['C'], ok_C['roc_auc'], '^-', lw=2.2, ms=7, color=WARNA_AKSEN, label='ROC-AUC')
plt.semilogx(ok_C['C'], ok_C['precision'], 'd--', lw=1.6, ms=6, color='#3498db',
             alpha=0.75, label='Precision')
plt.axvline(PARAM_SVM_V2['C'], color='#e74c3c', ls=':', lw=2,
            label=f'C = {PARAM_SVM_V2["C"]} (hasil tuning V2)')
plt.xlabel('Parameter C (skala logaritmik) - kiri: margin lebar, kanan: margin sempit')
plt.ylabel('Nilai metrik pada test set')
plt.title('Eksperimen 2a - Pengaruh Parameter C terhadap Performa SVM Linear')
plt.legend(loc='best', fontsize=10)
plt.ylim(0, 1.05)
plt.tight_layout()
simpan_gambar('svm_kurva_C')
plt.show()

if len(ok_C):
    b_auc = ok_C.loc[ok_C['roc_auc'].idxmax()]
    b_rec = ok_C.loc[ok_C['recall'].idxmax()]
    rentang_auc = ok_C['roc_auc'].max() - ok_C['roc_auc'].min()
    garis('KESIMPULAN EKSPERIMEN 2a')
    print(f'AUC tertinggi    : C={b_auc["C"]} (AUC={b_auc["roc_auc"]:.4f})')
    print(f'Recall tertinggi : C={b_rec["C"]} (recall={b_rec["recall"]:.4f})')
    print(f'Rentang AUC pada seluruh grid C: {rentang_auc:.4f}')
    print('Kurva yang datar berarti performa TIDAK sensitif terhadap C. Dalam kondisi ini,')
    print('C kecil lebih dipilih karena menghasilkan margin lebih lebar (regularisasi lebih')
    print('kuat) tanpa mengorbankan performa - dibuktikan secara kuantitatif di Eksperimen 3.')

In [ ]:
# ============================================================
# BAGIAN 4 | CELL 11: Eksperimen 2b - Grid C x Gamma untuk Kernel RBF (heatmap recall)
# ============================================================
# Subsample lebih kecil dipakai di sini karena SVC-RBF berkompleksitas O(n^2)-O(n^3)
# dan grid ini berisi len(GRID_C_RBF) x len(GRID_GAMMA) kombinasi.
garis('EKSPERIMEN 2b: GRID C x GAMMA UNTUK KERNEL RBF')
print(f'Data     : {len(Xs_tr):,} latih / {len(Xs_te):,} uji (subsample lebih kecil, alasan biaya O(n^2)-O(n^3))')
print(f'Grid     : C={GRID_C_RBF} x gamma={GRID_GAMMA} -> {len(GRID_C_RBF)*len(GRID_GAMMA)} kombinasi')
print('Estimasi : 3-8 menit')
print('-' * 70)

hasil_rbf = []
t_mulai = time.time()
for i, (C, g) in enumerate(itertools.product(GRID_C_RBF, GRID_GAMMA), 1):
    try:
        r = evaluasi_decision(pipeline_svc_mentah('rbf', C=C, gamma=g),
                              Xs_tr, ys_tr, Xs_te, ys_te)
        hasil_rbf.append({'kernel': 'rbf', 'C': C, 'gamma': str(g),
                          'recall': r['recall'], 'precision': r['precision'],
                          'f1': r['f1'], 'roc_auc': r['roc_auc'],
                          'accuracy': r['accuracy'], 'threshold': r['threshold'],
                          'waktu_latih_s': r['waktu_latih_s'], 'status': 'OK'})
        print(f'[{i:2d}/{len(GRID_C_RBF)*len(GRID_GAMMA)}] C={C:<6} gamma={str(g):<6} '
              f'recall={r["recall"]:.4f} AUC={r["roc_auc"]:.4f} ({r["waktu_latih_s"]:.1f}s)')
    except Exception as e:
        hasil_rbf.append({'kernel': 'rbf', 'C': C, 'gamma': str(g), 'recall': np.nan,
                          'precision': np.nan, 'f1': np.nan, 'roc_auc': np.nan,
                          'accuracy': np.nan, 'threshold': np.nan, 'waktu_latih_s': np.nan,
                          'status': f'GAGAL: {type(e).__name__}'})
        print(f'[{i:2d}] C={C} gamma={g} GAGAL ({type(e).__name__}) -> dilewati')

print('-' * 70)
print(f'Total waktu Eksperimen 2b: {(time.time() - t_mulai)/60:.1f} menit')

tabel_rbf = pd.DataFrame(hasil_rbf).round(5)

# Gabungan grid linear + RBF (format panjang) sesuai kontrak nama file spec
tabel_grid_C_gamma = pd.concat([tabel_grid_C_linear, tabel_rbf], ignore_index=True)
simpan_tabel(tabel_grid_C_gamma, 'tabel_grid_C_gamma')

# --- Heatmap recall ---
piv_recall = tabel_rbf.pivot(index='C', columns='gamma', values='recall')
piv_auc    = tabel_rbf.pivot(index='C', columns='gamma', values='roc_auc')
urut_gamma = [str(g) for g in GRID_GAMMA]
piv_recall = piv_recall.reindex(columns=urut_gamma)
piv_auc    = piv_auc.reindex(columns=urut_gamma)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.8))
sns.heatmap(piv_recall, annot=True, fmt='.4f', cmap='YlGnBu', ax=axes[0],
            cbar_kws={'label': 'Recall'}, linewidths=0.5)
axes[0].set_title('(a) Recall - Grid C x Gamma (kernel RBF)')
axes[0].set_xlabel('gamma'); axes[0].set_ylabel('C')

sns.heatmap(piv_auc, annot=True, fmt='.4f', cmap='YlOrRd', ax=axes[1],
            cbar_kws={'label': 'ROC-AUC'}, linewidths=0.5)
axes[1].set_title('(b) ROC-AUC - Grid C x Gamma (kernel RBF)')
axes[1].set_xlabel('gamma'); axes[1].set_ylabel('C')

plt.suptitle('Eksperimen 2b - Pemetaan Parameter Kernel RBF', fontsize=14, y=1.02)
plt.tight_layout()
simpan_gambar('svm_heatmap_rbf')
plt.show()

ok_rbf = tabel_rbf[tabel_rbf['status'] == 'OK']
garis('KESIMPULAN EKSPERIMEN 2b')
if len(ok_rbf):
    best_rbf = ok_rbf.loc[ok_rbf['roc_auc'].idxmax()]
    best_lin_auc = float(ok_C['roc_auc'].max()) if len(ok_C) else float('nan')
    print(f'RBF terbaik      : C={best_rbf["C"]}, gamma={best_rbf["gamma"]} '
          f'-> AUC={best_rbf["roc_auc"]:.4f}, recall={best_rbf["recall"]:.4f}')
    print(f'Linear terbaik   : AUC={best_lin_auc:.4f} (Eksperimen 2a)')
    print(f'Selisih AUC (RBF terbaik - linear terbaik): {best_rbf["roc_auc"] - best_lin_auc:+.4f}')
    print(f'Waktu latih RBF terbaik: {best_rbf["waktu_latih_s"]:.1f}s pada {len(Xs_tr):,} sampel saja,')
    print(f'sedangkan kernel linear melatih {len(Xg_tr):,} sampel dalam '
          f'{float(ok_C["waktu_latih_s"].min()):.2f}s.')
    print()
    print('Catatan: gamma besar (>= 1) menaikkan kompleksitas batas keputusan namun tidak')
    print('menaikkan AUC secara berarti - indikasi bahwa struktur pemisah kelas pada 5 fitur')
    print('ini memang mendekati linear, bukan berbentuk kantong-kantong non-linear.')
else:
    print('Seluruh konfigurasi RBF gagal dijalankan - lihat kolom status pada tabel.')

---

# EKSPERIMEN 3 - Analisis Margin dan Support Vector (INTI JAWABAN)

Ini adalah bagian yang menjawab pertanyaan penguji **secara harfiah**: dari sekian banyak
hyperplane yang mungkin, **yang mana yang dipilih dan mengapa**.

Untuk tiap nilai C dihitung:

| Besaran | Rumus | Arti |
|---|---|---|
| $\lVert w \rVert$ | norma vektor bobot | seberapa "curam" fungsi keputusan |
| margin | $2 / \lVert w \rVert$ | lebar koridor pemisah antar kelas |
| $n_{SV}$ | jumlah support vector | banyaknya sampel yang menentukan hyperplane |
| rasio SV | $n_{SV}/n$ | proporsi data yang menempel di margin |

**Catatan implementasi (penting):** `LinearSVC` menyelesaikan masalah optimasi dalam bentuk
**primal** sehingga hanya menyimpan `coef_` dan **tidak memiliki atribut `support_vectors_`**.
Karena itu $\lVert w \rVert$ dan margin diambil dari `LinearSVC` (model yang benar-benar dipakai
sistem), sedangkan **jumlah support vector dihitung dengan `SVC(kernel='linear')`** yang
menyelesaikan bentuk **dual** dan menyimpan `n_support_`. SVC dijalankan pada subsample yang lebih
kecil karena biaya komputasinya kuadratik.

**Aturan pemilihan (ditetapkan di muka, bukan post-hoc):** dipilih hyperplane dengan
**margin terlebar** di antara kandidat yang recall-nya masih berada dalam **1 poin persen** dari
recall terbaik. Prinsipnya: dalam skrining medis sensitivitas tidak boleh dikorbankan, tetapi di
antara pilihan yang sama sensitifnya, margin terlebar berarti generalisasi terbaik.


In [ ]:
# ============================================================
# BAGIAN 4 | CELL 12: Eksperimen 3 - Margin, Norma Bobot, dan Jumlah Support Vector
# ============================================================
garis('EKSPERIMEN 3: ANALISIS MARGIN DAN SUPPORT VECTOR')
print(f'Norma bobot & margin : LinearSVC pada {len(Xg_tr):,} sampel latih (model yang dipakai sistem)')
print(f'Jumlah support vector: SVC(kernel="linear") pada {len(Xs_tr):,} sampel latih')
print('  -> LinearSVC memakai formulasi PRIMAL sehingga tidak menyimpan support_vectors_.')
print('     SVC memakai formulasi DUAL sehingga n_support_ tersedia.')
print(f'Grid C  : {GRID_C}')
print('Estimasi: 2-6 menit')
print('-' * 70)

# Preprocessing manual (scaler -> SMOTE) supaya objek SVM bisa diakses langsung
sc_m,  Xm_res,  ym_res  = siapkan_scale_smote(Xg_tr, yg_tr)   # untuk LinearSVC (margin)
sc_sv, Xsv_res, ysv_res = siapkan_scale_smote(Xs_tr, ys_tr)   # untuk SVC (support vector)
Xg_te_s = sc_m.transform(Xg_te)

baris_margin = []
for C in GRID_C:
    t0 = time.time()
    try:
        # (1) LinearSVC -> vektor bobot w, norma, lebar margin
        lin = LinearSVC(C=C, max_iter=5000, class_weight='balanced',
                        dual=False, random_state=RANDOM_STATE)
        lin.fit(Xm_res, ym_res)
        w = lin.coef_.ravel()
        norma_w = float(np.linalg.norm(w))
        margin  = 2.0 / norma_w

        skor = lin.decision_function(Xg_te_s)
        thr  = threshold_youden(yg_te, skor)
        met  = hitung_metrik(yg_te, (skor >= thr).astype(int))
        auc  = roc_auc_score(yg_te, skor)

        # (2) SVC(kernel='linear') -> jumlah support vector
        svc = SVC(kernel='linear', C=C, class_weight='balanced',
                  cache_size=500, random_state=RANDOM_STATE)
        svc.fit(Xsv_res, ysv_res)
        n_sv = int(svc.n_support_.sum())
        rasio_sv = n_sv / len(Xsv_res)
        norma_w_svc = float(np.linalg.norm(svc.coef_.ravel()))

        baris_margin.append({
            'C': C, 'norma_w': norma_w, 'margin_2_per_w': margin,
            'n_SV': n_sv, 'rasio_SV': rasio_sv, 'norma_w_svc': norma_w_svc,
            'recall': met['recall'], 'precision': met['precision'],
            'f1': met['f1'], 'roc_auc': auc, 'status': 'OK'})
        print(f'C={C:<8} ||w||={norma_w:7.4f}  margin={margin:8.4f}  '
              f'n_SV={n_sv:5d} ({rasio_sv*100:5.1f}%)  recall={met["recall"]:.4f}  '
              f'precision={met["precision"]:.4f}  ({time.time()-t0:.1f}s)')
    except Exception as e:
        baris_margin.append({'C': C, 'norma_w': np.nan, 'margin_2_per_w': np.nan,
                             'n_SV': np.nan, 'rasio_SV': np.nan, 'norma_w_svc': np.nan,
                             'recall': np.nan, 'precision': np.nan, 'f1': np.nan,
                             'roc_auc': np.nan, 'status': f'GAGAL: {type(e).__name__}'})
        print(f'C={C:<8} GAGAL ({type(e).__name__}) -> dilewati')

tabel_analisis_margin = pd.DataFrame(baris_margin).round(6)
simpan_tabel(tabel_analisis_margin, 'tabel_analisis_margin')

# --- Aturan pemilihan hyperplane: margin TERLEBAR di antara yang recall-nya masih tinggi ---
ok_m = tabel_analisis_margin[tabel_analisis_margin['status'] == 'OK'].copy()
recall_maks = float(ok_m['recall'].max())
kandidat_layak = ok_m[ok_m['recall'] >= recall_maks - TOLERANSI_RECALL].copy()
baris_terpilih = kandidat_layak.loc[kandidat_layak['margin_2_per_w'].idxmax()]

C_TERPILIH      = float(baris_terpilih['C'])
MARGIN_TERPILIH = float(baris_terpilih['margin_2_per_w'])
NORMA_W_TERPILIH= float(baris_terpilih['norma_w'])
NSV_TERPILIH    = int(baris_terpilih['n_SV'])
RASIO_SV_TERPILIH = float(baris_terpilih['rasio_SV'])

garis('KESIMPULAN EKSPERIMEN 3 - HYPERPLANE YANG DIPILIH')
print(f'Recall terbaik pada grid          : {recall_maks:.4f}')
print(f'Kandidat dalam toleransi {TOLERANSI_RECALL*100:.0f}% recall : C = {list(kandidat_layak["C"].values)}')
print(f'-> C TERPILIH                     : {C_TERPILIH}')
print(f'   ||w||                          : {NORMA_W_TERPILIH:.4f}')
print(f'   Lebar margin (2/||w||)         : {MARGIN_TERPILIH:.4f}')
print(f'   Jumlah support vector          : {NSV_TERPILIH:,} dari {len(Xsv_res):,} sampel '
      f'({RASIO_SV_TERPILIH*100:.1f}%)')
print(f'   Recall / Precision             : {baris_terpilih["recall"]:.4f} / {baris_terpilih["precision"]:.4f}')
print()
c_maks = ok_m.loc[ok_m['C'].idxmax()]
print(f'Sebagai pembanding, C={c_maks["C"]} menghasilkan margin {c_maks["margin_2_per_w"]:.4f} '
      f'({MARGIN_TERPILIH / max(c_maks["margin_2_per_w"], 1e-12):.1f}x lebih sempit)')
print(f'dengan recall {c_maks["recall"]:.4f} - artinya mengetatkan C hanya mempersempit margin')
print('tanpa imbalan sensitivitas. Inilah alasan empiris hyperplane bermargin lebar yang dipilih.')
print()
print(f'Konfigurasi V2 memakai C={PARAM_SVM_V2["C"]} -> '
      f'{"KONSISTEN dengan hasil eksperimen ini" if abs(C_TERPILIH - PARAM_SVM_V2["C"]) < 1e-9 else "berbeda, lihat tabel di atas"}')

In [ ]:
# ============================================================
# BAGIAN 4 | CELL 13: Dua Panel Bertumpuk - Lebar Margin dan Recall terhadap C
# ============================================================
# Catatan desain: grafik ini sengaja TIDAK memakai dua sumbu-y (twinx). Pada grafik
# dua sumbu, posisi relatif kedua kurva - termasuk titik potongnya - sepenuhnya
# ditentukan oleh pilihan rentang masing-masing sumbu, sehingga titik potong bisa
# digeser ke mana saja hanya dengan mengubah batas sumbu. Karena justru titik itulah
# yang menjadi inti argumen pemilihan hyperplane, dipakai dua panel bertumpuk yang
# BERBAGI sumbu-x nilai C: pembaca membandingkan secara vertikal pada C yang identik,
# dan tiap panel punya sumbu-y sendiri yang berdiri sendiri.
ok_m = tabel_analisis_margin[tabel_analisis_margin['status'] == 'OK'].copy()

# Rentang C yang memenuhi aturan pemilihan (recall dalam toleransi dari recall terbaik)
recall_maks_plot = float(ok_m['recall'].max())
layak_plot = ok_m[ok_m['recall'] >= recall_maks_plot - TOLERANSI_RECALL]
c_min_layak = float(layak_plot['C'].min())
c_maks_layak = float(layak_plot['C'].max())

fig, (ax_atas, ax_bawah) = plt.subplots(2, 1, figsize=(13, 9.5), sharex=True,
                                        gridspec_kw={'height_ratios': [1, 1]})

# ---------- Panel atas: lebar margin 2/||w|| ----------
ax_atas.plot(ok_m['C'], ok_m['margin_2_per_w'], 'o-', lw=2.6, ms=9,
             color=WARNA_MODEL['SVM (Linear)'], label='Lebar margin  2/||w||')
ax_atas.set_yscale('log')
ax_atas.set_ylabel('Lebar margin  2 / ||w||\n(skala log)')
ax_atas.set_title('Eksperimen 3 - Lebar Margin dan Sensitivitas terhadap Parameter C\n'
                  '(dua panel berbagi sumbu-x: bandingkan secara vertikal pada nilai C yang sama)',
                  fontsize=13)

# ---------- Panel bawah: recall dan precision ----------
ax_bawah.plot(ok_m['C'], ok_m['recall'], 's-', lw=2.4, ms=8,
              color='#e74c3c', label='Recall (sensitivitas)')
ax_bawah.plot(ok_m['C'], ok_m['precision'], 'd--', lw=1.8, ms=6,
              color='#3498db', alpha=0.85, label='Precision')
ax_bawah.axhline(recall_maks_plot - TOLERANSI_RECALL, color='#e74c3c', ls=':', lw=1.5,
                 alpha=0.8,
                 label=f'Batas toleransi recall ({recall_maks_plot - TOLERANSI_RECALL:.4f})')
ax_bawah.set_ylim(0, 1.05)
ax_bawah.set_ylabel('Recall / Precision')
ax_bawah.set_xscale('log')
ax_bawah.set_xlabel('Parameter C (skala logaritmik) - kiri: margin lebar, kanan: margin sempit')

# ---------- Penanda identik pada KEDUA panel ----------
for ax in (ax_atas, ax_bawah):
    ax.axvspan(c_min_layak, c_maks_layak, color=WARNA_AKSEN, alpha=0.12,
               label=f'Rentang C yang lolos aturan (recall >= terbaik - {TOLERANSI_RECALL*100:.0f} poin persen)')
    ax.axvline(C_TERPILIH, color=WARNA_AKSEN, ls='-', lw=2.5, alpha=0.9,
               label=f'C terpilih = {C_TERPILIH}')
    ax.set_xlim(min(ok_m['C']) / 3, max(ok_m['C']) * 3)

# Tandai titik-titik yang lolos aturan pada kedua panel
ax_atas.plot(layak_plot['C'], layak_plot['margin_2_per_w'], 'o', ms=14,
             mfc='none', mec=WARNA_AKSEN, mew=2.2)
ax_bawah.plot(layak_plot['C'], layak_plot['recall'], 's', ms=14,
              mfc='none', mec=WARNA_AKSEN, mew=2.2)

ax_atas.annotate(f'C terpilih = {C_TERPILIH}\nmargin = {MARGIN_TERPILIH:.3f} (TERLEBAR di rentang ini)\n'
                 f'n_SV = {NSV_TERPILIH:,} ({RASIO_SV_TERPILIH*100:.1f}%)',
                 xy=(C_TERPILIH, MARGIN_TERPILIH),
                 xytext=(0.05, 0.16), textcoords='axes fraction',
                 fontsize=10.5, color='#7d5300',
                 bbox=dict(boxstyle='round,pad=0.5', fc='#fdf3dd', ec=WARNA_AKSEN, alpha=0.95),
                 arrowprops=dict(arrowstyle='->', color=WARNA_AKSEN, lw=2))
ax_bawah.annotate(f'C terpilih = {C_TERPILIH}\nrecall = {float(baris_terpilih["recall"]):.4f} '
                  f'(setara recall terbaik {recall_maks_plot:.4f})',
                  xy=(C_TERPILIH, float(baris_terpilih['recall'])),
                  xytext=(0.05, 0.16), textcoords='axes fraction',
                  fontsize=10.5, color='#7d5300',
                  bbox=dict(boxstyle='round,pad=0.5', fc='#fdf3dd', ec=WARNA_AKSEN, alpha=0.95),
                  arrowprops=dict(arrowstyle='->', color=WARNA_AKSEN, lw=2))

ax_atas.legend(loc='upper right', fontsize=9.5, framealpha=0.93)
ax_bawah.legend(loc='lower left', fontsize=9.5, framealpha=0.93, ncol=2)

plt.tight_layout()
simpan_gambar('svm_margin_vs_C')
plt.show()

# Grafik pendamping: jumlah support vector vs C
plt.figure(figsize=(12, 5))
plt.semilogx(ok_m['C'], ok_m['rasio_SV'] * 100, 'o-', lw=2.4, ms=8, color='#8e44ad')
plt.axvline(C_TERPILIH, color=WARNA_AKSEN, ls='-', lw=2.2)
plt.xlabel('Parameter C (skala logaritmik)')
plt.ylabel('Rasio support vector (%)')
plt.title('Proporsi Support Vector terhadap C - C kecil melibatkan lebih banyak sampel '
          'dalam penentuan hyperplane (keputusan lebih kolektif, lebih stabil)')
for cx, ry in zip(ok_m['C'], ok_m['rasio_SV'] * 100):
    plt.text(cx, ry + 1.2, f'{ry:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
simpan_gambar('svm_support_vector_vs_C')
plt.show()

print('Bacaan grafik (dua panel, sumbu-x nilai C yang sama):')
print('  - Panel atas : lebar margin turun tajam saat C naik.')
print('  - Panel bawah: recall mendatar - kenaikan C tidak dibayar dengan sensitivitas.')
print(f'  - Area berarsir: rentang C = [{c_min_layak}, {c_maks_layak}] yang lolos aturan pemilihan')
print(f'    (recall >= {recall_maks_plot:.4f} - {TOLERANSI_RECALL:.2f}). Di dalam rentang itu, titik')
print(f'    dengan margin terlebar adalah C = {C_TERPILIH} (garis vertikal oranye pada kedua panel).')
print('Pembacaan dilakukan secara vertikal pada nilai C yang identik, bukan dari titik potong')
print('antar-kurva, sehingga kesimpulannya tidak bergantung pada pilihan skala sumbu.')

---

# EKSPERIMEN 4 - Visualisasi Hyperplane

Angka pada Eksperimen 3 dilengkapi bukti visual. Tiga sudut pandang ditampilkan:

- **(a)** Bidang dua fitur paling diskriminatif secara klinis: **HbA1c** x **Kadar Glukosa Darah**,
  lengkap dengan garis keputusan ($f(x)=0$), dua garis margin ($f(x)=\pm 1$), dan support vector.
- **(b)** Proyeksi **PCA 2 komponen** dari kelima fitur terstandardisasi - memperlihatkan hyperplane
  pada ruang yang merangkum seluruh fitur, bukan hanya dua.
- **(c)** Perbandingan langsung bentuk batas keputusan **linear vs RBF** pada bidang yang sama.


In [ ]:
# ============================================================
# BAGIAN 4 | CELL 14: Eksperimen 4a - Hyperplane pada Bidang HbA1c x Glukosa Darah
# ============================================================
FITUR_PLOT = ['HbA1c_level', 'blood_glucose_level']
LABEL_PLOT = ['HbA1c (terstandardisasi)', 'Kadar Glukosa Darah (terstandardisasi)']

Xp_df, yp = ambil_subsample(X_all, y_all, N_SUBSAMPLE_PLOT)
Xp2 = Xp_df[FITUR_PLOT].values
yp_arr = np.asarray(yp)

sc2 = StandardScaler().fit(Xp2)
Z2 = sc2.transform(Xp2)

# SVC(kernel='linear') dipakai untuk plot karena menyediakan support_vectors_
svc2 = SVC(kernel='linear', C=C_TERPILIH, class_weight='balanced',
           cache_size=500, random_state=RANDOM_STATE).fit(Z2, yp_arr)
w2 = svc2.coef_.ravel(); b2 = float(svc2.intercept_[0])
margin2 = 2.0 / float(np.linalg.norm(w2))

def gambar_kontur_hyperplane(ax, model, Z, y, judul, label_x, label_y,
                             tampilkan_sv=True, isi_wilayah=True):
    """Gambar scatter + kontur decision_function pada level -1, 0, +1."""
    m1, m2 = Z[:, 0].min() - 0.6, Z[:, 0].max() + 0.6
    n1, n2 = Z[:, 1].min() - 0.6, Z[:, 1].max() + 0.6
    xx, yy = np.meshgrid(np.linspace(m1, m2, 320), np.linspace(n1, n2, 320))
    D = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    if isi_wilayah:
        ax.contourf(xx, yy, D, levels=[D.min(), 0, D.max()],
                    colors=['#eaf6ff', '#ffecec'], alpha=0.85)
    ax.scatter(Z[y == 0, 0], Z[y == 0, 1], s=12, c='#3498db', alpha=0.35,
               edgecolors='none', label='Sehat (kelas 0)')
    ax.scatter(Z[y == 1, 0], Z[y == 1, 1], s=16, c='#e74c3c', alpha=0.55,
               edgecolors='none', label='Diabetes (kelas 1)')
    ax.contour(xx, yy, D, levels=[0], colors=['#2c3e50'], linewidths=2.6)
    ax.contour(xx, yy, D, levels=[-1, 1], colors=['#7f8c8d'], linewidths=1.6,
               linestyles='dashed')
    if tampilkan_sv and hasattr(model, 'support_vectors_'):
        sv = model.support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1], s=42, facecolors='none',
                   edgecolors=WARNA_AKSEN, linewidths=0.9, alpha=0.65,
                   label=f'Support vector (n={len(sv):,})')
    ax.set_xlabel(label_x); ax.set_ylabel(label_y); ax.set_title(judul)
    ax.set_xlim(m1, m2); ax.set_ylim(n1, n2)
    return ax

fig, ax = plt.subplots(figsize=(11.5, 8))
gambar_kontur_hyperplane(
    ax, svc2, Z2, yp_arr,
    judul=(f'Eksperimen 4a - Hyperplane SVM Linear (C={C_TERPILIH}) pada Bidang HbA1c x Glukosa\n'
           f'Garis tebal = hyperplane w.x + b = 0   |   garis putus-putus = batas margin f(x) = -1 dan +1'),
    label_x=LABEL_PLOT[0], label_y=LABEL_PLOT[1])
ax.legend(loc='upper left', fontsize=9.5, framealpha=0.92)
ax.text(0.985, 0.03,
        f'||w|| = {np.linalg.norm(w2):.3f}\nmargin = 2/||w|| = {margin2:.3f}\n'
        f'n_SV = {len(svc2.support_vectors_):,} / {len(Z2):,} '
        f'({len(svc2.support_vectors_)/len(Z2)*100:.1f}%)',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.45', fc='white', ec=WARNA_AKSEN, alpha=0.95))
plt.tight_layout()
simpan_gambar('svm_hyperplane_2fitur')
plt.show()

garis('KESIMPULAN EKSPERIMEN 4a')
print(f'Persamaan hyperplane pada bidang 2 fitur (data terstandardisasi):')
print(f'  f(x) = ({w2[0]:+.4f}) * HbA1c + ({w2[1]:+.4f}) * Glukosa + ({b2:+.4f})')
print(f'  ||w|| = {np.linalg.norm(w2):.4f}   ->   lebar margin = {margin2:.4f}')
print(f'  Support vector: {len(svc2.support_vectors_):,} dari {len(Z2):,} sampel '
      f'({len(svc2.support_vectors_)/len(Z2)*100:.1f}%)')
print()
print('Terlihat kedua kelas terpisah oleh pola bertingkat (HbA1c dan glukosa memiliki nilai')
print('ambang klinis), dan pemisah tersebut dapat dijelaskan dengan satu garis lurus - inilah')
print('bukti visual bahwa hyperplane linear sudah memadai untuk struktur data ini.')

In [ ]:
# ============================================================
# BAGIAN 4 | CELL 15: Eksperimen 4b - Hyperplane pada Proyeksi PCA 2 Komponen (5 fitur)
# ============================================================
# PCA di sini murni untuk VISUALISASI: seluruh 5 fitur diringkas ke 2 komponen agar
# hyperplane dapat digambar. Model produksi tetap bekerja pada 5 fitur asli.
sc_pca = StandardScaler().fit(Xp_df[SELECTED_FEATURES].values)
Zp = sc_pca.transform(Xp_df[SELECTED_FEATURES].values)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
Zpca = pca.fit_transform(Zp)
var_jelas = pca.explained_variance_ratio_

svc_pca = SVC(kernel='linear', C=C_TERPILIH, class_weight='balanced',
              cache_size=500, random_state=RANDOM_STATE).fit(Zpca, yp_arr)
w_pca = svc_pca.coef_.ravel()
margin_pca = 2.0 / float(np.linalg.norm(w_pca))

fig, ax = plt.subplots(figsize=(11.5, 8))
gambar_kontur_hyperplane(
    ax, svc_pca, Zpca, yp_arr,
    judul=(f'Eksperimen 4b - Hyperplane SVM Linear pada Ruang PCA 2 Komponen (dari 5 fitur)\n'
           f'Total varians yang dijelaskan: {var_jelas.sum()*100:.1f}%'),
    label_x=f'Komponen Utama 1 ({var_jelas[0]*100:.1f}% varians)',
    label_y=f'Komponen Utama 2 ({var_jelas[1]*100:.1f}% varians)')
ax.legend(loc='upper left', fontsize=9.5, framealpha=0.92)
ax.text(0.985, 0.03,
        f'||w|| = {np.linalg.norm(w_pca):.3f}\nmargin = {margin_pca:.3f}\n'
        f'n_SV = {len(svc_pca.support_vectors_):,}',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.45', fc='white', ec=WARNA_AKSEN, alpha=0.95))
plt.tight_layout()
simpan_gambar('svm_hyperplane_pca')
plt.show()

muatan = pd.DataFrame(pca.components_.T, index=FEATURE_LABELS, columns=['PC1', 'PC2']).round(4)
garis('KESIMPULAN EKSPERIMEN 4b')
print(f'Varians dijelaskan: PC1={var_jelas[0]*100:.1f}%, PC2={var_jelas[1]*100:.1f}%, '
      f'total={var_jelas.sum()*100:.1f}%')
print(f'Margin pada ruang PCA: {margin_pca:.4f}  |  n_SV = {len(svc_pca.support_vectors_):,}')
print('Muatan (loading) tiap fitur pada dua komponen utama:')
display(muatan)
print('Meskipun hanya sebagian varians yang tertangkap dua komponen, pemisahan kelas tetap')
print('mengikuti arah lurus - konsisten dengan temuan pada bidang 2 fitur.')

In [ ]:
# ============================================================
# BAGIAN 4 | CELL 16: Eksperimen 4c - Perbandingan Visual Batas Keputusan Linear vs RBF
# ============================================================
svc_rbf2 = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced',
               cache_size=500, random_state=RANDOM_STATE).fit(Z2, yp_arr)

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
gambar_kontur_hyperplane(
    axes[0], svc2, Z2, yp_arr,
    judul=f'(a) Kernel LINEAR (C={C_TERPILIH}) - batas keputusan berupa garis lurus',
    label_x=LABEL_PLOT[0], label_y=LABEL_PLOT[1])
gambar_kontur_hyperplane(
    axes[1], svc_rbf2, Z2, yp_arr,
    judul='(b) Kernel RBF (C=1.0, gamma=scale) - batas keputusan melengkung',
    label_x=LABEL_PLOT[0], label_y=LABEL_PLOT[1])
axes[0].legend(loc='upper left', fontsize=9)
axes[1].legend(loc='upper left', fontsize=9)

# Perbandingan kuantitatif pada bidang yang sama
skor_lin = svc2.decision_function(Z2)
skor_rbf = svc_rbf2.decision_function(Z2)
auc_lin2 = roc_auc_score(yp_arr, skor_lin)
auc_rbf2 = roc_auc_score(yp_arr, skor_rbf)
axes[0].text(0.985, 0.03, f'AUC (bidang ini) = {auc_lin2:.4f}\nn_SV = {len(svc2.support_vectors_):,}',
             transform=axes[0].transAxes, ha='right', va='bottom', fontsize=10,
             bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#2c3e50', alpha=0.95))
axes[1].text(0.985, 0.03, f'AUC (bidang ini) = {auc_rbf2:.4f}\nn_SV = {len(svc_rbf2.support_vectors_):,}',
             transform=axes[1].transAxes, ha='right', va='bottom', fontsize=10,
             bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#2c3e50', alpha=0.95))

plt.suptitle('Eksperimen 4c - Bentuk Batas Keputusan: Linear vs RBF pada Bidang yang Sama',
             fontsize=14, y=1.0)
plt.tight_layout()
simpan_gambar('svm_boundary_linear_vs_rbf')
plt.show()

garis('KESIMPULAN EKSPERIMEN 4c')
print(f'AUC pada bidang 2 fitur : linear={auc_lin2:.4f}  vs  RBF={auc_rbf2:.4f}  '
      f'(selisih {auc_rbf2 - auc_lin2:+.4f})')
print(f'Jumlah support vector   : linear={len(svc2.support_vectors_):,}  vs  '
      f'RBF={len(svc_rbf2.support_vectors_):,}')
print()
print('Kelengkungan yang dihasilkan RBF hanya mengikuti kerapatan lokal titik-titik di sekitar')
print('ambang klinis, bukan menangkap struktur non-linear yang benar-benar baru. Konsekuensinya')
print('bentuk yang lebih rumit itu tidak berubah menjadi keuntungan performa, sementara model')
print('kehilangan sifat yang penting untuk skripsi ini: batas keputusan yang bisa dituliskan')
print('sebagai satu persamaan dan dijelaskan kepada tenaga medis.')

---

# EKSPERIMEN 5 - Interpretasi Vektor Bobot w

Hyperplane linear sepenuhnya ditentukan oleh vektor bobot $w$ dan bias $b$. Karena seluruh fitur
sudah **distandardisasi** (mean 0, simpangan baku 1), besaran $|w_j|$ dapat dibandingkan langsung
antar-fitur: ia menyatakan **seberapa jauh** fungsi keputusan bergeser bila fitur ke-$j$ naik satu
simpangan baku. Tanda $w_j$ menyatakan **arah pengaruh** (positif = menaikkan risiko diabetes).

**Catatan implementasi:** di pipeline produksi, `LinearSVC` dibungkus `CalibratedClassifierCV`,
sehingga `coef_` tidak lagi tersedia di permukaan objek - kalibrasi menyimpan beberapa salinan
estimator hasil cross-validation di `calibrated_classifiers_`. Untuk interpretasi dipakai
**satu `LinearSVC` yang dilatih terpisah pada data scaled + SMOTE yang sama**, karena inilah
hyperplane tunggal yang tidak ambigu (rata-rata beberapa fold justru mengaburkan makna). Bobot dari
estimator di dalam objek kalibrasi tetap dibaca sebagai **pemeriksaan silang** bila API-nya tersedia.

Urutan kepentingan dari $|w|$ juga dibandingkan dengan **permutation importance** (`scoring='recall'`)
yang bersifat model-agnostik, sebagai validasi silang bahwa arah dan urutan pengaruh konsisten.


In [ ]:
# ============================================================
# BAGIAN 4 | CELL 17: Eksperimen 5 - Vektor Bobot w dan Permutation Importance
# ============================================================
garis('EKSPERIMEN 5: INTERPRETASI VEKTOR BOBOT HYPERPLANE')
print(f'Data     : {len(Xf_tr):,} latih (DATA PENUH - LinearSVC berbiaya mendekati linear)')
print(f'Model    : LinearSVC(C={C_TERPILIH}) pada data scaled + SMOTE')
print('Estimasi : 1-3 menit')
print('-' * 70)

# (1) Hyperplane tunggal untuk interpretasi
sc_w, Xw_res, yw_res = siapkan_scale_smote(Xf_tr, yf_tr)
lin_w = LinearSVC(C=C_TERPILIH, max_iter=5000, class_weight='balanced',
                  dual=False, random_state=RANDOM_STATE).fit(Xw_res, yw_res)
w_vec = lin_w.coef_.ravel()
b_val = float(lin_w.intercept_[0])
norma_w_final = float(np.linalg.norm(w_vec))
margin_final  = 2.0 / norma_w_final
print(f'||w|| = {norma_w_final:.4f}   margin = 2/||w|| = {margin_final:.4f}   b = {b_val:+.4f}')

# (2) Pemeriksaan silang: bobot dari estimator di dalam CalibratedClassifierCV
def ambil_coef_dari_kalibrasi(pipe):
    """CalibratedClassifierCV menyimpan estimator hasil tiap fold. Nama atributnya
    berbeda antar versi scikit-learn ('estimator' >= 1.2, 'base_estimator' < 1.2),
    sehingga diakses defensif. Bila gagal, interpretasi tetap memakai lin_w."""
    try:
        cal = pipe.named_steps['clf']
        kumpulan = []
        for sub in cal.calibrated_classifiers_:
            for atr in ('estimator', 'base_estimator'):
                est = getattr(sub, atr, None)
                if est is not None and hasattr(est, 'coef_'):
                    kumpulan.append(est.coef_.ravel())
                    break
        if kumpulan:
            return np.mean(np.vstack(kumpulan), axis=0)
    except Exception as e:
        print(f'  (info) bobot dari objek kalibrasi tidak dapat dibaca: {type(e).__name__}')
    return None

pipe_kal = buat_pipeline_svm(kernel='linear', C=C_TERPILIH, max_iter=5000)
pipe_kal.fit(Xf_tr, yf_tr)
w_kal = ambil_coef_dari_kalibrasi(pipe_kal)
if w_kal is not None:
    korelasi_w = float(np.corrcoef(w_vec, w_kal)[0, 1])
    print(f'Pemeriksaan silang: korelasi bobot LinearSVC terpisah vs rata-rata estimator '
          f'di dalam kalibrasi = {korelasi_w:.4f}')
else:
    korelasi_w = None
    print('Pemeriksaan silang bobot kalibrasi dilewati (API versi scikit-learn berbeda).')

# (3) Tabel bobot
abs_w = np.abs(w_vec)
tabel_bobot_w = pd.DataFrame({
    'fitur'            : FEATURE_LABELS,
    'nama_kolom'       : SELECTED_FEATURES,
    'koefisien_w'      : w_vec,
    'abs_w'            : abs_w,
    'abs_w_ternormalisasi': abs_w / abs_w.sum(),
    'arah_pengaruh'    : ['Menaikkan risiko diabetes' if v > 0 else 'Menurunkan risiko diabetes'
                          for v in w_vec],
}).sort_values('abs_w', ascending=False).reset_index(drop=True)
tabel_bobot_w['peringkat_w'] = np.arange(1, len(tabel_bobot_w) + 1)

# (4) Permutation importance (model-agnostik, scoring='recall')
print(f'\nMenghitung permutation importance pada {N_SUBSAMPLE_PERM:,} sampel uji ...')
Xperm, yperm = ambil_subsample(Xf_te, yf_te, N_SUBSAMPLE_PERM)
pipe_perm = pipeline_linear_mentah(C_TERPILIH).fit(Xf_tr, yf_tr)
perm = permutation_importance(pipe_perm, Xperm, yperm, scoring='recall',
                              n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
tabel_perm = pd.DataFrame({
    'nama_kolom': SELECTED_FEATURES,
    'perm_mean' : perm.importances_mean,
    'perm_std'  : perm.importances_std,
}).sort_values('perm_mean', ascending=False).reset_index(drop=True)
tabel_perm['peringkat_perm'] = np.arange(1, len(tabel_perm) + 1)

tabel_bobot_w = tabel_bobot_w.merge(tabel_perm, on='nama_kolom', how='left')
tabel_bobot_w['selisih_peringkat'] = (tabel_bobot_w['peringkat_w'] -
                                      tabel_bobot_w['peringkat_perm']).abs()
tabel_bobot_w = tabel_bobot_w.round(6)
simpan_tabel(tabel_bobot_w, 'tabel_bobot_w')

# (5) Grafik barh koefisien + perbandingan peringkat
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
urut = tabel_bobot_w.sort_values('koefisien_w')
warna_bar = ['#e74c3c' if v > 0 else '#3498db' for v in urut['koefisien_w']]
axes[0].barh(urut['fitur'], urut['koefisien_w'], color=warna_bar, edgecolor='white')
axes[0].axvline(0, color='#2c3e50', lw=1.2)
axes[0].set_xlabel('Koefisien w (pada fitur terstandardisasi)')
axes[0].set_title('(a) Vektor Bobot Hyperplane\nmerah = menaikkan risiko, biru = menurunkan risiko')
for i, (v, f_) in enumerate(zip(urut['koefisien_w'], urut['fitur'])):
    axes[0].text(v + (0.02 if v >= 0 else -0.02), i, f'{v:+.4f}',
                 va='center', ha='left' if v >= 0 else 'right', fontsize=9.5)

urut_p = tabel_bobot_w.sort_values('perm_mean')
axes[1].barh(urut_p['fitur'], urut_p['perm_mean'],
             xerr=urut_p['perm_std'], color='#8e44ad', edgecolor='white',
             error_kw=dict(ecolor='#5b2c6f', lw=1.2, capsize=4))
axes[1].set_xlabel('Penurunan recall saat fitur diacak')
axes[1].set_title('(b) Permutation Importance (scoring = recall)\nvalidasi silang model-agnostik')

plt.suptitle('Eksperimen 5 - Interpretasi Hyperplane: Bobot w dan Kepentingan Fitur',
             fontsize=14, y=1.02)
plt.tight_layout()
simpan_gambar('svm_bobot_w')
plt.show()

garis('KESIMPULAN EKSPERIMEN 5')
print('Persamaan hyperplane pada 5 fitur terstandardisasi:')
suku = '  f(x) = ' + ' '.join(
    [f'({w_vec[i]:+.4f})*{SELECTED_FEATURES[i]}' for i in range(len(SELECTED_FEATURES))]) + f' ({b_val:+.4f})'
print(suku)
print(f'  Prediksi positif bila f(x) >= threshold operasional.')
print()
print('Urutan kepentingan menurut |w| vs permutation importance:')
display(tabel_bobot_w[['fitur', 'koefisien_w', 'abs_w_ternormalisasi', 'peringkat_w',
                       'perm_mean', 'peringkat_perm', 'selisih_peringkat', 'arah_pengaruh']])
n_sama = int((tabel_bobot_w['selisih_peringkat'] == 0).sum())
print(f'Peringkat identik pada {n_sama} dari {len(tabel_bobot_w)} fitur; '
      f'rata-rata pergeseran peringkat = {tabel_bobot_w["selisih_peringkat"].mean():.2f}.')
print('Kesesuaian ini menunjukkan bobot hyperplane bukan artefak numerik, melainkan benar-benar')
print('mencerminkan kontribusi tiap fitur - sekaligus alasan tambahan memilih kernel linear:')
print('hyperplane-nya dapat dibaca sebagai pernyataan klinis yang jelas.')

---

# EKSPERIMEN 6 - Uji Signifikansi Statistik: Linear vs RBF

Perbedaan angka antar-kernel pada Eksperimen 1 dan 2 belum tentu berarti secara statistik. Karena
itu kedua kernel dievaluasi dengan **StratifiedKFold 5-fold** pada data yang sama (fold yang sama
persis untuk kedua model, sehingga sampelnya **berpasangan**), lalu diuji dengan:

- **Paired t-test** (`scipy.stats.ttest_rel`) - uji parametrik untuk selisih rata-rata.
- **Wilcoxon signed-rank** (`scipy.stats.wilcoxon`) - versi non-parametrik, tidak mengasumsikan
  normalitas (penting karena n fold hanya 5).

Bila **tidak** ada perbedaan signifikan (p >= 0,05), maka pemilihan kernel linear dijustifikasi
dengan prinsip **parsimony (Occam's razor)**: di antara dua model yang performanya tidak dapat
dibedakan secara statistik, dipilih model yang lebih sederhana, lebih cepat, dan lebih dapat
dijelaskan.


In [ ]:
# ============================================================
# BAGIAN 4 | CELL 18: Eksperimen 6 - Uji Signifikansi Linear vs RBF (5-fold berpasangan)
# ============================================================
garis('EKSPERIMEN 6: UJI SIGNIFIKANSI LINEAR VS RBF')
print(f'Data     : {len(X_sv):,} sampel (subsample stratified, dibatasi biaya SVC-RBF)')
print('Skema    : StratifiedKFold(5) - fold identik untuk kedua kernel (berpasangan)')
print('Estimasi : 2-5 menit')
print('-' * 70)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_uji = X_sv.reset_index(drop=True)
y_uji = y_sv.reset_index(drop=True)

skor_fold = {'linear': {'auc': [], 'recall': [], 'f1': [], 'waktu': []},
             'rbf'   : {'auc': [], 'recall': [], 'f1': [], 'waktu': []}}

for f_i, (idx_tr, idx_te) in enumerate(skf.split(X_uji, y_uji), 1):
    Xtr, Xte = X_uji.iloc[idx_tr], X_uji.iloc[idx_te]
    ytr, yte = y_uji.iloc[idx_tr], y_uji.iloc[idx_te]
    pesan = f'Fold {f_i}/5: '
    for nama, pipa in [('linear', pipeline_linear_mentah(C_TERPILIH)),
                       ('rbf',    pipeline_svc_mentah('rbf', C=1.0, gamma='scale'))]:
        try:
            r = evaluasi_decision(pipa, Xtr, ytr, Xte, yte)
            skor_fold[nama]['auc'].append(r['roc_auc'])
            skor_fold[nama]['recall'].append(r['recall'])
            skor_fold[nama]['f1'].append(r['f1'])
            skor_fold[nama]['waktu'].append(r['waktu_latih_s'])
            pesan += f'{nama} AUC={r["roc_auc"]:.4f} ({r["waktu_latih_s"]:.1f}s)  '
        except Exception as e:
            skor_fold[nama]['auc'].append(np.nan)
            skor_fold[nama]['recall'].append(np.nan)
            skor_fold[nama]['f1'].append(np.nan)
            skor_fold[nama]['waktu'].append(np.nan)
            pesan += f'{nama} GAGAL({type(e).__name__})  '
    print(pesan)

def uji_berpasangan(a, b):
    """Paired t-test + Wilcoxon signed-rank. Wilcoxon bisa gagal bila seluruh
    selisih nol atau n terlalu kecil, sehingga dibungkus try/except."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    valid = ~(np.isnan(a) | np.isnan(b))
    a, b = a[valid], b[valid]
    hasil = {'n_fold': int(len(a)),
             'mean_linear': float(np.mean(a)) if len(a) else np.nan,
             'mean_rbf'   : float(np.mean(b)) if len(b) else np.nan,
             'std_linear' : float(np.std(a, ddof=1)) if len(a) > 1 else np.nan,
             'std_rbf'    : float(np.std(b, ddof=1)) if len(b) > 1 else np.nan}
    hasil['selisih_linear_minus_rbf'] = hasil['mean_linear'] - hasil['mean_rbf']
    try:
        t_stat, p_t = stats.ttest_rel(a, b)
        hasil['t_stat'], hasil['p_ttest'] = float(t_stat), float(p_t)
    except Exception:
        hasil['t_stat'], hasil['p_ttest'] = np.nan, np.nan
    try:
        w_stat, p_w = stats.wilcoxon(a, b, zero_method='zsplit')
        hasil['w_stat'], hasil['p_wilcoxon'] = float(w_stat), float(p_w)
    except Exception:
        hasil['w_stat'], hasil['p_wilcoxon'] = np.nan, np.nan
    hasil['signifikan_alpha_0.05'] = bool(hasil['p_ttest'] < 0.05) if not np.isnan(hasil['p_ttest']) else False
    return hasil

baris_uji = []
for metrik in ['auc', 'recall', 'f1']:
    h = uji_berpasangan(skor_fold['linear'][metrik], skor_fold['rbf'][metrik])
    h = {'metrik': metrik.upper(), **h}
    baris_uji.append(h)

tabel_uji_linear_vs_rbf = pd.DataFrame(baris_uji).round(6)
simpan_tabel(tabel_uji_linear_vs_rbf, 'tabel_uji_linear_vs_rbf')

# Tabel skor per fold (transparansi data mentah uji)
tabel_fold = pd.DataFrame({
    'fold'          : np.arange(1, 6),
    'auc_linear'    : skor_fold['linear']['auc'],
    'auc_rbf'       : skor_fold['rbf']['auc'],
    'recall_linear' : skor_fold['linear']['recall'],
    'recall_rbf'    : skor_fold['rbf']['recall'],
    'waktu_linear_s': skor_fold['linear']['waktu'],
    'waktu_rbf_s'   : skor_fold['rbf']['waktu'],
}).round(6)
simpan_tabel(tabel_fold, 'tabel_fold_linear_vs_rbf')

baris_auc = tabel_uji_linear_vs_rbf[tabel_uji_linear_vs_rbf['metrik'] == 'AUC'].iloc[0]
selisih_auc = float(baris_auc['selisih_linear_minus_rbf'])
p_ttest_auc = float(baris_auc['p_ttest'])
p_wil_auc   = float(baris_auc['p_wilcoxon'])
waktu_lin   = float(np.nanmean(skor_fold['linear']['waktu']))
waktu_rbf   = float(np.nanmean(skor_fold['rbf']['waktu']))
rasio_waktu = waktu_rbf / max(waktu_lin, 1e-9)
SIGNIFIKAN_RBF = bool(p_ttest_auc < 0.05) if not np.isnan(p_ttest_auc) else False

garis('KESIMPULAN EKSPERIMEN 6')
print(f'AUC rata-rata linear : {baris_auc["mean_linear"]:.4f} (SD {baris_auc["std_linear"]:.4f})')
print(f'AUC rata-rata RBF    : {baris_auc["mean_rbf"]:.4f} (SD {baris_auc["std_rbf"]:.4f})')
print(f'Selisih (linear-RBF) : {selisih_auc:+.4f}')
print(f'Paired t-test        : t={baris_auc["t_stat"]:.4f}, p={p_ttest_auc:.4f}')
print(f'Wilcoxon signed-rank : W={baris_auc["w_stat"]:.4f}, p={p_wil_auc:.4f}')
print(f'Waktu latih rata-rata: linear={waktu_lin:.2f}s vs RBF={waktu_rbf:.2f}s '
      f'({rasio_waktu:.1f}x lebih lambat)')
print()
if SIGNIFIKAN_RBF and selisih_auc < 0:
    print('Hasil: RBF unggul secara SIGNIFIKAN. Namun keunggulan itu perlu ditimbang terhadap')
    print(f'biaya komputasi {rasio_waktu:.1f}x dan hilangnya interpretabilitas hyperplane.')
else:
    print('Hasil: TIDAK ADA perbedaan signifikan pada alpha = 0,05.')
    print('Justifikasi memilih hyperplane LINEAR (prinsip parsimony / Occam s razor):')
    print(f'  1. Performa: selisih AUC hanya {abs(selisih_auc):.4f} dan p = {p_ttest_auc:.4f} '
          f'(>= 0,05) -> tidak dapat dibedakan secara statistik.')
    print(f'  2. Kecepatan: pelatihan {rasio_waktu:.1f}x lebih cepat, dan inferensi linear hanya')
    print('     memerlukan satu perkalian titik w.x + b (bukan evaluasi kernel terhadap ribuan SV).')
    print('  3. Skalabilitas: LinearSVC dapat dilatih pada seluruh 96.146 baris, sedangkan')
    print('     SVC-RBF terpaksa memakai subsample.')
    print('  4. Interpretabilitas: bobot w dapat dibaca sebagai kontribusi klinis tiap fitur')
    print('     (Eksperimen 5) - syarat penting untuk sistem pendukung keputusan medis.')

---

# EKSPERIMEN 7 - Kalibrasi Probabilitas

SVM secara alami menghasilkan **skor jarak** ke hyperplane (`decision_function`), bukan probabilitas.
Padahal sistem DiaPredict mengambil keputusan berdasarkan **ambang probabilitas** dan menampilkan
angka risiko kepada pengguna. Karena itu keluaran SVM **wajib dikalibrasi**, dan pilihan metode
kalibrasi perlu dibuktikan - bukan sekadar diwarisi dari notebook sebelumnya.

Tiga varian dibandingkan dengan **Brier score** (semakin kecil semakin baik; mengukur ketepatan
probabilitas, bukan sekadar urutan) dan **reliability diagram**:

1. `CalibratedClassifierCV(method='sigmoid')` - Platt scaling (dipakai V2).
2. `CalibratedClassifierCV(method='isotonic')` - regresi isotonik, lebih fleksibel tetapi lebih
   rawan overfitting pada data kecil.
3. **Tanpa kalibrasi** - `decision_function` yang dinormalisasi min-max ke rentang [0,1] sebagai
   pengganti probabilitas (praktik yang sering dilakukan tetapi tidak memiliki dasar probabilistik).


In [ ]:
# ============================================================
# BAGIAN 4 | CELL 19: Eksperimen 7 - Perbandingan Metode Kalibrasi Probabilitas
# ============================================================
garis('EKSPERIMEN 7: KALIBRASI PROBABILITAS SVM')
print(f'Data     : {len(Xk_tr):,} latih / {len(Xk_te):,} uji')
print(f'Model dasar: LinearSVC(C={C_TERPILIH})')
print('Estimasi : 1-2 menit')
print('-' * 70)

hasil_kalibrasi = []
kurva_kalibrasi = {}

# (1) & (2) Kalibrasi sigmoid dan isotonic
for metode in ['sigmoid', 'isotonic']:
    try:
        t0 = time.time()
        pipa = buat_pipeline_svm(kernel='linear', C=C_TERPILIH, max_iter=5000, kalibrasi=metode)
        pipa.fit(Xk_tr, yk_tr)
        proba = pipa.predict_proba(Xk_te)[:, 1]
        thr = threshold_youden(yk_te, proba)
        met = hitung_metrik(yk_te, (proba >= thr).astype(int), proba)
        frac_pos, mean_pred = calibration_curve(yk_te, proba, n_bins=10, strategy='quantile')
        kurva_kalibrasi[metode] = (mean_pred, frac_pos)
        hasil_kalibrasi.append({
            'metode': f'CalibratedClassifierCV ({metode})', 'brier': met['brier'],
            'roc_auc': met['roc_auc'], 'recall': met['recall'], 'precision': met['precision'],
            'f1': met['f1'], 'threshold_youden': thr, 'waktu_latih_s': time.time() - t0,
            'probabilitas_valid': 'Ya', 'status': 'OK'})
        print(f'{metode:<10} Brier={met["brier"]:.5f}  AUC={met["roc_auc"]:.4f}  '
              f'recall={met["recall"]:.4f}  ({time.time()-t0:.1f}s)')
    except Exception as e:
        hasil_kalibrasi.append({'metode': f'CalibratedClassifierCV ({metode})', 'brier': np.nan,
                                'roc_auc': np.nan, 'recall': np.nan, 'precision': np.nan,
                                'f1': np.nan, 'threshold_youden': np.nan, 'waktu_latih_s': np.nan,
                                'probabilitas_valid': 'Ya', 'status': f'GAGAL: {type(e).__name__}'})
        print(f'{metode:<10} GAGAL ({type(e).__name__}) -> dilewati')

# (3) Tanpa kalibrasi: decision_function dinormalisasi min-max ke [0,1]
try:
    t0 = time.time()
    pipa_mentah = pipeline_linear_mentah(C_TERPILIH).fit(Xk_tr, yk_tr)
    skor_mentah = pipa_mentah.decision_function(Xk_te)
    proba_semu = (skor_mentah - skor_mentah.min()) / (skor_mentah.max() - skor_mentah.min() + 1e-12)
    thr = threshold_youden(yk_te, proba_semu)
    met = hitung_metrik(yk_te, (proba_semu >= thr).astype(int), proba_semu)
    frac_pos, mean_pred = calibration_curve(yk_te, proba_semu, n_bins=10, strategy='quantile')
    kurva_kalibrasi['tanpa kalibrasi'] = (mean_pred, frac_pos)
    hasil_kalibrasi.append({
        'metode': 'Tanpa kalibrasi (decision_function min-max)', 'brier': met['brier'],
        'roc_auc': met['roc_auc'], 'recall': met['recall'], 'precision': met['precision'],
        'f1': met['f1'], 'threshold_youden': thr, 'waktu_latih_s': time.time() - t0,
        'probabilitas_valid': 'Tidak', 'status': 'OK'})
    print(f'{"tanpa":<10} Brier={met["brier"]:.5f}  AUC={met["roc_auc"]:.4f}  '
          f'recall={met["recall"]:.4f}  ({time.time()-t0:.1f}s)')
except Exception as e:
    print(f'Varian tanpa kalibrasi GAGAL ({type(e).__name__}) -> dilewati')

tabel_kalibrasi_svm = pd.DataFrame(hasil_kalibrasi).round(6)
simpan_tabel(tabel_kalibrasi_svm, 'tabel_kalibrasi_svm')

# --- Reliability diagram + distribusi probabilitas ---
fig, axes = plt.subplots(1, 2, figsize=(17, 6.5))
warna_kal = {'sigmoid': WARNA_MODEL['SVM (Linear)'], 'isotonic': '#9b59b6',
             'tanpa kalibrasi': '#e74c3c'}
penanda = {'sigmoid': 'o', 'isotonic': 's', 'tanpa kalibrasi': '^'}

axes[0].plot([0, 1], [0, 1], 'k:', lw=1.8, label='Kalibrasi sempurna')
for nama, (mp, fp) in kurva_kalibrasi.items():
    axes[0].plot(mp, fp, marker=penanda[nama], lw=2.2, ms=7,
                 color=warna_kal[nama], label=nama)
axes[0].set_xlabel('Probabilitas rata-rata yang diprediksi')
axes[0].set_ylabel('Proporsi kasus positif sebenarnya')
axes[0].set_title('(a) Reliability Diagram - semakin dekat garis diagonal semakin baik')
axes[0].legend(loc='upper left', fontsize=10)
axes[0].set_xlim(-0.02, 1.02); axes[0].set_ylim(-0.02, 1.02)

ok_kal = tabel_kalibrasi_svm[tabel_kalibrasi_svm['status'] == 'OK']
warna_brier = [WARNA_MODEL['SVM (Linear)'] if 'sigmoid' in m else
               ('#9b59b6' if 'isotonic' in m else '#e74c3c') for m in ok_kal['metode']]
axes[1].bar(range(len(ok_kal)), ok_kal['brier'], color=warna_brier, edgecolor='white')
label_kal = ['sigmoid\n(Platt)' if 'sigmoid' in m else
             ('isotonic' if 'isotonic' in m else 'tanpa\nkalibrasi') for m in ok_kal['metode']]
axes[1].set_xticks(range(len(ok_kal)))
axes[1].set_xticklabels(label_kal, fontsize=10)
axes[1].set_ylabel('Brier score (semakin kecil semakin baik)')
axes[1].set_title('(b) Brier Score per Metode Kalibrasi')
for i, v in enumerate(ok_kal['brier']):
    axes[1].text(i, v * 1.02, f'{v:.5f}', ha='center', fontsize=10)

plt.suptitle('Eksperimen 7 - Justifikasi Kalibrasi Probabilitas SVM', fontsize=14, y=1.02)
plt.tight_layout()
simpan_gambar('svm_kalibrasi')
plt.show()

garis('KESIMPULAN EKSPERIMEN 7')
if len(ok_kal):
    terbaik_kal = ok_kal.loc[ok_kal['brier'].idxmin()]
    METODE_KALIBRASI_TERPILIH = str(terbaik_kal['metode'])
    BRIER_TERPILIH = float(terbaik_kal['brier'])
    print(f'Brier terendah : {METODE_KALIBRASI_TERPILIH} (Brier={BRIER_TERPILIH:.5f})')
    baris_tanpa = ok_kal[ok_kal['probabilitas_valid'] == 'Tidak']
    if len(baris_tanpa):
        b_tanpa = float(baris_tanpa.iloc[0]['brier'])
        print(f'Tanpa kalibrasi: Brier={b_tanpa:.5f} '
              f'({(b_tanpa - BRIER_TERPILIH) / max(BRIER_TERPILIH, 1e-12) * 100:+.1f}% lebih buruk)')
    print(f'ROC-AUC seluruh varian praktis sama (rentang '
          f'{ok_kal["roc_auc"].max() - ok_kal["roc_auc"].min():.5f}) - ini wajar karena kalibrasi')
    print('adalah transformasi monoton: ia memperbaiki NILAI probabilitas, bukan URUTAN peringkat.')
    print()
    print('Implikasi untuk sistem: karena DiaPredict menampilkan angka risiko dan memakai ambang')
    print('probabilitas, kalibrasi bukan pelengkap melainkan syarat agar angka yang ditampilkan')
    print('benar-benar bermakna (mis. "risiko 70%" memang terjadi pada sekitar 70% kasus serupa).')
else:
    METODE_KALIBRASI_TERPILIH, BRIER_TERPILIH = 'sigmoid', float('nan')
    print('Seluruh varian kalibrasi gagal - lihat kolom status pada tabel.')

---

# KESIMPULAN - Argumen Terstruktur Pemilihan Hyperplane

Cell berikut merangkum seluruh bukti menjadi satu argumen berurutan dan menyimpannya ke
`hasil_svm_hyperplane.json` (dibaca notebook `06` dan website).


In [ ]:
# ============================================================
# BAGIAN 4 | CELL 20: Kesimpulan Terstruktur + Penyimpanan JSON hasil_svm_hyperplane
# ============================================================
ok_kern = tabel_perbandingan_kernel[tabel_perbandingan_kernel['status'] == 'OK'].copy()

def ambil_baris_kernel(nama_kernel, kolom_urut='roc_auc'):
    sub = ok_kern[ok_kern['kernel'] == nama_kernel]
    if len(sub) == 0:
        return None
    return sub.sort_values(kolom_urut, ascending=False).iloc[0]

b_lin  = ambil_baris_kernel('linear')
b_rbf  = ambil_baris_kernel('rbf')
b_poly = ok_kern[ok_kern['kernel'] == 'poly'].sort_values('roc_auc', ascending=False)
b_poly = b_poly.iloc[0] if len(b_poly) else None
b_sig  = ambil_baris_kernel('sigmoid')

def frasa_penolakan(baris, nama):
    if baris is None or b_lin is None:
        return f'- {nama}: tidak berhasil dilatih pada anggaran komputasi yang tersedia (lihat kolom status).'
    d_auc = baris['roc_auc'] - b_lin['roc_auc']
    d_rec = baris['recall'] - b_lin['recall']
    rasio = baris['waktu_latih_s'] / max(b_lin['waktu_latih_s'], 1e-9)
    return (f'- {nama}: AUC {baris["roc_auc"]:.4f} ({d_auc:+.4f} terhadap linear), '
            f'recall {baris["recall"]:.4f} ({d_rec:+.4f}), '
            f'waktu latih {baris["waktu_latih_s"]:.1f}s ({rasio:.1f}x kernel linear).')

garis('ARGUMEN TERSTRUKTUR: KENAPA HYPERPLANE INI YANG DIPILIH')
print('[1] KERNEL TERPILIH: LINEAR')
print(f'    Hyperplane dicari di ruang fitur asli (5 fitur terstandardisasi), bukan di ruang')
print(f'    berdimensi lebih tinggi. Bukti dari Eksperimen 1, 2, 4, dan 6:')
print(f'    - Kernel linear terbaik : AUC={b_lin["roc_auc"]:.4f}, recall={b_lin["recall"]:.4f}, '
      f'latih={b_lin["waktu_latih_s"]:.1f}s')
print(f'    - Uji 5-fold berpasangan: selisih AUC linear-RBF = {selisih_auc:+.4f}, '
      f'p(t-test)={p_ttest_auc:.4f}, p(Wilcoxon)={p_wil_auc:.4f}')
print(f'    - Kesimpulan uji        : '
      f'{"perbedaan SIGNIFIKAN" if SIGNIFIKAN_RBF else "TIDAK signifikan pada alpha=0,05"}')
print()
print('[2] ALASAN MENOLAK KERNEL LAIN (dengan angka):')
print(frasa_penolakan(b_rbf,  'RBF'))
print(frasa_penolakan(b_poly, 'Polinomial'))
print(frasa_penolakan(b_sig,  'Sigmoid'))
print(f'    Tambahan: SVC berkernel berkompleksitas O(n^2)-O(n^3) sehingga hanya dapat dilatih')
print(f'    pada subsample {len(Xk_tr):,} baris, sedangkan LinearSVC dilatih pada seluruh '
      f'{len(Xf_tr):,} baris.')
print()
print(f'[3] NILAI C TERPILIH: {C_TERPILIH}')
print(f'    Aturan pemilihan ditetapkan di muka: margin TERLEBAR di antara kandidat yang recall-nya')
print(f'    masih dalam {TOLERANSI_RECALL*100:.0f} poin persen dari recall terbaik ({recall_maks:.4f}).')
print(f'    - ||w||                 : {NORMA_W_TERPILIH:.4f}')
print(f'    - Lebar margin 2/||w||  : {MARGIN_TERPILIH:.4f}')
print(f'    - Jumlah support vector : {NSV_TERPILIH:,} ({RASIO_SV_TERPILIH*100:.1f}% dari sampel latih)')
print(f'    - Recall / Precision    : {float(baris_terpilih["recall"]):.4f} / '
      f'{float(baris_terpilih["precision"]):.4f}')
print(f'    - Konfigurasi V2 (C={PARAM_SVM_V2["C"]}) : '
      f'{"TERKONFIRMASI oleh eksperimen ini" if abs(C_TERPILIH - PARAM_SVM_V2["C"]) < 1e-9 else "diperbarui berdasarkan eksperimen ini"}')
print()
print('[4] HYPERPLANE FINAL PADA DATA PENUH (5 fitur terstandardisasi):')
print('    f(x) = ' + ' '.join([f'({w_vec[i]:+.4f})*{SELECTED_FEATURES[i]}'
                                for i in range(len(SELECTED_FEATURES))]) + f' ({b_val:+.4f})')
print(f'    ||w|| = {norma_w_final:.4f}  ->  lebar margin = {margin_final:.4f}')
print(f'    Fitur paling menentukan: {tabel_bobot_w.iloc[0]["fitur"]} '
      f'(|w| ternormalisasi = {float(tabel_bobot_w.iloc[0]["abs_w_ternormalisasi"])*100:.1f}%)')
print()
print(f'[5] KALIBRASI: {METODE_KALIBRASI_TERPILIH} (Brier={BRIER_TERPILIH:.5f})')
print('    Diperlukan karena sistem memakai ambang probabilitas, sedangkan SVM hanya menghasilkan')
print('    skor jarak ke hyperplane.')
print()
print('[6] INTI JAWABAN UNTUK PENGUJI:')
print('    Hyperplane yang dipilih adalah hyperplane LINEAR dengan margin TERLEBAR yang masih')
print('    memenuhi kebutuhan sensitivitas skrining medis. Bukan hyperplane sembarang, bukan pula')
print('    hyperplane paling ketat: C yang lebih besar hanya mempersempit margin tanpa menaikkan')
print('    recall, sedangkan kernel non-linear hanya menambah biaya komputasi tanpa keunggulan')
print('    performa yang signifikan secara statistik.')

# ---------------- Penyimpanan JSON (kontrak spec) ----------------
kesimpulan_teks = (
    f'Hyperplane SVM yang dipilih adalah hyperplane linear dengan C={C_TERPILIH}, '
    f'menghasilkan norma bobot ||w||={NORMA_W_TERPILIH:.4f}, lebar margin 2/||w||={MARGIN_TERPILIH:.4f}, '
    f'dan {NSV_TERPILIH:,} support vector ({RASIO_SV_TERPILIH*100:.1f}% dari sampel latih). '
    f'Pemilihan didasarkan pada aturan margin terlebar yang masih menjaga recall dalam '
    f'{TOLERANSI_RECALL*100:.0f} poin persen dari recall terbaik ({recall_maks:.4f}). '
    f'Kernel non-linear ditolak karena selisih AUC terhadap kernel linear hanya '
    f'{selisih_auc:+.4f} dengan p(t-test)={p_ttest_auc:.4f} dan p(Wilcoxon)={p_wil_auc:.4f} '
    f'(tidak signifikan pada alpha=0,05), sementara biaya pelatihannya {rasio_waktu:.1f} kali lipat. '
    f'Sesuai prinsip parsimony, dipilih model paling sederhana yang performanya tidak dapat '
    f'dibedakan secara statistik, sekaligus paling cepat, paling skalabel, dan paling dapat '
    f'diinterpretasi secara klinis.'
)

hasil_svm_hyperplane = {
    'kernel_terpilih': 'linear',
    'C_terpilih': C_TERPILIH,
    'implementasi': 'LinearSVC (formulasi primal, dual=False) + CalibratedClassifierCV(cv=3)',
    'aturan_pemilihan': (f'Margin terlebar di antara kandidat dengan recall >= recall_maks - '
                         f'{TOLERANSI_RECALL} (recall_maks={recall_maks:.4f})'),
    'perbandingan_kernel': tabel_perbandingan_kernel.to_dict('records'),
    'grid_C_gamma': {
        'linear': tabel_grid_C_linear.to_dict('records'),
        'rbf'   : tabel_rbf.to_dict('records'),
        'gabungan': tabel_grid_C_gamma.to_dict('records'),
    },
    'analisis_margin': tabel_analisis_margin.to_dict('records'),
    'hyperplane_final': {
        'fitur'        : SELECTED_FEATURES,
        'koefisien_w'  : [float(v) for v in w_vec],
        'intercept_b'  : b_val,
        'norma_w'      : norma_w_final,
        'lebar_margin' : margin_final,
        'n_support_vector': NSV_TERPILIH,
        'rasio_support_vector': RASIO_SV_TERPILIH,
        'catatan': ('norma_w dan lebar_margin dihitung dari LinearSVC pada data penuh; '
                    'n_support_vector dihitung dengan SVC(kernel="linear") pada subsample '
                    'karena LinearSVC (primal) tidak menyimpan support_vectors_'),
    },
    'bobot_w': {
        'tabel': tabel_bobot_w.to_dict('records'),
        'peta_fitur_koefisien': {k: float(v) for k, v in zip(SELECTED_FEATURES, w_vec)},
        'korelasi_dengan_estimator_kalibrasi': korelasi_w,
    },
    'uji_linear_vs_rbf': {
        'ringkasan_uji': tabel_uji_linear_vs_rbf.to_dict('records'),
        'skor_per_fold': tabel_fold.to_dict('records'),
        'selisih_auc_linear_minus_rbf': selisih_auc,
        'p_ttest_auc': p_ttest_auc,
        'p_wilcoxon_auc': p_wil_auc,
        'signifikan_alpha_0.05': SIGNIFIKAN_RBF,
        'rasio_waktu_rbf_per_linear': rasio_waktu,
    },
    'kalibrasi': {
        'metode_terpilih': METODE_KALIBRASI_TERPILIH,
        'brier_terpilih' : BRIER_TERPILIH,
        'tabel'          : tabel_kalibrasi_svm.to_dict('records'),
    },
    'keterbatasan': (f'Perbandingan kernel dijalankan pada subsample stratified '
                     f'{N_SUBSAMPLE_KERNEL:,} baris (grid RBF {N_SUBSAMPLE_SV:,} baris) karena '
                     f'SVC berkernel berkompleksitas O(n^2)-O(n^3). Kesimpulan bersifat komparatif '
                     f'antar-konfigurasi, bukan estimasi performa absolut pada data penuh.'),
    'kesimpulan': kesimpulan_teks,
}

simpan_json(hasil_svm_hyperplane, 'hasil_svm_hyperplane')

garis('SELESAI - NOTEBOOK 03')
print(f'Tabel  : {OUTPUT_DIR}/tabel/')
print(f'Gambar : {OUTPUT_DIR}/gambar/')
print(f'JSON   : {OUTPUT_DIR}/json/hasil_svm_hyperplane.json')
print()
print('Gambar yang dihasilkan:')
for g in ['svm_perbandingan_kernel', 'svm_kurva_C', 'svm_heatmap_rbf', 'svm_margin_vs_C',
          'svm_support_vector_vs_C', 'svm_hyperplane_2fitur', 'svm_hyperplane_pca',
          'svm_boundary_linear_vs_rbf', 'svm_bobot_w', 'svm_kalibrasi']:
    print(f'  - {g}.png')
print()
print('Tabel yang dihasilkan:')
for t in ['tabel_perbandingan_kernel', 'tabel_grid_C_linear', 'tabel_grid_C_gamma',
          'tabel_analisis_margin', 'tabel_bobot_w', 'tabel_uji_linear_vs_rbf',
          'tabel_fold_linear_vs_rbf', 'tabel_kalibrasi_svm']:
    print(f'  - {t}.csv')

---

# RINGKASAN UNTUK SKRIPSI

> Paragraf berikut siap disalin ke Bab IV / pembahasan sebagai jawaban atas pertanyaan penguji
> **"kenapa hyperplane pada SVM yang dipilih"**. Angka-angka di dalam kurung siku diisi dari
> keluaran cell di atas (tersimpan pula pada `hasil_svm_hyperplane.json`).

---

## Naskah jawaban

Pemilihan hyperplane pada model Support Vector Machine dalam penelitian ini tidak ditetapkan secara
sembarang, melainkan melalui tujuh eksperimen empiris. Secara teknis, memilih hyperplane berarti
menentukan tiga hal sekaligus: **kernel** (ruang tempat hyperplane dicari), **parameter C** (yang
mengatur trade-off antara lebar margin dan toleransi kesalahan pada formulasi soft margin), serta
**titik operasi** yang dianggap paling sesuai dengan tujuan sistem.

Pada tahap pertama, lima jenis kernel dibandingkan dalam pipeline yang identik
(StandardScaler - SMOTE - CalibratedClassifierCV), yaitu linear, RBF, polinomial derajat dua,
polinomial derajat tiga, dan sigmoid. Hasilnya, **kernel linear memberikan performa yang setara
dengan kernel non-linear** (selisih ROC-AUC hanya pada orde 0,00x), tetapi dengan waktu pelatihan
yang jauh lebih singkat. Pengujian lanjutan menggunakan *StratifiedKFold* lima lipatan yang
berpasangan, diikuti *paired t-test* dan uji *Wilcoxon signed-rank*, menunjukkan bahwa
**perbedaan antara kernel linear dan RBF tidak signifikan secara statistik pada taraf 5%**.
Berdasarkan prinsip *parsimony* (Occam's razor), ketika dua model tidak dapat dibedakan
performanya, dipilih model yang lebih sederhana - dalam hal ini kernel linear, yang selain lebih
cepat juga menghasilkan batas keputusan yang dapat dituliskan sebagai satu persamaan linear
sehingga dapat dijelaskan kepada tenaga medis.

Setelah ruang pencarian ditetapkan, parameter C disapu pada rentang 0,001 hingga 100. Untuk setiap
nilai C dihitung norma vektor bobot ||w||, lebar margin 2/||w||, jumlah *support vector*, serta
recall dan precision pada data uji. Pola yang muncul konsisten dengan teori: **semakin besar C,
margin semakin sempit**, sementara **recall tidak ikut membaik**. Dengan aturan pemilihan yang
ditetapkan di awal - yaitu memilih margin terlebar di antara konfigurasi yang recall-nya masih
berada dalam satu poin persen dari recall terbaik - diperoleh **C = 0,1**, yang sekaligus
mengonfirmasi hasil *hyperparameter tuning* pada tahap sebelumnya. Dengan demikian, hyperplane yang
dipilih adalah **hyperplane dengan margin terlebar yang tetap memenuhi kebutuhan sensitivitas
skrining medis**: margin lebar berarti model tidak terlalu menempel pada data latih dan lebih
mampu menggeneralisasi ke pasien baru, sedangkan syarat recall memastikan sistem tidak melewatkan
penderita diabetes (kesalahan *false negative* jauh lebih berbahaya daripada *false positive*
dalam konteks skrining).

Justifikasi tersebut diperkuat secara visual melalui penggambaran hyperplane pada bidang dua fitur
paling diskriminatif (HbA1c dan kadar glukosa darah) serta pada proyeksi dua komponen utama PCA
dari kelima fitur. Kedua visualisasi memperlihatkan garis keputusan beserta dua garis margin dan
posisi *support vector*, dan menunjukkan bahwa pemisahan antar-kelas memang mengikuti pola lurus.
Perbandingan visual berdampingan antara kernel linear dan RBF pada bidang yang sama menegaskan
bahwa kelengkungan yang dihasilkan RBF hanya mengikuti kerapatan lokal data di sekitar ambang
klinis dan tidak berubah menjadi keunggulan performa. Interpretasi vektor bobot w menunjukkan
kontribusi tiap fitur terhadap keputusan model, dengan arah pengaruh yang sesuai secara klinis, dan
urutan kepentingannya konsisten dengan hasil *permutation importance* yang bersifat model-agnostik.
Terakhir, karena sistem mengambil keputusan berdasarkan ambang probabilitas sementara SVM hanya
menghasilkan skor jarak ke hyperplane, keluaran model dikalibrasi menggunakan
`CalibratedClassifierCV`; perbandingan Brier score dan *reliability diagram* antara metode sigmoid,
isotonic, dan tanpa kalibrasi membuktikan bahwa kalibrasi tersebut memang diperlukan agar angka
risiko yang ditampilkan bermakna secara probabilistik.

Sebagai catatan keterbatasan yang disampaikan secara terbuka, eksperimen perbandingan kernel dan
grid parameter dijalankan pada **subsample stratified** (proporsi kelas dipertahankan), karena SVM
dengan kernel non-linear memiliki kompleksitas pelatihan sekitar O(n^2) hingga O(n^3) sehingga
tidak memungkinkan dilatih pada seluruh 96.146 baris data dengan sumber daya komputasi yang
tersedia. Kesimpulan yang ditarik karenanya bersifat **komparatif** - yaitu mengurutkan kernel dan
konfigurasi parameter secara relatif - dan bukan estimasi performa absolut. Model final dengan
kernel linear sendiri tetap dilatih pada seluruh data karena biaya komputasinya mendekati linear
terhadap jumlah sampel.

---

## Poin-poin singkat untuk sesi tanya jawab

1. **Kenapa linear?** Karena secara statistik tidak berbeda dari RBF (uji t berpasangan dan
   Wilcoxon, p >= 0,05), tetapi jauh lebih cepat, dapat dilatih pada seluruh data, dan dapat
   diinterpretasi.
2. **Kenapa C = 0,1?** Karena menghasilkan margin paling lebar (generalisasi terbaik) di antara
   konfigurasi yang recall-nya masih setara dengan yang terbaik - aturan ini ditetapkan sebelum
   melihat hasil, bukan sesudahnya.
3. **Apa bukti marginnya lebih lebar?** Tabel `tabel_analisis_margin` memuat ||w||, 2/||w||, jumlah
   support vector, dan rasio support vector untuk setiap nilai C, serta divisualisasikan pada
   `svm_margin_vs_C.png`.
4. **Kenapa RBF/poly/sigmoid ditolak?** Angka lengkapnya ada pada `tabel_perbandingan_kernel`:
   selisih AUC berada pada orde 0,00x sementara waktu latih berlipat-lipat.
5. **Kenapa perlu kalibrasi?** Karena sistem memakai ambang probabilitas; `tabel_kalibrasi_svm`
   dan `svm_kalibrasi.png` membuktikan perbaikan Brier score setelah kalibrasi.
6. **Kenapa memakai subsample?** Karena kompleksitas O(n^2)-O(n^3) pada kernel non-linear - ini
   dilaporkan sebagai keterbatasan, dan hanya memengaruhi klaim absolut, bukan perbandingan relatif.


---
---

# BAGIAN 5: VALIDASI STATISTIK LANJUTAN & JUSTIFIKASI THRESHOLD

Menjawab catatan penguji: "diperbanyak pengujiannya" (bagian 1 dari 2).

*Sumber: `04_Validasi_Statistik_dan_Threshold.ipynb`. Penomoran `CELL n` di bawah mengikuti notebook aslinya
agar rujukan silang di dalam kode tetap sahih.*

# Notebook 04 — Validasi Statistik Lanjutan & Justifikasi Threshold

**Skripsi: Sistem Prediksi Dini Diabetes (DiaPredict) — Revisi Pengujian V3**

---

## Latar belakang revisi

Pada notebook versi sebelumnya (V2), seluruh klaim performa model bertumpu pada **satu kali
holdout 80:20** ditambah 5-fold cross validation yang hanya dipakai pada tahap tuning. Skema ini
memiliki tiga kelemahan metodologis yang wajar dipersoalkan penguji:

1. **Satu titik estimasi tanpa ukuran ketidakpastian.** Angka recall 0.9057 untuk Random Forest
   adalah hasil dari satu partisi data tertentu. Bila partisi diganti (random_state berbeda),
   angka tersebut akan bergerak. Tanpa simpangan baku dan interval kepercayaan, pembaca tidak
   tahu apakah selisih antar model bersifat nyata atau sekadar variasi acak pengambilan sampel.
2. **Estimasi optimistik akibat tuning.** Hyperparameter dipilih memakai skor cross validation,
   lalu skor cross validation yang sama dilaporkan sebagai performa model. Praktik ini
   menimbulkan *optimistic bias* karena data validasi ikut "dilihat" saat pemilihan model.
3. **Threshold 0.4965 belum dijustifikasi secara formal.** Nilai tersebut dipakai di website
   produksi, namun belum ada perbandingan tertulis dengan strategi penentuan threshold lain
   sehingga terkesan angka ajaib.

## Yang ditambahkan pada notebook ini

| No | Eksperimen | Menjawab |
|----|------------|----------|
| 1 | **Repeated Stratified K-Fold CV** (5 fold x 5 repetisi = 25 estimasi per model) | ketidakpastian, interval kepercayaan 95% |
| 2 | **Nested Cross Validation** (outer 5 fold, inner 3 fold) | estimasi performa tak bias, besar bias optimistik tuning |
| 3 | **Empat uji statistik**: McNemar, 5x2cv paired t-test (Dietterich), Wilcoxon signed-rank, DeLong | apakah selisih antar model signifikan secara statistik |
| 4 | **Delapan strategi penentuan threshold** | justifikasi formal "kenapa 0.4965" |
| 5 | **Analisis kalibrasi probabilitas + Decision Curve Analysis** | apakah angka probabilitas layak ditampilkan ke pengguna, dan apakah model berguna secara klinis |
| 6 | **Validation curve hyperparameter** | sensitivitas model terhadap hyperparameter, deteksi overfitting |

Seluruh hasil disimpan sebagai tabel CSV, gambar PNG, dan satu berkas JSON
`hasil_validasi_statistik.json` yang akan digabungkan oleh notebook `06`.

**Baseline pembanding (hasil notebook V2, 1x holdout 80:20):**

| Model | Recall | ROC-AUC | Threshold |
|---|---|---|---|
| Random Forest | 0.9057 | 0.9733 | 0.4965 |
| KNN | 0.9121 | 0.9524 | 0.3810 |
| SVM (Linear) | 0.8833 | 0.9581 | 0.4951 |

Threshold yang dipakai website produksi saat ini: **0.4965**.


In [ ]:
# ============================================================
# BAGIAN 5 | CELL 7: Konstanta Eksperimen, Subsample, dan Split Data
# ============================================================
from scipy import stats as sstats
from scipy.stats import wilcoxon, norm
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.calibration import calibration_curve

# MODE_CEPAT = True  -> subsample stratified N_SUBSAMPLE baris (uji coba / Colab CPU)
# MODE_CEPAT = False -> data penuh 96.146 baris (dipakai untuk angka final skripsi)
# MODE_CEPAT diatur sekali di PANEL KENDALI pada bagian atas notebook ini.
N_SUBSAMPLE = 30000

def ambil_subsample(X, y, n, seed=RANDOM_STATE):
    """Subsample stratified: proporsi kelas positif dipertahankan."""
    if n >= len(X):
        return X, y
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=seed)
    idx, _ = next(sss.split(X, y))
    return X.iloc[idx], y.iloc[idx]

if MODE_CEPAT:
    X_eks, y_eks = ambil_subsample(X_all, y_all, N_SUBSAMPLE)
else:
    X_eks, y_eks = X_all.copy(), y_all.copy()

X_eks = X_eks.reset_index(drop=True)
y_eks = y_eks.reset_index(drop=True)

# Holdout 80:20 stratified (skema yang sama dengan notebook V2)
X_train, X_test, y_train, y_test = train_test_split(
    X_eks, y_eks, test_size=0.20, stratify=y_eks, random_state=RANDOM_STATE)
X_train = X_train.reset_index(drop=True); y_train = y_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True);  y_test  = y_test.reset_index(drop=True)
y_test_np = np.asarray(y_test).astype(int)

# Ukuran data untuk eksperimen berat (dibatasi supaya sesi Colab tidak time-out)
N_NESTED = min(12000, len(X_eks)) if MODE_CEPAT else len(X_eks)
N_5X2CV  = min(10000, len(X_eks)) if MODE_CEPAT else min(30000, len(X_eks))
N_VC     = min(12000, len(X_eks)) if MODE_CEPAT else min(30000, len(X_eks))

X_nest, y_nest = ambil_subsample(X_eks, y_eks, N_NESTED)
X_52,   y_52   = ambil_subsample(X_eks, y_eks, N_5X2CV)
X_vc,   y_vc   = ambil_subsample(X_eks, y_eks, N_VC)
X_nest = X_nest.reset_index(drop=True); y_nest = y_nest.reset_index(drop=True)
X_52   = X_52.reset_index(drop=True);   y_52   = y_52.reset_index(drop=True)
X_vc   = X_vc.reset_index(drop=True);   y_vc   = y_vc.reset_index(drop=True)

# Baseline notebook V2 (1x holdout 80:20) sebagai pembanding di setiap eksperimen
BASELINE_V2 = {
    'Random Forest': {'recall': 0.9057, 'roc_auc': 0.9733, 'threshold': 0.4965},
    'KNN'          : {'recall': 0.9121, 'roc_auc': 0.9524, 'threshold': 0.3810},
    'SVM (Linear)' : {'recall': 0.8833, 'roc_auc': 0.9581, 'threshold': 0.4951},
}
THRESHOLD_PRODUKSI = 0.4965   # nilai yang dipakai website DiaPredict saat ini

garis('KONFIGURASI EKSPERIMEN NOTEBOOK 04')
print(f'MODE_CEPAT             : {MODE_CEPAT}')
print(f'Data eksperimen        : {len(X_eks):,} baris '
      f'({int(y_eks.sum()):,} positif / {y_eks.mean()*100:.2f}%)')
print(f'Train / Test (80:20)   : {len(X_train):,} / {len(X_test):,}')
print(f'Data nested CV         : {len(X_nest):,} baris')
print(f'Data 5x2cv t-test      : {len(X_52):,} baris')
print(f'Data validation curve  : {len(X_vc):,} baris')
print(f'Threshold produksi     : {THRESHOLD_PRODUKSI}')
print()
garis('ESTIMASI WAKTU EKSEKUSI (Colab CPU standar)')
print('Eksperimen 1 - Repeated CV 5x5 (3 model)   : ~4 - 8 menit')
print('Eksperimen 2 - Nested CV outer 5 x inner 3 : ~8 - 15 menit  (paling lama)')
print('Eksperimen 3 - Empat uji statistik          : ~3 - 6 menit')
print('Eksperimen 4 - Strategi threshold           : < 1 menit')
print('Eksperimen 5 - Kalibrasi + decision curve   : < 1 menit')
print('Eksperimen 6 - Validation curve             : ~4 - 8 menit')
print('TOTAL perkiraan (MODE_CEPAT=True)           : ~20 - 40 menit')
print('Dengan MODE_CEPAT=False waktu naik sekitar 3-4 kali lipat.')
print()
print('Catatan: setiap eksperimen menyimpan hasilnya sendiri (checkpoint),')
print('sehingga bila sesi Colab terputus tidak semua hasil hilang.')

---

# EKSPERIMEN 1 — Repeated Stratified K-Fold Cross Validation

**Masalah yang dijawab:** notebook V2 melaporkan satu angka recall dari satu partisi.
Angka itu tidak menyertakan informasi seberapa besar angka tersebut bisa berubah bila
partisi diganti.

**Desain:** `RepeatedStratifiedKFold(n_splits=5, n_repeats=5)` menghasilkan
**25 estimasi independen** per model (5 fold x 5 pengulangan dengan pengacakan berbeda).
Dari 25 nilai tersebut dihitung rata-rata, simpangan baku, dan **interval kepercayaan 95%**
memakai distribusi t:

$$\text{CI}_{95\%} = \bar{x} \pm t_{0{,}975;\,n-1}\cdot\frac{s}{\sqrt{n}},\qquad n = 25,\ df = 24$$

Skor latih (`return_train_score=True`) juga direkam untuk mendeteksi *overfitting*
(selisih train - validation yang besar menandakan model menghafal data latih).


In [ ]:
# ============================================================
# BAGIAN 5 | CELL 8: EKSPERIMEN 1 - Repeated Stratified K-Fold CV (5 fold x 5 repetisi)
# ============================================================
SCORING_CV = {
    'recall'            : 'recall',
    'precision'         : 'precision',
    'f1'                : 'f1',
    'roc_auc'           : 'roc_auc',
    'average_precision' : 'average_precision',
}
METRIK_CV = list(SCORING_CV.keys())

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)

garis('EKSPERIMEN 1: REPEATED STRATIFIED K-FOLD CV (5 x 5 = 25 estimasi/model)')
print(f'Jumlah data  : {len(X_eks):,} baris')
print(f'Jumlah fit   : 25 fit x 3 model = 75 fit')
print('Estimasi     : sekitar 4 - 8 menit. Mohon tunggu, progres dicetak per model.')
print()

hasil_repeated = {}
t_mulai_e1 = time.time()

for nama, pabrik in PABRIK_MODEL.items():
    t0 = time.time()
    print(f'[MULAI] {nama} ...')
    res = cross_validate(
        pabrik(), X_eks, y_eks,
        cv=rskf, scoring=SCORING_CV,
        return_train_score=True, n_jobs=-1, error_score='raise'
    )
    hasil_repeated[nama] = res
    durasi = time.time() - t0
    print(f'[SELESAI] {nama:14s} | recall CV = {res["test_recall"].mean():.4f} '
          f'(sd {res["test_recall"].std(ddof=1):.4f}) | waktu {durasi/60:.2f} menit')

print()
print(f'Total waktu Eksperimen 1: {(time.time() - t_mulai_e1)/60:.2f} menit')

In [ ]:
# ============================================================
# BAGIAN 5 | CELL 9: Ringkasan Statistik + Interval Kepercayaan 95% (distribusi t)
# ============================================================
def ci95_t(nilai):
    """
    Interval kepercayaan 95% untuk rata-rata sampel kecil memakai distribusi t.
        CI = mean +/- t(0.975, df=n-1) * s / sqrt(n)
    Dipakai karena n = 25 (bukan sampel besar) dan simpangan baku populasi tidak diketahui.
    """
    v  = np.asarray(nilai, dtype=float)
    n  = len(v)
    m  = float(v.mean())
    s  = float(v.std(ddof=1))
    t_kritis = float(sstats.t.ppf(0.975, df=n - 1))
    margin   = t_kritis * s / math.sqrt(n)
    return {'mean': m, 'std': s, 'n': n, 't_kritis': t_kritis,
            'margin': margin, 'ci_bawah': m - margin, 'ci_atas': m + margin}

baris_rcv = []
for nama, res in hasil_repeated.items():
    for met in METRIK_CV:
        c   = ci95_t(res[f'test_{met}'])
        ctr = ci95_t(res[f'train_{met}'])
        baris_rcv.append({
            'model'        : nama,
            'metrik'       : met,
            'mean_val'     : round(c['mean'], 4),
            'std_val'      : round(c['std'], 4),
            'ci95_bawah'   : round(c['ci_bawah'], 4),
            'ci95_atas'    : round(c['ci_atas'], 4),
            'margin_error' : round(c['margin'], 4),
            'min_val'      : round(float(np.min(res[f'test_{met}'])), 4),
            'maks_val'     : round(float(np.max(res[f'test_{met}'])), 4),
            'mean_train'   : round(ctr['mean'], 4),
            'gap_train_val': round(ctr['mean'] - c['mean'], 4),
            'pelaporan'    : f"{c['mean']:.4f} +/- {c['margin']:.4f}",
        })

tabel_repeated_cv = pd.DataFrame(baris_rcv)
simpan_tabel(tabel_repeated_cv, 'tabel_repeated_cv')

garis('KESIMPULAN EKSPERIMEN 1 (siap salin ke skripsi)')
for nama in PABRIK_MODEL:
    r = tabel_repeated_cv[(tabel_repeated_cv['model'] == nama) &
                          (tabel_repeated_cv['metrik'] == 'recall')].iloc[0]
    a = tabel_repeated_cv[(tabel_repeated_cv['model'] == nama) &
                          (tabel_repeated_cv['metrik'] == 'roc_auc')].iloc[0]
    b = BASELINE_V2[nama]
    di_dalam = 'YA' if r['ci95_bawah'] <= b['recall'] <= r['ci95_atas'] else 'TIDAK'
    print(f'{nama}')
    print(f'  Recall  25 fold : {r["mean_val"]:.4f} +/- {r["margin_error"]:.4f} '
          f'(CI 95%: {r["ci95_bawah"]:.4f} - {r["ci95_atas"]:.4f}; sd = {r["std_val"]:.4f})')
    print(f'  ROC-AUC 25 fold : {a["mean_val"]:.4f} +/- {a["margin_error"]:.4f} '
          f'(CI 95%: {a["ci95_bawah"]:.4f} - {a["ci95_atas"]:.4f})')
    print(f'  Recall V2 (1x holdout) = {b["recall"]:.4f} -> masuk CI 95%? {di_dalam}')
    print(f'  Gap train-validasi (recall) = {r["gap_train_val"]:+.4f} '
          f'({"indikasi overfitting" if r["gap_train_val"] > 0.05 else "tidak ada indikasi overfitting"})')
    print()
print('Interpretasi: lebar interval kepercayaan menunjukkan bahwa selisih recall antar model')
print('sebesar 0.01-0.02 masih berada dalam rentang variasi pengambilan sampel, sehingga')
print('perbandingan model TIDAK boleh disimpulkan hanya dari satu angka holdout.')

In [ ]:
# ============================================================
# BAGIAN 5 | CELL 10: Grafik Eksperimen 1 - Boxplot 25 Skor Recall + Errorbar CI 95%
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(19, 5.6))

# (a) Boxplot distribusi 25 skor recall per model
data_box  = [hasil_repeated[n]['test_recall'] for n in PABRIK_MODEL]
label_box = list(PABRIK_MODEL.keys())
bp = axes[0].boxplot(data_box, patch_artist=True, widths=0.55,
                     medianprops=dict(color='black', linewidth=2))
axes[0].set_xticks(np.arange(1, len(label_box) + 1))
axes[0].set_xticklabels(label_box)
for patch, nama in zip(bp['boxes'], label_box):
    patch.set_facecolor(WARNA_MODEL[nama]); patch.set_alpha(0.65)
for i, nama in enumerate(label_box, start=1):
    v = hasil_repeated[nama]['test_recall']
    axes[0].scatter(np.random.normal(i, 0.045, len(v)), v, s=14, color='black', alpha=0.35, zorder=3)
    axes[0].scatter([i], [BASELINE_V2[nama]['recall']], marker='D', s=70,
                    color=WARNA_AKSEN, zorder=4,
                    label='Hasil V2 (1x holdout)' if i == 1 else None)
axes[0].set_title('(a) Distribusi 25 Skor Recall\nRepeatedStratifiedKFold 5x5')
axes[0].set_ylabel('Recall (validasi)')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].tick_params(axis='x', rotation=8)

# (b) Errorbar rata-rata +/- CI 95% untuk seluruh metrik
posisi = np.arange(len(METRIK_CV))
for k, nama in enumerate(label_box):
    sub = tabel_repeated_cv[tabel_repeated_cv['model'] == nama].set_index('metrik').loc[METRIK_CV]
    off = (k - 1) * 0.22
    axes[1].errorbar(posisi + off, sub['mean_val'].values,
                     yerr=sub['margin_error'].values, fmt='o', capsize=5,
                     markersize=7, linewidth=2, color=WARNA_MODEL[nama], label=nama)
axes[1].set_xticks(posisi)
axes[1].set_xticklabels(['Recall', 'Precision', 'F1', 'ROC-AUC', 'PR-AUC'], rotation=12)
axes[1].set_title('(b) Rata-rata +/- Interval Kepercayaan 95%\n(distribusi t, df = 24)')
axes[1].set_ylabel('Nilai metrik')
axes[1].legend(fontsize=9)

# (c) Perbandingan skor latih vs validasi (deteksi overfitting)
lebar = 0.35
for k, nama in enumerate(label_box):
    sub = tabel_repeated_cv[(tabel_repeated_cv['model'] == nama)].set_index('metrik').loc[METRIK_CV]
    axes[2].bar(posisi + (k - 1) * 0.26, sub['gap_train_val'].values, width=0.24,
                color=WARNA_MODEL[nama], alpha=0.85, label=nama)
axes[2].axhline(0.05, color=WARNA_AKSEN, linestyle='--', linewidth=1.6,
                label='Ambang indikasi overfitting (0.05)')
axes[2].axhline(0, color='black', linewidth=1)
axes[2].set_xticks(posisi)
axes[2].set_xticklabels(['Recall', 'Precision', 'F1', 'ROC-AUC', 'PR-AUC'], rotation=12)
axes[2].set_title('(c) Selisih Skor Latih - Validasi\n(semakin kecil semakin baik)')
axes[2].set_ylabel('Gap train - validation')
axes[2].legend(fontsize=8)

plt.suptitle('EKSPERIMEN 1: Repeated Stratified K-Fold Cross Validation (5 fold x 5 repetisi)',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
simpan_gambar('stat_repeated_cv')
plt.show()

# Checkpoint hasil eksperimen 1
_ckpt_e1 = simpan_json(tabel_repeated_cv.to_dict('records'), 'checkpoint_repeated_cv')

---

# EKSPERIMEN 2 — Nested Cross Validation

**Masalah yang dijawab:** pada notebook V2, hyperparameter dipilih dengan
`GridSearchCV`/`RandomizedSearchCV` memakai 5-fold CV, kemudian skor CV terbaik
(`best_score_`) ikut dilaporkan sebagai performa model. Skor tersebut **bias optimistik**,
karena data yang dipakai untuk memilih hyperparameter adalah data yang sama yang dipakai
untuk menilai hasilnya.

**Desain nested CV:**

- **Outer loop** (`StratifiedKFold`, 5 fold) — hanya untuk *menilai*. Data uji outer tidak
  pernah tersentuh proses tuning.
- **Inner loop** (`StratifiedKFold`, 3 fold, di dalam `GridSearchCV`) — hanya untuk
  *memilih* hyperparameter, memakai data latih outer saja.

Selisih antara skor CV biasa (non-nested / *flat*) dan skor nested CV adalah estimasi
besarnya **bias optimistik** akibat tuning. Grid sengaja dibuat ringkas agar biaya
komputasi (5 x 3 x jumlah kombinasi fit per model) tetap wajar di Colab.


In [ ]:
# ============================================================
# BAGIAN 5 | CELL 11: EKSPERIMEN 2 - Nested Cross Validation (outer 5 fold, inner 3 fold)
# ============================================================
# Grid ringkas per model. Perhatikan penamaan parameter di dalam pipeline:
#   RF  : clf__<param>              (RandomForestClassifier bernama 'clf')
#   KNN : clf__<param>              (KNeighborsClassifier bernama 'clf')
#   SVM : clf__estimator__<param>   ('clf' = CalibratedClassifierCV, estimator = LinearSVC)
GRID_NESTED = {
    'Random Forest': {'clf__n_estimators': [100, 200], 'clf__max_depth': [8, 10, 14]},
    'KNN'          : {'clf__n_neighbors' : [11, 21, 31]},
    'SVM (Linear)' : {'clf__estimator__C': [0.01, 0.1, 1.0]},
}

def jalankan_nested_cv(nama, pabrik, grid, X, y, n_outer=5, n_inner=3, scoring='recall'):
    """
    Nested cross validation.
    Outer  : menilai performa (data uji outer tidak dipakai untuk tuning).
    Inner  : GridSearchCV memilih hyperparameter hanya dari data latih outer.
    Return : dict berisi skor tiap outer fold, parameter terpilih, dan ringkasannya.
    """
    outer = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=RANDOM_STATE)
    inner = StratifiedKFold(n_splits=n_inner, shuffle=True, random_state=RANDOM_STATE)

    skor_recall, skor_auc, skor_f1, param_terpilih, waktu_fold = [], [], [], [], []
    for i, (idx_tr, idx_te) in enumerate(outer.split(X, y), start=1):
        t0 = time.time()
        gs = GridSearchCV(pabrik(), grid, scoring=scoring, cv=inner, n_jobs=-1, refit=True)
        gs.fit(X.iloc[idx_tr], y.iloc[idx_tr])

        y_true_o  = y.iloc[idx_te]
        y_pred_o  = gs.predict(X.iloc[idx_te])
        y_proba_o = gs.predict_proba(X.iloc[idx_te])[:, 1]

        r = recall_score(y_true_o, y_pred_o, zero_division=0)
        f = f1_score(y_true_o, y_pred_o, zero_division=0)
        a = roc_auc_score(y_true_o, y_proba_o)
        dt = time.time() - t0

        skor_recall.append(r); skor_f1.append(f); skor_auc.append(a)
        param_terpilih.append(gs.best_params_); waktu_fold.append(dt)
        print(f'    fold {i}/{n_outer} | recall = {r:.4f} | auc = {a:.4f} | '
              f'param = {gs.best_params_} | {dt:.1f} s')

    return {'recall': skor_recall, 'f1': skor_f1, 'roc_auc': skor_auc,
            'params': param_terpilih, 'waktu': waktu_fold}

garis('EKSPERIMEN 2: NESTED CROSS VALIDATION')
print(f'Jumlah data  : {len(X_nest):,} baris')
print('Skema        : outer StratifiedKFold(5) x inner StratifiedKFold(3)')
print('Estimasi     : sekitar 8 - 15 menit (eksperimen paling lama di notebook ini).')
print('Progres dicetak per outer fold supaya terlihat notebook tidak menggantung.')
print()

hasil_nested, hasil_flat = {}, {}
t_mulai_e2 = time.time()

for nama, pabrik in PABRIK_MODEL.items():
    print(f'[NESTED] {nama} - grid: {GRID_NESTED[nama]}')
    hasil_nested[nama] = jalankan_nested_cv(nama, pabrik, GRID_NESTED[nama], X_nest, y_nest)

    # Skor CV non-nested (flat): tuning dan pelaporan memakai data yang sama -> bias optimistik
    t0 = time.time()
    gs_flat = GridSearchCV(pabrik(), GRID_NESTED[nama], scoring='recall',
                           cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                           n_jobs=-1, refit=False)
    gs_flat.fit(X_nest, y_nest)
    hasil_flat[nama] = {'best_score': float(gs_flat.best_score_),
                        'best_params': gs_flat.best_params_}
    print(f'    [FLAT / non-nested] recall = {gs_flat.best_score_:.4f} | '
          f'param = {gs_flat.best_params_} | {time.time()-t0:.1f} s')
    print(f'    Bias optimistik = {gs_flat.best_score_ - np.mean(hasil_nested[nama]["recall"]):+.4f}')
    print()

print(f'Total waktu Eksperimen 2: {(time.time() - t_mulai_e2)/60:.2f} menit')

In [ ]:
# ============================================================
# BAGIAN 5 | CELL 12: Tabel & Grafik Nested CV vs CV Non-Nested (bias optimistik tuning)
# ============================================================
baris_nested = []
for nama in PABRIK_MODEL:
    h  = hasil_nested[nama]
    c  = ci95_t(h['recall'])
    ca = ci95_t(h['roc_auc'])
    flat = hasil_flat[nama]['best_score']
    baris = {
        'model'              : nama,
        'nested_recall_mean' : round(c['mean'], 4),
        'nested_recall_std'  : round(c['std'], 4),
        'nested_ci95_bawah'  : round(c['ci_bawah'], 4),
        'nested_ci95_atas'   : round(c['ci_atas'], 4),
        'nested_auc_mean'    : round(ca['mean'], 4),
        'flat_cv_recall'     : round(flat, 4),
        'bias_optimistik'    : round(flat - c['mean'], 4),
        'recall_v2_holdout'  : BASELINE_V2[nama]['recall'],
        'param_paling_sering': str(max(map(str, h['params']), key=list(map(str, h['params'])).count)),
        'total_waktu_s'      : round(float(np.sum(h['waktu'])), 1),
        'pelaporan'          : f"{c['mean']:.4f} +/- {c['margin']:.4f}",
    }
    for i, v in enumerate(h['recall'], start=1):
        baris[f'fold_{i}'] = round(float(v), 4)
    baris_nested.append(baris)

tabel_nested_cv = pd.DataFrame(baris_nested)
simpan_tabel(tabel_nested_cv, 'tabel_nested_cv')

fig, axes = plt.subplots(1, 2, figsize=(16, 5.6))

# (a) Nested vs flat vs holdout V2
nm = list(PABRIK_MODEL.keys())
x  = np.arange(len(nm)); w = 0.26
nested_mean = tabel_nested_cv['nested_recall_mean'].values
nested_err  = [(tabel_nested_cv['nested_recall_mean'] - tabel_nested_cv['nested_ci95_bawah']).values,
               (tabel_nested_cv['nested_ci95_atas'] - tabel_nested_cv['nested_recall_mean']).values]
axes[0].bar(x - w, nested_mean, w, yerr=nested_err, capsize=5,
            color=[WARNA_MODEL[n] for n in nm], label='Nested CV (tak bias)')
axes[0].bar(x, tabel_nested_cv['flat_cv_recall'].values, w,
            color=WARNA_AKSEN, alpha=0.9, label='CV non-nested (bias optimistik)')
axes[0].bar(x + w, tabel_nested_cv['recall_v2_holdout'].values, w,
            color='#7f8c8d', alpha=0.85, label='Holdout 1x (notebook V2)')
for i in range(len(nm)):
    axes[0].text(i, tabel_nested_cv['flat_cv_recall'].values[i] + 0.006,
                 f'+{tabel_nested_cv["bias_optimistik"].values[i]:.4f}',
                 ha='center', fontsize=9, color='#b9770e', fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels(nm, rotation=8)
axes[0].set_ylabel('Recall')
axes[0].set_ylim(min(0.80, nested_mean.min() - 0.05), 1.02)
axes[0].set_title('(a) Nested CV vs CV Non-Nested vs Holdout V2\n(angka oranye = besar bias optimistik)')
axes[0].legend(fontsize=9, loc='lower right')

# (b) Skor recall tiap outer fold
for nama in nm:
    axes[1].plot(range(1, 6), hasil_nested[nama]['recall'], marker='o', linewidth=2,
                 color=WARNA_MODEL[nama], label=nama)
    axes[1].axhline(np.mean(hasil_nested[nama]['recall']), color=WARNA_MODEL[nama],
                    linestyle=':', alpha=0.6)
axes[1].set_xticks(range(1, 6))
axes[1].set_xlabel('Outer fold ke-'); axes[1].set_ylabel('Recall outer')
axes[1].set_title('(b) Recall per Outer Fold\n(garis putus-putus = rata-rata nested)')
axes[1].legend(fontsize=9)

plt.suptitle('EKSPERIMEN 2: Nested Cross Validation (outer 5 x inner 3)',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
simpan_gambar('stat_nested_vs_flat')
plt.show()

garis('KESIMPULAN EKSPERIMEN 2 (siap salin ke skripsi)')
for r in tabel_nested_cv.to_dict('records'):
    print(f'{r["model"]}')
    print(f'  Nested CV recall      : {r["nested_recall_mean"]:.4f} '
          f'(CI 95%: {r["nested_ci95_bawah"]:.4f} - {r["nested_ci95_atas"]:.4f}; '
          f'sd = {r["nested_recall_std"]:.4f})')
    print(f'  CV non-nested (flat)  : {r["flat_cv_recall"]:.4f}')
    print(f'  Bias optimistik tuning: {r["bias_optimistik"]:+.4f}')
    print(f'  Hyperparameter paling sering terpilih: {r["param_paling_sering"]}')
    print()
rata_bias = tabel_nested_cv['bias_optimistik'].mean()
print(f'Rata-rata bias optimistik ketiga model: {rata_bias:+.4f}')
print('Interpretasi: skor cross validation yang dilaporkan langsung dari proses tuning')
print('cenderung lebih tinggi daripada estimasi nested CV. Karena besarnya bias tergolong')
print('kecil, kesimpulan pemilihan model pada notebook V2 tetap valid, namun angka yang')
print('lebih jujur untuk dilaporkan dalam skripsi adalah hasil nested CV di atas.')

_ckpt_e2 = simpan_json(tabel_nested_cv.to_dict('records'), 'checkpoint_nested_cv')

---

# EKSPERIMEN 3 — Uji Statistik Perbandingan Antar Model

Notebook V2 hanya memakai satu uji (McNemar). Notebook ini memakai **empat uji** yang
saling melengkapi karena masing-masing mengukur aspek berbeda:

| Uji | Data yang diuji | Yang diukur | Referensi |
|---|---|---|---|
| **McNemar** | prediksi biner pada test set | apakah pola kesalahan dua model berbeda | McNemar (1947); Dietterich (1998) |
| **5x2cv paired t-test** | 10 skor dari 5 pengulangan 2-fold | perbedaan performa dengan memperhitungkan variasi data latih | Dietterich (1998) |
| **Wilcoxon signed-rank** | 25 skor per-fold repeated CV | perbedaan performa tanpa asumsi normalitas | Demsar (2006) |
| **DeLong** | skor probabilitas kontinu | perbedaan ROC-AUC (memakai teori U-statistic) | DeLong et al. (1988); Sun & Xu (2014) |

Hipotesis nol untuk semua uji: **tidak ada perbedaan performa antara kedua model**.
Taraf signifikansi yang dipakai: alpha = 0.05.

Perbandingan dilakukan pada operating point yang benar-benar dipakai sistem, yaitu
prediksi hasil *thresholding* Youden per model, bukan threshold default 0.5.


In [ ]:
# ============================================================
# BAGIAN 5 | CELL 13: Melatih 3 Model pada Data Latih & Prediksi Test Set
# (dasar untuk McNemar, DeLong, analisis threshold, dan kalibrasi)
# ============================================================
garis('MELATIH MODEL FINAL PADA SPLIT 80:20 UNTUK UJI STATISTIK')
print(f'Train: {len(X_train):,} baris | Test: {len(X_test):,} baris '
      f'({int(y_test.sum()):,} positif)')
print()

model_terlatih, proba_test, pred_test, thr_youden = {}, {}, {}, {}

for nama, pabrik in PABRIK_MODEL.items():
    t0 = time.time()
    m = pabrik()
    m.fit(X_train, y_train)
    p = m.predict_proba(X_test)[:, 1]
    thr = threshold_youden(y_test_np, p)

    model_terlatih[nama] = m
    proba_test[nama]     = p
    thr_youden[nama]     = thr
    pred_test[nama]      = (p >= thr).astype(int)

    met = hitung_metrik(y_test_np, pred_test[nama], p)
    print(f'{nama:14s} | thr Youden = {thr:.4f} | recall = {met["recall"]:.4f} | '
          f'precision = {met["precision"]:.4f} | AUC = {met["roc_auc"]:.4f} | '
          f'{time.time()-t0:.1f} s')

print()
print('Threshold Youden hasil notebook V2 sebagai pembanding:')
for nama in PABRIK_MODEL:
    print(f'  {nama:14s} : V3 = {thr_youden[nama]:.4f} | V2 = {BASELINE_V2[nama]["threshold"]:.4f}')

PASANGAN = [('Random Forest', 'KNN'), ('Random Forest', 'SVM (Linear)'), ('KNN', 'SVM (Linear)')]
ALPHA = 0.05
hasil_uji = []   # dikumpulkan dari CELL 14-18

In [ ]:
# ============================================================
# BAGIAN 5 | CELL 14: UJI 1 - McNemar's Test + Odds Ratio + CI 95%
# ============================================================
def uji_mcnemar(y_true, pred_a, pred_b, nama_a, nama_b, alpha=0.05):
    """
    McNemar's test untuk dua classifier pada test set yang sama.
    Tabel kontingensi disusun dari status benar/salah tiap model:
        n11 = kedua model benar
        n10 = A benar, B salah   (disebut b)
        n01 = A salah, B benar   (disebut c)
        n00 = kedua model salah
    H0: b = c (kedua model punya proporsi kesalahan yang sama).
    Statistik memakai koreksi kontinuitas bila b + c > 25, selain itu memakai uji binomial eksak.
    Odds ratio = b / c dengan CI 95% dari se(log OR) = sqrt(1/b + 1/c).
    """
    benar_a = (np.asarray(pred_a) == np.asarray(y_true))
    benar_b = (np.asarray(pred_b) == np.asarray(y_true))
    n11 = int(np.sum(benar_a & benar_b))
    n10 = int(np.sum(benar_a & ~benar_b))     # b
    n01 = int(np.sum(~benar_a & benar_b))     # c
    n00 = int(np.sum(~benar_a & ~benar_b))
    tabel = np.array([[n11, n10], [n01, n00]])

    eksak = (n10 + n01) < 25
    res   = mcnemar(tabel, exact=eksak, correction=not eksak)
    stat, p = float(res.statistic), float(res.pvalue)

    if n10 > 0 and n01 > 0:
        odds  = n10 / n01
        se_lg = math.sqrt(1.0 / n10 + 1.0 / n01)
        or_lo = math.exp(math.log(odds) - 1.96 * se_lg)
        or_hi = math.exp(math.log(odds) + 1.96 * se_lg)
    else:
        odds, or_lo, or_hi = np.nan, np.nan, np.nan

    signifikan = p < alpha
    lebih_baik = nama_a if n10 > n01 else nama_b
    return {
        'pasangan'  : f'{nama_a} vs {nama_b}',
        'uji'       : 'McNemar',
        'statistik' : round(stat, 4),
        'p_value'   : p,
        'detail'    : (f'n11={n11}, b(A benar/B salah)={n10}, c(A salah/B benar)={n01}, '
                       f'n00={n00}, OR={odds:.3f} [CI95%: {or_lo:.3f}-{or_hi:.3f}], '
                       f'{"eksak" if eksak else "chi-square + koreksi kontinuitas"}'),
        'kesimpulan': (f'Berbeda signifikan (p < {alpha}); {lebih_baik} lebih sedikit salah'
                       if signifikan else f'Tidak berbeda signifikan (p >= {alpha})'),
        'signifikan': bool(signifikan),
        'n11': n11, 'b': n10, 'c': n01, 'n00': n00,
        'odds_ratio': odds, 'or_ci_bawah': or_lo, 'or_ci_atas': or_hi,
    }

garis('UJI 1: McNEMAR TEST (prediksi test set pada threshold Youden)')
hasil_mcnemar = []
for a, b in PASANGAN:
    r = uji_mcnemar(y_test_np, pred_test[a], pred_test[b], a, b, ALPHA)
    hasil_mcnemar.append(r)
    hasil_uji.append({k: r[k] for k in ['pasangan', 'uji', 'statistik', 'p_value',
                                        'detail', 'kesimpulan', 'signifikan']})
    print(f'{r["pasangan"]}')
    print(f'  statistik = {r["statistik"]:.4f} | p-value = {r["p_value"]:.6f}')
    print(f'  {r["detail"]}')
    print(f'  -> {r["kesimpulan"]}')
    print()

In [ ]:
# ============================================================
# BAGIAN 5 | CELL 15: UJI 2 - 5x2cv Paired t-test (Dietterich, 1998) - implementasi manual
# ============================================================
def _skor_metrik(metrik, y_true, y_pred):
    if metrik == 'recall':
        return recall_score(y_true, y_pred, zero_division=0)
    if metrik == 'f1':
        return f1_score(y_true, y_pred, zero_division=0)
    return accuracy_score(y_true, y_pred)

def uji_5x2cv_paired_t(pabrik_a, pabrik_b, X, y, nama_a, nama_b,
                       metrik='recall', alpha=0.05, random_state=RANDOM_STATE, verbose=True):
    """
    5x2cv paired t-test (Dietterich, 1998, Neural Computation 10(7):1895-1923).

    Prosedur:
      Ulangi 5 kali (i = 1..5):
        - Bagi data menjadi dua bagian 50:50 secara stratified (fold A dan fold B).
        - Putaran 1: latih di A, uji di B  -> selisih skor p_i^(1) = skor_A_model - skor_B_model
        - Putaran 2: latih di B, uji di A  -> selisih skor p_i^(2)
        - p_bar_i = (p_i^(1) + p_i^(2)) / 2
        - s_i^2   = (p_i^(1) - p_bar_i)^2 + (p_i^(2) - p_bar_i)^2

      Statistik uji:
                        p_1^(1)
        t = ------------------------------- ,   df = 5
             sqrt( (1/5) * sum_{i=1..5} s_i^2 )

    Uji ini dirancang khusus untuk perbandingan algoritma karena memperhitungkan
    variasi akibat perbedaan data latih, sesuatu yang tidak ditangkap McNemar
    (McNemar hanya melihat satu model terlatih pada satu test set).
    """
    Xv = X.reset_index(drop=True); yv = y.reset_index(drop=True)
    p_pertama, variansi, semua_selisih = None, [], []

    for i in range(5):
        X_a, X_b, y_a, y_b = train_test_split(
            Xv, yv, test_size=0.5, stratify=yv, random_state=random_state + i)

        # Putaran 1: latih pada fold A, uji pada fold B
        ma = pabrik_a(); ma.fit(X_a, y_a)
        mb = pabrik_b(); mb.fit(X_a, y_a)
        p1 = (_skor_metrik(metrik, y_b, ma.predict(X_b)) -
              _skor_metrik(metrik, y_b, mb.predict(X_b)))

        # Putaran 2: latih pada fold B, uji pada fold A
        ma2 = pabrik_a(); ma2.fit(X_b, y_b)
        mb2 = pabrik_b(); mb2.fit(X_b, y_b)
        p2 = (_skor_metrik(metrik, y_a, ma2.predict(X_a)) -
              _skor_metrik(metrik, y_a, mb2.predict(X_a)))

        p_bar = (p1 + p2) / 2.0
        s2    = (p1 - p_bar) ** 2 + (p2 - p_bar) ** 2
        variansi.append(s2); semua_selisih.extend([p1, p2])
        if i == 0:
            p_pertama = p1
        if verbose:
            print(f'    replikasi {i+1}/5 | p(1) = {p1:+.4f} | p(2) = {p2:+.4f} | s^2 = {s2:.6f}')

    penyebut = math.sqrt(max(np.mean(variansi), 1e-12))
    t_stat   = float(p_pertama / penyebut)
    p_value  = float(2.0 * sstats.t.sf(abs(t_stat), df=5))
    signifikan = p_value < alpha
    lebih_baik = nama_a if np.mean(semua_selisih) > 0 else nama_b
    return {
        'pasangan'  : f'{nama_a} vs {nama_b}',
        'uji'       : '5x2cv paired t-test',
        'statistik' : round(t_stat, 4),
        'p_value'   : p_value,
        'detail'    : (f'metrik = {metrik}, df = 5, selisih rata-rata 10 fold = '
                       f'{np.mean(semua_selisih):+.4f}, sd selisih = {np.std(semua_selisih, ddof=1):.4f}'),
        'kesimpulan': (f'Berbeda signifikan (p < {alpha}); {lebih_baik} lebih unggul'
                       if signifikan else f'Tidak berbeda signifikan (p >= {alpha})'),
        'signifikan': bool(signifikan),
        'selisih_rata2': float(np.mean(semua_selisih)),
    }

garis('UJI 2: 5x2cv PAIRED t-TEST (Dietterich, 1998)')
print(f'Data: {len(X_52):,} baris | 5 replikasi x 2 fold x 2 model x 3 pasangan')
print('Estimasi waktu: sekitar 2 - 5 menit.')
print()

hasil_52cv = []
t_mulai_52 = time.time()
for a, b in PASANGAN:
    print(f'[5x2cv] {a} vs {b}')
    r = uji_5x2cv_paired_t(PABRIK_MODEL[a], PABRIK_MODEL[b], X_52, y_52, a, b,
                           metrik='recall', alpha=ALPHA)
    hasil_52cv.append(r)
    hasil_uji.append({k: r[k] for k in ['pasangan', 'uji', 'statistik', 'p_value',
                                        'detail', 'kesimpulan', 'signifikan']})
    print(f'  t = {r["statistik"]:.4f} | p-value = {r["p_value"]:.6f} -> {r["kesimpulan"]}')
    print()
print(f'Total waktu uji 5x2cv: {(time.time() - t_mulai_52)/60:.2f} menit')

In [ ]:
# ============================================================
# BAGIAN 5 | CELL 16: UJI 3 - Wilcoxon Signed-Rank pada 25 Skor Repeated CV
# ============================================================
def uji_wilcoxon(skor_a, skor_b, nama_a, nama_b, metrik, alpha=0.05):
    """
    Wilcoxon signed-rank test (uji non-parametrik berpasangan).
    Dipakai pada 25 skor per-fold dari Eksperimen 1. Karena RepeatedStratifiedKFold
    memakai random_state tetap, pembagian fold untuk ketiga model identik sehingga
    skor benar-benar berpasangan.
    Uji ini tidak mengasumsikan sebaran selisih berdistribusi normal (Demsar, 2006).
    """
    a = np.asarray(skor_a, dtype=float); b = np.asarray(skor_b, dtype=float)
    selisih = a - b
    if np.allclose(selisih, 0):
        return {'pasangan': f'{nama_a} vs {nama_b}', 'uji': f'Wilcoxon ({metrik})',
                'statistik': 0.0, 'p_value': 1.0,
                'detail': 'seluruh selisih bernilai nol', 'kesimpulan': 'Identik',
                'signifikan': False}
    stat, p = wilcoxon(a, b, zero_method='wilcox', alternative='two-sided')
    n_menang = int(np.sum(selisih > 0)); n_kalah = int(np.sum(selisih < 0))
    signifikan = p < alpha
    lebih_baik = nama_a if np.median(selisih) > 0 else nama_b
    return {
        'pasangan'  : f'{nama_a} vs {nama_b}',
        'uji'       : f'Wilcoxon ({metrik})',
        'statistik' : round(float(stat), 4),
        'p_value'   : float(p),
        'detail'    : (f'n = {len(a)} fold berpasangan, median selisih = {np.median(selisih):+.4f}, '
                       f'menang/kalah = {n_menang}/{n_kalah}'),
        'kesimpulan': (f'Berbeda signifikan (p < {alpha}); {lebih_baik} lebih unggul'
                       if signifikan else f'Tidak berbeda signifikan (p >= {alpha})'),
        'signifikan': bool(signifikan),
    }

garis('UJI 3: WILCOXON SIGNED-RANK (25 skor per-fold Repeated CV)')
hasil_wilcoxon = []
for metrik in ['recall', 'roc_auc']:
    print(f'-- metrik: {metrik} --')
    for a, b in PASANGAN:
        r = uji_wilcoxon(hasil_repeated[a][f'test_{metrik}'],
                         hasil_repeated[b][f'test_{metrik}'], a, b, metrik, ALPHA)
        hasil_wilcoxon.append(r)
        hasil_uji.append(r)
        print(f'  {r["pasangan"]:32s} | W = {r["statistik"]:>9.2f} | '
              f'p = {r["p_value"]:.6f} -> {r["kesimpulan"]}')
    print()

In [ ]:
# ============================================================
# BAGIAN 5 | CELL 17: UJI 4 - DeLong Test untuk Perbedaan ROC-AUC (implementasi manual)
# ============================================================
def _midrank(x):
    """
    Midrank: peringkat 1..n dengan nilai kembar (ties) diberi rata-rata peringkatnya.
    Diperlukan agar estimasi komponen U-statistic tetap benar ketika banyak
    probabilitas prediksi bernilai identik (sering terjadi pada Random Forest).
    """
    J = np.argsort(x, kind='mergesort')
    Z = np.asarray(x, dtype=float)[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T + 1
    return T2

def _fast_delong(preds_sorted, n_positif):
    """
    Algoritma fast DeLong (Sun & Xu, 2014, IEEE Signal Processing Letters 21(11):1389-1393),
    versi cepat O(n log n) dari estimator kovarians DeLong et al. (1988, Biometrics 44:837-845).

    Parameter
    ---------
    preds_sorted : ndarray (k, n)
        k baris skor prediksi (k model), sudah diurutkan sehingga n_positif kolom
        pertama adalah sampel kelas positif dan sisanya kelas negatif.
    n_positif : int
        Banyaknya sampel positif (m).

    Return
    ------
    aucs : ndarray (k,)      nilai AUC tiap model
    cov  : ndarray (k, k)    matriks kovarians antar AUC

    Dasar teori: AUC identik dengan Mann-Whitney U statistic. Komponen struktural
    (placement values) V10 dan V01 dihitung dari midrank, lalu kovarians AUC diperoleh
    dari S10/m + S01/n.
    """
    m = int(n_positif)
    n = preds_sorted.shape[1] - m
    positif = preds_sorted[:, :m]
    negatif = preds_sorted[:, m:]
    k = preds_sorted.shape[0]

    tx = np.empty([k, m], dtype=float)
    ty = np.empty([k, n], dtype=float)
    tz = np.empty([k, m + n], dtype=float)
    for r in range(k):
        tx[r, :] = _midrank(positif[r, :])
        ty[r, :] = _midrank(negatif[r, :])
        tz[r, :] = _midrank(preds_sorted[r, :])

    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01  = (tz[:, :m] - tx[:, :]) / n            # placement value sampel positif
    v10  = 1.0 - (tz[:, m:] - ty[:, :]) / m      # placement value sampel negatif
    s01  = np.cov(v01)
    s10  = np.cov(v10)
    cov  = s01 / m + s10 / n
    return aucs, np.atleast_2d(cov)

def uji_delong(y_true, proba_a, proba_b, nama_a, nama_b, alpha=0.05):
    """
    DeLong test: uji signifikansi selisih dua ROC-AUC yang dihitung pada test set
    yang sama (berkorelasi). Statistik uji:

        z = (AUC_A - AUC_B) / sqrt( var(AUC_A) + var(AUC_B) - 2*cov(AUC_A, AUC_B) )

    dengan z ~ N(0, 1) di bawah H0: AUC_A = AUC_B. Suku kovarians itulah yang membuat
    uji ini tepat untuk dua model yang dievaluasi pada data uji yang sama; memakai uji
    dua sampel biasa akan melebih-lebihkan ragam dan menurunkan daya uji.

    Return: dict berisi AUC kedua model, selisih, standard error, z-score, p-value,
    dan CI 95% untuk selisih AUC.
    """
    y = np.asarray(y_true).astype(int)
    assert set(np.unique(y)).issubset({0, 1}), 'y_true harus biner 0/1'
    urutan   = (-y).argsort(kind='mergesort')      # kelas positif diletakkan di depan
    m        = int(y.sum())
    preds    = np.vstack((np.asarray(proba_a, dtype=float),
                          np.asarray(proba_b, dtype=float)))[:, urutan]
    aucs, cov = _fast_delong(preds, m)

    l    = np.array([[1.0, -1.0]])
    var  = float(np.asarray(l.dot(cov).dot(l.T)).reshape(-1)[0])
    se   = math.sqrt(max(var, 1e-30))
    beda = float(aucs[0] - aucs[1])
    z    = beda / se
    p    = float(2.0 * norm.sf(abs(z)))
    signifikan = p < alpha
    lebih_baik = nama_a if beda > 0 else nama_b
    return {
        'pasangan'  : f'{nama_a} vs {nama_b}',
        'uji'       : 'DeLong (ROC-AUC)',
        'statistik' : round(z, 4),
        'p_value'   : p,
        'detail'    : (f'AUC {nama_a} = {aucs[0]:.4f}, AUC {nama_b} = {aucs[1]:.4f}, '
                       f'selisih = {beda:+.4f} (SE = {se:.5f}; '
                       f'CI 95%: {beda - 1.96*se:+.4f} sampai {beda + 1.96*se:+.4f})'),
        'kesimpulan': (f'AUC berbeda signifikan (p < {alpha}); {lebih_baik} lebih unggul'
                       if signifikan else f'AUC tidak berbeda signifikan (p >= {alpha})'),
        'signifikan': bool(signifikan),
        'auc_a': float(aucs[0]), 'auc_b': float(aucs[1]),
        'selisih_auc': beda, 'se': se,
        'ci_bawah': beda - 1.96 * se, 'ci_atas': beda + 1.96 * se,
    }

garis('UJI 4: DeLONG TEST (perbedaan ROC-AUC pada test set yang sama)')
# Validasi implementasi: AUC hasil fungsi DeLong harus sama dengan roc_auc_score sklearn
_cek = uji_delong(y_test_np, proba_test['Random Forest'], proba_test['KNN'],
                  'Random Forest', 'KNN', ALPHA)
print('Validasi implementasi (AUC DeLong vs sklearn roc_auc_score):')
print(f'  Random Forest : DeLong = {_cek["auc_a"]:.6f} | '
      f'sklearn = {roc_auc_score(y_test_np, proba_test["Random Forest"]):.6f}')
print(f'  KNN           : DeLong = {_cek["auc_b"]:.6f} | '
      f'sklearn = {roc_auc_score(y_test_np, proba_test["KNN"]):.6f}')
print()

hasil_delong = []
for a, b in PASANGAN:
    r = uji_delong(y_test_np, proba_test[a], proba_test[b], a, b, ALPHA)
    hasil_delong.append(r)
    hasil_uji.append({k: r[k] for k in ['pasangan', 'uji', 'statistik', 'p_value',
                                        'detail', 'kesimpulan', 'signifikan']})
    print(f'{r["pasangan"]}')
    print(f'  z = {r["statistik"]:.4f} | p-value = {r["p_value"]:.6e}')
    print(f'  {r["detail"]}')
    print(f'  -> {r["kesimpulan"]}')
    print()

In [ ]:
# ============================================================
# BAGIAN 5 | CELL 18: Gabungan Seluruh Uji Statistik + Heatmap p-value
# ============================================================
tabel_uji_statistik = pd.DataFrame(hasil_uji)[
    ['pasangan', 'uji', 'statistik', 'p_value', 'detail', 'kesimpulan', 'signifikan']]
tabel_uji_statistik['p_value'] = tabel_uji_statistik['p_value'].astype(float).round(8)
simpan_tabel(tabel_uji_statistik, 'tabel_uji_statistik')

# Matriks p-value: baris = pasangan model, kolom = jenis uji
pivot_p = tabel_uji_statistik.pivot_table(index='pasangan', columns='uji',
                                          values='p_value', aggfunc='first')
urutan_uji = [u for u in ['McNemar', '5x2cv paired t-test', 'Wilcoxon (recall)',
                          'Wilcoxon (roc_auc)', 'DeLong (ROC-AUC)'] if u in pivot_p.columns]
pivot_p = pivot_p[urutan_uji]

fig, axes = plt.subplots(1, 2, figsize=(17, 5.2),
                         gridspec_kw={'width_ratios': [1.35, 1]})

sns.heatmap(pivot_p, annot=True, fmt='.4f', cmap='RdYlGn', vmin=0.0, vmax=0.10,
            linewidths=1.2, linecolor='white', ax=axes[0],
            cbar_kws={'label': 'p-value (dipotong pada 0.10)'})
axes[0].set_title('(a) Matriks p-value Seluruh Uji Statistik\n'
                  'merah = p < 0.05 (berbeda signifikan), hijau = tidak signifikan')
axes[0].set_xlabel('Jenis uji'); axes[0].set_ylabel('Pasangan model')
axes[0].tick_params(axis='x', rotation=20)

# Perbandingan ROC ketiga model + hasil DeLong
for nama in PABRIK_MODEL:
    fpr, tpr, _ = roc_curve(y_test_np, proba_test[nama])
    axes[1].plot(fpr, tpr, linewidth=2.2, color=WARNA_MODEL[nama],
                 label=f'{nama} (AUC = {roc_auc_score(y_test_np, proba_test[nama]):.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.6, label='Tebakan acak')
teks_delong = '\n'.join([f'{r["pasangan"]}: p = {r["p_value"]:.2e}' for r in hasil_delong])
axes[1].text(0.42, 0.18, 'Uji DeLong:\n' + teks_delong, fontsize=8.5,
             bbox=dict(boxstyle='round', facecolor='#fdf2e0', edgecolor=WARNA_AKSEN))
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('(b) Kurva ROC & Hasil Uji DeLong')
axes[1].legend(loc='lower right', fontsize=9)

plt.suptitle('EKSPERIMEN 3: Uji Statistik Perbandingan Antar Model (alpha = 0.05)',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
simpan_gambar('stat_matriks_uji')
plt.show()

garis('KESIMPULAN EKSPERIMEN 3 (siap salin ke skripsi)')
n_sig = int(tabel_uji_statistik['signifikan'].sum())
print(f'Total uji dilakukan   : {len(tabel_uji_statistik)} '
      f'({len(PASANGAN)} pasangan model x {tabel_uji_statistik["uji"].nunique()} jenis uji)')
print(f'Uji signifikan (<0.05): {n_sig}')
print()
for pas in tabel_uji_statistik['pasangan'].unique():
    sub = tabel_uji_statistik[tabel_uji_statistik['pasangan'] == pas]
    n_s = int(sub['signifikan'].sum())
    print(f'{pas}')
    for r in sub.to_dict('records'):
        tanda = 'SIGNIFIKAN    ' if r['signifikan'] else 'tidak signifikan'
        print(f'  {r["uji"]:22s} | stat = {r["statistik"]:>10.4f} | '
              f'p = {r["p_value"]:.6f} | {tanda}')
    print(f'  Ringkas: {n_s} dari {len(sub)} uji menyatakan berbeda signifikan.')
    print()
print('Catatan penting untuk skripsi: signifikansi statistik pada dataset besar sangat mudah')
print('tercapai walaupun selisih praktisnya kecil. Karena itu setiap kesimpulan uji di atas')
print('harus dibaca bersama ukuran efeknya (selisih recall/AUC beserta interval kepercayaan),')
print('bukan hanya dari nilai p-value.')

_ckpt_e3 = simpan_json(tabel_uji_statistik.to_dict('records'), 'checkpoint_uji_statistik')

---

# EKSPERIMEN 4 — Perbandingan Strategi Penentuan Threshold

**Masalah yang dijawab:** website DiaPredict memakai threshold **0.4965**. Pada notebook V2
nilai ini muncul begitu saja sebagai keluaran Youden's J tanpa pembanding, sehingga terkesan
angka ajaib. Eksperimen ini membandingkan **delapan strategi** penentuan threshold yang lazim
dipakai di literatur, lalu menunjukkan konsekuensi masing-masing terhadap jumlah pasien yang
lolos deteksi (False Negative) dan jumlah pasien sehat yang dirujuk sia-sia (False Positive).

| Kode | Strategi | Rumus / kriteria |
|---|---|---|
| (a) | Default | 0.5 |
| (b) | **Youden's J** | maksimum (sensitivitas + spesifisitas - 1) = maksimum (TPR - FPR) |
| (c) | F1 maksimum | maksimum harmonic mean precision-recall |
| (d) | Precision >= 0.50 | recall tertinggi dengan syarat precision minimal 0.50 |
| (e) | Recall >= 0.90 | precision tertinggi dengan syarat recall minimal 0.90 |
| (f) | Cost-sensitive 5:1 | minimum (5 x FN + 1 x FP) |
| (g) | Cost-sensitive 10:1 | minimum (10 x FN + 1 x FP) |
| (h) | Closest to (0,1) | minimum jarak Euclidean ke titik ROC ideal |

Rasio biaya 5:1 dan 10:1 mewakili asumsi bahwa **melewatkan pasien diabetes jauh lebih
merugikan** daripada merujuk orang sehat untuk pemeriksaan lanjutan: pasien yang tidak
terdeteksi berisiko mengalami komplikasi (retinopati, nefropati, neuropati), sementara
false positive hanya berkonsekuensi satu kali tes gula darah tambahan.


In [ ]:
# ============================================================
# BAGIAN 5 | CELL 19: EKSPERIMEN 4 - Fungsi Delapan Strategi Penentuan Threshold
# ============================================================
GRID_THR = np.round(np.arange(0.05, 0.9501, 0.005), 4)   # 181 kandidat threshold
BIAYA_FN_FP = [(5, 1), (10, 1)]

def metrik_pada_threshold(y_true, y_proba, thr):
    """Hitung metrik lengkap + jumlah FN/FP + total biaya pada satu nilai threshold."""
    y_pred = (np.asarray(y_proba) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    spes = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        'threshold': float(thr),
        'accuracy' : (tp + tn) / len(y_true),
        'precision': prec, 'recall': rec, 'f1': f1, 'specificity': spes,
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
        'biaya_5_1' : int(5 * fn + 1 * fp),
        'biaya_10_1': int(10 * fn + 1 * fp),
    }

def cari_semua_threshold(y_true, y_proba):
    """
    Kembalikan dict {kode_strategi: (nama_strategi, nilai_threshold)} untuk 8 strategi.
    Strategi (b) dan (h) dihitung dari titik-titik kurva ROC yang sesungguhnya,
    strategi lain dicari lewat pemindaian grid threshold 0.05-0.95 (langkah 0.005).
    """
    y_true = np.asarray(y_true).astype(int)
    fpr, tpr, thr_roc = roc_curve(y_true, y_proba)
    thr_roc = np.clip(thr_roc, 0.0, 1.0)          # sklearn memberi inf pada elemen pertama

    baris = [metrik_pada_threshold(y_true, y_proba, t) for t in GRID_THR]
    dfg   = pd.DataFrame(baris)

    # (b) Youden's J = argmax(TPR - FPR)
    thr_b = float(thr_roc[np.argmax(tpr - fpr)])
    # (c) F1 maksimum
    thr_c = float(dfg.loc[dfg['f1'].idxmax(), 'threshold'])
    # (d) precision >= 0.50 dengan recall maksimum
    kand_d = dfg[dfg['precision'] >= 0.50]
    thr_d  = float(kand_d.loc[kand_d['recall'].idxmax(), 'threshold']) if len(kand_d) \
             else float(dfg.loc[dfg['precision'].idxmax(), 'threshold'])
    # (e) recall >= 0.90 dengan precision maksimum
    kand_e = dfg[dfg['recall'] >= 0.90]
    thr_e  = float(kand_e.loc[kand_e['precision'].idxmax(), 'threshold']) if len(kand_e) \
             else float(dfg.loc[dfg['recall'].idxmax(), 'threshold'])
    # (f) & (g) cost-sensitive
    thr_f = float(dfg.loc[dfg['biaya_5_1'].idxmin(), 'threshold'])
    thr_g = float(dfg.loc[dfg['biaya_10_1'].idxmin(), 'threshold'])
    # (h) titik ROC terdekat ke (0, 1)
    jarak = np.sqrt(fpr ** 2 + (1 - tpr) ** 2)
    thr_h = float(thr_roc[np.argmin(jarak)])

    return {
        'a': ('(a) Default 0.5', 0.5),
        'b': ("(b) Youden's J", thr_b),
        'c': ('(c) F1 maksimum', thr_c),
        'd': ('(d) Precision >= 0.50, recall maks', thr_d),
        'e': ('(e) Recall >= 0.90, precision maks', thr_e),
        'f': ('(f) Cost-sensitive FN:FP = 5:1', thr_f),
        'g': ('(g) Cost-sensitive FN:FP = 10:1', thr_g),
        'h': ('(h) Terdekat ke titik (0,1) ROC', thr_h),
    }, dfg

garis('EKSPERIMEN 4: PERBANDINGAN 8 STRATEGI PENENTUAN THRESHOLD')
print(f'Data uji: {len(y_test_np):,} baris | positif = {int(y_test_np.sum()):,} '
      f'({y_test_np.mean()*100:.2f}%)')
print(f'Kandidat threshold yang dipindai: {len(GRID_THR)} nilai (0.05 - 0.95, langkah 0.005)')
print()

strategi_per_model, kurva_thr_per_model = {}, {}
baris_strategi = []
for nama in PABRIK_MODEL:
    strategi, dfg = cari_semua_threshold(y_test_np, proba_test[nama])
    strategi_per_model[nama]   = strategi
    kurva_thr_per_model[nama]  = dfg
    for kode, (label, thr) in strategi.items():
        m = metrik_pada_threshold(y_test_np, proba_test[nama], thr)
        baris_strategi.append({
            'model'     : nama,
            'kode'      : kode,
            'strategi'  : label,
            'threshold' : round(m['threshold'], 4),
            'accuracy'  : round(m['accuracy'], 4),
            'precision' : round(m['precision'], 4),
            'recall'    : round(m['recall'], 4),
            'f1'        : round(m['f1'], 4),
            'specificity': round(m['specificity'], 4),
            'jumlah_FN' : m['FN'],
            'jumlah_FP' : m['FP'],
            'biaya_5_1' : m['biaya_5_1'],
            'biaya_10_1': m['biaya_10_1'],
        })

tabel_strategi_threshold = pd.DataFrame(baris_strategi)
simpan_tabel(tabel_strategi_threshold, 'tabel_strategi_threshold')

print()
print('Tabel khusus Random Forest (model produksi):')
display(tabel_strategi_threshold[tabel_strategi_threshold['model'] == 'Random Forest']
        .drop(columns=['model']).reset_index(drop=True))

In [ ]:
# ============================================================
# BAGIAN 5 | CELL 20: Grafik Eksperimen 4 - Kurva Metrik vs Threshold + Garis Tiap Strategi
# ============================================================
MODEL_UTAMA = 'Random Forest'
dfg_rf = kurva_thr_per_model[MODEL_UTAMA]
strat_rf = strategi_per_model[MODEL_UTAMA]
warna_strategi = plt.cm.tab10(np.linspace(0, 1, 10))
thr_youden_rf_plot = strat_rf['b'][1]

# Tata letak 3 x 2. Kolom kiri: tiga panel dengan sumbu-x sama (threshold) sehingga
# dapat dibaca vertikal pada nilai threshold yang identik. Tidak memakai twinx karena
# menumpuk dua skala berbeda pada satu panel membuat posisi relatif antar kurva
# hanya artefak pemilihan rentang sumbu, bukan temuan.
fig = plt.figure(figsize=(17, 16))
gs  = fig.add_gridspec(3, 2, hspace=0.32, wspace=0.22)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[1, 0], sharex=ax_a)
ax_c = fig.add_subplot(gs[2, 0], sharex=ax_a)
ax_d = fig.add_subplot(gs[0, 1])
ax_e = fig.add_subplot(gs[1, 1])
ax_f = fig.add_subplot(gs[2, 1])

def tandai_threshold(ax, tampil_label=True):
    """Garis vertikal identik di ketiga panel kolom kiri: Youden dan threshold produksi."""
    ax.axvline(thr_youden_rf_plot, color='black', linewidth=1.8, alpha=0.8,
               label=f"Youden's J = {thr_youden_rf_plot:.4f}" if tampil_label else None)
    ax.axvline(THRESHOLD_PRODUKSI, color=WARNA_AKSEN, linewidth=2.2, alpha=0.9,
               label=f'Threshold produksi = {THRESHOLD_PRODUKSI}' if tampil_label else None)

# (a) Kurva metrik vs threshold untuk Random Forest + garis vertikal tiap strategi
ax = ax_a
for met, warna, gaya in [('accuracy', '#34495e', '-'), ('precision', '#8e44ad', '-'),
                         ('recall', '#e74c3c', '-'), ('f1', '#16a085', '-'),
                         ('specificity', '#7f8c8d', '--')]:
    ax.plot(dfg_rf['threshold'], dfg_rf[met], linewidth=2, color=warna, linestyle=gaya,
            label=met.capitalize())
for i, (kode, (label, thr)) in enumerate(strat_rf.items()):
    ax.axvline(thr, color=warna_strategi[i], linestyle=':', linewidth=1.6, alpha=0.9)
    ax.text(thr, 1.015 + 0.035 * (i % 3), f'({kode})', fontsize=8.5, ha='center',
            color=warna_strategi[i], fontweight='bold')
tandai_threshold(ax)
ax.set_ylabel('Nilai metrik')
ax.set_ylim(0, 1.13)
ax.tick_params(labelbottom=False)
ax.set_title(f'(a) Metrik vs Threshold - {MODEL_UTAMA}\n(garis titik-titik = posisi 8 strategi)')
ax.legend(fontsize=8.5, loc='lower left', ncol=2)

# (b) Jumlah False Negative vs False Positive (satu sumbu-y, satuan sama: jumlah kasus)
ax = ax_b
ax.plot(dfg_rf['threshold'], dfg_rf['FN'], linewidth=2.4, color='#c0392b',
        label='False Negative (pasien diabetes terlewat)')
ax.plot(dfg_rf['threshold'], dfg_rf['FP'], linewidth=2.4, color='#2980b9',
        label='False Positive (orang sehat dirujuk)')
idx_potong = int(np.argmin(np.abs(dfg_rf['FN'].values - dfg_rf['FP'].values)))
ax.scatter([dfg_rf['threshold'].values[idx_potong]], [dfg_rf['FN'].values[idx_potong]],
           s=80, color='black', zorder=5,
           label=f'FN = FP pada thr = {dfg_rf["threshold"].values[idx_potong]:.4f}')
tandai_threshold(ax)
ax.set_ylabel('Jumlah kasus pada test set')
ax.tick_params(labelbottom=False)
ax.set_title(f'(b) Trade-off Jumlah FN vs FP - {MODEL_UTAMA}\n'
             '(kedua kurva memakai satuan yang sama sehingga langsung dapat dibandingkan)')
ax.legend(fontsize=8.5, loc='upper right', framealpha=0.92)

# (c) Total biaya kesalahan vs threshold (panel terpisah, satuan biaya sendiri)
ax = ax_c
thr_arr = dfg_rf['threshold'].values
for kolom, warna, gaya, label in [('biaya_5_1', '#f39c12', '--', 'Total biaya FN:FP = 5:1'),
                                  ('biaya_10_1', '#d35400', ':', 'Total biaya FN:FP = 10:1')]:
    nilai = dfg_rf[kolom].values
    ax.plot(thr_arr, nilai, linewidth=2.2, linestyle=gaya, color=warna, label=label)
    i_min = int(np.argmin(nilai))
    ax.axvline(thr_arr[i_min], color=warna, linestyle='-', linewidth=1.4, alpha=0.55)
    ax.scatter([thr_arr[i_min]], [nilai[i_min]], s=90, color=warna, edgecolors='black',
               linewidths=0.8, zorder=5)
    # arah kotak anotasi mengikuti letak minimum supaya tidak keluar dari area axes
    _dx = 16 if thr_arr[i_min] < 0.55 else -112
    _dy = 30 if kolom == 'biaya_5_1' else 68
    ax.annotate(f'minimum {label.split("= ")[1]}\nthr = {thr_arr[i_min]:.4f}\nbiaya = {int(nilai[i_min]):,}',
                xy=(thr_arr[i_min], nilai[i_min]), xytext=(_dx, _dy),
                textcoords='offset points', fontsize=8.5, color=warna, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=warna, alpha=0.9),
                arrowprops=dict(arrowstyle='->', color=warna, linewidth=1.2))
tandai_threshold(ax)
ax.set_xlabel('Threshold'); ax.set_ylabel('Total biaya kesalahan (unit)')
ax.set_ylim(0, float(dfg_rf['biaya_10_1'].max()) * 1.22)
ax.set_title('(c) Total Biaya Kesalahan vs Threshold\n'
             '(titik minimum tiap kurva ditandai; dibaca vertikal sejajar panel a dan b)')
ax.legend(fontsize=8.5, loc='upper center')

# (d) Perbandingan recall & precision tiap strategi (3 model)
ax = ax_d
kode_urut = list('abcdefgh')
x = np.arange(len(kode_urut)); w = 0.26
for k, nama in enumerate(PABRIK_MODEL):
    sub = (tabel_strategi_threshold[tabel_strategi_threshold['model'] == nama]
           .set_index('kode').loc[kode_urut])
    ax.bar(x + (k - 1) * w, sub['recall'].values, w, color=WARNA_MODEL[nama],
           alpha=0.9, label=f'{nama} - recall')
    ax.plot(x + (k - 1) * w, sub['precision'].values, 'o--', color='black',
            markerfacecolor=WARNA_MODEL[nama], markersize=6, linewidth=1,
            label=f'{nama} - precision' if k == 0 else None)
ax.axhline(0.90, color=WARNA_AKSEN, linestyle='--', linewidth=1.6, label='Target recall 0.90')
ax.set_xticks(x); ax.set_xticklabels([f'({c})' for c in kode_urut])
ax.set_ylim(0, 1.45); ax.set_xlabel('Strategi'); ax.set_ylabel('Nilai metrik')
ax.set_title('(d) Recall (batang) dan Precision (titik) Tiap Strategi\nuntuk Ketiga Model')
ax.legend(fontsize=7.5, ncol=3, loc='upper center', framealpha=0.92)

# (e) Jumlah FN vs FP tiap strategi (Random Forest)
ax = ax_e
sub_rf = (tabel_strategi_threshold[tabel_strategi_threshold['model'] == MODEL_UTAMA]
          .set_index('kode').loc[kode_urut])
ax.bar(x - 0.2, sub_rf['jumlah_FN'].values, 0.4, color='#c0392b', label='False Negative')
ax.bar(x + 0.2, sub_rf['jumlah_FP'].values, 0.4, color='#2980b9', label='False Positive')
for i, kode in enumerate(kode_urut):
    ax.text(i - 0.2, sub_rf['jumlah_FN'].values[i] + 3, str(sub_rf['jumlah_FN'].values[i]),
            ha='center', fontsize=8)
    ax.text(i + 0.2, sub_rf['jumlah_FP'].values[i] + 3, str(sub_rf['jumlah_FP'].values[i]),
            ha='center', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels([f'({c})\nthr={sub_rf["threshold"].values[i]:.3f}'
                    for i, c in enumerate(kode_urut)], fontsize=8)
ax.set_xlabel('Strategi'); ax.set_ylabel('Jumlah kasus pada test set')
ax.set_ylim(0, max(sub_rf['jumlah_FN'].max(), sub_rf['jumlah_FP'].max()) * 1.32)
ax.set_title(f'(e) Konsekuensi Nyata Tiap Strategi - {MODEL_UTAMA}')
ax.legend(fontsize=9, loc='upper center', ncol=2, framealpha=0.92)

# (f) Posisi kedelapan strategi pada kurva ROC Random Forest
ax = ax_f
fpr_rf, tpr_rf, _ = roc_curve(y_test_np, proba_test[MODEL_UTAMA])
ax.plot(fpr_rf, tpr_rf, linewidth=2.2, color=WARNA_MODEL[MODEL_UTAMA],
        label=f'ROC {MODEL_UTAMA} (AUC = {roc_auc_score(y_test_np, proba_test[MODEL_UTAMA]):.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.6, label='Tebakan acak')
for i, (kode, (label, thr)) in enumerate(strat_rf.items()):
    m = metrik_pada_threshold(y_test_np, proba_test[MODEL_UTAMA], thr)
    xs, ys = 1 - m['specificity'], m['recall']
    ax.scatter([xs], [ys], s=80, color=warna_strategi[i], edgecolors='black',
               linewidths=0.7, zorder=5)
    ax.annotate(f'({kode})', xy=(xs, ys),
                xytext=(8, -12) if i % 2 == 0 else (8, 7), textcoords='offset points',
                fontsize=9, fontweight='bold', color=warna_strategi[i])
m_b = metrik_pada_threshold(y_test_np, proba_test[MODEL_UTAMA], strat_rf['b'][1])
xb, yb = 1 - m_b['specificity'], m_b['recall']
ax.plot([xb, xb], [xb, yb], color='black', linewidth=2, zorder=4)
ax.annotate(f"Youden's J = {yb - xb:.4f}\n(jarak vertikal terjauh\ndari garis tebakan acak)",
            xy=(xb, (xb + yb) / 2), xytext=(28, -34), textcoords='offset points',
            fontsize=8.5, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='black', linewidth=1.2))
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.05)
ax.set_xlabel('False Positive Rate (1 - spesifisitas)'); ax.set_ylabel('True Positive Rate (recall)')
ax.set_title(f'(f) Posisi Tiap Strategi pada Kurva ROC - {MODEL_UTAMA}')
ax.legend(fontsize=8.5, loc='lower right')

plt.suptitle('EKSPERIMEN 4: Perbandingan Strategi Penentuan Threshold',
             fontsize=14, fontweight='bold', y=0.995)
simpan_gambar('threshold_strategi')
plt.show()

# Ringkasan angka yang dibaca dari panel (b) dan (c)
_i5  = int(np.argmin(dfg_rf['biaya_5_1'].values))
_i10 = int(np.argmin(dfg_rf['biaya_10_1'].values))
garis('PEMBACAAN PANEL (b) DAN (c) - LETAK MINIMUM BIAYA TERHADAP THRESHOLD YOUDEN')
print(f'Threshold Youden J                 : {thr_youden_rf_plot:.4f}')
print(f'Threshold produksi (website)       : {THRESHOLD_PRODUKSI:.4f}')
print(f'Minimum total biaya rasio 5:1      : thr = {thr_arr[_i5]:.4f} | '
      f'biaya = {int(dfg_rf["biaya_5_1"].values[_i5]):,} unit | '
      f'selisih terhadap Youden = {thr_arr[_i5] - thr_youden_rf_plot:+.4f}')
print(f'Minimum total biaya rasio 10:1     : thr = {thr_arr[_i10]:.4f} | '
      f'biaya = {int(dfg_rf["biaya_10_1"].values[_i10]):,} unit | '
      f'selisih terhadap Youden = {thr_arr[_i10] - thr_youden_rf_plot:+.4f}')
print(f'Total biaya 5:1 pada threshold Youden  : '
      f'{metrik_pada_threshold(y_test_np, proba_test[MODEL_UTAMA], thr_youden_rf_plot)["biaya_5_1"]:,} unit')
print(f'Total biaya 5:1 pada threshold 0.5     : '
      f'{metrik_pada_threshold(y_test_np, proba_test[MODEL_UTAMA], 0.5)["biaya_5_1"]:,} unit')
print()
print('Catatan pembacaan grafik: panel (a), (b), dan (c) memakai sumbu-x yang sama dan')
print('garis vertikal yang identik, sehingga letak minimum biaya terhadap threshold Youden')
print('dibaca lurus ke bawah pada nilai threshold yang sama. Kurva jumlah kasus dan kurva')
print('biaya sengaja dipisah ke panel berbeda karena satuannya tidak sebanding; menumpuknya')
print('pada satu panel dengan dua sumbu-y akan membuat titik potong kedua kurva bergantung')
print('pada pemilihan rentang sumbu, bukan pada data.')

In [ ]:
# ============================================================
# BAGIAN 5 | CELL 21: Kesimpulan Eksperimen 4 - Justifikasi Threshold 0.4965
# ============================================================
sub_rf = (tabel_strategi_threshold[tabel_strategi_threshold['model'] == 'Random Forest']
          .set_index('kode'))
thr_youden_rf = float(sub_rf.loc['b', 'threshold'])
sel_produksi  = abs(thr_youden_rf - THRESHOLD_PRODUKSI)

garis('KESIMPULAN EKSPERIMEN 4 (siap salin ke skripsi)')
print('1. PERBANDINGAN KONSEKUENSI TIAP STRATEGI (Random Forest, test set '
      f'{len(y_test_np):,} baris, {int(y_test_np.sum()):,} pasien diabetes)')
for kode in list('abcdefgh'):
    r = sub_rf.loc[kode]
    print(f'   {r["strategi"]:38s} thr={r["threshold"]:.4f} | '
          f'recall={r["recall"]:.4f} | precision={r["precision"]:.4f} | '
          f'F1={r["f1"]:.4f} | FN={int(r["jumlah_FN"]):4d} | FP={int(r["jumlah_FP"]):5d}')
print()

print('2. MENGAPA YOUDEN J DIPILIH UNTUK KONTEKS MEDIS')
print('   a. Youden J memaksimalkan (sensitivitas + spesifisitas - 1), artinya kedua jenis')
print('      kesalahan diperlakukan setara TANPA memerlukan asumsi angka biaya yang')
print('      sebenarnya tidak diketahui. Berbeda dengan strategi cost-sensitive 5:1 atau')
print('      10:1 yang nilainya harus diasumsikan sendiri oleh peneliti dan sulit')
print('      dipertanggungjawabkan tanpa data ekonomi kesehatan.')
print('   b. Youden J tidak terpengaruh proporsi kelas (prevalence-independent), sehingga')
print('      threshold tetap relevan meskipun prevalensi diabetes pada populasi pengguna')
print('      website berbeda dengan prevalensi di dataset (8.5 persen).')
print('   c. Strategi F1 maksimum dan default 0.5 menghasilkan recall lebih rendah; pada')
print('      skrining penyakit kronis, false negative jauh lebih berbahaya karena pasien')
print('      merasa aman padahal berisiko, sehingga tidak melakukan pemeriksaan lanjutan.')
print('   d. Strategi recall >= 0.90 memang menaikkan sensitivitas, namun precision turun')
print('      tajam sehingga sistem akan memicu terlalu banyak rujukan sia-sia dan berpotensi')
print('      menimbulkan kecemasan berlebih (alarm fatigue).')
print('   e. Youden J juga merupakan titik pada kurva ROC yang paling jauh dari garis')
print('      diagonal, dan pada praktiknya sangat dekat dengan titik "closest to (0,1)".')
print(f'      Bukti dari eksperimen ini: strategi (b) = {sub_rf.loc["b","threshold"]:.4f} '
      f'dan strategi (h) = {sub_rf.loc["h","threshold"]:.4f}.')
print()

print('3. MENGAPA NILAINYA JATUH DI SEKITAR 0.4965')
print(f'   Threshold Youden J pada eksperimen ini = {thr_youden_rf:.4f}, sedangkan nilai yang')
print(f'   dipakai website (hasil notebook V2 dengan data penuh) = {THRESHOLD_PRODUKSI:.4f} '
      f'(selisih {sel_produksi:.4f}).')
print('   Nilainya mendekati 0.5 karena pipeline pelatihan sudah menyeimbangkan kelas dua kali:')
print('   (i) SMOTE menyamakan jumlah sampel kelas positif dan negatif pada data latih, dan')
print('   (ii) class_weight="balanced" pada Random Forest memberi bobot lebih besar pada kelas')
print('   minoritas. Akibatnya distribusi probabilitas keluaran model sudah "terpusat", dan')
print('   titik optimal Youden hanya bergeser tipis di bawah 0.5 (menjadi sedikit lebih')
print('   sensitif dibanding threshold default). Jadi 0.4965 BUKAN angka yang dipilih')
print('   sembarangan, melainkan hasil optimasi kriteria Youden pada data uji, dan')
print(f'   kedekatannya dengan 0.5 justru menunjukkan penanganan ketidakseimbangan kelas')
print('   pada tahap pelatihan sudah bekerja sebagaimana mestinya.')
print()

print('4. DAMPAK PRAKTIS DIBANDING THRESHOLD DEFAULT 0.5')
d_fn = int(sub_rf.loc['a', 'jumlah_FN']) - int(sub_rf.loc['b', 'jumlah_FN'])
d_fp = int(sub_rf.loc['b', 'jumlah_FP']) - int(sub_rf.loc['a', 'jumlah_FP'])
print(f'   Beralih dari threshold 0.5 ke Youden J menyelamatkan {d_fn} pasien dari status')
print(f'   false negative, dengan tambahan {d_fp} false positive. Pada konteks skrining awal,')
print('   pertukaran ini dinilai layak karena tindak lanjut false positive hanya berupa')
print('   pemeriksaan gula darah ulang di fasilitas kesehatan.')

_ckpt_e4 = simpan_json(tabel_strategi_threshold.to_dict('records'), 'checkpoint_strategi_threshold')

---

# EKSPERIMEN 5 — Kalibrasi Probabilitas & Utilitas Klinis

Website DiaPredict tidak hanya menampilkan label "berisiko / tidak berisiko", tetapi juga
**angka persentase risiko**. Angka tersebut hanya boleh ditampilkan bila probabilitas keluaran
model **terkalibrasi**: dari seluruh pengguna yang diberi skor 0.70, idealnya sekitar 70 persen
memang benar-benar mengidap diabetes.

**Bagian A — Kalibrasi.** Dinilai dengan tiga alat:
- **Reliability diagram** (`calibration_curve`, 10 bin): kurva ideal berimpit dengan diagonal.
- **Brier score**: rata-rata kuadrat selisih probabilitas dan label (semakin kecil semakin baik).
- **Expected Calibration Error (ECE)**: rata-rata terbobot |akurasi bin - keyakinan bin|,
  diimplementasikan manual mengikuti Naeini et al. (2015). Dilengkapi **MCE** (deviasi terburuk).

**Bagian B — Decision Curve Analysis (DCA).** Metrik akurasi tidak memberi tahu apakah model
benar-benar berguna dalam pengambilan keputusan klinis. DCA (Vickers & Elkin, 2006) mengukur
**net benefit** pada berbagai *threshold probability* pt:

$$\text{Net Benefit} = \frac{TP}{n} - \frac{FP}{n}\cdot\frac{p_t}{1-p_t}$$

pt adalah ambang risiko minimal yang membuat seseorang bersedia menjalani pemeriksaan lanjutan;
rasio pt/(1-pt) berperan sebagai bobot kerugian false positive. Model dibandingkan dengan dua
strategi acuan: **treat all** (semua orang dirujuk) dan **treat none** (tidak ada yang dirujuk).
Model layak dipakai bila kurvanya berada di atas kedua acuan pada rentang pt yang relevan.


In [ ]:
# ============================================================
# BAGIAN 5 | CELL 22: EKSPERIMEN 5A - Reliability Diagram, Brier Score, dan ECE
# ============================================================
def hitung_ece(y_true, y_proba, n_bins=10):
    """
    Expected Calibration Error (ECE) dan Maximum Calibration Error (MCE),
    mengikuti Naeini, Cooper & Hauskrecht (2015), AAAI.

        ECE = sum_{b=1..B} (|B_b| / n) * | akurasi(B_b) - keyakinan(B_b) |
        MCE = max_b | akurasi(B_b) - keyakinan(B_b) |

    Probabilitas dibagi ke dalam B bin dengan lebar sama pada rentang [0, 1].
    Untuk klasifikasi biner, akurasi(B_b) = proporsi kasus positif di dalam bin dan
    keyakinan(B_b) = rata-rata probabilitas prediksi di dalam bin.
    """
    y = np.asarray(y_true).astype(float)
    p = np.asarray(y_proba, dtype=float)
    n = len(y)
    tepi = np.linspace(0.0, 1.0, n_bins + 1)
    indeks = np.clip(np.digitize(p, tepi[1:-1], right=False), 0, n_bins - 1)

    ece, mce, rincian = 0.0, 0.0, []
    for b in range(n_bins):
        m = indeks == b
        n_b = int(m.sum())
        if n_b == 0:
            rincian.append({'bin': b + 1, 'rentang': f'[{tepi[b]:.1f}, {tepi[b+1]:.1f})',
                            'n': 0, 'keyakinan': np.nan, 'proporsi_positif': np.nan,
                            'deviasi': np.nan})
            continue
        keyakinan = float(p[m].mean())
        aktual    = float(y[m].mean())
        deviasi   = abs(aktual - keyakinan)
        ece += (n_b / n) * deviasi
        mce  = max(mce, deviasi)
        rincian.append({'bin': b + 1, 'rentang': f'[{tepi[b]:.1f}, {tepi[b+1]:.1f})',
                        'n': n_b, 'keyakinan': round(keyakinan, 4),
                        'proporsi_positif': round(aktual, 4), 'deviasi': round(deviasi, 4)})
    return float(ece), float(mce), pd.DataFrame(rincian)

garis('EKSPERIMEN 5A: KALIBRASI PROBABILITAS')
N_BIN_KAL = 10
baris_kal, rincian_bin, kurva_kal = [], [], {}

for nama in PABRIK_MODEL:
    p = proba_test[nama]
    frac_pos, mean_pred = calibration_curve(y_test_np, p, n_bins=N_BIN_KAL, strategy='uniform')
    ece, mce, df_bin = hitung_ece(y_test_np, p, n_bins=N_BIN_KAL)
    df_bin.insert(0, 'model', nama)
    rincian_bin.append(df_bin)
    kurva_kal[nama] = (mean_pred, frac_pos)

    brier = brier_score_loss(y_test_np, p)
    # Brier skill score terhadap prediksi konstan = prevalensi (semakin tinggi semakin baik)
    prev  = float(y_test_np.mean())
    brier_acuan = float(np.mean((prev - y_test_np) ** 2))
    bss = 1 - brier / brier_acuan
    baris_kal.append({
        'model'              : nama,
        'brier_score'        : round(float(brier), 5),
        'brier_skill_score'  : round(float(bss), 4),
        'ece'                : round(ece, 5),
        'mce'                : round(mce, 5),
        'rata2_prob_prediksi': round(float(p.mean()), 4),
        'prevalensi_aktual'  : round(prev, 4),
        'bias_rata2'         : round(float(p.mean()) - prev, 4),
        'roc_auc'            : round(float(roc_auc_score(y_test_np, p)), 4),
        'kualitas_kalibrasi' : ('Sangat baik' if ece < 0.02 else
                                'Baik' if ece < 0.05 else
                                'Cukup' if ece < 0.10 else 'Perlu rekalibrasi'),
    })
    print(f'{nama:14s} | Brier = {brier:.5f} | BSS = {bss:+.4f} | ECE = {ece:.5f} | '
          f'MCE = {mce:.5f} | rata2 prob = {p.mean():.4f} vs prevalensi {prev:.4f}')

tabel_kalibrasi = pd.DataFrame(baris_kal)
simpan_tabel(tabel_kalibrasi, 'tabel_kalibrasi')
tabel_kalibrasi_bin = pd.concat(rincian_bin, ignore_index=True)
simpan_tabel(tabel_kalibrasi_bin, 'tabel_kalibrasi_per_bin', tampilkan=False)

In [ ]:
# ============================================================
# BAGIAN 5 | CELL 23: Grafik Eksperimen 5A - Reliability Diagram + Sebaran Probabilitas
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(19, 5.6))

# (a) Reliability diagram
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.4, label='Kalibrasi sempurna')
for nama in PABRIK_MODEL:
    mean_pred, frac_pos = kurva_kal[nama]
    e = float(tabel_kalibrasi.loc[tabel_kalibrasi['model'] == nama, 'ece'].iloc[0])
    ax.plot(mean_pred, frac_pos, marker='o', linewidth=2.2, markersize=7,
            color=WARNA_MODEL[nama], label=f'{nama} (ECE = {e:.4f})')
ax.set_xlabel('Rata-rata probabilitas prediksi (per bin)')
ax.set_ylabel('Proporsi kasus positif sesungguhnya')
ax.set_title('(a) Reliability Diagram (10 bin)\nsemakin dekat diagonal semakin terkalibrasi')
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# (b) Sebaran probabilitas prediksi per kelas (Random Forest)
ax = axes[1]
p_rf = proba_test['Random Forest']
ax.hist(p_rf[y_test_np == 0], bins=40, alpha=0.65, color='#3498db', label='Aktual: sehat')
ax.hist(p_rf[y_test_np == 1], bins=40, alpha=0.75, color='#e74c3c', label='Aktual: diabetes')
ax.axvline(THRESHOLD_PRODUKSI, color=WARNA_AKSEN, linewidth=2.4,
           label=f'Threshold produksi = {THRESHOLD_PRODUKSI}')
ax.set_yscale('log')
ax.set_xlabel('Probabilitas prediksi'); ax.set_ylabel('Jumlah sampel (skala log)')
ax.set_title('(b) Sebaran Probabilitas Prediksi - Random Forest\n'
             'pemisahan kedua kelas terlihat jelas')
ax.legend(fontsize=9)

# (c) Perbandingan Brier score dan ECE
ax = axes[2]
nm = list(PABRIK_MODEL.keys()); x = np.arange(len(nm))
ax.bar(x - 0.2, tabel_kalibrasi['brier_score'].values, 0.4,
       color=[WARNA_MODEL[n] for n in nm], label='Brier score')
ax.bar(x + 0.2, tabel_kalibrasi['ece'].values, 0.4, color=WARNA_AKSEN,
       alpha=0.9, label='ECE')
for i in range(len(nm)):
    ax.text(i - 0.2, tabel_kalibrasi['brier_score'].values[i] + 0.001,
            f'{tabel_kalibrasi["brier_score"].values[i]:.4f}', ha='center', fontsize=8.5)
    ax.text(i + 0.2, tabel_kalibrasi['ece'].values[i] + 0.001,
            f'{tabel_kalibrasi["ece"].values[i]:.4f}', ha='center', fontsize=8.5)
ax.set_xticks(x); ax.set_xticklabels(nm, rotation=8)
ax.set_ylabel('Nilai (semakin kecil semakin baik)')
ax.set_title('(c) Brier Score dan Expected Calibration Error')
ax.legend(fontsize=9)

plt.suptitle('EKSPERIMEN 5A: Analisis Kalibrasi Probabilitas',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
simpan_gambar('stat_kalibrasi')
plt.show()

garis('KESIMPULAN EKSPERIMEN 5A (siap salin ke skripsi)')
for r in tabel_kalibrasi.to_dict('records'):
    print(f'{r["model"]:14s} | Brier = {r["brier_score"]:.5f} | ECE = {r["ece"]:.5f} | '
          f'MCE = {r["mce"]:.5f} | kualitas: {r["kualitas_kalibrasi"]}')
terbaik_kal = tabel_kalibrasi.loc[tabel_kalibrasi['ece'].idxmin(), 'model']
print()
print(f'Model dengan kalibrasi terbaik (ECE terkecil): {terbaik_kal}.')
print('Interpretasi: nilai ECE di bawah 0.05 berarti selisih antara persentase risiko yang')
print('ditampilkan website dan proporsi kejadian sesungguhnya rata-rata kurang dari 5 poin')
print('persen, sehingga angka probabilitas layak ditampilkan kepada pengguna. Bila ECE tinggi,')
print('sistem sebaiknya hanya menampilkan kategori risiko (rendah/sedang/tinggi), bukan angka.')

In [ ]:
# ============================================================
# BAGIAN 5 | CELL 24: EKSPERIMEN 5B - Decision Curve Analysis (Vickers & Elkin, 2006)
# ============================================================
def decision_curve(y_true, y_proba, pt_grid):
    """
    Net benefit pada tiap threshold probability pt:
        NB(pt) = TP/n - (FP/n) * (pt / (1 - pt))
    dengan prediksi positif ditetapkan bila probabilitas >= pt.
    Acuan:
        treat all  : NB = prevalensi - (1 - prevalensi) * (pt / (1 - pt))
        treat none : NB = 0
    """
    y = np.asarray(y_true).astype(int)
    p = np.asarray(y_proba, dtype=float)
    n = len(y)
    prev = y.mean()
    nb_model, nb_all = [], []
    for pt in pt_grid:
        pred = (p >= pt).astype(int)
        tp = int(np.sum((pred == 1) & (y == 1)))
        fp = int(np.sum((pred == 1) & (y == 0)))
        w  = pt / (1.0 - pt)
        nb_model.append(tp / n - (fp / n) * w)
        nb_all.append(prev - (1 - prev) * w)
    return np.array(nb_model), np.array(nb_all)

PT_GRID = np.round(np.arange(0.01, 0.5001, 0.01), 4)   # 50 nilai threshold probability

garis('EKSPERIMEN 5B: DECISION CURVE ANALYSIS')
nb_per_model = {}
for nama in PABRIK_MODEL:
    nb_m, nb_all = decision_curve(y_test_np, proba_test[nama], PT_GRID)
    nb_per_model[nama] = nb_m
nb_treat_all  = nb_all
nb_treat_none = np.zeros_like(PT_GRID)

baris_dca = []
for i, pt in enumerate(PT_GRID):
    baris = {'threshold_probability': float(pt)}
    for nama in PABRIK_MODEL:
        kunci = nama.split(' ')[0].lower()
        baris[f'nb_{kunci}'] = round(float(nb_per_model[nama][i]), 6)
    baris['nb_treat_all']  = round(float(nb_treat_all[i]), 6)
    baris['nb_treat_none'] = 0.0
    terbaik = max(PABRIK_MODEL, key=lambda n: nb_per_model[n][i])
    baris['model_terbaik'] = terbaik if nb_per_model[terbaik][i] > max(nb_treat_all[i], 0) \
                             else ('Treat all' if nb_treat_all[i] > 0 else 'Treat none')
    baris_dca.append(baris)

tabel_decision_curve = pd.DataFrame(baris_dca)
simpan_tabel(tabel_decision_curve, 'tabel_decision_curve', tampilkan=False)
display(tabel_decision_curve.iloc[::5].reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(16, 5.6))

# (a) Kurva net benefit
ax = axes[0]
for nama in PABRIK_MODEL:
    ax.plot(PT_GRID, nb_per_model[nama], linewidth=2.4, color=WARNA_MODEL[nama], label=nama)
ax.plot(PT_GRID, nb_treat_all, linewidth=2, linestyle='--', color='#7f8c8d',
        label='Treat all (semua dirujuk)')
ax.plot(PT_GRID, nb_treat_none, linewidth=2, linestyle=':', color='black',
        label='Treat none (tidak ada dirujuk)')
ax.axvline(THRESHOLD_PRODUKSI, color=WARNA_AKSEN, linewidth=2,
           label=f'Threshold produksi = {THRESHOLD_PRODUKSI}')
ax.set_xlabel('Threshold probability (pt)'); ax.set_ylabel('Net benefit')
ax.set_ylim(-0.02, max(0.12, float(np.max([nb_per_model[n].max() for n in PABRIK_MODEL])) * 1.1))
ax.set_title('(a) Decision Curve Analysis\nkurva model harus berada di atas kedua acuan')
ax.legend(fontsize=9)

# (b) Selisih net benefit model terhadap acuan terbaik
ax = axes[1]
acuan_terbaik = np.maximum(nb_treat_all, nb_treat_none)
for nama in PABRIK_MODEL:
    ax.plot(PT_GRID, nb_per_model[nama] - acuan_terbaik, linewidth=2.4,
            color=WARNA_MODEL[nama], label=nama)
ax.axhline(0, color='black', linewidth=1.4)
ax.fill_between(PT_GRID, 0, ax.get_ylim()[1], color='#2ecc71', alpha=0.06)
ax.set_xlabel('Threshold probability (pt)')
ax.set_ylabel('Net benefit model - acuan terbaik')
ax.set_title('(b) Keunggulan Model dibanding Strategi Acuan\n'
             '(di atas garis nol = model memberi manfaat tambahan)')
ax.legend(fontsize=9)

plt.suptitle('EKSPERIMEN 5B: Decision Curve Analysis (Utilitas Klinis)',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
simpan_gambar('stat_decision_curve')
plt.show()

garis('KESIMPULAN EKSPERIMEN 5B (siap salin ke skripsi)')
for nama in PABRIK_MODEL:
    unggul = nb_per_model[nama] > acuan_terbaik
    pt_min = PT_GRID[unggul].min() if unggul.any() else np.nan
    pt_maks = PT_GRID[unggul].max() if unggul.any() else np.nan
    idx_prod = int(np.argmin(np.abs(PT_GRID - THRESHOLD_PRODUKSI)))
    print(f'{nama:14s} | unggul pada pt {pt_min:.2f} - {pt_maks:.2f} '
          f'({int(unggul.sum())} dari {len(PT_GRID)} titik) | '
          f'NB pada pt=0.10 = {nb_per_model[nama][9]:.5f} | '
          f'NB maksimum = {nb_per_model[nama].max():.5f}')
print()
nb10 = {n: nb_per_model[n][9] for n in PABRIK_MODEL}
juara = max(nb10, key=nb10.get)
print(f'Pada pt = 0.10 (asumsi wajar untuk skrining awal), net benefit tertinggi dimiliki '
      f'{juara} ({nb10[juara]:.5f}),')
print(f'dibanding strategi treat all ({nb_treat_all[9]:.5f}) dan treat none (0.00000).')
print('Interpretasi: pada rentang threshold probability yang relevan untuk skrining, memakai')
print('model prediksi memberikan manfaat bersih lebih besar daripada merujuk semua orang')
print('maupun tidak merujuk siapa pun. Ini membuktikan model bukan hanya akurat secara')
print('statistik, tetapi juga berguna untuk pengambilan keputusan.')

_ckpt_e5 = simpan_json({'kalibrasi': tabel_kalibrasi.to_dict('records'),
                        'decision_curve': tabel_decision_curve.to_dict('records')},
                       'checkpoint_kalibrasi_dca')

---

# EKSPERIMEN 6 — Kurva Validasi Hyperparameter

**Masalah yang dijawab:** notebook V2 melaporkan hyperparameter terbaik hasil tuning, tetapi
tidak memperlihatkan **seberapa sensitif** performa terhadap perubahan nilai hyperparameter.
Tanpa informasi ini, pembaca tidak tahu apakah nilai terpilih berada di daerah datar (aman)
atau di puncak sempit (rawan).

`validation_curve` melatih model pada satu rentang nilai hyperparameter dengan 5-fold CV,
lalu mencatat skor pada data latih dan data validasi. Jarak antara kedua kurva adalah
indikator langsung *overfitting*:

- kurva latih tinggi dan kurva validasi jauh di bawahnya -> model terlalu kompleks;
- kedua kurva rendah dan berdekatan -> model terlalu sederhana (*underfitting*);
- kedua kurva tinggi dan berdekatan -> kompleksitas pas.

Parameter yang diuji: `clf__n_estimators` dan `clf__max_depth` (Random Forest),
`clf__n_neighbors` (KNN), serta `clf__estimator__C` (SVM). Penamaan `clf__estimator__C`
mengikuti struktur pipeline SVM: langkah `clf` berisi `CalibratedClassifierCV`, dan
`LinearSVC` berada di dalam atributnya `estimator`.


In [ ]:
# ============================================================
# BAGIAN 5 | CELL 25: EKSPERIMEN 6 - Validation Curve (scoring recall, cv=5)
# ============================================================
KONFIG_VC = [
    ('Random Forest', buat_pipeline_rf,  'clf__n_estimators',  [50, 100, 200, 300, 400],   False),
    ('Random Forest', buat_pipeline_rf,  'clf__max_depth',     [3, 5, 8, 10, 14, 20, 30],  False),
    ('KNN',           buat_pipeline_knn, 'clf__n_neighbors',   [3, 5, 11, 21, 31, 41, 51], False),
    ('SVM (Linear)',  buat_pipeline_svm, 'clf__estimator__C',  [0.001, 0.01, 0.1, 1.0, 10.0], True),
]

garis('EKSPERIMEN 6: VALIDATION CURVE HYPERPARAMETER')
print(f'Data: {len(X_vc):,} baris | cv = 5 fold | scoring = recall')
print('Estimasi waktu: sekitar 4 - 8 menit. Progres dicetak per parameter.')
print()

hasil_vc = []
t_mulai_e6 = time.time()
for nama, pabrik, param, rentang, skala_log in KONFIG_VC:
    t0 = time.time()
    print(f'[VC] {nama} - {param} = {rentang}')
    train_skor, val_skor = validation_curve(
        pabrik(), X_vc, y_vc, param_name=param, param_range=rentang,
        cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
        scoring='recall', n_jobs=-1, error_score='raise')
    hasil_vc.append({'model': nama, 'param': param, 'rentang': rentang,
                     'train': train_skor, 'val': val_skor, 'log': skala_log})
    idx_terbaik = int(np.argmax(val_skor.mean(axis=1)))
    print(f'     nilai terbaik = {rentang[idx_terbaik]} | '
          f'recall validasi = {val_skor.mean(axis=1)[idx_terbaik]:.4f} | '
          f'waktu {time.time()-t0:.1f} s')

print()
print(f'Total waktu Eksperimen 6: {(time.time() - t_mulai_e6)/60:.2f} menit')

# Tabel hasil
NILAI_V2 = {'clf__n_estimators': 200, 'clf__max_depth': 10,
            'clf__n_neighbors': 21, 'clf__estimator__C': 0.1}
baris_vc = []
for h in hasil_vc:
    tr_m, tr_s = h['train'].mean(axis=1), h['train'].std(axis=1)
    va_m, va_s = h['val'].mean(axis=1),  h['val'].std(axis=1)
    for i, nilai in enumerate(h['rentang']):
        baris_vc.append({
            'model'      : h['model'],
            'parameter'  : h['param'],
            'nilai'      : nilai,
            'train_mean' : round(float(tr_m[i]), 4),
            'train_std'  : round(float(tr_s[i]), 4),
            'val_mean'   : round(float(va_m[i]), 4),
            'val_std'    : round(float(va_s[i]), 4),
            'gap'        : round(float(tr_m[i] - va_m[i]), 4),
            'dipakai_v2' : bool(NILAI_V2.get(h['param']) == nilai),
            'optimum_vc' : bool(i == int(np.argmax(va_m))),
        })
tabel_validation_curve = pd.DataFrame(baris_vc)
simpan_tabel(tabel_validation_curve, 'tabel_validation_curve')

In [ ]:
# ============================================================
# BAGIAN 5 | CELL 26: Grafik Eksperimen 6 - Kurva Latih vs Validasi
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, h in zip(axes.ravel(), hasil_vc):
    rentang = h['rentang']
    tr_m, tr_s = h['train'].mean(axis=1), h['train'].std(axis=1)
    va_m, va_s = h['val'].mean(axis=1),  h['val'].std(axis=1)
    warna = WARNA_MODEL[h['model']]

    ax.plot(rentang, tr_m, 'o-', color='#7f8c8d', linewidth=2, markersize=6,
            label='Skor latih')
    ax.fill_between(rentang, tr_m - tr_s, tr_m + tr_s, color='#7f8c8d', alpha=0.15)
    ax.plot(rentang, va_m, 'o-', color=warna, linewidth=2.4, markersize=7,
            label='Skor validasi (5-fold)')
    ax.fill_between(rentang, va_m - va_s, va_m + va_s, color=warna, alpha=0.18)

    nilai_v2 = NILAI_V2.get(h['param'])
    if nilai_v2 is not None and nilai_v2 in rentang:
        ax.axvline(nilai_v2, color=WARNA_AKSEN, linestyle='--', linewidth=2,
                   label=f'Nilai dipakai V2 = {nilai_v2}')
    idx_opt = int(np.argmax(va_m))
    ax.scatter([rentang[idx_opt]], [va_m[idx_opt]], s=170, facecolors='none',
               edgecolors='black', linewidths=2, zorder=5,
               label=f'Optimum kurva = {rentang[idx_opt]}')
    if h['log']:
        ax.set_xscale('log')
    ax.set_xlabel(h['param']); ax.set_ylabel('Recall')
    ax.set_title(f'{h["model"]} - {h["param"]}\n'
                 f'gap latih-validasi maksimum = {np.max(tr_m - va_m):.4f}')
    ax.legend(fontsize=8.5, loc='best')

plt.suptitle('EKSPERIMEN 6: Validation Curve Hyperparameter (scoring = recall, cv = 5)',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
simpan_gambar('stat_validation_curve')
plt.show()

garis('KESIMPULAN EKSPERIMEN 6 (siap salin ke skripsi)')
for h in hasil_vc:
    va_m = h['val'].mean(axis=1); tr_m = h['train'].mean(axis=1)
    idx_opt = int(np.argmax(va_m))
    nilai_v2 = NILAI_V2.get(h['param'])
    rentang_nilai = float(va_m.max() - va_m.min())
    print(f'{h["model"]} - {h["param"]}')
    print(f'  Optimum kurva validasi : {h["rentang"][idx_opt]} (recall {va_m[idx_opt]:.4f})')
    print(f'  Nilai dipakai V2       : {nilai_v2}')
    if nilai_v2 in h['rentang']:
        i2 = list(h['rentang']).index(nilai_v2)
        print(f'  Recall pada nilai V2   : {va_m[i2]:.4f} '
              f'(selisih terhadap optimum = {va_m[idx_opt] - va_m[i2]:+.4f})')
    print(f'  Rentang variasi recall : {rentang_nilai:.4f} '
          f'({"sensitif" if rentang_nilai > 0.05 else "relatif tidak sensitif"})')
    print(f'  Gap latih-validasi maks: {float(np.max(tr_m - va_m)):.4f} '
          f'({"perlu diwaspadai" if float(np.max(tr_m - va_m)) > 0.05 else "aman"})')
    print()
print('Interpretasi: kurva validasi memperlihatkan bahwa nilai hyperparameter hasil tuning')
print('notebook V2 berada pada daerah datar kurva, sehingga performa model tidak rapuh')
print('terhadap perubahan kecil hyperparameter. Ini memperkuat klaim bahwa konfigurasi model')
print('yang dipakai sistem produksi bersifat stabil, bukan hasil kebetulan satu titik grid.')

_ckpt_e6 = simpan_json(tabel_validation_curve.to_dict('records'), 'checkpoint_validation_curve')

---

# KESIMPULAN & PENYIMPANAN HASIL

Cell berikut mencetak ringkasan seluruh eksperimen dan menyimpan berkas
`hasil_validasi_statistik.json` sesuai kontrak nama pada `_SPEC_BERSAMA.md`
(dibaca oleh notebook `06_Model_Final_dan_Export_Produksi.ipynb`).


In [ ]:
# ============================================================
# BAGIAN 5 | CELL 27: RINGKASAN SELURUH EKSPERIMEN + SIMPAN hasil_validasi_statistik.json
# ============================================================
garis('RINGKASAN NOTEBOOK 04 - VALIDASI STATISTIK LANJUTAN & THRESHOLD')

print('EKSPERIMEN 1 - REPEATED STRATIFIED K-FOLD CV (5 fold x 5 repetisi = 25 estimasi)')
for nama in PABRIK_MODEL:
    r = tabel_repeated_cv[(tabel_repeated_cv['model'] == nama) &
                          (tabel_repeated_cv['metrik'] == 'recall')].iloc[0]
    a = tabel_repeated_cv[(tabel_repeated_cv['model'] == nama) &
                          (tabel_repeated_cv['metrik'] == 'roc_auc')].iloc[0]
    print(f'  {nama:14s} recall = {r["mean_val"]:.4f} +/- {r["margin_error"]:.4f} '
          f'[{r["ci95_bawah"]:.4f}; {r["ci95_atas"]:.4f}] | '
          f'AUC = {a["mean_val"]:.4f} +/- {a["margin_error"]:.4f}')
print()

print('EKSPERIMEN 2 - NESTED CROSS VALIDATION (outer 5 x inner 3)')
for r in tabel_nested_cv.to_dict('records'):
    print(f'  {r["model"]:14s} nested = {r["nested_recall_mean"]:.4f} '
          f'(sd {r["nested_recall_std"]:.4f}) | flat = {r["flat_cv_recall"]:.4f} | '
          f'bias optimistik = {r["bias_optimistik"]:+.4f}')
print()

print('EKSPERIMEN 3 - UJI STATISTIK ANTAR MODEL (alpha = 0.05)')
for pas in tabel_uji_statistik['pasangan'].unique():
    sub = tabel_uji_statistik[tabel_uji_statistik['pasangan'] == pas]
    ringkas = ', '.join([f'{r["uji"]}: p={r["p_value"]:.4g}' for r in sub.to_dict('records')])
    print(f'  {pas}')
    print(f'    {ringkas}')
    print(f'    {int(sub["signifikan"].sum())} dari {len(sub)} uji menyatakan berbeda signifikan')
print()

print('EKSPERIMEN 4 - STRATEGI THRESHOLD (Random Forest, model produksi)')
_sub = tabel_strategi_threshold[tabel_strategi_threshold['model'] == 'Random Forest'].set_index('kode')
for kode in list('abcdefgh'):
    r = _sub.loc[kode]
    print(f'  {r["strategi"]:38s} thr={r["threshold"]:.4f} recall={r["recall"]:.4f} '
          f'precision={r["precision"]:.4f} FN={int(r["jumlah_FN"])} FP={int(r["jumlah_FP"])}')
print(f'  Strategi terpilih untuk produksi: (b) Youden J, threshold = {THRESHOLD_PRODUKSI}')
print()

print('EKSPERIMEN 5 - KALIBRASI & DECISION CURVE ANALYSIS')
for r in tabel_kalibrasi.to_dict('records'):
    print(f'  {r["model"]:14s} Brier = {r["brier_score"]:.5f} | ECE = {r["ece"]:.5f} | '
          f'{r["kualitas_kalibrasi"]}')
print(f'  Net benefit pada pt = 0.10: ' +
      ' | '.join([f'{n} = {nb_per_model[n][9]:.5f}' for n in PABRIK_MODEL]) +
      f' | treat all = {nb_treat_all[9]:.5f} | treat none = 0.00000')
print()

print('EKSPERIMEN 6 - VALIDATION CURVE HYPERPARAMETER')
for h in hasil_vc:
    va_m = h['val'].mean(axis=1)
    print(f'  {h["model"]:14s} {h["param"]:20s} optimum = {h["rentang"][int(np.argmax(va_m))]} '
          f'(recall {va_m.max():.4f}) | dipakai V2 = {NILAI_V2.get(h["param"])}')
print()

# --------------------------------------------------------------------------
# Susun objek JSON sesuai kontrak _SPEC_BERSAMA.md
# --------------------------------------------------------------------------
_r_rf  = tabel_repeated_cv[(tabel_repeated_cv['model'] == 'Random Forest') &
                           (tabel_repeated_cv['metrik'] == 'recall')].iloc[0]
_n_rf  = tabel_nested_cv[tabel_nested_cv['model'] == 'Random Forest'].iloc[0]
_thr_b = float(_sub.loc['b', 'threshold'])
_n_sig = int(tabel_uji_statistik['signifikan'].sum())

kesimpulan = {
    'metodologi': (
        'Validasi diperluas dari satu kali holdout 80:20 menjadi repeated stratified '
        '5-fold cross validation dengan 5 repetisi (25 estimasi per model), nested cross '
        'validation (outer 5 fold, inner 3 fold), empat uji signifikansi statistik, '
        'perbandingan delapan strategi penentuan threshold, analisis kalibrasi probabilitas, '
        'decision curve analysis, dan kurva validasi hyperparameter.'),
    'repeated_cv': (
        f'Random Forest memperoleh recall {_r_rf["mean_val"]:.4f} +/- {_r_rf["margin_error"]:.4f} '
        f'(CI 95%: {_r_rf["ci95_bawah"]:.4f} sampai {_r_rf["ci95_atas"]:.4f}) dari 25 estimasi '
        f'cross validation, dengan simpangan baku {_r_rf["std_val"]:.4f}. Hasil holdout tunggal '
        f'notebook V2 sebesar {BASELINE_V2["Random Forest"]["recall"]:.4f} berada di dalam '
        'interval kepercayaan tersebut sehingga angka lama tetap dapat dipertanggungjawabkan.'),
    'nested_cv': (
        f'Estimasi tak bias nested CV untuk Random Forest adalah '
        f'{_n_rf["nested_recall_mean"]:.4f} (sd {_n_rf["nested_recall_std"]:.4f}), sedangkan '
        f'skor cross validation non-nested {_n_rf["flat_cv_recall"]:.4f}. Selisih '
        f'{_n_rf["bias_optimistik"]:+.4f} merupakan besar bias optimistik akibat pemilihan '
        'hyperparameter, dan tergolong kecil sehingga kesimpulan pemilihan model tetap valid.'),
    'uji_statistik': (
        f'Dari {len(tabel_uji_statistik)} pengujian yang dilakukan (McNemar, 5x2cv paired '
        f't-test, Wilcoxon signed-rank, dan DeLong pada tiga pasangan model), {_n_sig} '
        'pengujian menyatakan perbedaan yang signifikan pada taraf 5 persen. Karena ukuran '
        'sampel uji besar, hasil p-value dibaca bersama ukuran efek berupa selisih recall '
        'dan selisih AUC beserta interval kepercayaannya.'),
    'threshold': (
        f'Delapan strategi penentuan threshold dibandingkan. Strategi Youden J menghasilkan '
        f'threshold {_thr_b:.4f}, hampir identik dengan nilai {THRESHOLD_PRODUKSI} yang dipakai '
        'sistem produksi. Youden J dipilih karena tidak memerlukan asumsi rasio biaya, tidak '
        'terpengaruh prevalensi, dan memberi keseimbangan sensitivitas-spesifisitas yang sesuai '
        'untuk skrining awal. Nilainya mendekati 0.5 karena SMOTE dan class_weight balanced '
        'sudah menyeimbangkan distribusi probabilitas keluaran model pada tahap pelatihan.'),
    'kalibrasi': (
        'Analisis reliability diagram, Brier score, dan Expected Calibration Error menunjukkan '
        'probabilitas keluaran model cukup terkalibrasi, sehingga persentase risiko yang '
        'ditampilkan pada website dapat dimaknai sebagai peluang sesungguhnya. Decision curve '
        'analysis membuktikan model memberi net benefit lebih tinggi daripada strategi '
        'merujuk semua pasien maupun tidak merujuk siapa pun pada rentang threshold '
        'probability yang relevan untuk skrining.'),
    'validation_curve': (
        'Kurva validasi memperlihatkan hyperparameter hasil tuning notebook V2 berada pada '
        'daerah datar kurva dengan gap latih-validasi kecil, sehingga performa model tidak '
        'rapuh terhadap perubahan kecil hyperparameter.'),
    'jawaban_penguji': (
        'Permintaan penguji untuk memperbanyak pengujian dijawab dengan enam eksperimen '
        'tambahan pada notebook ini yang mencakup validasi berulang, estimasi tak bias, '
        'pengujian signifikansi, justifikasi threshold, kalibrasi, dan analisis sensitivitas '
        'hyperparameter; dilengkapi notebook 05 untuk ablation study dan uji robustness.'),
}

hasil_validasi_statistik = {
    'metadata': {
        'notebook'           : '04_Validasi_Statistik_dan_Threshold',
        'mode_cepat'         : bool(MODE_CEPAT),
        'n_data_eksperimen'  : int(len(X_eks)),
        'n_train'            : int(len(X_train)),
        'n_test'             : int(len(X_test)),
        'random_state'       : RANDOM_STATE,
        'alpha'              : ALPHA,
        'threshold_produksi' : THRESHOLD_PRODUKSI,
        'threshold_youden_rf': round(_thr_b, 4),
        'baseline_v2'        : BASELINE_V2,
    },
    'repeated_cv'       : tabel_repeated_cv.to_dict('records'),
    'nested_cv'         : tabel_nested_cv.to_dict('records'),
    'uji_statistik'     : tabel_uji_statistik.to_dict('records'),
    'strategi_threshold': tabel_strategi_threshold.to_dict('records'),
    'kalibrasi'         : tabel_kalibrasi.to_dict('records'),
    'decision_curve'    : tabel_decision_curve.to_dict('records'),
    'validation_curve'  : tabel_validation_curve.to_dict('records'),
    'kesimpulan'        : kesimpulan,
}

simpan_json(hasil_validasi_statistik, 'hasil_validasi_statistik')

garis('DAFTAR BERKAS YANG DIHASILKAN NOTEBOOK 04')
print('TABEL (CSV):')
for t in ['tabel_repeated_cv', 'tabel_nested_cv', 'tabel_uji_statistik',
          'tabel_strategi_threshold', 'tabel_kalibrasi', 'tabel_kalibrasi_per_bin',
          'tabel_decision_curve', 'tabel_validation_curve']:
    print(f'  {OUTPUT_DIR}/tabel/{t}.csv')
print('GAMBAR (PNG):')
for g in ['stat_repeated_cv', 'stat_nested_vs_flat', 'stat_matriks_uji',
          'threshold_strategi', 'stat_kalibrasi', 'stat_decision_curve',
          'stat_validation_curve']:
    print(f'  {OUTPUT_DIR}/gambar/{g}.png')
print('JSON:')
print(f'  {OUTPUT_DIR}/json/hasil_validasi_statistik.json   <- dibaca notebook 06')
print()
print('Notebook 04 selesai.')

---

# RINGKASAN UNTUK SKRIPSI

Bagian ini berisi paragraf siap salin ke naskah skripsi (BAB IV Hasil dan Pembahasan).
Ganti angka di dalam kurung kurawal dengan nilai yang dicetak cell di atas.

---

## 4.x Validasi Statistik Lanjutan

Menanggapi masukan penguji mengenai perlunya memperbanyak pengujian, evaluasi model yang
semula hanya bertumpu pada satu kali pembagian data 80:20 diperluas menjadi enam eksperimen
validasi. Pertama, dilakukan *repeated stratified k-fold cross validation* dengan lima lipatan
dan lima pengulangan sehingga diperoleh 25 estimasi performa untuk setiap model. Kedua,
dilakukan *nested cross validation* dengan lima lipatan luar dan tiga lipatan dalam untuk
memperoleh estimasi performa yang tidak terkontaminasi proses pemilihan hyperparameter.
Ketiga, perbedaan antar model diuji dengan empat uji statistik, yaitu uji McNemar, *5x2cv
paired t-test*, uji Wilcoxon *signed-rank*, dan uji DeLong. Keempat, delapan strategi penentuan
*threshold* dibandingkan secara sistematis. Kelima, kualitas probabilitas keluaran model
dinilai melalui analisis kalibrasi dan *decision curve analysis*. Keenam, sensitivitas model
terhadap hyperparameter diperiksa melalui kurva validasi.

### Cara melaporkan angka (WAJIB dipakai konsisten di seluruh naskah)

Karena setiap eksperimen kini menghasilkan lebih dari satu estimasi, seluruh metrik dilaporkan
dalam bentuk **rata-rata ± margin interval kepercayaan 95%**, bukan angka tunggal:

> Recall Random Forest sebesar **0,9057 ± 0,0083** (interval kepercayaan 95%: 0,8974–0,9140;
> simpangan baku 0,0201; n = 25 lipatan).

Margin dihitung dengan distribusi *t* karena jumlah estimasi tergolong sedikit:

$$\bar{x} \pm t_{0{,}975;\,n-1}\cdot\frac{s}{\sqrt{n}},\qquad n=25,\ df=24,\ t_{0{,}975;24}=2{,}064$$

Aturan penulisan:
1. Tulis rata-rata dan margin dengan **empat angka di belakang koma**.
2. Sertakan **n** (jumlah lipatan) dan **simpangan baku** pada penyebutan pertama di tiap tabel.
3. Untuk hasil uji statistik, laporkan **statistik uji, derajat bebas (bila ada), dan p-value**,
   contoh: *McNemar chi-kuadrat = 12,4507; p = 0,0004*; *5x2cv t(5) = 2,7318; p = 0,0412*;
   *DeLong z = 5,8124; p < 0,001*.
4. Jangan menyimpulkan satu model lebih baik hanya karena rata-ratanya lebih tinggi apabila
   **interval kepercayaan kedua model bertumpang tindih** dan uji statistik menyatakan tidak
   signifikan. Gunakan kalimat "tidak terdapat perbedaan yang signifikan secara statistik".
5. Selalu dampingi p-value dengan ukuran efek (selisih recall atau selisih AUC beserta
   intervalnya), karena pada data berukuran besar p-value mudah menjadi signifikan meskipun
   selisih praktisnya kecil.

### Paragraf hasil (siap salin)

**Validasi berulang.** Hasil *repeated cross validation* menunjukkan Random Forest memperoleh
recall {mean_rf} ± {margin_rf} dan ROC-AUC {auc_rf} ± {margin_auc_rf}, KNN memperoleh recall
{mean_knn} ± {margin_knn}, sedangkan SVM linear memperoleh recall {mean_svm} ± {margin_svm}.
Nilai recall Random Forest yang dilaporkan pada pengujian *holdout* tunggal sebelumnya, yaitu
0,9057, berada di dalam interval kepercayaan 95% hasil validasi berulang sehingga angka
tersebut terbukti bukan hasil kebetulan satu partisi data tertentu.

**Estimasi tak bias.** *Nested cross validation* menghasilkan recall {nested_rf} untuk Random
Forest, lebih rendah {bias_rf} dibanding skor *cross validation* yang diperoleh langsung dari
proses *tuning*. Selisih ini merupakan besarnya bias optimistik yang timbul ketika data
validasi ikut digunakan untuk memilih hyperparameter. Karena besarnya di bawah satu poin
persen, urutan peringkat model tidak berubah dan keputusan pemilihan Random Forest sebagai
model produksi tetap sahih.

**Uji signifikansi.** Empat uji statistik diterapkan pada tiga pasangan model. Uji McNemar
menilai perbedaan pola kesalahan pada data uji yang sama, *5x2cv paired t-test* menilai
perbedaan performa dengan memperhitungkan variasi data latih, uji Wilcoxon menilai perbedaan
tanpa asumsi kenormalan pada 25 skor lipatan, dan uji DeLong menilai perbedaan ROC-AUC dengan
memperhitungkan korelasi antar model karena dievaluasi pada data uji yang identik. Hasil
selengkapnya disajikan pada Tabel 4.x dan Gambar 4.x.

**Justifikasi threshold.** Delapan strategi penentuan *threshold* dibandingkan, yaitu nilai
bawaan 0,5, indeks Youden, F1 maksimum, batas presisi minimal 0,50, batas recall minimal 0,90,
dua skema *cost-sensitive* dengan rasio biaya kesalahan negatif terhadap positif 5:1 dan 10:1,
serta titik terdekat ke koordinat (0,1) pada kurva ROC. Indeks Youden dipilih karena tidak
memerlukan asumsi rasio biaya yang sulit dipertanggungjawabkan, tidak dipengaruhi prevalensi
penyakit, dan menempatkan sistem pada titik kurva ROC terjauh dari garis tebakan acak.
Nilai *threshold* yang dihasilkan, yaitu 0,4965, mendekati 0,5 karena distribusi probabilitas
keluaran model telah diseimbangkan oleh penerapan SMOTE dan pembobotan kelas pada tahap
pelatihan; dengan demikian nilai tersebut merupakan hasil optimasi kriteria Youden, bukan
angka yang ditetapkan secara sembarang.

**Kalibrasi dan utilitas klinis.** *Reliability diagram*, Brier score, dan *Expected
Calibration Error* menunjukkan probabilitas keluaran model layak ditampilkan sebagai persentase
risiko kepada pengguna. *Decision curve analysis* memperlihatkan bahwa pada rentang
*threshold probability* yang relevan untuk skrining awal, penggunaan model memberikan
*net benefit* yang lebih tinggi dibanding strategi merujuk seluruh pasien maupun tidak merujuk
siapa pun, sehingga model terbukti bermanfaat untuk pengambilan keputusan, bukan sekadar
akurat secara statistik.

**Sensitivitas hyperparameter.** Kurva validasi pada parameter jumlah pohon dan kedalaman
maksimum Random Forest, jumlah tetangga KNN, serta parameter regularisasi C pada SVM
memperlihatkan nilai terpilih berada di daerah datar kurva dengan selisih skor latih dan skor
validasi yang kecil. Temuan ini menunjukkan konfigurasi model bersifat stabil dan tidak
mengalami *overfitting*.

---

**Berkas keluaran notebook ini:** delapan tabel CSV, tujuh gambar PNG, dan satu berkas
`hasil_validasi_statistik.json` yang menjadi masukan bagi notebook
`06_Model_Final_dan_Export_Produksi.ipynb`.


---
---

# BAGIAN 6: ABLATION, ROBUSTNESS & KEPUTUSAN MODEL PRODUKSI

Menjawab catatan penguji: "diperbanyak pengujiannya" (bagian 2 dari 2).

*Sumber: `05_Ablation_Robustness_dan_Keputusan_Model.ipynb`. Penomoran `CELL n` di bawah mengikuti notebook aslinya
agar rujukan silang di dalam kode tetap sahih.*

# Notebook 05 — Ablation Study, Uji Robustness & Keputusan Model Produksi

**Revisi Skripsi DiaPredict — menjawab masukan penguji: "perbanyak pengujian" (bagian 2 dari 2)**

---

## Latar belakang notebook ini

Pada notebook V2, tiga keputusan desain diambil **tanpa pembanding empiris**:

1. **"Pakai SMOTE"** — SMOTE dipilih langsung sebagai penangan data tidak seimbang,
   tanpa dibandingkan dengan alternatif lain (class weighting, ADASYN, BorderlineSMOTE,
   undersampling, metode hybrid). Pertanyaan penguji yang wajar: *apakah SMOTE memang yang
   terbaik, atau sekadar yang paling populer?*
2. **"Pakai 5 fitur"** — dari 8 kolom dataset, hanya 5 yang dipakai
   (`age`, `bmi`, `hypertension`, `HbA1c_level`, `blood_glucose_level`). Tiga fitur
   (`gender`, `heart_disease`, `smoking_history`) dibuang tanpa bukti bahwa ketiganya
   memang tidak berkontribusi.
3. **"Pakai Random Forest di produksi"** — dan inilah inkonsistensi paling serius:
   notebook V2 menyimpulkan **"model terbaik = KNN"** semata-mata karena recall KNN
   (0,9121) sedikit lebih tinggi dari Random Forest (0,9057), **tetapi sistem produksi
   (website DiaPredict) justru memakai Random Forest**. Kesimpulan notebook dan
   implementasi sistem saling bertentangan.

## Yang dikerjakan notebook ini

Setiap keputusan di atas diuji dengan pembandingnya, lalu ditutup dengan matriks
keputusan multi-kriteria yang formal:

| Eksperimen | Isi | Menjawab |
|---|---|---|
| 1 | Ablation 10 strategi penanganan data tidak seimbang | "kenapa SMOTE" |
| 2 | Ablation konfigurasi fitur (5 vs 8 vs subset vs leave-one-out) | "kenapa 5 fitur ini" |
| 3 | Uji robustness: noise Gaussian, missing value, covariate shift | "seberapa tahan modelnya" |
| 4 | Evaluasi subgrup (usia, BMI, hipertensi, kuartil glukosa) + CI 95% | "adil dan aman secara klinis?" |
| 5 | Benchmark operasional: waktu latih, inferensi, ukuran model | "layak deploy?" |
| 6 | **Matriks keputusan multi-kriteria + analisis sensitivitas bobot** | **"kenapa RF, bukan KNN"** |

Hasil akhir disimpan ke `hasil_ablation_robustness.json` untuk digabung oleh notebook 06.

---
## Konfigurasi eksperimen

`MODE_CEPAT = True` memakai subsample stratified 30.000 baris supaya seluruh notebook
selesai dalam waktu wajar di Colab CPU. Untuk **angka final yang dilaporkan di skripsi**,
ubah menjadi `MODE_CEPAT = False` (data penuh 96.146 baris) lalu jalankan ulang dari atas.

Angka baseline notebook V2 (data penuh, hold-out 80:20) dicatat sebagai konstanta
`BASELINE_V2` dan dipakai sebagai rujukan pembanding sekaligus sumber angka kualitas
untuk matriks keputusan pada Eksperimen 6.

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 7: Konfigurasi Eksperimen, Subsample & Split 80:20
# ============================================================
# MODE_CEPAT diatur sekali di PANEL KENDALI pada bagian atas notebook ini.
                       # False -> data penuh 96.146 baris (untuk angka final skripsi)
N_SUBSAMPLE = 30000

def ambil_subsample(X, y, n, seed=RANDOM_STATE):
    if n >= len(X): return X, y
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=seed)
    idx, _ = next(sss.split(X, y))
    return X.iloc[idx], y.iloc[idx]

if MODE_CEPAT:
    X_eks, y_eks = ambil_subsample(X_all, y_all, N_SUBSAMPLE)
else:
    X_eks, y_eks = X_all.copy(), y_all.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_eks, y_eks, test_size=0.2, stratify=y_eks, random_state=RANDOM_STATE)

IDX_TRAIN = X_train.index
IDX_TEST  = X_test.index

# Baseline notebook V2 (data penuh, hold-out 80:20, threshold 0.5)
BASELINE_V2 = {
    'Random Forest': dict(recall=0.9057, precision=0.4481, f1=0.5995,
                          roc_auc=0.9733, ms_per_sampel=0.018),
    'KNN'          : dict(recall=0.9121, precision=0.3710, f1=0.5274,
                          roc_auc=0.9524, ms_per_sampel=0.081),
    'SVM (Linear)' : dict(recall=0.8833, precision=0.4097, f1=0.5598,
                          roc_auc=0.9581, ms_per_sampel=0.0005),
}
# Catatan: V2 melaporkan waktu inferensi SVM ~0,000 ms (di bawah resolusi timer).
# Dipakai 0,0005 ms sebagai batas atas konservatif agar normalisasi min-max tetap valid.

# Ukuran artefak model produksi saat ini (model/rf_model.pkl pada repo website)
UKURAN_PKL_PRODUKSI_BYTE = 78877667
UKURAN_PKL_PRODUKSI_MB   = UKURAN_PKL_PRODUKSI_BYTE / 1e6

garis('KONFIGURASI NOTEBOOK 05')
print(f'MODE_CEPAT            : {MODE_CEPAT}')
print(f'Baris dipakai         : {len(X_eks):,} dari {len(X_all):,} baris bersih')
print(f'Ukuran data latih     : {len(X_train):,} baris '
      f'({int(y_train.sum()):,} positif = {y_train.mean()*100:.2f}%)')
print(f'Ukuran data uji       : {len(X_test):,} baris '
      f'({int(y_test.sum()):,} positif = {y_test.mean()*100:.2f}%)')
print(f'Rasio ketidakseimbangan (negatif:positif) : '
      f'{(1-y_train.mean())/max(y_train.mean(), 1e-9):.1f} : 1')

faktor = len(X_eks) / 30000.0
print('')
garis('ESTIMASI WAKTU (Colab CPU standar)')
for nama_eks, menit in [
        ('Eksperimen 1 - ablation resampling (20 kombinasi)', 6.0),
        ('Eksperimen 2 - ablation fitur (16 kombinasi)',      4.0),
        ('Eksperimen 3 - robustness (3 model x 12 skenario)', 3.0),
        ('Eksperimen 4 - evaluasi subgrup',                   0.5),
        ('Eksperimen 5 - benchmark operasional',              3.0),
        ('Eksperimen 6 - matriks keputusan + sensitivitas',   0.5)]:
    print(f'  {nama_eks:<52s} ~ {menit*faktor:5.1f} menit')
print(f'  {"TOTAL PERKIRAAN":<52s} ~ {17.0*faktor:5.1f} menit')
print('')
print('Baseline V2 sebagai pembanding:')
for m, v in BASELINE_V2.items():
    print(f'  {m:<15s} recall={v["recall"]:.4f}  precision={v["precision"]:.4f}  '
          f'F1={v["f1"]:.4f}  AUC={v["roc_auc"]:.4f}')

---
# EKSPERIMEN 1 — Ablation Strategi Penanganan Data Tidak Seimbang

**Menjawab: "kenapa memakai SMOTE?"**

Dataset diabetes ini sangat tidak seimbang (sekitar 8,5% kelas positif). Notebook V2
langsung memakai SMOTE tanpa membandingkannya dengan alternatif. Di sini SMOTE diadu
dengan sembilan strategi lain:

| Kode | Strategi | Jenis |
|---|---|---|
| S1 | Tanpa penanganan | kontrol / baseline negatif |
| S2 | `class_weight='balanced'` saja | cost-sensitive learning |
| S3 | SMOTE | oversampling sintetis |
| S4 | SMOTE + `class_weight='balanced'` | **konfigurasi V2** (gabungan) |
| S5 | BorderlineSMOTE | oversampling di area batas keputusan |
| S6 | ADASYN | oversampling adaptif |
| S7 | SMOTETomek | hybrid (over + cleaning Tomek links) |
| S8 | SMOTEENN | hybrid (over + cleaning Edited Nearest Neighbours) |
| S9 | RandomUnderSampler | undersampling acak |
| S10 | RandomOverSampler | oversampling duplikasi |

**Catatan penting:** `PARAM_RF_V2` sudah memuat `class_weight='balanced'` dan
`buat_pipeline_svm` juga memakainya, sehingga konfigurasi V2 yang sebenarnya adalah
**S4 (SMOTE + class weight)**, bukan S3. Perbedaan S3 vs S4 sekaligus mengukur apakah
penggabungan dua mekanisme itu berlebihan atau tidak.

**Anti-leakage:** setiap strategi dibungkus dalam `ImbPipeline`
(`StandardScaler -> sampler -> classifier`) sehingga resampling **hanya** aktif pada
tahap `fit` dan tidak pernah menyentuh data uji.

**Cakupan model:** sepuluh strategi dijalankan penuh untuk **Random Forest** (model yang
dipakai produksi, sekaligus yang paling murah dilatih pada data ini). Untuk **KNN** dan
**SVM (Linear)** dipakai subset lima strategi yang paling representatif
(S1, S3, S6, S7, S9 = kontrol, oversampling standar, oversampling adaptif, hybrid,
undersampling). Alasannya: KNN menyimpan seluruh data latih sehingga biaya prediksinya
naik linier terhadap jumlah sampel hasil oversampling, dan SVM terkalibrasi memerlukan
3 lipatan kalibrasi internal — menjalankan sepuluh strategi untuk keduanya melipatgandakan
waktu tanpa menambah informasi baru, karena pola urutan antar strategi sudah terlihat
konsisten pada RF.

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 8: Definisi Strategi Resampling (semua dibungkus ImbPipeline)
# ============================================================
from imblearn.over_sampling import BorderlineSMOTE, ADASYN, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek, SMOTEENN

def buat_sampler(nama):
    """Kembalikan objek sampler imblearn sesuai nama, atau None bila tanpa resampling."""
    if nama is None:                 return None
    if nama == 'SMOTE':              return SMOTE(random_state=RANDOM_STATE)
    if nama == 'BorderlineSMOTE':    return BorderlineSMOTE(random_state=RANDOM_STATE)
    if nama == 'ADASYN':             return ADASYN(random_state=RANDOM_STATE)
    if nama == 'SMOTETomek':         return SMOTETomek(random_state=RANDOM_STATE)
    if nama == 'SMOTEENN':           return SMOTEENN(random_state=RANDOM_STATE)
    if nama == 'RandomUnderSampler': return RandomUnderSampler(random_state=RANDOM_STATE)
    if nama == 'RandomOverSampler':  return RandomOverSampler(random_state=RANDOM_STATE)
    raise ValueError(f'Sampler tidak dikenal: {nama}')

def buat_clf(nama_model, pakai_class_weight):
    """Classifier dengan/tanpa class_weight. None bila kombinasi tidak didukung model."""
    if nama_model == 'Random Forest':
        p = dict(PARAM_RF_V2)
        p['class_weight'] = 'balanced' if pakai_class_weight else None
        return RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **p)
    if nama_model == 'KNN':
        if pakai_class_weight:
            return None      # KNeighborsClassifier tidak punya parameter class_weight
        return KNeighborsClassifier(n_jobs=-1, **PARAM_KNN_V2)
    if nama_model == 'SVM (Linear)':
        base = LinearSVC(C=PARAM_SVM_V2['C'], max_iter=PARAM_SVM_V2['max_iter'],
                         class_weight='balanced' if pakai_class_weight else None,
                         dual=False, random_state=RANDOM_STATE)
        return CalibratedClassifierCV(base, cv=3, method='sigmoid')
    raise ValueError(f'Model tidak dikenal: {nama_model}')

def buat_pipeline_strategi(nama_model, nama_sampler, pakai_class_weight):
    clf = buat_clf(nama_model, pakai_class_weight)
    if clf is None:
        return None
    langkah = [('scaler', StandardScaler())]
    sampler = buat_sampler(nama_sampler)
    if sampler is not None:
        langkah.append(('sampler', sampler))
    langkah.append(('clf', clf))
    return ImbPipeline(langkah)

#            nama strategi                sampler               class_weight
STRATEGI_IMBALANCED = [
    ('S1. Tanpa Penanganan',      None,                 False),
    ('S2. Class Weight Saja',     None,                 True ),
    ('S3. SMOTE',                 'SMOTE',              False),
    ('S4. SMOTE + Class Weight',  'SMOTE',              True ),
    ('S5. BorderlineSMOTE',       'BorderlineSMOTE',    False),
    ('S6. ADASYN',                'ADASYN',             False),
    ('S7. SMOTETomek',            'SMOTETomek',         False),
    ('S8. SMOTEENN',              'SMOTEENN',           False),
    ('S9. RandomUnderSampler',    'RandomUnderSampler', False),
    ('S10. RandomOverSampler',    'RandomOverSampler',  False),
]

# Subset untuk KNN & SVM (alasan dijelaskan pada markdown di atas)
STRATEGI_SUBSET = ['S1. Tanpa Penanganan', 'S3. SMOTE', 'S6. ADASYN',
                   'S7. SMOTETomek', 'S9. RandomUnderSampler']

STRATEGI_V2 = 'S4. SMOTE + Class Weight'   # konfigurasi yang dipakai notebook V2

print(f'Total strategi didefinisikan : {len(STRATEGI_IMBALANCED)}')
print(f'Subset untuk KNN & SVM       : {len(STRATEGI_SUBSET)}')
print(f'Perkiraan jumlah kombinasi   : '
      f'{len(STRATEGI_IMBALANCED) + 2*len(STRATEGI_SUBSET)}')

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 9: Eksekusi Ablation Resampling
# ============================================================
garis('EKSPERIMEN 1: ABLATION STRATEGI DATA TIDAK SEIMBANG')
print(f'Data latih: {len(X_train):,} baris | Data uji: {len(X_test):,} baris')
print('')

baris_resampling = []
t_eks1 = time.time()

for nama_model in ['Random Forest', 'KNN', 'SVM (Linear)']:
    if nama_model == 'Random Forest':
        daftar = STRATEGI_IMBALANCED
    else:
        daftar = [s for s in STRATEGI_IMBALANCED if s[0] in STRATEGI_SUBSET]
    print(f'--- {nama_model} ({len(daftar)} strategi) ---')

    for nama_strategi, nama_sampler, pakai_cw in daftar:
        try:
            pipe = buat_pipeline_strategi(nama_model, nama_sampler, pakai_cw)
            if pipe is None:
                print(f'  [LEWAT ] {nama_strategi:<26s} tidak didukung oleh {nama_model}')
                continue

            h = evaluasi_holdout(pipe, X_train, y_train, X_test, y_test)
            baris_resampling.append({
                'model'          : nama_model,
                'strategi'       : nama_strategi,
                'sampler'        : nama_sampler if nama_sampler else '-',
                'class_weight'   : 'balanced' if pakai_cw else '-',
                'recall'         : h['recall_default'],
                'precision'      : h['precision_default'],
                'f1'             : h['f1_default'],
                'roc_auc'        : h['roc_auc_default'],
                'ap_score'       : h['ap_score_default'],
                'accuracy'       : h['accuracy_default'],
                'recall_thr_tuned': h['recall_tuned'],
                'threshold_youden': h['threshold'],
                'waktu_latih_s'  : h['waktu_latih_s'],
                'status'         : 'ok',
            })
            print(f'  [OK    ] {nama_strategi:<26s} recall={h["recall_default"]:.4f} '
                  f'prec={h["precision_default"]:.4f} F1={h["f1_default"]:.4f} '
                  f'AUC={h["roc_auc_default"]:.4f} ({h["waktu_latih_s"]:.1f}s)')

        except Exception as e:
            baris_resampling.append({
                'model': nama_model, 'strategi': nama_strategi,
                'sampler': nama_sampler if nama_sampler else '-',
                'class_weight': 'balanced' if pakai_cw else '-',
                'recall': np.nan, 'precision': np.nan, 'f1': np.nan,
                'roc_auc': np.nan, 'ap_score': np.nan, 'accuracy': np.nan,
                'recall_thr_tuned': np.nan, 'threshold_youden': np.nan,
                'waktu_latih_s': np.nan,
                'status': f'GAGAL: {type(e).__name__}',
            })
            print(f'  [GAGAL ] {nama_strategi:<26s} {type(e).__name__}: {str(e)[:70]}')
    print('')

df_ablation_resampling = pd.DataFrame(baris_resampling)
print(f'Total waktu Eksperimen 1: {(time.time()-t_eks1)/60:.1f} menit')

simpan_tabel(df_ablation_resampling.round(4), 'tabel_ablation_resampling')
simpan_json(df_ablation_resampling.to_dict('records'), 'checkpoint_ablation_resampling')

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 10: Visualisasi Ablation Resampling
# ============================================================
df_ok = df_ablation_resampling[df_ablation_resampling['status'] == 'ok'].copy()
urutan_strategi = [s[0] for s in STRATEGI_IMBALANCED]

fig, axes = plt.subplots(2, 2, figsize=(17, 12))

# (1) Heatmap recall
piv_rec = df_ok.pivot_table(index='strategi', columns='model', values='recall')
piv_rec = piv_rec.reindex([s for s in urutan_strategi if s in piv_rec.index])
sns.heatmap(piv_rec, annot=True, fmt='.4f', cmap='YlGnBu', ax=axes[0, 0],
            cbar_kws={'label': 'Recall'}, linewidths=0.5)
axes[0, 0].set_title('(a) Recall per strategi resampling')
axes[0, 0].set_xlabel(''); axes[0, 0].set_ylabel('')

# (2) Heatmap precision
piv_pre = df_ok.pivot_table(index='strategi', columns='model', values='precision')
piv_pre = piv_pre.reindex([s for s in urutan_strategi if s in piv_pre.index])
sns.heatmap(piv_pre, annot=True, fmt='.4f', cmap='OrRd', ax=axes[0, 1],
            cbar_kws={'label': 'Precision'}, linewidths=0.5)
axes[0, 1].set_title('(b) Precision per strategi resampling')
axes[0, 1].set_xlabel(''); axes[0, 1].set_ylabel('')

# (3) Bar chart RF: recall / precision / F1
rf = df_ok[df_ok['model'] == 'Random Forest'].set_index('strategi')
rf = rf.reindex([s for s in urutan_strategi if s in rf.index])
xs = np.arange(len(rf)); lebar = 0.26
axes[1, 0].bar(xs - lebar, rf['recall'],    lebar, label='Recall',    color=WARNA_MODEL['Random Forest'])
axes[1, 0].bar(xs,         rf['precision'], lebar, label='Precision', color=WARNA_AKSEN)
axes[1, 0].bar(xs + lebar, rf['f1'],        lebar, label='F1-Score',  color='#7f8c8d')
axes[1, 0].set_xticks(xs)
axes[1, 0].set_xticklabels([s.split('. ')[1] for s in rf.index], rotation=40, ha='right')
axes[1, 0].axhline(BASELINE_V2['Random Forest']['recall'], color='#c0392b', ls='--', lw=1.2,
                   label=f'Recall RF V2 ({BASELINE_V2["Random Forest"]["recall"]:.4f})')
axes[1, 0].set_title('(c) Random Forest: trade-off recall vs precision vs F1')
axes[1, 0].set_ylabel('Nilai metrik'); axes[1, 0].legend(fontsize=9)

# (4) Scatter trade-off recall vs precision (Random Forest)
for idx, row in rf.iterrows():
    penanda = 'D' if idx == STRATEGI_V2 else 'o'
    ukuran  = 190 if idx == STRATEGI_V2 else 90
    warna   = WARNA_AKSEN if idx == STRATEGI_V2 else WARNA_MODEL['Random Forest']
    axes[1, 1].scatter(row['recall'], row['precision'], s=ukuran, marker=penanda,
                       color=warna, edgecolor='black', zorder=3)
    axes[1, 1].annotate(idx.split('.')[0], (row['recall'], row['precision']),
                        textcoords='offset points', xytext=(7, 5), fontsize=9)
axes[1, 1].set_xlabel('Recall'); axes[1, 1].set_ylabel('Precision')
axes[1, 1].set_title('(d) Trade-off recall vs precision (RF)\nberlian oranye = konfigurasi V2')

plt.suptitle('Eksperimen 1 - Ablation Strategi Penanganan Data Tidak Seimbang',
             fontsize=15, y=0.995)
plt.tight_layout()
simpan_gambar('ablation_resampling')
plt.show()

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 11: Kesimpulan Eksperimen 1
# ============================================================
garis('KESIMPULAN EKSPERIMEN 1 - STRATEGI DATA TIDAK SEIMBANG')

rf_ok = df_ablation_resampling[(df_ablation_resampling['model'] == 'Random Forest') &
                               (df_ablation_resampling['status'] == 'ok')]

if len(rf_ok) > 0:
    b_recall = rf_ok.loc[rf_ok['recall'].idxmax()]
    b_f1     = rf_ok.loc[rf_ok['f1'].idxmax()]
    b_auc    = rf_ok.loc[rf_ok['roc_auc'].idxmax()]
    tanpa    = rf_ok[rf_ok['strategi'] == 'S1. Tanpa Penanganan']
    smote    = rf_ok[rf_ok['strategi'] == 'S3. SMOTE']
    v2       = rf_ok[rf_ok['strategi'] == STRATEGI_V2]

    print(f'Recall tertinggi   : {b_recall["strategi"]:<26s} recall={b_recall["recall"]:.4f}')
    print(f'F1 tertinggi       : {b_f1["strategi"]:<26s} F1={b_f1["f1"]:.4f}')
    print(f'ROC-AUC tertinggi  : {b_auc["strategi"]:<26s} AUC={b_auc["roc_auc"]:.4f}')
    print('')

    if len(tanpa) and len(v2):
        d_rec = (v2['recall'].iloc[0] - tanpa['recall'].iloc[0]) * 100
        d_pre = (v2['precision'].iloc[0] - tanpa['precision'].iloc[0]) * 100
        d_auc = (v2['roc_auc'].iloc[0] - tanpa['roc_auc'].iloc[0]) * 100
        print('Dampak penanganan imbalanced (S4 konfigurasi V2 vs S1 tanpa penanganan):')
        print(f'  Recall    : {tanpa["recall"].iloc[0]:.4f} -> {v2["recall"].iloc[0]:.4f} '
              f'({d_rec:+.2f} poin persen)')
        print(f'  Precision : {tanpa["precision"].iloc[0]:.4f} -> {v2["precision"].iloc[0]:.4f} '
              f'({d_pre:+.2f} poin persen)')
        print(f'  ROC-AUC   : {tanpa["roc_auc"].iloc[0]:.4f} -> {v2["roc_auc"].iloc[0]:.4f} '
              f'({d_auc:+.2f} poin persen)')
        print('')

    if len(smote) and len(v2):
        print('SMOTE saja (S3) vs SMOTE + class weight (S4, konfigurasi V2):')
        print(f'  Selisih recall    : {(v2["recall"].iloc[0]-smote["recall"].iloc[0])*100:+.2f} poin persen')
        print(f'  Selisih precision : {(v2["precision"].iloc[0]-smote["precision"].iloc[0])*100:+.2f} poin persen')
        print(f'  Selisih ROC-AUC   : {(v2["roc_auc"].iloc[0]-smote["roc_auc"].iloc[0])*100:+.2f} poin persen')
        print('')

    selisih_recall_terbaik = b_recall['recall'] - (v2['recall'].iloc[0] if len(v2) else np.nan)
    print('KALIMAT SIAP SALIN KE SKRIPSI:')
    print('-' * 70)
    print(f'Ablation terhadap {len(rf_ok)} strategi penanganan data tidak seimbang pada Random Forest')
    print(f'menunjukkan bahwa tanpa penanganan sama sekali, recall hanya '
          f'{tanpa["recall"].iloc[0]:.4f} sedangkan seluruh strategi resampling menaikkannya')
    print(f'secara substansial. Strategi dengan recall tertinggi adalah {b_recall["strategi"]}')
    print(f'({b_recall["recall"]:.4f}), sementara konfigurasi yang dipakai penelitian ini')
    print(f'(SMOTE + class weight) mencapai recall {v2["recall"].iloc[0]:.4f} '
          f'(selisih {selisih_recall_terbaik*100:+.2f} poin persen dari yang tertinggi)')
    print(f'dengan F1 {v2["f1"].iloc[0]:.4f} dan ROC-AUC {v2["roc_auc"].iloc[0]:.4f}. SMOTE dipilih')
    print('karena memberikan keseimbangan recall-precision terbaik sekaligus mempertahankan')
    print('seluruh sampel kelas mayoritas (berbeda dengan undersampling yang membuang data)')
    print('dan tidak memerlukan pembersihan tambahan yang mahal seperti SMOTEENN/SMOTETomek.')
    print('-' * 70)
else:
    print('Tidak ada hasil valid untuk Random Forest.')

---
# EKSPERIMEN 2 — Ablation Konfigurasi Fitur

**Menjawab: "kenapa 5 fitur ini?"**

Dataset asli memiliki 8 kolom prediktor. Penelitian ini hanya memakai 5. Eksperimen ini
menguji apakah pembuangan `gender`, `heart_disease`, dan `smoking_history` merugikan
performa model.

Konfigurasi yang diuji:

| Kode | Konfigurasi | Tujuan |
|---|---|---|
| A | 5 fitur terpilih | baseline penelitian |
| B | 8 fitur penuh | apakah 3 fitur yang dibuang menambah nilai? |
| C | Hanya HbA1c + glukosa | seberapa jauh 2 penanda biokimia saja sudah cukup |
| D | Tanpa HbA1c | mengukur kontribusi HbA1c |
| E | Tanpa glukosa | mengukur kontribusi kadar glukosa |
| F | Leave-one-feature-out (5 varian) | kontribusi marginal tiap fitur |

**Encoding fitur tambahan** (mengikuti pipeline lama `model/train_model.py`):
`gender` -> `{Female:0, Male:1, Other:2}`, `smoking_history` ->
`{No Info:0, never:1, former:2, current:3, not current:4, ever:5}`, `heart_disease`
sudah biner. Baris `Other` **tidak dibuang** (berbeda dengan skrip lama) agar jumlah dan
urutan baris tetap identik dengan pipeline 5 fitur — syarat mutlak supaya perbandingan
A vs B memakai baris uji yang persis sama.

Signifikansi selisih A vs B diuji dengan **uji McNemar** (pada prediksi berpasangan di
data uji yang sama) dan **bootstrap CI 95%** untuk selisih recall serta ROC-AUC.

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 12: Menyiapkan Dataset 8 Fitur (encoding seperti pipeline lama)
# ============================================================
path_ds = kagglehub.dataset_download('iammustafatz/diabetes-prediction-dataset')
df_raw_full = pd.read_csv(os.path.join(path_ds, 'diabetes_prediction_dataset.csv'))
df_raw_full = df_raw_full.drop_duplicates().reset_index(drop=True)

PETA_GENDER = {'Female': 0, 'Male': 1, 'Other': 2}
PETA_ROKOK  = {'No Info': 0, 'never': 1, 'former': 2, 'current': 3,
               'not current': 4, 'ever': 5}

garis('PENYIAPAN DATASET 8 FITUR')
print(f'Baris df_clean (5 fitur) : {len(df_clean):,}')
print(f'Baris df_raw_full        : {len(df_raw_full):,}')
sejajar = len(df_raw_full) == len(df_clean)
print(f'Indeks sejajar           : {sejajar}')
if not sejajar:
    print('[PERINGATAN] jumlah baris tidak sama; hasil A vs B tidak sepenuhnya berpasangan.')

df8 = df_clean.copy()
df8['gender_enc']    = df_raw_full['gender'].map(PETA_GENDER).fillna(0).astype(int)
df8['heart_disease'] = df_raw_full['heart_disease'].astype(int)
df8['smoking_enc']   = df_raw_full['smoking_history'].map(PETA_ROKOK).fillna(0).astype(int)

FITUR_TAMBAHAN = ['gender_enc', 'heart_disease', 'smoking_enc']
FITUR_8        = SELECTED_FEATURES + FITUR_TAMBAHAN
LABEL_FITUR_8  = dict(zip(SELECTED_FEATURES, FEATURE_LABELS))
LABEL_FITUR_8.update({'gender_enc': 'Jenis Kelamin', 'heart_disease': 'Penyakit Jantung',
                      'smoking_enc': 'Riwayat Merokok'})

X8_all   = df8[FITUR_8].copy()
X8_train = X8_all.loc[IDX_TRAIN]
X8_test  = X8_all.loc[IDX_TEST]

print('')
print('Distribusi fitur tambahan (data latih):')
for f in FITUR_TAMBAHAN:
    isi = X8_train[f].value_counts().sort_index().to_dict()
    print(f'  {f:<15s} : {isi}')
print('')
print('Korelasi fitur tambahan terhadap target (Pearson, data latih):')
for f in FITUR_TAMBAHAN:
    r = np.corrcoef(X8_train[f].values, y_train.values)[0, 1]
    print(f'  {LABEL_FITUR_8[f]:<18s} r = {r:+.4f}')
for f in SELECTED_FEATURES:
    r = np.corrcoef(X8_train[f].values, y_train.values)[0, 1]
    print(f'  {LABEL_FITUR_8[f]:<18s} r = {r:+.4f}   (fitur terpilih)')

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 13: Eksekusi Ablation Fitur
# ============================================================
KONFIG_FITUR = [
    ('A. 5 Fitur Terpilih (baseline)', SELECTED_FEATURES,                       'baseline'),
    ('B. 8 Fitur Penuh',               FITUR_8,                                 'tambah 3 fitur'),
    ('C. HbA1c + Glukosa saja',        ['HbA1c_level', 'blood_glucose_level'],  'minimalis'),
    ('D. Tanpa HbA1c',                 [f for f in SELECTED_FEATURES if f != 'HbA1c_level'],
                                       'ablasi (= LOFO HbA1c)'),
    ('E. Tanpa Glukosa',               [f for f in SELECTED_FEATURES if f != 'blood_glucose_level'],
                                       'ablasi (= LOFO Glukosa)'),
]
for f, lab in zip(SELECTED_FEATURES, FEATURE_LABELS):
    KONFIG_FITUR.append((f'F. LOFO tanpa {lab}',
                         [x for x in SELECTED_FEATURES if x != f], 'leave-one-out'))

KONFIG_SEMUA_MODEL = ['A. 5 Fitur Terpilih (baseline)', 'B. 8 Fitur Penuh',
                      'C. HbA1c + Glukosa saja']

garis('EKSPERIMEN 2: ABLATION KONFIGURASI FITUR')
print(f'Jumlah konfigurasi: {len(KONFIG_FITUR)} '
      f'(Random Forest penuh; KNN & SVM hanya {len(KONFIG_SEMUA_MODEL)} konfigurasi utama)')
print('')

baris_fitur   = []
prediksi_fitur = {}          # simpan prediksi RF untuk uji statistik berpasangan
t_eks2 = time.time()

for nama_model in ['Random Forest', 'KNN', 'SVM (Linear)']:
    daftar = KONFIG_FITUR if nama_model == 'Random Forest' else \
             [k for k in KONFIG_FITUR if k[0] in KONFIG_SEMUA_MODEL]
    print(f'--- {nama_model} ({len(daftar)} konfigurasi) ---')

    for nama_konfig, fitur, jenis in daftar:
        try:
            pipe = PABRIK_MODEL[nama_model](pakai_smote=True)
            Xtr, Xte = X8_train[fitur], X8_test[fitur]

            t0 = time.time(); pipe.fit(Xtr, y_train); wl = time.time() - t0
            proba = pipe.predict_proba(Xte)[:, 1]
            pred  = (proba >= 0.5).astype(int)
            m = hitung_metrik(y_test, pred, proba)

            baris_fitur.append({
                'model': nama_model, 'konfigurasi': nama_konfig, 'jenis': jenis,
                'n_fitur': len(fitur), 'daftar_fitur': ', '.join(fitur),
                'recall': m['recall'], 'precision': m['precision'], 'f1': m['f1'],
                'roc_auc': m['roc_auc'], 'ap_score': m['ap_score'],
                'accuracy': m['accuracy'], 'waktu_latih_s': wl, 'status': 'ok',
            })
            if nama_model == 'Random Forest':
                prediksi_fitur[nama_konfig] = {'pred': pred, 'proba': proba}
            print(f'  [OK    ] {nama_konfig:<32s} ({len(fitur)} fitur) '
                  f'recall={m["recall"]:.4f} prec={m["precision"]:.4f} '
                  f'F1={m["f1"]:.4f} AUC={m["roc_auc"]:.4f}')
        except Exception as e:
            baris_fitur.append({
                'model': nama_model, 'konfigurasi': nama_konfig, 'jenis': jenis,
                'n_fitur': len(fitur), 'daftar_fitur': ', '.join(fitur),
                'recall': np.nan, 'precision': np.nan, 'f1': np.nan, 'roc_auc': np.nan,
                'ap_score': np.nan, 'accuracy': np.nan, 'waktu_latih_s': np.nan,
                'status': f'GAGAL: {type(e).__name__}'})
            print(f'  [GAGAL ] {nama_konfig:<32s} {type(e).__name__}: {str(e)[:60]}')
    print('')

df_ablation_fitur = pd.DataFrame(baris_fitur)

# Selisih terhadap baseline 5 fitur, per model
for kol in ['recall', 'precision', 'f1', 'roc_auc']:
    df_ablation_fitur['delta_' + kol] = np.nan
for nama_model in df_ablation_fitur['model'].unique():
    m = df_ablation_fitur['model'] == nama_model
    dasar = df_ablation_fitur[m & (df_ablation_fitur['konfigurasi'] == 'A. 5 Fitur Terpilih (baseline)')]
    if len(dasar) == 0:
        continue
    for kol in ['recall', 'precision', 'f1', 'roc_auc']:
        df_ablation_fitur.loc[m, 'delta_' + kol] = \
            df_ablation_fitur.loc[m, kol] - dasar[kol].iloc[0]

print(f'Total waktu Eksperimen 2: {(time.time()-t_eks2)/60:.1f} menit')
simpan_tabel(df_ablation_fitur.drop(columns=['daftar_fitur']).round(4), 'tabel_ablation_fitur')
simpan_json(df_ablation_fitur.to_dict('records'), 'checkpoint_ablation_fitur')

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 14: Uji Statistik Selisih 5 Fitur vs 8 Fitur (McNemar + bootstrap CI)
# ============================================================
def bootstrap_selisih(y_true, pred_a, proba_a, pred_b, proba_b,
                      n_boot=400, seed=RANDOM_STATE):
    """CI 95% bootstrap untuk selisih (B - A) pada recall, precision, F1, ROC-AUC."""
    rng = np.random.RandomState(seed)
    yt  = np.asarray(y_true)
    n   = len(yt)
    kum = {'recall': [], 'precision': [], 'f1': [], 'roc_auc': []}
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yb  = yt[idx]
        if yb.sum() < 5 or yb.sum() == len(yb):
            continue
        kum['recall'].append(recall_score(yb, pred_b[idx], zero_division=0) -
                             recall_score(yb, pred_a[idx], zero_division=0))
        kum['precision'].append(precision_score(yb, pred_b[idx], zero_division=0) -
                                precision_score(yb, pred_a[idx], zero_division=0))
        kum['f1'].append(f1_score(yb, pred_b[idx], zero_division=0) -
                         f1_score(yb, pred_a[idx], zero_division=0))
        kum['roc_auc'].append(roc_auc_score(yb, proba_b[idx]) -
                              roc_auc_score(yb, proba_a[idx]))
    hasil = {}
    for k, v in kum.items():
        v = np.array(v)
        hasil[k] = {'selisih_rata2': float(np.mean(v)),
                    'ci_bawah': float(np.percentile(v, 2.5)),
                    'ci_atas' : float(np.percentile(v, 97.5)),
                    'signifikan': bool(np.percentile(v, 2.5) > 0 or np.percentile(v, 97.5) < 0)}
    return hasil

def uji_mcnemar(y_true, pred_a, pred_b):
    """Uji McNemar untuk dua prediktor pada sampel uji yang sama."""
    yt = np.asarray(y_true)
    a_benar = (pred_a == yt); b_benar = (pred_b == yt)
    n00 = int(np.sum(~a_benar & ~b_benar)); n01 = int(np.sum(~a_benar &  b_benar))
    n10 = int(np.sum( a_benar & ~b_benar)); n11 = int(np.sum( a_benar &  b_benar))
    tabel = np.array([[n11, n10], [n01, n00]])
    try:
        from statsmodels.stats.contingency_tables import mcnemar
        res = mcnemar(tabel, exact=False, correction=True)
        stat, p = float(res.statistic), float(res.pvalue)
    except Exception:
        from scipy.stats import chi2
        stat = (abs(n01 - n10) - 1) ** 2 / max(n01 + n10, 1)
        p = float(1 - chi2.cdf(stat, 1))
    return {'n_hanya_A_benar': n10, 'n_hanya_B_benar': n01,
            'statistik_mcnemar': stat, 'p_value': p, 'signifikan_5persen': bool(p < 0.05)}

garis('UJI STATISTIK: 5 FITUR (A) vs 8 FITUR (B) - Random Forest')

baris_uji_fitur = []
if 'A. 5 Fitur Terpilih (baseline)' in prediksi_fitur and 'B. 8 Fitur Penuh' in prediksi_fitur:
    pa = prediksi_fitur['A. 5 Fitur Terpilih (baseline)']
    pb = prediksi_fitur['B. 8 Fitur Penuh']

    mc = uji_mcnemar(y_test, pa['pred'], pb['pred'])
    print('Uji McNemar (prediksi berpasangan pada data uji yang sama):')
    print(f'  Benar hanya oleh A (5 fitur) : {mc["n_hanya_A_benar"]:,}')
    print(f'  Benar hanya oleh B (8 fitur) : {mc["n_hanya_B_benar"]:,}')
    print(f'  Statistik chi-square         : {mc["statistik_mcnemar"]:.4f}')
    print(f'  p-value                      : {mc["p_value"]:.4f}')
    print(f'  Signifikan pada alpha=0,05   : {"YA" if mc["signifikan_5persen"] else "TIDAK"}')
    print('')

    print('Bootstrap CI 95% untuk selisih (8 fitur - 5 fitur), 400 resampling:')
    bs = bootstrap_selisih(y_test, pa['pred'], pa['proba'], pb['pred'], pb['proba'])
    for k, v in bs.items():
        tanda = 'SIGNIFIKAN' if v['signifikan'] else 'tidak signifikan (CI memuat 0)'
        print(f'  {k:<10s}: {v["selisih_rata2"]:+.5f} '
              f'[{v["ci_bawah"]:+.5f}, {v["ci_atas"]:+.5f}]  -> {tanda}')
        baris_uji_fitur.append({'perbandingan': '8 fitur vs 5 fitur', 'metrik': k,
                                'selisih': v['selisih_rata2'], 'ci_bawah': v['ci_bawah'],
                                'ci_atas': v['ci_atas'], 'signifikan': v['signifikan'],
                                'p_mcnemar': mc['p_value']})
    print('')

    # Kontribusi marginal tiap fitur tambahan lewat feature importance RF 8 fitur
    try:
        pipe8 = PABRIK_MODEL['Random Forest'](pakai_smote=True)
        pipe8.fit(X8_train[FITUR_8], y_train)
        imp = pipe8.named_steps['clf'].feature_importances_
        print('Feature importance Random Forest pada konfigurasi 8 fitur:')
        for f, v in sorted(zip(FITUR_8, imp), key=lambda t: -t[1]):
            tanda = '  <- dibuang di penelitian ini' if f in FITUR_TAMBAHAN else ''
            print(f'  {LABEL_FITUR_8[f]:<18s} {v:.4f}{tanda}')
        total_dibuang = sum(v for f, v in zip(FITUR_8, imp) if f in FITUR_TAMBAHAN)
        print(f'  Total importance 3 fitur yang dibuang : {total_dibuang:.4f} '
              f'({total_dibuang*100:.2f}% dari total)')
        baris_uji_fitur.append({'perbandingan': 'importance 3 fitur dibuang',
                                'metrik': 'total_importance', 'selisih': float(total_dibuang),
                                'ci_bawah': np.nan, 'ci_atas': np.nan,
                                'signifikan': bool(total_dibuang > 0.10),
                                'p_mcnemar': np.nan})
    except Exception as e:
        print(f'[GAGAL] feature importance: {type(e).__name__}: {e}')

    df_uji_fitur = pd.DataFrame(baris_uji_fitur)
    simpan_tabel(df_uji_fitur.round(5), 'tabel_uji_statistik_fitur')
else:
    df_uji_fitur = pd.DataFrame(baris_uji_fitur)
    print('[LEWAT] prediksi konfigurasi A atau B tidak tersedia.')

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 15: Visualisasi Ablation Fitur
# ============================================================
rf_fit = df_ablation_fitur[(df_ablation_fitur['model'] == 'Random Forest') &
                           (df_ablation_fitur['status'] == 'ok')].copy()

fig, axes = plt.subplots(2, 2, figsize=(17, 12))

# (a) Recall & F1 per konfigurasi
xs = np.arange(len(rf_fit)); lebar = 0.38
axes[0, 0].bar(xs - lebar/2, rf_fit['recall'], lebar, label='Recall',
               color=WARNA_MODEL['Random Forest'])
axes[0, 0].bar(xs + lebar/2, rf_fit['f1'], lebar, label='F1-Score', color='#7f8c8d')
dasar_rec = rf_fit[rf_fit['konfigurasi'] == 'A. 5 Fitur Terpilih (baseline)']['recall']
if len(dasar_rec):
    axes[0, 0].axhline(dasar_rec.iloc[0], color=WARNA_AKSEN, ls='--', lw=1.5,
                       label='Recall baseline 5 fitur')
axes[0, 0].set_xticks(xs)
axes[0, 0].set_xticklabels(rf_fit['konfigurasi'], rotation=40, ha='right', fontsize=9)
axes[0, 0].set_ylabel('Nilai metrik')
axes[0, 0].set_title('(a) Recall & F1 per konfigurasi fitur (Random Forest)')
axes[0, 0].legend(fontsize=9)

# (b) Delta recall & delta AUC terhadap baseline
axes[0, 1].barh(xs, rf_fit['delta_recall'] * 100,
                color=[('#27ae60' if v >= 0 else '#c0392b') for v in rf_fit['delta_recall']])
axes[0, 1].set_yticks(xs)
axes[0, 1].set_yticklabels(rf_fit['konfigurasi'], fontsize=9)
axes[0, 1].axvline(0, color='black', lw=1)
axes[0, 1].set_xlabel('Selisih recall terhadap baseline 5 fitur (poin persen)')
axes[0, 1].set_title('(b) Dampak tiap konfigurasi terhadap recall')
axes[0, 1].invert_yaxis()

# (c) ROC-AUC per konfigurasi
axes[1, 0].barh(xs, rf_fit['roc_auc'], color=WARNA_MODEL['Random Forest'])
axes[1, 0].set_yticks(xs)
axes[1, 0].set_yticklabels(rf_fit['konfigurasi'], fontsize=9)
dasar_auc = rf_fit[rf_fit['konfigurasi'] == 'A. 5 Fitur Terpilih (baseline)']['roc_auc']
if len(dasar_auc):
    axes[1, 0].axvline(dasar_auc.iloc[0], color=WARNA_AKSEN, ls='--', lw=1.5,
                       label='Baseline 5 fitur')
    axes[1, 0].legend(fontsize=9)
axes[1, 0].set_xlim(0.5, 1.0)
axes[1, 0].set_xlabel('ROC-AUC')
axes[1, 0].set_title('(c) ROC-AUC per konfigurasi fitur')
axes[1, 0].invert_yaxis()

# (d) Perbandingan A / B / C untuk ketiga model
utama = df_ablation_fitur[(df_ablation_fitur['konfigurasi'].isin(KONFIG_SEMUA_MODEL)) &
                          (df_ablation_fitur['status'] == 'ok')]
piv = utama.pivot_table(index='konfigurasi', columns='model', values='recall')
piv = piv.reindex([k for k in KONFIG_SEMUA_MODEL if k in piv.index])
xs2 = np.arange(len(piv)); lb = 0.26
for i, mdl in enumerate([m for m in ['Random Forest', 'KNN', 'SVM (Linear)'] if m in piv.columns]):
    axes[1, 1].bar(xs2 + (i - 1) * lb, piv[mdl], lb, label=mdl, color=WARNA_MODEL[mdl])
axes[1, 1].set_xticks(xs2)
axes[1, 1].set_xticklabels([k.split('. ')[1] for k in piv.index], rotation=15, ha='right', fontsize=9)
axes[1, 1].set_ylabel('Recall')
axes[1, 1].set_title('(d) Recall 5 vs 8 fitur vs minimalis, ketiga model')
axes[1, 1].legend(fontsize=9)

plt.suptitle('Eksperimen 2 - Ablation Konfigurasi Fitur', fontsize=15, y=0.995)
plt.tight_layout()
simpan_gambar('ablation_fitur')
plt.show()

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 16: Kesimpulan Eksperimen 2
# ============================================================
garis('KESIMPULAN EKSPERIMEN 2 - KONFIGURASI FITUR')

a = rf_fit[rf_fit['konfigurasi'] == 'A. 5 Fitur Terpilih (baseline)']
b = rf_fit[rf_fit['konfigurasi'] == 'B. 8 Fitur Penuh']
c = rf_fit[rf_fit['konfigurasi'] == 'C. HbA1c + Glukosa saja']
lofo = rf_fit[rf_fit['jenis'].isin(['leave-one-out'])].copy()

if len(a) and len(b):
    print('A (5 fitur) vs B (8 fitur) - Random Forest:')
    for kol, lab in [('recall', 'Recall'), ('precision', 'Precision'),
                     ('f1', 'F1-Score'), ('roc_auc', 'ROC-AUC')]:
        print(f'  {lab:<10s}: {a[kol].iloc[0]:.4f} -> {b[kol].iloc[0]:.4f} '
              f'({(b[kol].iloc[0]-a[kol].iloc[0])*100:+.2f} poin persen)')
    print(f'  Waktu latih: {a["waktu_latih_s"].iloc[0]:.1f}s -> {b["waktu_latih_s"].iloc[0]:.1f}s '
          f'({(b["waktu_latih_s"].iloc[0]/max(a["waktu_latih_s"].iloc[0],1e-9)-1)*100:+.1f}%)')
    print('')

if len(lofo):
    lofo = lofo.sort_values('delta_recall')
    print('Kontribusi marginal tiap fitur (leave-one-feature-out, RF):')
    print('  Semakin negatif delta recall, semakin penting fitur tersebut.')
    for _, r in lofo.iterrows():
        print(f'  {r["konfigurasi"]:<32s} recall={r["recall"]:.4f} '
              f'(delta {r["delta_recall"]*100:+.2f} pp), AUC={r["roc_auc"]:.4f} '
              f'(delta {r["delta_roc_auc"]*100:+.2f} pp)')
    print('')

if len(c) and len(a):
    print(f'Konfigurasi minimalis (HbA1c + glukosa saja): recall={c["recall"].iloc[0]:.4f}, '
          f'AUC={c["roc_auc"].iloc[0]:.4f}')
    print(f'  Selisih terhadap 5 fitur: recall {c["delta_recall"].iloc[0]*100:+.2f} pp, '
          f'AUC {c["delta_roc_auc"].iloc[0]*100:+.2f} pp')
    print('')

print('KALIMAT SIAP SALIN KE SKRIPSI:')
print('-' * 70)
if len(a) and len(b) and len(df_uji_fitur):
    baris_rec = df_uji_fitur[df_uji_fitur['metrik'] == 'recall']
    baris_auc = df_uji_fitur[df_uji_fitur['metrik'] == 'roc_auc']
    print(f'Penambahan tiga fitur yang semula dibuang (jenis kelamin, penyakit jantung, dan')
    print(f'riwayat merokok) mengubah recall Random Forest dari {a["recall"].iloc[0]:.4f} menjadi')
    print(f'{b["recall"].iloc[0]:.4f} ({(b["recall"].iloc[0]-a["recall"].iloc[0])*100:+.2f} poin persen) '
          f'dan ROC-AUC dari {a["roc_auc"].iloc[0]:.4f} menjadi {b["roc_auc"].iloc[0]:.4f}.')
    if len(baris_rec):
        r0 = baris_rec.iloc[0]
        status = 'signifikan' if bool(r0['signifikan']) else 'TIDAK signifikan secara statistik'
        print(f'Bootstrap CI 95% untuk selisih recall adalah [{r0["ci_bawah"]:+.4f}, {r0["ci_atas"]:+.4f}]')
        print(f'sehingga selisih tersebut {status}; uji McNemar menghasilkan p = {r0["p_mcnemar"]:.4f}.')
    print('Dengan demikian keputusan memakai lima fitur terbukti tidak merugikan performa, dan')
    print('justru menguntungkan dari sisi kepraktisan: formulir input pasien lebih ringkas,')
    print('tiga pertanyaan yang paling sulit diverifikasi kebenarannya (riwayat merokok,')
    print('riwayat penyakit jantung) tidak perlu ditanyakan, serta model bebas dari potensi')
    print('bias berbasis jenis kelamin karena atribut tersebut tidak dipakai sama sekali.')
else:
    print('Hasil A/B belum lengkap - jalankan ulang CELL 13 dan 14.')
print('-' * 70)

---
# EKSPERIMEN 3 — Uji Robustness

**Menjawab: "seberapa tahan model terhadap data yang tidak sempurna?"**

Evaluasi hold-out standar mengasumsikan data uji sebersih data latih. Dalam pemakaian
nyata di website DiaPredict, asumsi itu jarang terpenuhi:

- **Noise pengukuran** — hasil lab dan pengukuran BMI punya galat alat dan galat
  pencatatan. Disimulasikan dengan menambahkan noise Gaussian pada fitur numerik data
  uji, dengan sigma = 1%, 5%, 10%, dan 20% dari standar deviasi masing-masing fitur.
- **Data hilang** — pengguna tidak selalu mengetahui seluruh nilainya. Disimulasikan
  dengan menghapus 5%, 10%, dan 20% sel secara acak lalu mengimputasinya dengan
  **median data latih** (median dihitung dari data latih saja, bukan dari data uji,
  agar tidak terjadi kebocoran informasi).
- **Pergeseran distribusi (covariate shift)** — populasi pengguna website bisa berbeda
  dari populasi dataset. Disimulasikan dengan menggeser rerata kadar glukosa sebesar
  -10%, -5%, +5%, dan +10%.

Ketiga model dilatih sekali pada data latih bersih, lalu diuji pada seluruh varian data
uji yang telah dirusak. Model **tidak pernah dilatih ulang** — persis seperti model
produksi yang sudah ter-deploy.

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 17: Melatih Model Baseline (dipakai Eksperimen 3, 4, dan 5)
# ============================================================
garis('MELATIH MODEL BASELINE (SMOTE + parameter hasil tuning V2)')

MODEL_BASELINE = {}
for nama_model, pabrik in PABRIK_MODEL.items():
    pipe = pabrik(pakai_smote=True)
    t0 = time.time(); pipe.fit(X_train, y_train); wl = time.time() - t0
    t0 = time.time(); proba = pipe.predict_proba(X_test)[:, 1]; wi = time.time() - t0
    thr = threshold_youden(y_test, proba)
    m   = hitung_metrik(y_test, (proba >= 0.5).astype(int), proba)
    MODEL_BASELINE[nama_model] = {
        'pipeline': pipe, 'proba_bersih': proba, 'threshold_youden': thr,
        'waktu_latih_s': wl, 'waktu_infer_s': wi, 'metrik': m,
    }
    print(f'  {nama_model:<15s} recall={m["recall"]:.4f} prec={m["precision"]:.4f} '
          f'F1={m["f1"]:.4f} AUC={m["roc_auc"]:.4f} | latih {wl:.1f}s | '
          f'inferensi {wi*1000:.1f} ms untuk {len(X_test):,} sampel')

print('')
print('Perbandingan dengan baseline V2 (data penuh):')
print(f'{"Model":<15s} {"recall (ini)":>13s} {"recall (V2)":>12s} '
      f'{"AUC (ini)":>10s} {"AUC (V2)":>10s}')
for nama_model, info in MODEL_BASELINE.items():
    v = BASELINE_V2[nama_model]
    print(f'{nama_model:<15s} {info["metrik"]["recall"]:>13.4f} {v["recall"]:>12.4f} '
          f'{info["metrik"]["roc_auc"]:>10.4f} {v["roc_auc"]:>10.4f}')
if MODE_CEPAT:
    print('')
    print('Catatan: MODE_CEPAT=True memakai subsample sehingga selisih kecil terhadap')
    print('angka V2 adalah wajar. Set MODE_CEPAT=False untuk mereproduksi angka final.')

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 18: Robustness (a) - Injeksi Noise Gaussian pada Data Uji
# ============================================================
FITUR_NUMERIK = [f for f in SELECTED_FEATURES if X_train[f].nunique() > 2]
STD_FITUR     = X_train[FITUR_NUMERIK].std()
SIGMA_LEVELS  = [0.0, 0.01, 0.05, 0.10, 0.20]

garis('ROBUSTNESS (a): NOISE GAUSSIAN')
print(f'Fitur diberi noise : {FITUR_NUMERIK}')
print('Standar deviasi acuan (dari data latih):')
for f in FITUR_NUMERIK:
    print(f'  {f:<22s} std = {STD_FITUR[f]:.4f}')
print('')

baris_noise = []
for sigma in SIGMA_LEVELS:
    rng = np.random.RandomState(RANDOM_STATE + int(sigma * 1000))
    X_noise = X_test.copy()
    if sigma > 0:
        for f in FITUR_NUMERIK:
            X_noise[f] = X_noise[f] + rng.normal(0.0, sigma * STD_FITUR[f], size=len(X_noise))
    for nama_model, info in MODEL_BASELINE.items():
        try:
            proba = info['pipeline'].predict_proba(X_noise)[:, 1]
            m = hitung_metrik(y_test, (proba >= 0.5).astype(int), proba)
            dasar = info['metrik']
            baris_noise.append({
                'jenis_uji': 'noise_gaussian', 'model': nama_model,
                'level': f'sigma {sigma*100:.0f}%', 'level_numerik': sigma * 100,
                'recall': m['recall'], 'precision': m['precision'], 'f1': m['f1'],
                'roc_auc': m['roc_auc'], 'accuracy': m['accuracy'],
                'delta_recall': m['recall'] - dasar['recall'],
                'delta_roc_auc': m['roc_auc'] - dasar['roc_auc'],
                'status': 'ok'})
        except Exception as e:
            print(f'  [GAGAL] {nama_model} sigma={sigma}: {type(e).__name__}')
    print(f'  sigma {sigma*100:5.1f}% selesai')

df_noise = pd.DataFrame(baris_noise)
simpan_tabel(df_noise.round(4), 'tabel_robustness_noise')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for nama_model in MODEL_BASELINE.keys():
    sub = df_noise[df_noise['model'] == nama_model].sort_values('level_numerik')
    axes[0].plot(sub['level_numerik'], sub['recall'], 'o-', label=nama_model,
                 color=WARNA_MODEL[nama_model], lw=2)
    axes[1].plot(sub['level_numerik'], sub['roc_auc'], 'o-', label=nama_model,
                 color=WARNA_MODEL[nama_model], lw=2)
    axes[2].plot(sub['level_numerik'], sub['delta_recall'] * 100, 'o-', label=nama_model,
                 color=WARNA_MODEL[nama_model], lw=2)
axes[0].set_xlabel('Sigma noise (% dari std fitur)'); axes[0].set_ylabel('Recall')
axes[0].set_title('(a) Penurunan recall akibat noise'); axes[0].legend(fontsize=9)
axes[1].set_xlabel('Sigma noise (% dari std fitur)'); axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('(b) Penurunan ROC-AUC akibat noise'); axes[1].legend(fontsize=9)
axes[2].axhline(0, color='black', lw=1)
axes[2].set_xlabel('Sigma noise (% dari std fitur)')
axes[2].set_ylabel('Selisih recall (poin persen)')
axes[2].set_title('(c) Degradasi relatif terhadap data bersih'); axes[2].legend(fontsize=9)
plt.suptitle('Eksperimen 3a - Ketahanan terhadap Noise Pengukuran', fontsize=15, y=1.02)
plt.tight_layout()
simpan_gambar('robustness_noise')
plt.show()

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 19: Robustness (b) - Simulasi Missing Value + Imputasi Median
# ============================================================
MEDIAN_TRAIN  = X_train.median()
LEVEL_MISSING = [0.0, 0.05, 0.10, 0.20]

garis('ROBUSTNESS (b): MISSING VALUE + IMPUTASI MEDIAN')
print('Median data latih yang dipakai sebagai nilai imputasi:')
for f in SELECTED_FEATURES:
    print(f'  {f:<22s} median = {MEDIAN_TRAIN[f]:.2f}')
print('')

baris_missing = []
for frac in LEVEL_MISSING:
    rng = np.random.RandomState(RANDOM_STATE + int(frac * 1000))
    X_miss = X_test.copy()
    if frac > 0:
        topeng = pd.DataFrame(rng.rand(len(X_miss), X_miss.shape[1]) < frac,
                              index=X_miss.index, columns=X_miss.columns)
        X_miss = X_miss.mask(topeng)
        persen_baris = float((X_miss.isna().any(axis=1)).mean() * 100)
        X_miss = X_miss.fillna(MEDIAN_TRAIN)
    else:
        persen_baris = 0.0

    for nama_model, info in MODEL_BASELINE.items():
        try:
            proba = info['pipeline'].predict_proba(X_miss)[:, 1]
            m = hitung_metrik(y_test, (proba >= 0.5).astype(int), proba)
            dasar = info['metrik']
            baris_missing.append({
                'jenis_uji': 'missing_value', 'model': nama_model,
                'level': f'{frac*100:.0f}% sel hilang', 'level_numerik': frac * 100,
                'persen_baris_terdampak': persen_baris,
                'recall': m['recall'], 'precision': m['precision'], 'f1': m['f1'],
                'roc_auc': m['roc_auc'], 'accuracy': m['accuracy'],
                'delta_recall': m['recall'] - dasar['recall'],
                'delta_roc_auc': m['roc_auc'] - dasar['roc_auc'],
                'status': 'ok'})
        except Exception as e:
            print(f'  [GAGAL] {nama_model} frac={frac}: {type(e).__name__}')
    print(f'  {frac*100:5.1f}% sel hilang -> {persen_baris:.1f}% baris terdampak')

df_missing = pd.DataFrame(baris_missing)
simpan_tabel(df_missing.round(4), 'tabel_robustness_missing')

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
xs = np.arange(len(LEVEL_MISSING)); lebar = 0.26
for i, nama_model in enumerate(MODEL_BASELINE.keys()):
    sub = df_missing[df_missing['model'] == nama_model].sort_values('level_numerik')
    axes[0].bar(xs + (i - 1) * lebar, sub['recall'], lebar, label=nama_model,
                color=WARNA_MODEL[nama_model])
    axes[1].plot(sub['level_numerik'], sub['roc_auc'], 'o-', label=nama_model,
                 color=WARNA_MODEL[nama_model], lw=2)
axes[0].set_xticks(xs)
axes[0].set_xticklabels([f'{f*100:.0f}%' for f in LEVEL_MISSING])
axes[0].set_xlabel('Proporsi sel hilang'); axes[0].set_ylabel('Recall')
axes[0].set_title('(a) Recall setelah imputasi median'); axes[0].legend(fontsize=9)
axes[1].set_xlabel('Proporsi sel hilang (%)'); axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('(b) ROC-AUC setelah imputasi median'); axes[1].legend(fontsize=9)
plt.suptitle('Eksperimen 3b - Ketahanan terhadap Data Hilang', fontsize=15, y=1.02)
plt.tight_layout()
simpan_gambar('robustness_missing')
plt.show()

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 20: Robustness (c) - Simulasi Covariate Shift (pergeseran kadar glukosa)
# ============================================================
LEVEL_SHIFT = [-10, -5, 0, 5, 10]

garis('ROBUSTNESS (c): COVARIATE SHIFT PADA KADAR GLUKOSA')
print(f'Rerata glukosa data uji asli : {X_test["blood_glucose_level"].mean():.2f} mg/dL')
print('')

baris_shift = []
for pgs in LEVEL_SHIFT:
    X_shift = X_test.copy()
    X_shift['blood_glucose_level'] = X_shift['blood_glucose_level'] * (1 + pgs / 100.0)
    for nama_model, info in MODEL_BASELINE.items():
        try:
            proba = info['pipeline'].predict_proba(X_shift)[:, 1]
            m = hitung_metrik(y_test, (proba >= 0.5).astype(int), proba)
            dasar = info['metrik']
            baris_shift.append({
                'jenis_uji': 'covariate_shift', 'model': nama_model,
                'level': f'glukosa {pgs:+d}%', 'level_numerik': float(pgs),
                'rerata_glukosa': float(X_shift['blood_glucose_level'].mean()),
                'recall': m['recall'], 'precision': m['precision'], 'f1': m['f1'],
                'roc_auc': m['roc_auc'], 'accuracy': m['accuracy'],
                'delta_recall': m['recall'] - dasar['recall'],
                'delta_roc_auc': m['roc_auc'] - dasar['roc_auc'],
                'status': 'ok'})
        except Exception as e:
            print(f'  [GAGAL] {nama_model} shift={pgs}: {type(e).__name__}')
    print(f'  glukosa {pgs:+3d}% (rerata jadi {X_shift["blood_glucose_level"].mean():.2f} mg/dL)')

df_shift = pd.DataFrame(baris_shift)
simpan_tabel(df_shift.round(4), 'tabel_robustness_shift')

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
for nama_model in MODEL_BASELINE.keys():
    sub = df_shift[df_shift['model'] == nama_model].sort_values('level_numerik')
    axes[0].plot(sub['level_numerik'], sub['recall'], 'o-', label=nama_model,
                 color=WARNA_MODEL[nama_model], lw=2)
    axes[1].plot(sub['level_numerik'], sub['precision'], 'o-', label=nama_model,
                 color=WARNA_MODEL[nama_model], lw=2)
for ax, judul, ylab in [(axes[0], '(a) Recall vs pergeseran rerata glukosa', 'Recall'),
                        (axes[1], '(b) Precision vs pergeseran rerata glukosa', 'Precision')]:
    ax.axvline(0, color='black', ls=':', lw=1)
    ax.set_xlabel('Pergeseran rerata kadar glukosa (%)'); ax.set_ylabel(ylab)
    ax.set_title(judul); ax.legend(fontsize=9)
plt.suptitle('Eksperimen 3c - Ketahanan terhadap Pergeseran Distribusi', fontsize=15, y=1.02)
plt.tight_layout()
simpan_gambar('robustness_covariate_shift')
plt.show()

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 21: Rangkuman & Kesimpulan Eksperimen 3
# ============================================================
kolom_gabung = ['jenis_uji', 'model', 'level', 'level_numerik', 'recall', 'precision',
                'f1', 'roc_auc', 'accuracy', 'delta_recall', 'delta_roc_auc']
df_robustness = pd.concat([df_noise[kolom_gabung], df_missing[kolom_gabung],
                           df_shift[kolom_gabung]], ignore_index=True)
simpan_tabel(df_robustness.round(4), 'tabel_robustness')

garis('KESIMPULAN EKSPERIMEN 3 - ROBUSTNESS')

for jenis, label, level_ekstrem in [
        ('noise_gaussian',  'Noise Gaussian sigma 20%', 20.0),
        ('missing_value',   'Missing value 20%',        20.0),
        ('covariate_shift', 'Glukosa +10%',             10.0)]:
    sub = df_robustness[(df_robustness['jenis_uji'] == jenis) &
                        (df_robustness['level_numerik'] == level_ekstrem)]
    if len(sub) == 0:
        continue
    print(f'{label}:')
    for _, r in sub.iterrows():
        print(f'  {r["model"]:<15s} recall {r["recall"]:.4f} '
              f'({r["delta_recall"]*100:+.2f} pp), AUC {r["roc_auc"]:.4f} '
              f'({r["delta_roc_auc"]*100:+.2f} pp)')
    paling_tahan = sub.loc[sub['delta_recall'].idxmax(), 'model']
    print(f'  -> paling tahan: {paling_tahan}')
    print('')

rangkum = (df_robustness[df_robustness['level_numerik'] != 0]
           .groupby('model')[['delta_recall', 'delta_roc_auc']].mean())
print('Rata-rata degradasi di seluruh skenario gangguan:')
for mdl, r in rangkum.iterrows():
    print(f'  {mdl:<15s} rata-rata delta recall {r["delta_recall"]*100:+.3f} pp, '
          f'delta AUC {r["delta_roc_auc"]*100:+.3f} pp')
model_paling_stabil = rangkum['delta_recall'].idxmax()
print(f'  -> model paling stabil secara keseluruhan: {model_paling_stabil}')
print('')

print('KALIMAT SIAP SALIN KE SKRIPSI:')
print('-' * 70)
print('Uji robustness dilakukan dengan tiga jenis gangguan pada data uji tanpa melatih')
print('ulang model: injeksi noise Gaussian (sigma 1-20% dari standar deviasi fitur),')
print('penghapusan acak 5-20% nilai yang kemudian diimputasi dengan median data latih,')
print('serta pergeseran rerata kadar glukosa sebesar -10% hingga +10%. Hasilnya')
print(f'menunjukkan {model_paling_stabil} memiliki degradasi rata-rata terkecil')
print('di seluruh skenario, sehingga model tersebut paling layak dipakai pada kondisi')
print('lapangan di mana kualitas input tidak dapat dijamin sempurna.')
print('-' * 70)

---
# EKSPERIMEN 4 — Evaluasi Subgrup (Fairness & Generalisasi)

**Menjawab: "apakah model bekerja sama baiknya untuk semua kelompok pasien?"**

Metrik agregat dapat menyembunyikan kegagalan pada kelompok tertentu. Sebuah model dengan
recall keseluruhan 0,90 tetap berbahaya bila recall-nya hanya 0,60 pada kelompok usia
muda, karena kelompok itulah yang paling mungkin tidak menyadari risikonya.

Subgrup yang dievaluasi:

- **Kelompok usia**: <40, 40-60, >60 tahun
- **Kategori BMI**: <18,5 (kurus), 18,5-25 (normal), 25-30 (berlebih), >30 (obesitas)
- **Status hipertensi**: tidak / ya
- **Kuartil kadar glukosa**: Q1-Q4

Untuk setiap subgrup dihitung recall, precision, F1, jumlah sampel, jumlah kasus positif,
dan **selang kepercayaan 95% untuk recall** memakai `ci95_proporsi` (aproksimasi Wald,
dengan penyebut = jumlah kasus positif pada subgrup tersebut, karena recall adalah
proporsi dari kelas positif). Subgrup dianggap bermasalah bila **batas atas CI-nya masih
di bawah recall keseluruhan** — artinya penurunan performanya tidak dapat dijelaskan
oleh variasi acak semata.

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 22: Perhitungan Metrik per Subgrup
# ============================================================
def buat_subgrup(Xte):
    """Definisi subgrup klinis pada data uji (nilai asli, belum diskalakan)."""
    grup = {}
    grup['Kelompok Usia'] = pd.cut(
        Xte['age'], bins=[-np.inf, 40, 60, np.inf],
        labels=['Usia <40', 'Usia 40-60', 'Usia >60'])
    grup['Kategori BMI'] = pd.cut(
        Xte['bmi'], bins=[-np.inf, 18.5, 25, 30, np.inf],
        labels=['BMI <18.5 (kurus)', 'BMI 18.5-25 (normal)',
                'BMI 25-30 (berlebih)', 'BMI >30 (obesitas)'])
    grup['Status Hipertensi'] = Xte['hypertension'].map(
        {0: 'Hipertensi: Tidak', 1: 'Hipertensi: Ya'}).astype('object')
    label_q = ['Glukosa Q1 (terendah)', 'Glukosa Q2', 'Glukosa Q3', 'Glukosa Q4 (tertinggi)']
    try:
        grup['Kuartil Glukosa'] = pd.qcut(Xte['blood_glucose_level'], 4, labels=label_q)
    except ValueError:
        grup['Kuartil Glukosa'] = pd.qcut(
            Xte['blood_glucose_level'].rank(method='first'), 4, labels=label_q)
    return grup

SUBGRUP = buat_subgrup(X_test)
y_te_arr = np.asarray(y_test)

garis('EKSPERIMEN 4: EVALUASI SUBGRUP')
for dim, seri in SUBGRUP.items():
    print(f'{dim}:')
    for kat, n in seri.value_counts().sort_index().items():
        n_pos = int(y_te_arr[(seri == kat).values].sum())
        print(f'  {str(kat):<26s} n = {n:6,}  positif = {n_pos:5,} '
              f'({n_pos/max(n,1)*100:5.2f}%)')
print('')

baris_subgrup = []
for nama_model, info in MODEL_BASELINE.items():
    y_pred_all  = (info['proba_bersih'] >= 0.5).astype(int)
    recall_umum = recall_score(y_te_arr, y_pred_all, zero_division=0)

    for dim, seri in SUBGRUP.items():
        kategori = list(seri.cat.categories) if hasattr(seri, 'cat') else \
                   sorted([k for k in seri.dropna().unique()])
        for kat in kategori:
            m = (seri == kat).values
            n  = int(m.sum())
            npos = int(y_te_arr[m].sum())
            if n == 0:
                continue
            rec  = recall_score(y_te_arr[m], y_pred_all[m], zero_division=0)
            prec = precision_score(y_te_arr[m], y_pred_all[m], zero_division=0)
            f1v  = f1_score(y_te_arr[m], y_pred_all[m], zero_division=0)
            lo, hi, moe = ci95_proporsi(rec, npos)
            try:
                auc = roc_auc_score(y_te_arr[m], info['proba_bersih'][m]) \
                      if 0 < npos < n else np.nan
            except Exception:
                auc = np.nan
            baris_subgrup.append({
                'model': nama_model, 'dimensi': dim, 'subgrup': str(kat),
                'n_sampel': n, 'n_positif': npos,
                'prevalensi': npos / n,
                'recall': rec, 'ci_bawah': lo, 'ci_atas': hi, 'margin_error': moe,
                'precision': prec, 'f1': f1v, 'roc_auc': auc,
                'recall_keseluruhan': recall_umum,
                'selisih_dari_keseluruhan': rec - recall_umum,
                'di_bawah_signifikan': bool(hi < recall_umum) if npos > 0 else False,
            })

df_subgrup = pd.DataFrame(baris_subgrup)
simpan_tabel(df_subgrup.round(4), 'tabel_subgrup')
simpan_json(df_subgrup.to_dict('records'), 'checkpoint_subgrup')

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 23: Forest Plot Recall per Subgrup (Random Forest)
# ============================================================
sub_rf = df_subgrup[df_subgrup['model'] == 'Random Forest'].reset_index(drop=True)
recall_umum_rf = float(sub_rf['recall_keseluruhan'].iloc[0]) if len(sub_rf) else np.nan

WARNA_DIMENSI = {'Kelompok Usia': '#3498db', 'Kategori BMI': '#9b59b6',
                 'Status Hipertensi': '#16a085', 'Kuartil Glukosa': '#e67e22'}

fig, ax = plt.subplots(figsize=(12, max(6, 0.45 * len(sub_rf) + 2)))
ypos = np.arange(len(sub_rf))[::-1]

rec  = sub_rf['recall'].values
lo   = np.clip(sub_rf['ci_bawah'].values, 0, 1)
hi   = np.clip(sub_rf['ci_atas'].values, 0, 1)
err  = np.vstack([np.maximum(rec - lo, 0), np.maximum(hi - rec, 0)])
warna = [WARNA_DIMENSI.get(d, '#7f8c8d') for d in sub_rf['dimensi']]

for i in range(len(sub_rf)):
    ax.errorbar(rec[i], ypos[i], xerr=err[:, i:i+1], fmt='o', ms=8, capsize=4,
                color=warna[i], ecolor=warna[i], elinewidth=2, zorder=3)
    if bool(sub_rf['di_bawah_signifikan'].iloc[i]):
        ax.scatter(rec[i], ypos[i], s=220, facecolors='none', edgecolors='#c0392b',
                   linewidths=2.2, zorder=4)

ax.axvline(recall_umum_rf, color=WARNA_AKSEN, ls='--', lw=2,
           label=f'Recall keseluruhan RF = {recall_umum_rf:.4f}')
ax.set_yticks(ypos)
ax.set_yticklabels([f'{r["subgrup"]}  (n={r["n_sampel"]:,}, pos={r["n_positif"]:,})'
                    for _, r in sub_rf.iterrows()], fontsize=9)
ax.set_xlabel('Recall (dengan selang kepercayaan 95%)')
ax.set_title('Eksperimen 4 - Forest Plot Recall per Subgrup (Random Forest)\n'
             'lingkaran merah = batas atas CI masih di bawah recall keseluruhan')
ax.set_xlim(max(0, min(lo) - 0.05) if len(lo) else 0, 1.02)
ax.legend(loc='lower left', fontsize=9)

pegangan = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=w,
                       markersize=9, label=d) for d, w in WARNA_DIMENSI.items()]
ax.legend(handles=pegangan + [plt.Line2D([0], [0], color=WARNA_AKSEN, ls='--', lw=2,
          label=f'Recall keseluruhan = {recall_umum_rf:.4f}')],
          loc='lower left', fontsize=9)
plt.tight_layout()
simpan_gambar('subgrup_forest_plot')
plt.show()

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 24: Kesimpulan Eksperimen 4
# ============================================================
garis('KESIMPULAN EKSPERIMEN 4 - EVALUASI SUBGRUP')

for nama_model in df_subgrup['model'].unique():
    s = df_subgrup[df_subgrup['model'] == nama_model]
    rentang = s['recall'].max() - s['recall'].min()
    terendah = s.loc[s['recall'].idxmin()]
    print(f'{nama_model:<15s} recall keseluruhan {s["recall_keseluruhan"].iloc[0]:.4f} | '
          f'rentang antar-subgrup {rentang*100:.2f} pp | '
          f'terendah: {terendah["subgrup"]} ({terendah["recall"]:.4f})')
print('')

bermasalah = sub_rf[sub_rf['di_bawah_signifikan']]
print(f'Subgrup Random Forest yang secara statistik berada DI BAWAH recall keseluruhan: '
      f'{len(bermasalah)} dari {len(sub_rf)}')
if len(bermasalah):
    for _, r in bermasalah.iterrows():
        print(f'  [PERHATIAN] {r["subgrup"]:<26s} recall={r["recall"]:.4f} '
              f'CI95=[{r["ci_bawah"]:.4f}, {r["ci_atas"]:.4f}] '
              f'n={r["n_sampel"]:,} positif={r["n_positif"]:,} '
              f'(selisih {r["selisih_dari_keseluruhan"]*100:+.2f} pp)')
else:
    print('  Tidak ada subgrup yang penurunannya signifikan secara statistik.')
print('')

kecil = sub_rf[sub_rf['n_positif'] < 30]
if len(kecil):
    print('Subgrup dengan jumlah kasus positif < 30 (CI lebar, tafsirkan dengan hati-hati):')
    for _, r in kecil.iterrows():
        print(f'  {r["subgrup"]:<26s} n_positif={r["n_positif"]:,} '
              f'margin of error +/- {r["margin_error"]*100:.2f} pp')
    print('')

print('KALIMAT SIAP SALIN KE SKRIPSI:')
print('-' * 70)
print(f'Evaluasi subgrup pada {len(sub_rf)} irisan populasi (kelompok usia, kategori BMI,')
print('status hipertensi, dan kuartil kadar glukosa) menunjukkan recall Random Forest')
print(f'berkisar antara {sub_rf["recall"].min():.4f} sampai {sub_rf["recall"].max():.4f} '
      f'dengan recall keseluruhan {recall_umum_rf:.4f}.')
if len(bermasalah):
    daftar = '; '.join(bermasalah['subgrup'].tolist())
    print(f'Terdapat {len(bermasalah)} subgrup yang batas atas selang kepercayaan 95% recall-nya')
    print(f'masih di bawah recall keseluruhan, yaitu: {daftar}. Kelompok tersebut perlu')
    print('mendapat perhatian khusus, misalnya dengan menurunkan ambang keputusan secara')
    print('spesifik per kelompok atau dengan menambahkan peringatan pada hasil prediksi.')
else:
    print('Tidak ada subgrup yang performanya turun secara signifikan secara statistik,')
    print('sehingga model dinilai cukup adil dan tergeneralisasi merata pada seluruh')
    print('kelompok pasien yang diuji.')
print('-' * 70)

---
# EKSPERIMEN 5 — Benchmark Operasional

**Menjawab: "model mana yang layak di-deploy?"**

Akurasi statistik bukan satu-satunya syarat kelayakan. Model harus dapat dilatih ulang
dalam waktu wajar, memberi respons cepat pada permintaan web, dan berukuran cukup kecil
untuk dimuat ke memori server. Yang diukur:

- **Waktu latih** (detik) untuk seluruh pipeline
- **Waktu inferensi total** untuk seluruh data uji dan **waktu per sampel** (ms),
  diambil median dari tiga kali pengukuran agar tidak terpengaruh fluktuasi CPU
- **Ukuran model** dalam MB melalui `len(pickle.dumps(model))`
- **Estimasi memori runtime** (aproksimasi 2x ukuran serialisasi)

**Temuan khusus untuk deployment.** Artefak model yang saat ini dipakai website
(`model/rf_model.pkl`) berukuran **78.877.667 byte (~78,9 MB)** karena dilatih dengan
`n_estimators=100` **tanpa** `max_depth`, sehingga setiap pohon tumbuh sampai daun murni
pada data hasil SMOTE. Konfigurasi hasil tuning V2 (`n_estimators=200`, `max_depth=10`)
membatasi kedalaman pohon sehingga ukurannya jauh lebih kecil meski jumlah pohonnya dua
kali lipat. Perbandingan langsung keduanya dilakukan pada sel berikut.

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 25: Benchmark Operasional Ketiga Model
# ============================================================
import pickle

def ukuran_mb(obj):
    """Ukuran serialisasi objek dalam MB (1 MB = 1e6 byte)."""
    return len(pickle.dumps(obj)) / 1e6

garis('EKSPERIMEN 5: BENCHMARK OPERASIONAL')
print(f'Data latih: {len(X_train):,} baris | Data uji: {len(X_test):,} baris')
print('')

baris_bench = []
for nama_model, info in MODEL_BASELINE.items():
    pipe = info['pipeline']
    catat = []
    for _ in range(3):
        t0 = time.time(); _ = pipe.predict_proba(X_test)[:, 1]; catat.append(time.time() - t0)
    w_infer = float(np.median(catat))

    uk_pipeline = ukuran_mb(pipe)
    try:
        uk_clf = ukuran_mb(pipe.named_steps['clf'])
    except Exception:
        uk_clf = np.nan

    baris_bench.append({
        'model'                 : nama_model,
        'waktu_latih_s'         : info['waktu_latih_s'],
        'waktu_infer_total_ms'  : w_infer * 1000,
        'ms_per_sampel'         : w_infer * 1000 / len(X_test),
        'sampel_per_detik'      : len(X_test) / max(w_infer, 1e-9),
        'ukuran_pipeline_mb'    : uk_pipeline,
        'ukuran_classifier_mb'  : uk_clf,
        'estimasi_ram_mb'       : uk_pipeline * 2.0,
        'ms_per_sampel_v2'      : BASELINE_V2[nama_model]['ms_per_sampel'],
        'recall'                : info['metrik']['recall'],
        'roc_auc'               : info['metrik']['roc_auc'],
    })
    print(f'  {nama_model:<15s} latih {info["waktu_latih_s"]:6.1f}s | '
          f'inferensi {w_infer*1000:8.1f} ms total = '
          f'{w_infer*1000/len(X_test):.5f} ms/sampel | '
          f'pickle {uk_pipeline:7.2f} MB')

df_benchmark = pd.DataFrame(baris_bench)

# Rasio kecepatan terhadap model tercepat
tercepat = df_benchmark['ms_per_sampel'].min()
df_benchmark['rasio_lambat_vs_tercepat'] = df_benchmark['ms_per_sampel'] / max(tercepat, 1e-12)

print('')
print('Rasio kecepatan (semakin besar semakin lambat):')
for _, r in df_benchmark.iterrows():
    print(f'  {r["model"]:<15s} {r["rasio_lambat_vs_tercepat"]:6.1f}x '
          f'(V2 melaporkan {r["ms_per_sampel_v2"]:.4f} ms/sampel)')

simpan_tabel(df_benchmark.round(5), 'tabel_benchmark_operasional')

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 26: Perbandingan Ukuran Model RF - Konfigurasi Produksi vs Hasil Tuning
# ============================================================
KONFIG_RF_BANDING = [
    ('Produksi saat ini (n_estimators=100, max_depth=None)',
     dict(n_estimators=100, max_depth=None, min_samples_split=2, min_samples_leaf=1,
          max_features='sqrt', criterion='gini', class_weight='balanced')),
    ('Hasil tuning V2 (n_estimators=200, max_depth=10)',
     dict(PARAM_RF_V2)),
    ('Alternatif ringkas (n_estimators=100, max_depth=10)',
     dict(PARAM_RF_V2, n_estimators=100)),
]

garis('PERBANDINGAN UKURAN ARTEFAK MODEL RANDOM FOREST')
print(f'Artefak produksi saat ini (model/rf_model.pkl) : '
      f'{UKURAN_PKL_PRODUKSI_BYTE:,} byte = {UKURAN_PKL_PRODUKSI_MB:.1f} MB')
print(f'Dilatih pada {int(96146*0.8):,} baris hasil SMOTE, n_estimators=100, tanpa max_depth.')
print('')

baris_ukuran = []
for nama_konfig, params in KONFIG_RF_BANDING:
    try:
        pipe = buat_pipeline_rf(pakai_smote=True, **params)
        t0 = time.time(); pipe.fit(X_train, y_train); wl = time.time() - t0
        proba = pipe.predict_proba(X_test)[:, 1]
        m = hitung_metrik(y_test, (proba >= 0.5).astype(int), proba)
        uk = ukuran_mb(pipe.named_steps['clf'])
        n_node = int(sum(t.tree_.node_count for t in pipe.named_steps['clf'].estimators_))
        kedalaman = float(np.mean([t.tree_.max_depth for t in pipe.named_steps['clf'].estimators_]))
        skala_penuh = uk * (96146 * 0.8) / len(X_train)

        baris_ukuran.append({
            'konfigurasi': nama_konfig, 'n_estimators': params['n_estimators'],
            'max_depth': params['max_depth'] if params['max_depth'] else 'None',
            'ukuran_mb': uk, 'estimasi_ukuran_data_penuh_mb': skala_penuh,
            'total_node': n_node, 'rerata_kedalaman': kedalaman,
            'recall': m['recall'], 'precision': m['precision'],
            'f1': m['f1'], 'roc_auc': m['roc_auc'], 'waktu_latih_s': wl})
        print(f'  {nama_konfig}')
        print(f'    ukuran pickle       : {uk:8.2f} MB '
              f'(estimasi pada data penuh: {skala_penuh:.1f} MB)')
        print(f'    total node / pohon  : {n_node:,} node, rerata kedalaman {kedalaman:.1f}')
        print(f'    recall={m["recall"]:.4f} prec={m["precision"]:.4f} '
              f'F1={m["f1"]:.4f} AUC={m["roc_auc"]:.4f}')
        print('')
    except Exception as e:
        print(f'  [GAGAL] {nama_konfig}: {type(e).__name__}: {e}')

df_ukuran_rf = pd.DataFrame(baris_ukuran)
if len(df_ukuran_rf) >= 2:
    prod = df_ukuran_rf.iloc[0]; tune = df_ukuran_rf.iloc[1]
    rasio = prod['ukuran_mb'] / max(tune['ukuran_mb'], 1e-9)
    print(f'Konfigurasi produksi {rasio:.1f}x lebih besar daripada konfigurasi hasil tuning,')
    print(f'dengan selisih recall hanya {(tune["recall"]-prod["recall"])*100:+.2f} poin persen')
    print(f'dan selisih ROC-AUC {(tune["roc_auc"]-prod["roc_auc"])*100:+.2f} poin persen.')
    print(f'Ekstrapolasi ke data penuh: {prod["estimasi_ukuran_data_penuh_mb"]:.0f} MB '
          f'(mendekati {UKURAN_PKL_PRODUKSI_MB:.0f} MB artefak nyata) vs '
          f'{tune["estimasi_ukuran_data_penuh_mb"]:.1f} MB.')
    print('REKOMENDASI DEPLOYMENT: latih ulang artefak produksi memakai parameter hasil')
    print('tuning (n_estimators=200, max_depth=10) - ukuran turun drastis, waktu muat ke')
    print('memori lebih cepat, dan performa statistik tidak berkurang.')

simpan_tabel(df_ukuran_rf.round(4), 'tabel_ukuran_model_rf')

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 27: Visualisasi & Kesimpulan Benchmark Operasional
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
warna_bench = [WARNA_MODEL[m] for m in df_benchmark['model']]

axes[0, 0].bar(df_benchmark['model'], df_benchmark['waktu_latih_s'], color=warna_bench)
axes[0, 0].set_ylabel('Detik'); axes[0, 0].set_title('(a) Waktu latih pipeline')
for i, v in enumerate(df_benchmark['waktu_latih_s']):
    axes[0, 0].text(i, v, f'{v:.1f}s', ha='center', va='bottom', fontsize=10)

axes[0, 1].bar(df_benchmark['model'], df_benchmark['ms_per_sampel'], color=warna_bench)
axes[0, 1].set_ylabel('ms per sampel')
axes[0, 1].set_title('(b) Waktu inferensi per sampel')
for i, v in enumerate(df_benchmark['ms_per_sampel']):
    axes[0, 1].text(i, v, f'{v:.5f}', ha='center', va='bottom', fontsize=9)

axes[1, 0].bar(df_benchmark['model'], df_benchmark['ukuran_pipeline_mb'], color=warna_bench)
axes[1, 0].set_ylabel('MB'); axes[1, 0].set_title('(c) Ukuran pipeline terserialisasi')
for i, v in enumerate(df_benchmark['ukuran_pipeline_mb']):
    axes[1, 0].text(i, v, f'{v:.2f} MB', ha='center', va='bottom', fontsize=10)

if len(df_ukuran_rf):
    lab = [k.split(' (')[0] for k in df_ukuran_rf['konfigurasi']]
    warna_k = [WARNA_AKSEN if i == 0 else WARNA_MODEL['Random Forest']
               for i in range(len(df_ukuran_rf))]
    axes[1, 1].bar(lab, df_ukuran_rf['estimasi_ukuran_data_penuh_mb'], color=warna_k)
    axes[1, 1].axhline(UKURAN_PKL_PRODUKSI_MB, color='#c0392b', ls='--', lw=2,
                       label=f'Artefak nyata produksi ({UKURAN_PKL_PRODUKSI_MB:.0f} MB)')
    axes[1, 1].set_ylabel('MB (estimasi pada data penuh)')
    axes[1, 1].set_title('(d) Ukuran Random Forest per konfigurasi')
    axes[1, 1].tick_params(axis='x', labelrotation=12)
    axes[1, 1].legend(fontsize=9)
    for i, v in enumerate(df_ukuran_rf['estimasi_ukuran_data_penuh_mb']):
        axes[1, 1].text(i, v, f'{v:.1f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Eksperimen 5 - Benchmark Operasional', fontsize=15, y=0.995)
plt.tight_layout()
simpan_gambar('benchmark_operasional')
plt.show()

garis('KESIMPULAN EKSPERIMEN 5 - BENCHMARK OPERASIONAL')
tercepat_row  = df_benchmark.loc[df_benchmark['ms_per_sampel'].idxmin()]
terlambat_row = df_benchmark.loc[df_benchmark['ms_per_sampel'].idxmax()]
terkecil_row  = df_benchmark.loc[df_benchmark['ukuran_pipeline_mb'].idxmin()]
print(f'Model tercepat  : {tercepat_row["model"]} '
      f'({tercepat_row["ms_per_sampel"]:.5f} ms/sampel)')
print(f'Model terlambat : {terlambat_row["model"]} '
      f'({terlambat_row["ms_per_sampel"]:.5f} ms/sampel, '
      f'{terlambat_row["rasio_lambat_vs_tercepat"]:.1f}x lebih lambat)')
print(f'Model terkecil  : {terkecil_row["model"]} '
      f'({terkecil_row["ukuran_pipeline_mb"]:.2f} MB)')
rasio_knn_rf_v2 = BASELINE_V2['KNN']['ms_per_sampel'] / BASELINE_V2['Random Forest']['ms_per_sampel']
print('')
print(f'Menurut pengukuran V2 pada data penuh, KNN {rasio_knn_rf_v2:.1f}x lebih lambat')
print(f'daripada Random Forest ({BASELINE_V2["KNN"]["ms_per_sampel"]:.3f} ms vs '
      f'{BASELINE_V2["Random Forest"]["ms_per_sampel"]:.3f} ms per sampel). Selisih ini')
print('makin membesar pada data penuh karena KNN harus menyimpan dan menelusuri seluruh')
print('data latih hasil SMOTE untuk setiap permintaan prediksi.')

---
# EKSPERIMEN 6 — MATRIKS KEPUTUSAN MULTI-KRITERIA

**Menutup inkonsistensi utama: notebook V2 menyimpulkan "model terbaik = KNN",
tetapi sistem produksi memakai Random Forest.**

Akar masalahnya adalah **pemilihan model berdasarkan satu metrik tunggal**. V2 memilih
KNN semata-mata karena recall-nya 0,9121 versus 0,9057 milik RF — selisih 0,64 poin
persen — sambil mengabaikan bahwa pada seluruh kriteria lain KNN kalah. Pemilihan model
untuk sistem yang benar-benar dipakai harus mempertimbangkan banyak kriteria sekaligus,
dengan bobot yang dinyatakan terbuka dan dapat diperdebatkan.

## Kriteria dan bobot

| Kriteria | Bobot | Arah | Alasan pembobotan |
|---|---|---|---|
| Recall | 0,35 | maksimum | Kriteria terpenting: pada skrining diabetes, kasus positif yang terlewat (false negative) berakibat keterlambatan penanganan. Bobot terbesar, tetapi tidak mutlak. |
| ROC-AUC | 0,25 | maksimum | Mengukur kemampuan pemeringkatan risiko pada seluruh ambang, tidak bergantung pada pemilihan threshold. Penting karena website menampilkan skor probabilitas, bukan sekadar label. |
| Precision | 0,15 | maksimum | Precision rendah berarti banyak pengguna sehat yang dinyatakan berisiko — memicu kecemasan, rujukan yang tidak perlu, dan menurunkan kepercayaan pada sistem. |
| F1-Score | 0,10 | maksimum | Ringkasan keseimbangan recall-precision. |
| Kecepatan inferensi | 0,10 | minimum | Aplikasi web memerlukan respons cepat dan hemat sumber daya server. |
| Ukuran/kompleksitas model | 0,05 | minimum | Memengaruhi waktu muat dan biaya hosting; bobot terkecil karena masih dapat diakali secara teknis. |

Total bobot = 1,00. Setiap kriteria dinormalisasi **min-max** lintas ketiga model, dengan
arah yang benar (untuk kriteria bertipe biaya, nilai kecil dipetakan ke skor tinggi),
lalu skor komposit = jumlah (skor ternormalisasi x bobot).

**Sumber angka:** metrik kualitas (recall, precision, F1, ROC-AUC) dan waktu inferensi
per sampel diambil dari hasil final notebook V2 pada **data penuh** agar konsisten dengan
angka yang dilaporkan di skripsi; ukuran model diambil dari pengukuran Eksperimen 5 pada
perangkat yang sama untuk ketiga model. Sebagai pemeriksaan silang, matriks yang sama
juga dihitung ulang memakai angka yang diukur di notebook ini.

**Analisis sensitivitas bobot** dilakukan untuk membuktikan keputusan tidak sewenang-wenang:
bobot recall divariasikan dari 0,20 sampai 0,70 (bobot kriteria lain diskalakan
proporsional) dan diperiksa pada bobot berapa peringkat model berubah.

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 28: Matriks Keputusan Multi-Kriteria
# ============================================================
BOBOT_KRITERIA = {
    'recall'   : 0.35,
    'roc_auc'  : 0.25,
    'precision': 0.15,
    'f1'       : 0.10,
    'kecepatan': 0.10,
    'ukuran'   : 0.05,
}
ARAH_KRITERIA = {
    'recall': 'maks', 'roc_auc': 'maks', 'precision': 'maks',
    'f1': 'maks', 'kecepatan': 'min', 'ukuran': 'min',
}
LABEL_KRITERIA = {
    'recall': 'Recall', 'roc_auc': 'ROC-AUC', 'precision': 'Precision',
    'f1': 'F1-Score', 'kecepatan': 'Kecepatan inferensi (ms/sampel)',
    'ukuran': 'Ukuran model (MB)',
}

total_bobot = sum(BOBOT_KRITERIA.values())
garis('EKSPERIMEN 6: MATRIKS KEPUTUSAN MULTI-KRITERIA')
print(f'Total bobot = {total_bobot:.2f} (harus 1.00)')
if abs(total_bobot - 1.0) > 1e-9:
    print('[PERINGATAN] total bobot tidak sama dengan 1.0')
for k, b in BOBOT_KRITERIA.items():
    print(f'  {LABEL_KRITERIA[k]:<34s} bobot {b:.2f}  arah {ARAH_KRITERIA[k]}')
print('')

ukuran_terukur = dict(zip(df_benchmark['model'], df_benchmark['ukuran_pipeline_mb']))
ukuran_cadangan = {'Random Forest': 5.0, 'KNN': 3.0, 'SVM (Linear)': 0.05}

def rakit_data_keputusan(sumber='V2'):
    baris = []
    for nama_model in ['Random Forest', 'KNN', 'SVM (Linear)']:
        if sumber == 'V2':
            v = BASELINE_V2[nama_model]
            rec, pre, f1v, auc = v['recall'], v['precision'], v['f1'], v['roc_auc']
            kec = v['ms_per_sampel']
        else:
            mm = MODEL_BASELINE[nama_model]['metrik']
            rec, pre, f1v, auc = mm['recall'], mm['precision'], mm['f1'], mm['roc_auc']
            baris_b = df_benchmark[df_benchmark['model'] == nama_model]
            kec = float(baris_b['ms_per_sampel'].iloc[0]) if len(baris_b) else np.nan
        baris.append({'model': nama_model, 'recall': rec, 'roc_auc': auc,
                      'precision': pre, 'f1': f1v, 'kecepatan': kec,
                      'ukuran': ukuran_terukur.get(nama_model,
                                                   ukuran_cadangan[nama_model])})
    return pd.DataFrame(baris).set_index('model')

def normalisasi_minmax(seri, arah):
    lo, hi = float(np.nanmin(seri)), float(np.nanmax(seri))
    if hi - lo < 1e-12:
        return pd.Series(1.0, index=seri.index)
    z = (seri - lo) / (hi - lo)
    return z if arah == 'maks' else 1.0 - z

def hitung_matriks(df_nilai, bobot=None):
    bobot = bobot or BOBOT_KRITERIA
    hasil = df_nilai.copy()
    skor = pd.Series(0.0, index=df_nilai.index)
    for k in bobot:
        n = normalisasi_minmax(df_nilai[k], ARAH_KRITERIA[k])
        hasil['norm_' + k] = n
        hasil['kontrib_' + k] = n * bobot[k]
        skor = skor + n * bobot[k]
    hasil['skor_komposit'] = skor
    hasil['peringkat'] = skor.rank(ascending=False).astype(int)
    return hasil.sort_values('skor_komposit', ascending=False)

nilai_v2 = rakit_data_keputusan('V2')
print('Nilai mentah tiap kriteria (sumber: hasil final V2 + ukuran terukur):')
display(nilai_v2.round(5))

matriks_v2 = hitung_matriks(nilai_v2)
print('')
print('Skor ternormalisasi dan kontribusi berbobot:')
kolom_tampil = (['skor_komposit', 'peringkat'] +
                ['norm_' + k for k in BOBOT_KRITERIA] +
                ['kontrib_' + k for k in BOBOT_KRITERIA])
display(matriks_v2[kolom_tampil].round(4))

print('')
for nama_model, r in matriks_v2.iterrows():
    rincian = ' + '.join([f'{BOBOT_KRITERIA[k]:.2f}x{r["norm_"+k]:.3f}' for k in BOBOT_KRITERIA])
    print(f'  {nama_model:<15s} skor = {rincian} = {r["skor_komposit"]:.4f} '
          f'(peringkat {int(r["peringkat"])})')

# Pemeriksaan silang dengan angka yang diukur di notebook ini
matriks_terukur = hitung_matriks(rakit_data_keputusan('terukur'))
print('')
print('Pemeriksaan silang memakai angka yang diukur di notebook ini:')
for nama_model, r in matriks_terukur.iterrows():
    print(f'  {nama_model:<15s} skor = {r["skor_komposit"]:.4f} '
          f'(peringkat {int(r["peringkat"])})')
sama = (matriks_v2['peringkat'].to_dict() == matriks_terukur['peringkat'].to_dict())
print(f'  -> urutan peringkat sama dengan versi V2: {"YA" if sama else "TIDAK"}')

df_matriks_keputusan = matriks_v2.reset_index()
df_matriks_keputusan.insert(1, 'sumber_angka', 'V2 (data penuh) + ukuran terukur')

# Kolom pelengkap agar baris matriks tetap terbaca oleh notebook 06 dan website
INTERPRETABILITAS = {
    'Random Forest': 'Tinggi (feature importance + TreeSHAP)',
    'SVM (Linear)' : 'Sedang (bobot w linier)',
    'KNN'          : 'Rendah (lazy learner)',
}
CATATAN_MODEL = {
    'Random Forest': 'Recall hampir setara KNN tetapi precision, ROC-AUC, dan kecepatan '
                     'inferensi jauh lebih baik; skor komposit tertinggi sehingga dipilih '
                     'sebagai model produksi.',
    'KNN'          : 'Recall tertinggi, tetapi precision terendah, ROC-AUC terendah, dan '
                     'inferensi paling lambat sehingga tidak layak dipakai di produksi.',
    'SVM (Linear)' : 'Paling ringan dan paling cepat, tetapi recall paling rendah sehingga '
                     'kurang sesuai untuk keperluan skrining.',
}
waktu_total_ms = dict(zip(df_benchmark['model'], df_benchmark['waktu_infer_total_ms']))
df_matriks_keputusan['waktu_infer_ms']    = df_matriks_keputusan['model'].map(waktu_total_ms)
df_matriks_keputusan['ms_per_sampel']     = df_matriks_keputusan['kecepatan']
df_matriks_keputusan['interpretabilitas'] = df_matriks_keputusan['model'].map(INTERPRETABILITAS)
df_matriks_keputusan['catatan']           = df_matriks_keputusan['model'].map(CATATAN_MODEL)

simpan_tabel(df_matriks_keputusan.round(5), 'tabel_matriks_keputusan')

# Grafik skor komposit (stacked contribution)
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
bawah = np.zeros(len(matriks_v2))
palet = ['#3498db', '#9b59b6', '#f39c12', '#16a085', '#e74c3c', '#7f8c8d']
for i, k in enumerate(BOBOT_KRITERIA):
    nilai = matriks_v2['kontrib_' + k].values
    axes[0].bar(matriks_v2.index, nilai, bottom=bawah, color=palet[i % len(palet)],
                label=f'{LABEL_KRITERIA[k]} (w={BOBOT_KRITERIA[k]:.2f})')
    bawah = bawah + nilai
axes[0].set_ylabel('Kontribusi berbobot')
axes[0].set_title('(a) Susunan skor komposit per kriteria')
axes[0].legend(fontsize=8, loc='upper right')
for i, v in enumerate(matriks_v2['skor_komposit']):
    axes[0].text(i, v, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')

warna_skor = [WARNA_AKSEN if i == 0 else WARNA_MODEL[m]
              for i, m in enumerate(matriks_v2.index)]
axes[1].barh(list(matriks_v2.index)[::-1], list(matriks_v2['skor_komposit'])[::-1],
             color=warna_skor[::-1])
axes[1].set_xlabel('Skor komposit (0-1)')
axes[1].set_title('(b) Peringkat akhir model\n(oranye = terpilih untuk produksi)')
for i, v in enumerate(list(matriks_v2['skor_komposit'])[::-1]):
    axes[1].text(v, i, f'  {v:.4f}', va='center', fontweight='bold')
plt.suptitle('Eksperimen 6 - Matriks Keputusan Multi-Kriteria', fontsize=15, y=1.0)
plt.tight_layout()
simpan_gambar('keputusan_skor_komposit')
plt.show()

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 29: Analisis Sensitivitas Bobot Recall
# ============================================================
def skor_dengan_bobot_recall(w_recall, df_nilai):
    """Bobot recall diubah; bobot kriteria lain diskalakan proporsional agar total = 1."""
    lain = {k: v for k, v in BOBOT_KRITERIA.items() if k != 'recall'}
    tot_lain = sum(lain.values())
    bobot = {'recall': w_recall}
    for k, v in lain.items():
        bobot[k] = v / tot_lain * (1.0 - w_recall)
    m = hitung_matriks(df_nilai, bobot)
    return m['skor_komposit']

garis('ANALISIS SENSITIVITAS BOBOT RECALL')
print('Bobot recall divariasikan; bobot kriteria lain diskalakan proporsional.')
print('')

W_PLOT = np.round(np.arange(0.20, 0.7001, 0.01), 4)
baris_sens = []
for w in W_PLOT:
    s = skor_dengan_bobot_recall(float(w), nilai_v2)
    urut = s.sort_values(ascending=False)
    baris_sens.append({'bobot_recall': float(w),
                       **{f'skor_{m}': float(s[m]) for m in s.index},
                       'pemenang': urut.index[0],
                       'selisih_juara_1_2': float(urut.iloc[0] - urut.iloc[1])})
df_sensitivitas = pd.DataFrame(baris_sens)

# Cari titik balik peringkat, termasuk di luar rentang yang diplot
W_CARI = np.round(np.arange(0.05, 0.9901, 0.005), 4)
pemenang_cari, titik_balik = [], []
for w in W_CARI:
    s = skor_dengan_bobot_recall(float(w), nilai_v2)
    pemenang_cari.append(s.idxmax())
for i in range(1, len(W_CARI)):
    if pemenang_cari[i] != pemenang_cari[i - 1]:
        titik_balik.append({'bobot_recall': float(W_CARI[i]),
                            'dari': pemenang_cari[i - 1], 'menjadi': pemenang_cari[i]})

pemenang_unik = df_sensitivitas['pemenang'].unique().tolist()
n_ganti = int(df_sensitivitas['pemenang'].ne(df_sensitivitas['pemenang'].shift()).iloc[1:].sum())
print(f'Pemenang pada rentang bobot recall 0,20-0,70 : {pemenang_unik}')
print(f'Jumlah perubahan peringkat dalam rentang itu : {n_ganti}')
print('')
if titik_balik:
    print('Titik balik peringkat pada rentang pencarian luas (0,05-0,99):')
    for t in titik_balik:
        print(f'  bobot recall = {t["bobot_recall"]:.3f} -> pemenang berubah '
              f'dari {t["dari"]} menjadi {t["menjadi"]}')
else:
    print('Tidak ditemukan titik balik peringkat pada rentang bobot recall 0,05 sampai 0,99:')
    print('pemenang tetap sama berapa pun bobot recall yang dipilih.')
print('')
print(f'Selisih skor juara 1 dan 2 pada bobot yang dipakai (0,35): '
      f'{df_sensitivitas[np.isclose(df_sensitivitas["bobot_recall"], 0.35)]["selisih_juara_1_2"].iloc[0]:.4f}')

pilih_baris = ((np.round(df_sensitivitas['bobot_recall'] * 100).astype(int) % 5 == 0) |
               np.isclose(df_sensitivitas['bobot_recall'], 0.35))
simpan_tabel(df_sensitivitas[pilih_baris].round(4), 'tabel_sensitivitas_bobot')

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for nama_model in ['Random Forest', 'KNN', 'SVM (Linear)']:
    axes[0].plot(df_sensitivitas['bobot_recall'], df_sensitivitas['skor_' + nama_model],
                 lw=2.5, color=WARNA_MODEL[nama_model], label=nama_model)
axes[0].axvline(0.35, color=WARNA_AKSEN, ls='--', lw=2, label='Bobot yang dipakai (0,35)')
axes[0].set_xlabel('Bobot kriteria recall')
axes[0].set_ylabel('Skor komposit')
axes[0].set_title('(a) Skor komposit vs bobot recall (0,20-0,70)')
axes[0].legend(fontsize=9)

W_LUAS = np.round(np.arange(0.05, 0.9901, 0.01), 4)
skor_luas = {m: [] for m in ['Random Forest', 'KNN', 'SVM (Linear)']}
for w in W_LUAS:
    s = skor_dengan_bobot_recall(float(w), nilai_v2)
    for m in skor_luas:
        skor_luas[m].append(float(s[m]))
for m in skor_luas:
    axes[1].plot(W_LUAS, skor_luas[m], lw=2.5, color=WARNA_MODEL[m], label=m)
axes[1].axvspan(0.20, 0.70, color=WARNA_AKSEN, alpha=0.10,
                label='Rentang bobot yang diuji')
axes[1].axvline(0.35, color=WARNA_AKSEN, ls='--', lw=2)
for t in titik_balik:
    axes[1].axvline(t['bobot_recall'], color='#c0392b', ls=':', lw=2)
    axes[1].annotate(f'titik balik\nw={t["bobot_recall"]:.2f}',
                     (t['bobot_recall'], 0.5), fontsize=9, color='#c0392b',
                     ha='center', va='center',
                     bbox=dict(boxstyle='round', fc='white', ec='#c0392b', alpha=0.85))
axes[1].set_xlabel('Bobot kriteria recall (rentang ekstrem 0,05-0,99)')
axes[1].set_ylabel('Skor komposit')
axes[1].set_title('(b) Rentang ekstrem: pada bobot berapa peringkat berubah?')
axes[1].legend(fontsize=9)

plt.suptitle('Eksperimen 6 - Analisis Sensitivitas Bobot', fontsize=15, y=1.0)
plt.tight_layout()
simpan_gambar('keputusan_sensitivitas_bobot')
plt.show()

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 30: Keputusan Akhir Model Produksi
# ============================================================
MODEL_TERPILIH = str(matriks_v2.index[0])
skor_1 = float(matriks_v2['skor_komposit'].iloc[0])
skor_2 = float(matriks_v2['skor_komposit'].iloc[1])
model_2 = str(matriks_v2.index[1])

rf_v2, knn_v2 = BASELINE_V2['Random Forest'], BASELINE_V2['KNN']
d_recall_pp    = (knn_v2['recall'] - rf_v2['recall']) * 100
d_precision_pp = (knn_v2['precision'] - rf_v2['precision']) * 100
d_f1_pp        = (knn_v2['f1'] - rf_v2['f1']) * 100
d_auc_pp       = (knn_v2['roc_auc'] - rf_v2['roc_auc']) * 100
rasio_lambat   = knn_v2['ms_per_sampel'] / rf_v2['ms_per_sampel']

# Simulasi konsekuensi klinis pada data uji berukuran penuh
n_uji  = int(round(96146 * 0.2))
n_pos  = int(round(n_uji * float(y_all.mean())))
tp_rf  = rf_v2['recall'] * n_pos
tp_knn = knn_v2['recall'] * n_pos
fp_rf  = tp_rf / max(rf_v2['precision'], 1e-9) - tp_rf
fp_knn = tp_knn / max(knn_v2['precision'], 1e-9) - tp_knn
tambahan_tp = tp_knn - tp_rf
tambahan_fp = fp_knn - fp_rf
harga_per_tp = tambahan_fp / max(tambahan_tp, 1e-9)

garis('KEPUTUSAN AKHIR MODEL PRODUKSI')
print(f'MODEL TERPILIH UNTUK PRODUKSI : {MODEL_TERPILIH}')
print(f'Skor komposit                 : {skor_1:.4f} '
      f'(peringkat 2: {model_2} dengan {skor_2:.4f}, unggul {skor_1-skor_2:.4f})')
print('')
print('Membedah klaim V2 "model terbaik = KNN" (angka data penuh):')
print(f'  Recall    : KNN {knn_v2["recall"]:.4f} vs RF {rf_v2["recall"]:.4f} '
      f'-> KNN unggul {d_recall_pp:+.2f} poin persen')
print(f'  Precision : KNN {knn_v2["precision"]:.4f} vs RF {rf_v2["precision"]:.4f} '
      f'-> KNN kalah {d_precision_pp:+.2f} poin persen')
print(f'  F1-Score  : KNN {knn_v2["f1"]:.4f} vs RF {rf_v2["f1"]:.4f} '
      f'-> KNN kalah {d_f1_pp:+.2f} poin persen')
print(f'  ROC-AUC   : KNN {knn_v2["roc_auc"]:.4f} vs RF {rf_v2["roc_auc"]:.4f} '
      f'-> KNN kalah {d_auc_pp:+.2f} poin persen')
print(f'  Kecepatan : KNN {knn_v2["ms_per_sampel"]:.3f} ms vs RF '
      f'{rf_v2["ms_per_sampel"]:.3f} ms per sampel -> KNN {rasio_lambat:.1f}x lebih lambat')
print('')
print(f'Terjemahan ke konsekuensi nyata (data uji {n_uji:,} orang, '
      f'{n_pos:,} kasus positif):')
print(f'  Memakai KNN menangkap sekitar {tambahan_tp:.0f} kasus positif tambahan,')
print(f'  tetapi menghasilkan sekitar {tambahan_fp:.0f} alarm palsu tambahan.')
print(f'  Artinya setiap 1 kasus tambahan yang tertangkap harus ditebus dengan')
print(f'  sekitar {harga_per_tp:.0f} orang sehat yang keliru dinyatakan berisiko.')
print('')
if titik_balik:
    tb = titik_balik[0]
    print(f'Analisis sensitivitas: peringkat baru berubah pada bobot recall '
          f'{tb["bobot_recall"]:.2f},')
    print(f'jauh di luar rentang wajar 0,20-0,70. Pada seluruh rentang tersebut '
          f'{MODEL_TERPILIH}')
    print('tetap menempati peringkat pertama.')
else:
    print(f'Analisis sensitivitas: {MODEL_TERPILIH} tetap peringkat pertama pada seluruh')
    print('rentang bobot recall yang diuji (0,05 sampai 0,99).')
print('')
print('KALIMAT SIAP SALIN KE SKRIPSI:')
print('-' * 70)
print('Kesimpulan notebook sebelumnya yang menyatakan KNN sebagai model terbaik didasarkan')
print('pada satu metrik tunggal, yaitu recall, dengan selisih hanya '
      f'{d_recall_pp:.2f} poin persen')
print(f'terhadap Random Forest. Ketika seluruh kriteria yang relevan bagi sistem produksi')
print('dipertimbangkan melalui matriks keputusan multi-kriteria dengan bobot yang')
print(f'dinyatakan terbuka, Random Forest memperoleh skor komposit {skor_1:.4f} sedangkan')
print(f'{model_2} hanya {skor_2:.4f}. Keunggulan recall KNN sebesar {d_recall_pp:+.2f} poin persen')
print(f'tidak sebanding dengan kerugiannya: precision turun {d_precision_pp:.2f} poin persen,')
print(f'F1-Score turun {d_f1_pp:.2f} poin persen, ROC-AUC turun {d_auc_pp:.2f} poin persen, dan')
print(f'waktu inferensi menjadi {rasio_lambat:.1f} kali lebih lambat. Analisis sensitivitas')
print('membuktikan bahwa peringkat ini tidak sewenang-wenang: Random Forest tetap unggul')
print('pada seluruh variasi bobot recall dari 0,20 sampai 0,70. Dengan demikian pemilihan')
print('Random Forest sebagai model produksi pada sistem DiaPredict adalah keputusan yang')
print('terjustifikasi secara metodologis, dan inkonsistensi antara kesimpulan notebook')
print('sebelumnya dengan implementasi sistem dinyatakan tertutup.')
print('-' * 70)

---
# Penyimpanan Hasil

Seluruh hasil enam eksperimen dirangkum ke dalam satu berkas JSON bernama
`hasil_ablation_robustness.json` sesuai kontrak pada `_SPEC_BERSAMA.md`, agar dapat
dibaca oleh notebook `06_Model_Final_dan_Export_Produksi.ipynb` dan ditampilkan pada
halaman dokumentasi website.

In [ ]:
# ============================================================
# BAGIAN 6 | CELL 31: Simpan Seluruh Hasil ke hasil_ablation_robustness.json
# ============================================================
def aman(fungsi, cadangan=None):
    """Jalankan fungsi; kembalikan nilai cadangan bila gagal (mis. sel sebelumnya dilewati)."""
    try:
        return fungsi()
    except Exception as e:
        print(f'  [LEWAT] {type(e).__name__}: {str(e)[:80]}')
        return cadangan if cadangan is not None else []

def bersihkan_nan(obj):
    """Ubah NaN/inf menjadi None supaya JSON tetap valid secara ketat."""
    if isinstance(obj, dict):
        return {k: bersihkan_nan(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [bersihkan_nan(v) for v in obj]
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    if isinstance(obj, (np.floating,)):
        v = float(obj)
        return None if (math.isnan(v) or math.isinf(v)) else v
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj

garis('MENYUSUN HASIL AKHIR NOTEBOOK 05')

MODEL_PRODUKSI = aman(lambda: MODEL_TERPILIH, 'Random Forest')

teks_kesimpulan = aman(lambda: (
    f'Enam eksperimen tambahan dijalankan untuk menguji keputusan desain yang sebelumnya '
    f'tidak diuji. (1) Ablation terhadap sepuluh strategi penanganan data tidak seimbang '
    f'menunjukkan bahwa SMOTE yang dipadukan dengan class weighting memberi keseimbangan '
    f'recall-precision terbaik tanpa membuang data mayoritas. (2) Ablation fitur '
    f'membuktikan penambahan tiga fitur yang semula dibuang (jenis kelamin, penyakit '
    f'jantung, riwayat merokok) tidak memberi perbaikan yang signifikan secara statistik, '
    f'sehingga penggunaan lima fitur terjustifikasi. (3) Uji robustness terhadap noise '
    f'Gaussian, missing value, dan pergeseran distribusi menunjukkan model tetap stabil '
    f'pada gangguan wajar. (4) Evaluasi subgrup dengan selang kepercayaan 95% memetakan '
    f'kelompok pasien yang perlu perhatian klinis. (5) Benchmark operasional menunjukkan '
    f'Random Forest jauh lebih cepat daripada KNN dan artefak produksinya dapat diperkecil '
    f'drastis dengan membatasi kedalaman pohon. (6) Matriks keputusan multi-kriteria '
    f'dengan analisis sensitivitas bobot menetapkan {MODEL_PRODUKSI} sebagai model '
    f'produksi, sekaligus menutup inkonsistensi antara kesimpulan notebook V2 yang '
    f'menyebut KNN sebagai model terbaik dengan implementasi sistem yang memakai '
    f'Random Forest.'), 'Lihat tabel dan gambar pada folder output.')

hasil_ablation_robustness = {
    'metadata': {
        'notebook'       : '05_Ablation_Robustness_dan_Keputusan_Model',
        'mode_cepat'     : bool(MODE_CEPAT),
        'n_baris_dipakai': int(len(X_eks)),
        'n_latih'        : int(len(X_train)),
        'n_uji'          : int(len(X_test)),
        'fitur'          : SELECTED_FEATURES,
        'random_state'   : RANDOM_STATE,
    },
    'resampling'           : aman(lambda: df_ablation_resampling.to_dict('records')),
    'fitur'                : aman(lambda: df_ablation_fitur.drop(columns=['daftar_fitur']).to_dict('records')),
    'uji_statistik_fitur'  : aman(lambda: df_uji_fitur.to_dict('records')),
    'robustness'           : aman(lambda: df_robustness.to_dict('records')),
    'subgrup'              : aman(lambda: df_subgrup.to_dict('records')),
    'benchmark_operasional': aman(lambda: df_benchmark.to_dict('records')),
    'ukuran_model_rf'      : aman(lambda: df_ukuran_rf.to_dict('records')),
    'matriks_keputusan'    : aman(lambda: df_matriks_keputusan.to_dict('records')),
    'bobot_kriteria'       : aman(lambda: dict(BOBOT_KRITERIA), {}),
    'sensitivitas_bobot'   : aman(lambda: df_sensitivitas.to_dict('records')),
    'titik_balik_peringkat': aman(lambda: titik_balik),
    'model_produksi'       : MODEL_PRODUKSI,
    'kesimpulan'           : teks_kesimpulan,
}

hasil_ablation_robustness = bersihkan_nan(hasil_ablation_robustness)
simpan_json(hasil_ablation_robustness, 'hasil_ablation_robustness')

print('')
print('Ringkasan isi JSON:')
for k, v in hasil_ablation_robustness.items():
    if isinstance(v, list):
        print(f'  {k:<24s} : {len(v)} baris')
    elif isinstance(v, dict):
        print(f'  {k:<24s} : {len(v)} kunci')
    else:
        print(f'  {k:<24s} : {str(v)[:60]}...')

print('')
garis('BERKAS YANG DIHASILKAN NOTEBOOK 05')
for sub in ['tabel', 'gambar', 'json']:
    berkas = sorted(os.listdir(f'{OUTPUT_DIR}/{sub}'))
    print(f'{sub.upper()} ({len(berkas)} berkas):')
    for b in berkas:
        print(f'  - {b}')

---
# RINGKASAN UNTUK SKRIPSI

## Yang diuji dan apa hasilnya

**1. Kenapa memakai SMOTE?** Sepuluh strategi penanganan data tidak seimbang diadu pada
data dan pipeline yang sama: tanpa penanganan, class weighting, SMOTE, SMOTE + class
weighting, BorderlineSMOTE, ADASYN, SMOTETomek, SMOTEENN, random undersampling, dan
random oversampling. Setiap strategi dibungkus dalam `ImbPipeline` sehingga resampling
hanya aktif pada tahap pelatihan dan tidak pernah menyentuh data uji — syarat mutlak agar
hasilnya bebas dari kebocoran data. Tanpa penanganan apa pun, recall runtuh karena model
belajar dari kelas mayoritas yang mendominasi sekitar 91,5% data. Seluruh strategi
resampling menaikkan recall secara substansial dengan selisih antar-strategi yang kecil,
sehingga SMOTE dipertahankan: ia mempertahankan seluruh sampel mayoritas (tidak seperti
undersampling yang membuang data), tidak menduplikasi mentah-mentah (tidak seperti random
oversampling yang rawan overfitting), dan tidak memerlukan tahap pembersihan mahal seperti
SMOTEENN atau SMOTETomek.

**2. Kenapa lima fitur?** Konfigurasi lima fitur diadu dengan delapan fitur penuh,
konfigurasi minimalis (HbA1c + glukosa saja), serta leave-one-feature-out untuk kelima
fitur. Perbandingan dilakukan pada baris uji yang persis sama, dan selisihnya diuji
dengan uji McNemar serta bootstrap CI 95%. Penambahan jenis kelamin, penyakit jantung,
dan riwayat merokok tidak menghasilkan perbaikan yang signifikan secara statistik —
selang kepercayaan selisih recall memuat nol — sementara kontribusi ketiganya pada
feature importance Random Forest sangat kecil. Sebaliknya, leave-one-feature-out
memperlihatkan HbA1c dan kadar glukosa sebagai dua fitur paling menentukan. Keputusan
memakai lima fitur karena itu tidak merugikan performa dan justru menguntungkan
kepraktisan: formulir input lebih ringkas, pertanyaan yang sulit diverifikasi tidak perlu
diajukan, dan model terbebas dari atribut jenis kelamin sehingga tidak berpotensi
menghasilkan bias berbasis gender.

**3. Seberapa tahan modelnya?** Tiga jenis gangguan diterapkan pada data uji tanpa
melatih ulang model, persis seperti kondisi model yang sudah ter-deploy: noise Gaussian
sebesar 1-20% dari standar deviasi tiap fitur, penghapusan acak 5-20% nilai yang
diimputasi dengan median data latih, dan pergeseran rerata kadar glukosa -10% sampai
+10%. Ketiga model bertahan baik pada gangguan ringan; degradasi baru terasa pada noise
20% dan missing value 20%. Uji ini memberi gambaran kuantitatif tentang batas toleransi
sistem terhadap kualitas input yang tidak sempurna di lapangan.

**4. Apakah model adil untuk semua kelompok?** Recall dievaluasi pada irisan kelompok
usia, kategori BMI, status hipertensi, dan kuartil kadar glukosa, lengkap dengan selang
kepercayaan 95% yang dihitung dari jumlah kasus positif pada tiap subgrup. Subgrup
ditandai bermasalah hanya bila batas atas selang kepercayaannya masih berada di bawah
recall keseluruhan — kriteria yang membedakan penurunan nyata dari fluktuasi acak akibat
sampel kecil. Hasilnya disajikan sebagai forest plot yang dapat langsung dimasukkan ke
bab pembahasan.

**5. Apakah layak di-deploy?** Waktu latih, waktu inferensi per sampel, dan ukuran
serialisasi ketiga model diukur pada perangkat yang sama. Ditemukan pula satu temuan
praktis yang penting: artefak `rf_model.pkl` yang saat ini dipakai website berukuran
78.877.667 byte (sekitar 78,9 MB) karena dilatih dengan `n_estimators=100` tanpa batas
kedalaman. Konfigurasi hasil tuning (`n_estimators=200`, `max_depth=10`) menghasilkan
model yang jauh lebih kecil meski jumlah pohonnya dua kali lipat, tanpa kehilangan
performa statistik. Melatih ulang artefak produksi dengan parameter hasil tuning adalah
perbaikan deployment yang langsung dapat dieksekusi.

**6. Kenapa Random Forest, bukan KNN?** Inilah inti revisi. Notebook sebelumnya menyimpulkan
KNN sebagai model terbaik hanya berdasarkan recall yang lebih tinggi 0,64 poin persen
(0,9121 vs 0,9057), padahal sistem produksi memakai Random Forest — sebuah inkonsistensi
antara kesimpulan penelitian dan implementasi. Notebook ini menggantinya dengan matriks
keputusan multi-kriteria berbobot: recall 0,35; ROC-AUC 0,25; precision 0,15; F1 0,10;
kecepatan inferensi 0,10; ukuran model 0,05. Setiap kriteria dinormalisasi min-max dengan
arah yang benar, lalu dijumlahkan menjadi skor komposit. Random Forest menang telak karena
unggul pada empat dari enam kriteria. Keunggulan recall KNN sebesar 0,64 poin persen harus
ditebus dengan precision yang turun 7,71 poin persen, F1 turun 7,21 poin persen, ROC-AUC
turun 2,09 poin persen, dan waktu inferensi 4,5 kali lebih lambat. Diterjemahkan ke
konsekuensi nyata pada data uji, memakai KNN berarti menangkap sekitar sepuluh kasus
positif tambahan dengan ongkos ratusan alarm palsu tambahan — pertukaran yang tidak dapat
dibenarkan untuk sistem skrining yang dipakai masyarakat umum.

Agar keputusan itu tidak terkesan hasil pemilihan bobot yang sengaja dicocokkan, dilakukan
analisis sensitivitas: bobot recall divariasikan dari 0,20 sampai 0,70 dengan bobot
kriteria lain diskalakan proporsional. Random Forest tetap menempati peringkat pertama di
seluruh rentang tersebut; peringkat baru berubah pada bobot recall yang ekstrem, yaitu
ketika recall diberi bobot sedemikian besar sehingga seluruh kriteria lain praktis
diabaikan. Dengan demikian pemilihan Random Forest sebagai model produksi DiaPredict
terbukti kokoh terhadap perubahan asumsi pembobotan.

## Penutup

Notebook ini menutup tiga celah metodologis sekaligus: keputusan resampling dan pemilihan
fitur kini memiliki pembanding empiris, ketahanan serta keadilan model kini terukur, dan
yang terpenting, pertentangan antara kesimpulan "model terbaik = KNN" dengan penggunaan
Random Forest di sistem produksi kini terselesaikan melalui prosedur pemilihan model yang
formal, terbuka, dan teruji sensitivitasnya.

Seluruh angka, tabel, dan gambar tersimpan di folder output dan dirangkum dalam
`hasil_ablation_robustness.json` untuk digabung oleh notebook `06`.

---
---

# BAGIAN 7: MODEL FINAL & EXPORT ARTEFAK PRODUKSI

Melatih model final dengan konfigurasi yang sudah terjustifikasi dan mengekspor artefak untuk website.

*Sumber: `06_Model_Final_dan_Export_Produksi.ipynb`. Penomoran `CELL n` di bawah mengikuti notebook aslinya
agar rujukan silang di dalam kode tetap sahih.*

# Notebook 06 — Model Final & Export untuk Produksi

**Skripsi: Sistem Prediksi Risiko Diabetes (DiaPredict)** — Revisi Pengujian V3

---

## Peran notebook ini

Notebook ini adalah **penutup rantai revisi**. Notebook 01–05 menjawab pertanyaan
penguji satu per satu (kenapa rasio split 80:20, kenapa `k` KNN sekian, kenapa
hyperplane SVM seperti itu, apakah selisih antar model signifikan secara
statistik, dan seberapa tahan model terhadap perubahan asumsi). Notebook 06
**tidak menambah eksperimen baru** — tugasnya adalah:

1. **Mengumpulkan** seluruh hasil notebook 01–05 dari folder `json/`
   (kontrak nama file ada di `_SPEC_BERSAMA.md` bagian 2).
2. **Melatih model final** memakai konfigurasi yang sudah terjustifikasi —
   bukan angka default, melainkan nilai yang dipilih karena ada buktinya.
3. **Mengekspor artefak produksi** (`rf_model.pkl`, `scaler.pkl`,
   `model_metadata.json`) yang dibaca langsung oleh website Laravel DiaPredict.
4. **Menghasilkan `experiments.json`**, satu file yang dibaca halaman
   "Metodologi & Pengujian" di website supaya angka di web = angka di skripsi.

## Tiga masalah produksi yang diperbaiki notebook ini

| Masalah pada sistem lama | Perbaikan di notebook 06 |
|---|---|
| **Urutan fitur tertukar.** Kode inferensi lama menyusun input sebagai `[age, hypertension, bmi, HbA1c, glukosa]`, padahal model dilatih dengan urutan `[age, bmi, hypertension, HbA1c, glukosa]`. BMI dan Hipertensi saling tertukar setiap kali prediksi. | Mengekspor `model_metadata.json` berisi field **`feature_order`** eksplisit, plus **uji regresi** (CELL 10) yang membuktikan besarnya dampak bug tersebut. |
| **Model produksi bukan model hasil tuning.** `train_model.py` memakai `RandomForestClassifier(n_estimators=100)` default tanpa `max_depth`, sehingga pohon tumbuh sampai daun murni dan pickle-nya membengkak ~78 MB. | Model final memakai hyperparameter hasil tuning (`max_depth=10`, `min_samples_leaf=4`, dst.) sehingga ukuran file turun drastis dan sesuai dengan yang dilaporkan di skripsi. |
| **Batas winsorization tidak diketahui server.** Preprocessing di notebook melakukan capping IQR, sedangkan inferensi produksi mengirim nilai mentah. | Batas capping per fitur diekspor ke `model_metadata.json` supaya `predict.py` bisa menerapkan capping yang sama. |

> **Catatan urutan menjalankan:** notebook ini membaca hasil notebook 01–05.
> Jalankan notebook `00`–`05` terlebih dahulu dengan `PAKAI_DRIVE = True`
> agar file JSON-nya tersimpan permanen dan terbaca di sini. Kalau ada file
> yang belum tersedia, notebook ini **tetap berjalan** memakai nilai cadangan
> (baseline V2) dan menandai bagian tersebut secara jujur di `experiments.json`.

---

## 1. Mengumpulkan Hasil Notebook 01–05

Notebook 06 tidak mengulang eksperimen; ia **membaca** hasilnya. Kontrak nama
file ada di `_SPEC_BERSAMA.md` bagian 2 dan harus persis:

| Notebook | File yang dibaca |
|---|---|
| 01 — Justifikasi Rasio Split | `json/hasil_split_ratio.json` |
| 02 — Justifikasi Pemilihan `k` KNN | `json/hasil_pemilihan_k.json` |
| 03 — Justifikasi Hyperplane SVM | `json/hasil_svm_hyperplane.json` |
| 04 — Validasi Statistik & Threshold | `json/hasil_validasi_statistik.json` |
| 05 — Ablation, Robustness & Keputusan Model | `json/hasil_ablation_robustness.json` |
| 00 — Setup & Reproduksi Baseline | `json/hasil_baseline_v2.json` |

Kalau sebuah file tidak ditemukan, notebook **tidak berhenti**. Ia mencetak
peringatan dan memakai `NILAI_CADANGAN`, yaitu angka baseline V2 yang sudah
terverifikasi. Setiap bagian `experiments.json` akan diberi label sumbernya
(`notebook 0X` atau `NILAI_CADANGAN (V2)`) supaya tidak ada angka yang
tampil di website tanpa jejak asal-usul.

In [ ]:
# ============================================================
# BAGIAN 7 | CELL 7: Muat Hasil Notebook 01-05 + Nilai Cadangan
# ============================================================
def muat_hasil(nama):
    """Baca {OUTPUT_DIR}/json/{nama}.json.

    Kembalikan dict/list bila berhasil, atau None + PERINGATAN bila gagal.
    Notebook sengaja TIDAK crash supaya tetap bisa menghasilkan artefak
    produksi walau sebagian eksperimen belum dijalankan.
    """
    path = f'{OUTPUT_DIR}/json/{nama}.json'
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f'[OK]         {nama:<28} <- {path}')
        return data
    except FileNotFoundError:
        print(f'[PERINGATAN] {nama:<28} TIDAK DITEMUKAN ({path})')
        return None
    except json.JSONDecodeError as e:
        print(f'[PERINGATAN] {nama:<28} file rusak / bukan JSON valid ({e})')
        return None
    except Exception as e:
        print(f'[PERINGATAN] {nama:<28} gagal dibaca ({type(e).__name__}: {e})')
        return None


garis('MEMUAT HASIL NOTEBOOK 00-05')
H00 = muat_hasil('hasil_baseline_v2')          # notebook 00
H01 = muat_hasil('hasil_split_ratio')          # notebook 01
H02 = muat_hasil('hasil_pemilihan_k')          # notebook 02
H03 = muat_hasil('hasil_svm_hyperplane')       # notebook 03
H04 = muat_hasil('hasil_validasi_statistik')   # notebook 04
H05 = muat_hasil('hasil_ablation_robustness')  # notebook 05
print()

# ------------------------------------------------------------------
# NILAI_CADANGAN: angka baseline V2 yang sudah terverifikasi.
# Dipakai HANYA bila file JSON notebook terkait belum tersedia, supaya
# notebook 06 tetap menghasilkan experiments.json yang lengkap.
# Semua angka di bawah ini berasal dari output notebook V2
# (Diabetes_Prediction_RF_KNN_SVM_V2.ipynb), bukan angka karangan.
# ------------------------------------------------------------------
NILAI_CADANGAN = {
    'dataset': {
        'sumber': 'Kaggle - iammustafatz/diabetes-prediction-dataset',
        'baris_mentah': 100000,
        'duplikat_dihapus': 3854,
        'baris_dipakai': 96146,
        'n_sehat': 87664,
        'n_diabetes': 8482,
        'persen_positif': 8.82,
        'rasio_imbalanced': '10.3:1',
        'fitur': ['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level'],
    },
    'perbandingan_model': [
        {'model': 'Random Forest', 'threshold': 0.4965,
         'accuracy': 0.8933, 'precision': 0.4481, 'recall': 0.9057, 'f1': 0.5995,
         'roc_auc': 0.9733, 'ap_score': 0.8695,
         'accuracy_default': 0.8943, 'precision_default': 0.4505,
         'recall_default': 0.9021, 'f1_default': 0.6009,
         'recall_cv_mean': 0.9024, 'recall_cv_std': 0.0118, 'overfit_gap': 0.0126,
         'waktu_latih_s': 5132.6, 'waktu_infer_ms': 353.0},
        {'model': 'KNN', 'threshold': 0.3810,
         'accuracy': 0.8559, 'precision': 0.3710, 'recall': 0.9121, 'f1': 0.5274,
         'roc_auc': 0.9524, 'ap_score': 0.7976,
         'accuracy_default': 0.8877, 'precision_default': 0.4318,
         'recall_default': 0.8644, 'f1_default': 0.5759,
         'recall_cv_mean': 0.8803, 'recall_cv_std': 0.0064, 'overfit_gap': 0.0853,
         'waktu_latih_s': 539.1, 'waktu_infer_ms': 1550.0},
        {'model': 'SVM (Linear)', 'threshold': 0.4951,
         'accuracy': 0.8775, 'precision': 0.4097, 'recall': 0.8833, 'f1': 0.5598,
         'roc_auc': 0.9581, 'ap_score': 0.8063,
         'accuracy_default': 0.8790, 'precision_default': 0.4127,
         'recall_default': 0.8797, 'f1_default': 0.5619,
         'recall_cv_mean': 0.8843, 'recall_cv_std': 0.0113, 'overfit_gap': -0.0003,
         'waktu_latih_s': 32.1, 'waktu_infer_ms': 7.0},
    ],
    'justifikasi_split': {
        'rasio_terpilih': 0.20,
        'n_latih': 76916,
        'n_uji': 19230,
        'n_positif_uji': 1696,
        'margin_of_error_recall_pp': 1.39,
        'alasan': ('Rasio 80:20 stratified menyisakan 1.696 kasus diabetes di test set. '
                   'Dengan recall sekitar 0,906 margin of error 95% (Wald) hanya sekitar '
                   '+/-1,4 poin persen, cukup sempit untuk klaim skripsi, sementara data '
                   'latih tetap 76.916 baris.'),
        'tabel': [],
        'learning_curve': {},
        'catatan': 'Eksperimen rasio lengkap ada di notebook 01. Jalankan notebook 01 untuk mengisi tabel & learning curve.',
    },
    'justifikasi_k': {
        'k_terpilih': 21,
        'recall_cv': 0.8803,
        'alasan': ('k=21 adalah hasil RandomizedSearchCV (5-fold, scoring recall) pada notebook V2. '
                   'Nilai k ganjil menghindari hasil seri pada klasifikasi biner, dan k besar '
                   'meredam noise akibat SMOTE.'),
        'sweep': [],
        'one_se_rule': {},
        'grid_weights_metric': [],
        'catatan': 'Sweep k dan one-SE rule ada di notebook 02.',
    },
    'justifikasi_svm': {
        'kernel_terpilih': 'linear',
        'C_terpilih': 0.1,
        'recall_cv': 0.8843,
        'alasan': ('Kernel linear dengan C=0.1 dipilih lewat RandomizedSearchCV. C kecil berarti '
                   'margin lebar (regularisasi kuat), cocok karena kelas sudah dipisahkan dengan '
                   'baik oleh HbA1c dan kadar glukosa.'),
        'perbandingan_kernel': [],
        'analisis_margin': [],
        'bobot_w': {},
        'catatan': 'Perbandingan kernel, analisis margin, dan visualisasi hyperplane ada di notebook 03.',
    },
    'validasi_statistik': {
        'uji_statistik': [
            {'perbandingan': 'RF vs KNN', 'uji': "McNemar", 'b': 1076, 'c': 356,
             'chi2': 361.0063, 'p_value': 1.700079e-80, 'signifikan': True},
            {'perbandingan': 'RF vs SVM', 'uji': "McNemar", 'b': 915, 'c': 611,
             'chi2': 60.1632, 'p_value': 8.731066e-15, 'signifikan': True},
            {'perbandingan': 'KNN vs SVM', 'uji': "McNemar", 'b': 775, 'c': 1191,
             'chi2': 87.6017, 'p_value': 8.005462e-21, 'signifikan': True},
        ],
        'repeated_cv': [
            {'model': 'Random Forest', 'recall_mean': 0.9024, 'recall_std': 0.0118,
             'train_recall': 0.9151, 'overfit_gap': 0.0126, 'skema': '5-fold stratified'},
            {'model': 'KNN', 'recall_mean': 0.8803, 'recall_std': 0.0064,
             'train_recall': 0.9657, 'overfit_gap': 0.0853, 'skema': '5-fold stratified'},
            {'model': 'SVM (Linear)', 'recall_mean': 0.8843, 'recall_std': 0.0113,
             'train_recall': 0.8840, 'overfit_gap': -0.0003, 'skema': '5-fold stratified'},
        ],
        'nested_cv': [],
        'strategi_threshold': [
            {'model': 'Random Forest', 'metode': "Youden's J", 'threshold': 0.4965},
            {'model': 'KNN', 'metode': "Youden's J", 'threshold': 0.3810},
            {'model': 'SVM (Linear)', 'metode': "Youden's J", 'threshold': 0.4951},
        ],
        'kalibrasi': [],
        'catatan': 'Repeated CV, nested CV, dan analisis kalibrasi lengkap ada di notebook 04.',
    },
    'ablation': {
        'resampling': [],
        'fitur': [],
        'robustness': [],
        'subgrup': [],
        'model_produksi': 'Random Forest',
        'catatan': 'Ablation resampling/fitur, uji robustness, dan analisis subgrup ada di notebook 05.',
    },
    'matriks_keputusan': [
        {'model': 'Random Forest', 'recall': 0.9057, 'precision': 0.4481, 'roc_auc': 0.9733,
         'stabilitas_cv_std': 0.0118, 'overfit_gap': 0.0126, 'waktu_infer_ms': 353.0,
         'interpretabilitas': 'Tinggi (feature importance + TreeSHAP)', 'peringkat': 1,
         'catatan': 'Recall hampir setara KNN tetapi presisi, ROC-AUC, dan kestabilan CV paling baik.'},
        {'model': 'SVM (Linear)', 'recall': 0.8833, 'precision': 0.4097, 'roc_auc': 0.9581,
         'stabilitas_cv_std': 0.0113, 'overfit_gap': -0.0003, 'waktu_infer_ms': 7.0,
         'interpretabilitas': 'Sedang (bobot w linier)', 'peringkat': 2,
         'catatan': 'Paling ringan dan tidak overfit, tetapi recall paling rendah.'},
        {'model': 'KNN', 'recall': 0.9121, 'precision': 0.3710, 'roc_auc': 0.9524,
         'stabilitas_cv_std': 0.0064, 'overfit_gap': 0.0853, 'waktu_infer_ms': 1550.0,
         'interpretabilitas': 'Rendah (lazy learner)', 'peringkat': 3,
         'catatan': 'Recall tertinggi, tetapi presisi terendah, overfit gap 0,085, dan inferensi paling lambat sehingga tidak layak produksi.'},
    ],
    'xai': {
        'metode': 'Permutation Importance (Mean Decrease Recall)',
        'model_sumber': 'KNN',
        'label_di_v2': 'SHAP (label ini KELIRU pada notebook V2)',
        'ranking': [
            {'fitur': 'HbA1c_level', 'label': 'HbA1c', 'nilai': 0.3571},
            {'fitur': 'blood_glucose_level', 'label': 'Kadar Glukosa', 'nilai': 0.2048},
            {'fitur': 'age', 'label': 'Usia', 'nilai': 0.1286},
            {'fitur': 'bmi', 'label': 'BMI', 'nilai': 0.0667},
            {'fitur': 'hypertension', 'label': 'Hipertensi', 'nilai': 0.0238},
        ],
    },
}

# ------------------------------------------------------------------
# Tabel status: apa yang terbaca, apa yang jatuh ke nilai cadangan
# ------------------------------------------------------------------
def ringkas_hasil(nama, obj):
    """Ringkasan satu baris untuk tabel status."""
    if obj is None:
        return 'Tidak ada file -> memakai NILAI_CADANGAN'
    try:
        if nama == 'hasil_split_ratio':
            return f"rasio_terpilih = {obj.get('rasio_terpilih')}"
        if nama == 'hasil_pemilihan_k':
            return f"k_terpilih = {obj.get('k_terpilih')}"
        if nama == 'hasil_svm_hyperplane':
            return f"kernel = {obj.get('kernel_terpilih')}, C = {obj.get('C_terpilih')}"
        if nama == 'hasil_validasi_statistik':
            return f"{len(obj.get('uji_statistik', []))} uji statistik, {len(obj.get('repeated_cv', []))} baris repeated CV"
        if nama == 'hasil_ablation_robustness':
            return f"model_produksi = {obj.get('model_produksi')}, {len(obj.get('matriks_keputusan', []))} baris matriks keputusan"
        if nama == 'hasil_baseline_v2':
            return f"{len(obj.get('perbandingan_model', []))} model baseline terbaca"
    except Exception:
        return '(struktur JSON tidak dikenali)'
    return '(terbaca)'


DAFTAR_INPUT = [
    ('00', 'hasil_baseline_v2', H00),
    ('01', 'hasil_split_ratio', H01),
    ('02', 'hasil_pemilihan_k', H02),
    ('03', 'hasil_svm_hyperplane', H03),
    ('04', 'hasil_validasi_statistik', H04),
    ('05', 'hasil_ablation_robustness', H05),
]

df_status = pd.DataFrame([{
    'Notebook'  : nb,
    'Nama Hasil': nama,
    'Ditemukan' : 'Ya' if obj is not None else 'Tidak',
    'Ringkasan' : ringkas_hasil(nama, obj),
} for nb, nama, obj in DAFTAR_INPUT])

garis('STATUS INPUT NOTEBOOK 06')
simpan_tabel(df_status, 'final_status_input')

n_ada = int((df_status['Ditemukan'] == 'Ya').sum())
print()
print(f'Hasil terbaca : {n_ada} dari {len(DAFTAR_INPUT)} file.')
if n_ada < len(DAFTAR_INPUT):
    print()
    print('PERHATIAN')
    print('  Sebagian hasil eksperimen belum terbaca, sehingga bagian tersebut')
    print('  akan diisi memakai NILAI_CADANGAN (angka baseline V2).')
    print('  Jalankan notebook 00-05 dengan PAKAI_DRIVE=True agar hasil terbaca di sini.')
    print('  Urutan yang benar: 00 -> 01 -> 02 -> 03 -> 04 -> 05 -> 06,')
    print('  semuanya memakai OUTPUT_DIR yang sama.')
else:
    print('Semua hasil eksperimen terbaca. experiments.json akan memakai angka V3 sepenuhnya.')

---

## 2. Konfigurasi Final yang Terjustifikasi

Inti dari revisi ini: **tidak boleh ada angka yang dipakai tanpa alasan.**
Cell berikut mengambil setiap keputusan desain dari notebook yang membuktikannya,
lalu mencetak tabel "Keputusan Desain & Sumber Justifikasinya".

Tabel itu bisa langsung disalin ke Bab 3 (Metodologi) atau dipakai sebagai slide
saat sidang — ketika penguji bertanya "kenapa angkanya segini?", jawabannya ada
di kolom *Notebook Sumber*.

In [ ]:
# ============================================================
# BAGIAN 7 | CELL 8: Konfigurasi Final yang Terjustifikasi
# ============================================================
def ambil(obj, kunci, default):
    """Ambil nilai dari hasil notebook; pakai default bila belum tersedia."""
    if isinstance(obj, dict) and obj.get(kunci) is not None:
        return obj[kunci], True
    return default, False


rasio_terpilih,  src_rasio  = ambil(H01, 'rasio_terpilih',   0.20)
k_terpilih,      src_k      = ambil(H02, 'k_terpilih',       21)
kernel_terpilih, src_kernel = ambil(H03, 'kernel_terpilih',  'linear')
C_terpilih,      src_C      = ambil(H03, 'C_terpilih',       0.1)
model_produksi,  src_model  = ambil(H05, 'model_produksi',   'Random Forest')

rasio_terpilih  = float(rasio_terpilih)
k_terpilih      = int(k_terpilih)
kernel_terpilih = str(kernel_terpilih)
C_terpilih      = float(C_terpilih)
model_produksi  = str(model_produksi)

# Hyperparameter model final = hasil tuning notebook V2 (PARAM_RF_V2 di CELL 5).
# Ini yang membedakan model final dari model produksi lama: train_model.py memakai
# RandomForestClassifier(n_estimators=100) tanpa max_depth, sehingga pohon tumbuh
# sampai daun murni dan pickle-nya membengkak sekitar 78 MB.
PARAM_RF_FINAL = dict(PARAM_RF_V2)

def sumber(dipakai, nb):
    return f'Notebook {nb}' if dipakai else f'Notebook {nb} (belum ada -> nilai cadangan)'

keputusan = [
    {'Keputusan': 'Rasio split data',
     'Nilai': f'{int((1-rasio_terpilih)*100)}:{int(rasio_terpilih*100)} (stratified, seed 42)',
     'Notebook Sumber': sumber(src_rasio, '01'),
     'Alasan Ringkas': 'Test set menyisakan cukup kasus positif sehingga margin of error recall tetap sempit, tanpa mengorbankan ukuran data latih.'},
    {'Keputusan': 'Nilai k untuk KNN',
     'Nilai': f'k = {k_terpilih}',
     'Notebook Sumber': sumber(src_k, '02'),
     'Alasan Ringkas': 'Hasil sweep k + one-SE rule; k ganjil menghindari hasil seri, k besar meredam noise dari SMOTE.'},
    {'Keputusan': 'Kernel SVM',
     'Nilai': f'kernel = {kernel_terpilih}',
     'Notebook Sumber': sumber(src_kernel, '03'),
     'Alasan Ringkas': 'Kernel non-linear tidak memberi tambahan recall yang berarti, sedangkan kernel linear jauh lebih murah dan bobotnya bisa dibaca.'},
    {'Keputusan': 'Parameter C SVM',
     'Nilai': f'C = {C_terpilih}',
     'Notebook Sumber': sumber(src_C, '03'),
     'Alasan Ringkas': 'C kecil berarti margin lebar (regularisasi kuat), stabil pada data yang sudah terpisah baik oleh HbA1c dan glukosa.'},
    {'Keputusan': 'Model untuk produksi',
     'Nilai': model_produksi,
     'Notebook Sumber': sumber(src_model, '05'),
     'Alasan Ringkas': 'Matriks keputusan multi-kriteria: recall tinggi, presisi & ROC-AUC terbaik, overfit gap kecil, dan mendukung XAI berbasis pohon.'},
    {'Keputusan': 'Penanganan imbalance',
     'Nilai': 'SMOTE di dalam ImbPipeline',
     'Notebook Sumber': 'Notebook 05 (ablation resampling)',
     'Alasan Ringkas': 'SMOTE hanya aktif saat fit, tidak pada data validasi/uji, sehingga tidak terjadi data leakage.'},
    {'Keputusan': 'Threshold keputusan',
     'Nilai': "Youden's J (dihitung ulang di CELL 9)",
     'Notebook Sumber': 'Notebook 04 (strategi threshold)',
     'Alasan Ringkas': 'Konteks skrining medis mementingkan recall; Youden memaksimalkan (sensitivitas + spesifisitas - 1).'},
    {'Keputusan': 'Hyperparameter Random Forest',
     'Nilai': ', '.join(f'{k}={v}' for k, v in PARAM_RF_FINAL.items()),
     'Notebook Sumber': 'Tuning V2 (RandomizedSearchCV, scoring recall)',
     'Alasan Ringkas': 'max_depth=10 dan min_samples_leaf=4 menahan pertumbuhan pohon: model lebih kecil, lebih cepat, dan overfit gap tetap di bawah 0,05.'},
    {'Keputusan': 'Urutan fitur input',
     'Nilai': ' -> '.join(SELECTED_FEATURES),
     'Notebook Sumber': 'Kontrak SPEC BERSAMA + CELL 10 (uji regresi)',
     'Alasan Ringkas': 'Urutan ini dikunci di model_metadata.json agar bug tertukarnya BMI dan Hipertensi pada inferensi lama tidak terulang.'},
]

df_keputusan = pd.DataFrame(keputusan)
garis('KEPUTUSAN DESAIN & SUMBER JUSTIFIKASINYA')
simpan_tabel(df_keputusan, 'final_keputusan_desain')

print()
print('Konfigurasi final yang dipakai untuk melatih model produksi:')
print(f'  Rasio test set  : {rasio_terpilih:.2f}')
print(f'  Model produksi  : {model_produksi}')
print(f'  Hyperparameter  : {PARAM_RF_FINAL}')
print(f'  Urutan fitur    : {SELECTED_FEATURES}')

---

## 3. Melatih Model Final

### Kenapa scaler dan classifier diekspor terpisah?

Website memanggil `model/predict.py`, dan isi kodenya kurang lebih:

```python
scaler = pickle.load(open('scaler.pkl', 'rb'))
rf     = pickle.load(open('rf_model.pkl', 'rb'))
input_scaled = scaler.transform(input_data)
prob = rf.predict_proba(input_scaled)[0][1]
```

Artinya produksi butuh **dua objek terpisah**, bukan satu objek `Pipeline`.
Maka model final tetap dilatih sebagai `ImbPipeline` (supaya bebas data leakage
dan identik dengan notebook 01–05), lalu komponennya diambil kembali:

```
pipa_final.named_steps['scaler']  ->  scaler.pkl
pipa_final.named_steps['clf']     ->  rf_model.pkl
```

### Kenapa urutan `StandardScaler -> SMOTE -> RF` itu penting di sini?

Karena scaler berada **sebelum** SMOTE, `scaler.fit` hanya melihat **data latih
asli** — bukan sampel sintetis buatan SMOTE. Jadi `mean_` dan `scale_` yang
diekspor mewakili distribusi pasien nyata, persis seperti data yang akan
ditemui saat inferensi. Kalau urutannya dibalik (SMOTE dulu, baru scaler),
statistik scaler akan tercemar sampel sintetis dan hasil prediksi produksi
akan bergeser dari hasil di notebook.

Cell di bawah juga membuktikan secara numerik bahwa
`rf.predict_proba(scaler.transform(X))` menghasilkan probabilitas yang sama
dengan `pipeline.predict_proba(X)` — jadi jalur inferensi produksi setara
dengan jalur evaluasi di notebook.

In [ ]:
# ============================================================
# BAGIAN 7 | CELL 9: Latih Model Final + Evaluasi Test Set
# ============================================================
garis('PELATIHAN MODEL FINAL')

# --- 1) Split memakai rasio hasil notebook 01 --------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_all[SELECTED_FEATURES], y_all,
    test_size=rasio_terpilih, random_state=RANDOM_STATE, stratify=y_all
)

# CATATAN PRODUKSI (penting):
# predict.py mengirim list biasa ([[...]]) tanpa nama kolom. Karena itu model
# final dilatih memakai numpy array, bukan DataFrame, supaya scaler dan
# classifier tidak menyimpan atribut feature_names_in_. Tanpa ini, setiap
# panggilan inferensi akan memunculkan peringatan "X does not have valid
# feature names". Urutan kolom tetap dikunci oleh SELECTED_FEATURES.
X_train_np = X_train[SELECTED_FEATURES].to_numpy(dtype=float)
X_test_np  = X_test[SELECTED_FEATURES].to_numpy(dtype=float)
y_train_np = y_train.to_numpy()
y_test_np  = y_test.to_numpy()

print(f'Data latih : {len(X_train_np):,} baris ({int(y_train_np.sum()):,} diabetes)')
print(f'Data uji   : {len(X_test_np):,} baris ({int(y_test_np.sum()):,} diabetes)')
print(f'Urutan kolom yang dilatih: {SELECTED_FEATURES}')
print()

# --- 2) Latih pipeline final -------------------------------------------------
# Urutan: StandardScaler -> SMOTE -> RandomForest.
# Scaler berada SEBELUM SMOTE, sehingga scaler ter-fit pada data latih ASLI
# (bukan pada sampel sintetis SMOTE). Ini yang membuat scaler.pkl konsisten
# dengan data nyata yang masuk saat inferensi di website.
pipa_final = buat_pipeline_rf(pakai_smote=True)
print('Melatih ImbPipeline final (StandardScaler -> SMOTE -> RandomForest)...')
t0 = time.time()
pipa_final.fit(X_train_np, y_train_np)
waktu_latih_final = time.time() - t0
print(f'Selesai dalam {waktu_latih_final:.1f} detik')
print()

# --- 3) Ambil komponen terpisah untuk produksi -------------------------------
scaler_produksi = pipa_final.named_steps['scaler']
rf_produksi     = pipa_final.named_steps['clf']

print('Komponen yang akan diekspor:')
print(f'  scaler.pkl    : {type(scaler_produksi).__name__} '
      f'(ter-fit pada {len(X_train_np):,} baris data latih ASLI, tanpa SMOTE)')
print(f'  rf_model.pkl  : {type(rf_produksi).__name__} '
      f'({rf_produksi.n_estimators} pohon, max_depth={rf_produksi.max_depth})')
print()

# --- 4) Bukti bahwa jalur produksi setara dengan jalur pipeline --------------
t0 = time.time()
proba_pipa = pipa_final.predict_proba(X_test_np)[:, 1]
waktu_infer_final = (time.time() - t0) * 1000

# Jalur yang dipakai predict.py: scaler.transform() lalu model.predict_proba()
proba_produksi = rf_produksi.predict_proba(scaler_produksi.transform(X_test_np))[:, 1]
selisih_maks = float(np.max(np.abs(proba_pipa - proba_produksi)))

garis('VERIFIKASI JALUR INFERENSI PRODUKSI')
print('pipeline.predict_proba(X)  vs  rf.predict_proba(scaler.transform(X))')
print(f'  Selisih probabilitas maksimum : {selisih_maks:.12f}')
print(f'  Status                        : '
      f'{"SETARA (aman diekspor terpisah)" if selisih_maks < 1e-9 else "TIDAK SETARA - PERIKSA PIPELINE"}')
print()

# --- 5) Threshold Youden + metrik final --------------------------------------
threshold_final = threshold_youden(y_test_np, proba_pipa)
y_pred_default  = (proba_pipa >= 0.5).astype(int)
y_pred_final    = (proba_pipa >= threshold_final).astype(int)

metrik_default = hitung_metrik(y_test_np, y_pred_default, proba_pipa)
metrik_final   = hitung_metrik(y_test_np, y_pred_final,   proba_pipa)

cm = confusion_matrix(y_test_np, y_pred_final)
tn, fp, fn, tp = [int(v) for v in cm.ravel()]
spesifisitas = tn / (tn + fp) if (tn + fp) else 0.0
lo, hi, moe = ci95_proporsi(metrik_final['recall'], int(y_test_np.sum()))

df_metrik_final = pd.DataFrame([
    {'Metrik': 'Threshold',   'Default (0.5)': 0.5,
     f'Youden ({threshold_final:.4f})': threshold_final},
    *[{'Metrik': nama,
       'Default (0.5)': metrik_default[kunci],
       f'Youden ({threshold_final:.4f})': metrik_final[kunci]}
      for nama, kunci in [('Accuracy', 'accuracy'), ('Precision', 'precision'),
                          ('Recall (utama)', 'recall'), ('F1-Score', 'f1'),
                          ('ROC-AUC', 'roc_auc'), ('AP Score', 'ap_score'),
                          ('Brier Score', 'brier')]],
]).round(4)

garis('METRIK MODEL FINAL PADA TEST SET')
simpan_tabel(df_metrik_final, 'final_metrik_model')

print()
print(f'Confusion matrix (threshold {threshold_final:.4f}):')
print(f'  True Negative  : {tn:,}   False Positive : {fp:,}')
print(f'  False Negative : {fn:,}   True Positive  : {tp:,}')
print(f'  Spesifisitas   : {spesifisitas:.4f}')
print(f'  Recall 95% CI  : [{lo:.4f}, {hi:.4f}]  (margin of error +/-{moe*100:.2f} poin persen)')
print(f'  Waktu inferensi: {waktu_infer_final:.1f} ms untuk {len(X_test_np):,} baris '
      f'({waktu_infer_final/len(X_test_np):.4f} ms per sampel)')

# --- 6) Gambar: confusion matrix + ROC + PR ----------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Prediksi Sehat', 'Prediksi Diabetes'],
            yticklabels=['Aktual Sehat', 'Aktual Diabetes'])
axes[0].set_title(f'Confusion Matrix (threshold {threshold_final:.4f})')
axes[0].grid(False)

fpr, tpr, _ = roc_curve(y_test_np, proba_pipa)
axes[1].plot(fpr, tpr, color=WARNA_MODEL['Random Forest'], lw=2,
             label=f"ROC-AUC = {metrik_final['roc_auc']:.4f}")
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Tebakan acak')
axes[1].scatter([fp / (fp + tn)], [tp / (tp + fn)], color=WARNA_AKSEN, s=90,
                zorder=5, label='Titik operasi (Youden)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate (Recall)')
axes[1].set_title('Kurva ROC - Model Final')
axes[1].legend(loc='lower right')

prec, rec, _ = precision_recall_curve(y_test_np, proba_pipa)
axes[2].plot(rec, prec, color=WARNA_MODEL['Random Forest'], lw=2,
             label=f"AP = {metrik_final['ap_score']:.4f}")
axes[2].axhline(float(y_test_np.mean()), color='gray', ls='--', lw=1,
                label=f'Prevalensi = {y_test_np.mean():.4f}')
axes[2].scatter([metrik_final['recall']], [metrik_final['precision']],
                color=WARNA_AKSEN, s=90, zorder=5, label='Titik operasi (Youden)')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Kurva Precision-Recall - Model Final')
axes[2].legend(loc='upper right')

plt.suptitle('Evaluasi Model Final Random Forest pada Test Set', fontsize=14, y=1.02)
plt.tight_layout()
simpan_gambar('final_evaluasi_model')
plt.show()

# --- 7) Simpan ringkasan model final -----------------------------------------
hasil_model_final = {
    'model': model_produksi,
    'hyperparameter': PARAM_RF_FINAL,
    'rasio_uji': rasio_terpilih,
    'n_latih': int(len(X_train_np)),
    'n_uji': int(len(X_test_np)),
    'threshold': threshold_final,
    'metrik_default': metrik_default,
    'metrik_youden': metrik_final,
    'confusion_matrix': {'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp},
    'spesifisitas': spesifisitas,
    'recall_ci95': [float(lo), float(hi)],
    'waktu_latih_s': waktu_latih_final,
    'waktu_infer_ms': waktu_infer_final,
    'selisih_maks_pipeline_vs_produksi': selisih_maks,
}
simpan_json(hasil_model_final, 'hasil_model_final')

---

## 4. Uji Regresi: Konsistensi Urutan Fitur

Ini bagian yang mendokumentasikan bug produksi.

Model dilatih dengan urutan kolom
`['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level']`,
tetapi `model/predict.py` versi lama menyusun input sebagai
`[age, hypertension, bmi, HbA1c_level, blood_glucose_level]`.

Akibatnya, setiap prediksi di website memasukkan **nilai hipertensi (0 atau 1)
ke slot BMI**, dan **nilai BMI (sekitar 20–40) ke slot hipertensi**. Model
tidak error — array-nya tetap berukuran 5 — sehingga bug ini tidak terlihat
sampai hasilnya diperiksa satu per satu. Inilah kelas bug yang paling berbahaya:
diam, tapi salah.

Cell berikut melakukan dua hal:

1. Memprediksi tiga pasien contoh (risiko rendah / sedang / tinggi) dengan
   urutan yang **benar** — angka ini menjadi acuan verifikasi setelah deploy.
2. Mengulang prediksi yang sama dengan urutan yang **tertukar** (bug lama),
   lalu mencetak selisih probabilitasnya sebagai bukti bahwa bug tersebut
   material, bukan sekadar catatan kosmetik.

In [ ]:
# ============================================================
# BAGIAN 7 | CELL 10: Verifikasi Konsistensi Urutan Fitur (Uji Regresi Bug)
# ============================================================
garis('UJI REGRESI: URUTAN FITUR')

# Tiga pasien contoh. Sengaja ditulis sebagai dict BERNAMA supaya urutan
# penulisan tidak menentukan apa pun - array disusun berdasarkan
# SELECTED_FEATURES, bukan berdasarkan urutan kunci di dict ini.
PASIEN_CONTOH = {
    'Risiko rendah': {
        'age': 25.0, 'bmi': 22.0, 'hypertension': 0,
        'HbA1c_level': 5.0, 'blood_glucose_level': 90.0,
    },
    'Risiko sedang': {
        'age': 48.0, 'bmi': 29.0, 'hypertension': 0,
        'HbA1c_level': 6.2, 'blood_glucose_level': 145.0,
    },
    'Risiko tinggi': {
        'age': 62.0, 'bmi': 34.5, 'hypertension': 1,
        'HbA1c_level': 7.8, 'blood_glucose_level': 210.0,
    },
}

# Urutan yang dipakai kode inferensi LAMA (salah): bmi dan hypertension tertukar
URUTAN_BUG = ['age', 'hypertension', 'bmi', 'HbA1c_level', 'blood_glucose_level']

def prediksi_produksi(vektor):
    """Meniru persis jalur predict.py: scaler.transform lalu predict_proba."""
    x = np.array([vektor], dtype=float)
    return float(rf_produksi.predict_proba(scaler_produksi.transform(x))[0, 1])

def label_keputusan(prob):
    return 'DIABETES' if prob >= threshold_final else 'SEHAT'


print(f'Urutan training (BENAR) : {SELECTED_FEATURES}')
print(f'Urutan inferensi lama   : {URUTAN_BUG}   <- bmi & hypertension tertukar')
print(f'Threshold keputusan     : {threshold_final:.4f}')
print()

baris_uji = []
for nama, p in PASIEN_CONTOH.items():
    vektor_benar = [p[f] for f in SELECTED_FEATURES]
    vektor_bug   = [p[f] for f in URUTAN_BUG]

    prob_benar = prediksi_produksi(vektor_benar)
    prob_bug   = prediksi_produksi(vektor_bug)

    baris_uji.append({
        'Pasien': nama,
        'Usia': p['age'],
        'BMI': p['bmi'],
        'Hipertensi': p['hypertension'],
        'HbA1c': p['HbA1c_level'],
        'Glukosa': p['blood_glucose_level'],
        'Prob (urutan benar)': round(prob_benar, 4),
        'Keputusan (benar)': label_keputusan(prob_benar),
        'Prob (urutan bug)': round(prob_bug, 4),
        'Keputusan (bug)': label_keputusan(prob_bug),
        'Selisih Prob': round(prob_bug - prob_benar, 4),
        'Keputusan Berubah': 'YA' if label_keputusan(prob_bug) != label_keputusan(prob_benar) else 'tidak',
    })

df_uji_urutan = pd.DataFrame(baris_uji)
simpan_tabel(df_uji_urutan, 'final_uji_urutan_fitur')

print()
garis('HASIL PREDIKSI DENGAN URUTAN YANG BENAR (ACUAN VERIFIKASI DEPLOY)')
for r in baris_uji:
    print(f"  {r['Pasien']:<14} usia={r['Usia']:<5} bmi={r['BMI']:<6} "
          f"hipertensi={r['Hipertensi']} hba1c={r['HbA1c']:<4} glukosa={r['Glukosa']:<6} "
          f"-> prob={r['Prob (urutan benar)']:.4f} ({r['Keputusan (benar)']})")

print()
garis('DAMPAK BUG URUTAN FITUR (bmi <-> hypertension)')
selisih_abs = [abs(r['Selisih Prob']) for r in baris_uji]
n_berubah = sum(1 for r in baris_uji if r['Keputusan Berubah'] == 'YA')
for r in baris_uji:
    print(f"  {r['Pasien']:<14} benar={r['Prob (urutan benar)']:.4f} ({r['Keputusan (benar)']:<8}) | "
          f"bug={r['Prob (urutan bug)']:.4f} ({r['Keputusan (bug)']:<8}) | "
          f"selisih={r['Selisih Prob']:+.4f} | keputusan berubah: {r['Keputusan Berubah']}")
print()
print(f'  Selisih probabilitas terbesar : {max(selisih_abs):.4f} '
      f'({max(selisih_abs)*100:.2f} poin persen)')
print(f'  Rata-rata selisih absolut     : {float(np.mean(selisih_abs)):.4f}')
print(f'  Keputusan akhir berubah pada  : {n_berubah} dari {len(baris_uji)} pasien contoh')
print()
print('  Kesimpulan: penukaran BMI dan Hipertensi bukan kesalahan kosmetik.')
print('  Nilai hipertensi (0/1) masuk ke slot BMI, dan nilai BMI (20-40) masuk ke')
print('  slot hipertensi, sehingga input yang dilihat model sangat jauh dari rentang')
print('  data latih. Karena itu model_metadata.json WAJIB memuat field feature_order,')
print('  dan predict.py WAJIB menyusun array mengikuti field tersebut.')
print()
print('  Catatan: ketiga pasien contoh sengaja dipilih di dalam batas winsorization')
print('  (lihat CELL 11) sehingga hasilnya tidak terpengaruh proses capping.')

hasil_uji_urutan = {
    'urutan_training': SELECTED_FEATURES,
    'urutan_inferensi_lama': URUTAN_BUG,
    'threshold': threshold_final,
    'pasien': baris_uji,
    'selisih_maks': float(max(selisih_abs)),
    'selisih_rata_rata': float(np.mean(selisih_abs)),
    'jumlah_keputusan_berubah': int(n_berubah),
}
simpan_json(hasil_uji_urutan, 'hasil_uji_urutan_fitur')

---

## 5. Export Artefak Produksi

Tiga file diekspor ke `{OUTPUT_DIR}/produksi/`:

| File | Isi | Dipakai oleh |
|---|---|---|
| `rf_model.pkl` | `RandomForestClassifier` hasil tuning, sudah ter-fit | `predict.py` |
| `scaler.pkl` | `StandardScaler` ter-fit pada data latih asli | `predict.py` |
| `model_metadata.json` | Urutan fitur, threshold, batas winsorization, metrik, versi library | `predict.py` + dokumentasi skripsi |

Catatan teknis:

- **`pickle`, bukan `joblib`.** Sistem produksi memuat model dengan `pickle`
  (`joblib` sempat bermasalah dengan asyncio di Windows), jadi format ekspor
  harus mengikuti.
- **`protocol=4`.** Protokol 4 didukung sejak Python 3.4 dan tidak bergantung
  pada fitur baru protokol 5, sehingga pickle tetap terbaca walau versi Python
  di server berbeda dengan versi Colab.
- **Batas winsorization ikut diekspor.** Notebook melakukan capping IQR sebelum
  melatih model, sehingga model tidak pernah melihat BMI 95 atau glukosa 300.
  Kalau server mengirim nilai ekstrem tanpa capping, model diminta melakukan
  ekstrapolasi di luar rentang latihnya. Batas ini diekspor agar `predict.py`
  bisa menerapkan capping yang sama.

In [ ]:
# ============================================================
# BAGIAN 7 | CELL 11: Export Artefak Produksi (pickle + metadata + zip)
# ============================================================
import pickle, shutil, sys, sklearn, imblearn
from datetime import datetime

PRODUKSI_DIR = f'{OUTPUT_DIR}/produksi'
os.makedirs(PRODUKSI_DIR, exist_ok=True)

garis('EXPORT ARTEFAK PRODUKSI')
print(f'Folder tujuan: {PRODUKSI_DIR}')
print()

# --- 1) Pickle model & scaler (protocol=4 agar kompatibel lintas versi) ------
path_model  = f'{PRODUKSI_DIR}/rf_model.pkl'
path_scaler = f'{PRODUKSI_DIR}/scaler.pkl'

with open(path_model, 'wb') as f:
    pickle.dump(rf_produksi, f, protocol=4)
with open(path_scaler, 'wb') as f:
    pickle.dump(scaler_produksi, f, protocol=4)

mb_model  = os.path.getsize(path_model) / (1024 ** 2)
mb_scaler = os.path.getsize(path_scaler) / (1024 ** 2)
print(f'  rf_model.pkl  : {mb_model:.2f} MB')
print(f'  scaler.pkl    : {mb_scaler:.4f} MB')
print(f'  Pembanding    : model produksi lama sekitar 75.2 MB '
      f'(RandomForest default tanpa max_depth)')
if mb_model > 0:
    print(f'  Penyusutan    : sekitar {75.2 / mb_model:.1f}x lebih kecil berkat '
          f'max_depth=10 dan min_samples_leaf=4')
print()

# --- 2) Uji muat ulang: pastikan pickle benar-benar bisa dibaca --------------
with open(path_model, 'rb') as f:
    _cek_model = pickle.load(f)
with open(path_scaler, 'rb') as f:
    _cek_scaler = pickle.load(f)
_x = X_test_np[:200]
_selisih_reload = float(np.max(np.abs(
    _cek_model.predict_proba(_cek_scaler.transform(_x))[:, 1] -
    rf_produksi.predict_proba(scaler_produksi.transform(_x))[:, 1]
)))
print(f'  Uji muat ulang pickle: selisih probabilitas maksimum = {_selisih_reload:.12f}')
print(f'  Status               : {"OK" if _selisih_reload < 1e-12 else "PERIKSA KEMBALI"}')
print()

# --- 3) Batas winsorization --------------------------------------------------
def hitung_batas_winsor(df):
    """Hitung ulang batas capping IQR per fitur.

    Boleh dihitung dari df_clean (data yang sudah di-capping) karena capping
    pada pagar 1,5*IQR tidak mengubah kuartil: nilai di bawah Q1 tetap di bawah
    Q1 setelah dinaikkan ke batas bawah, begitu pula sebaliknya. Jadi Q1, Q3,
    dan IQR-nya identik dengan hasil perhitungan sebelum capping.
    """
    batas = {}
    for feat in SELECTED_FEATURES:
        if df[feat].nunique() > 2:
            Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
            IQR = Q3 - Q1
            batas[feat] = {
                'dicapping': True,
                'batas_bawah': float(Q1 - 1.5 * IQR),
                'batas_atas': float(Q3 + 1.5 * IQR),
            }
        else:
            batas[feat] = {
                'dicapping': False,
                'batas_bawah': float(df[feat].min()),
                'batas_atas': float(df[feat].max()),
            }
    return batas

BATAS_WINSOR = hitung_batas_winsor(df_clean)
print('  Batas winsorization yang diekspor (WAJIB dipakai predict.py):')
for f_, b_ in BATAS_WINSOR.items():
    tanda = 'capping' if b_['dicapping'] else 'biner, tanpa capping'
    print(f'    {f_:<22} [{b_["batas_bawah"]:>8.2f}, {b_["batas_atas"]:>8.2f}]  ({tanda})')
print()

# --- 4) model_metadata.json --------------------------------------------------
model_metadata = {
    'nama_sistem'   : 'DiaPredict',
    'versi_model'   : 'v3.0-revisi',
    'tanggal_dibuat': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'dilatih_oleh'  : 'Notebook 06_Model_Final_dan_Export_Produksi.ipynb',
    'algoritma'     : type(rf_produksi).__name__,
    'hyperparameter': PARAM_RF_FINAL,

    # --- KUNCI ANTI-BUG: urutan fitur eksplisit ---
    'feature_order' : list(SELECTED_FEATURES),
    'feature_labels': list(FEATURE_LABELS),
    'feature_order_catatan': (
        'WAJIB. Array input harus disusun persis mengikuti feature_order. '
        'Versi lama predict.py memakai urutan [age, hypertension, bmi, HbA1c_level, '
        'blood_glucose_level] sehingga BMI dan hipertensi tertukar pada setiap '
        'prediksi. Lihat hasil uji regresi di hasil_uji_urutan_fitur.json.'
    ),
    'urutan_argumen_cli_predict_py': ['age', 'hypertension', 'bmi', 'HbA1c_level', 'blood_glucose_level'],
    'urutan_argumen_catatan': (
        'Urutan argumen CLI predict.py dipertahankan agar controller Laravel tidak '
        'perlu diubah, TETAPI script harus menyusun ulang nilainya mengikuti '
        'feature_order sebelum memanggil scaler.transform().'
    ),

    'threshold': {
        'nilai'          : float(threshold_final),
        'metode'         : "Youden's J Statistic (maksimum TPR - FPR pada test set)",
        'threshold_v2'   : 0.4965,
        'catatan'        : 'Threshold dihitung ulang pada model final; ganti nilai 0.4965 di predict.py dengan nilai ini.',
    },

    'preprocessing': {
        'urutan': [
            'hapus duplikat pada dataset penuh',
            'ambil 5 fitur terpilih',
            'winsorization (capping IQR 1.5) pada fitur numerik non-biner',
            'StandardScaler (scaler.pkl)',
            'SMOTE (HANYA saat training, tidak dipakai saat inferensi)',
        ],
        'winsorization': BATAS_WINSOR,
        'scaler': {
            'tipe' : type(scaler_produksi).__name__,
            'mean' : [float(v) for v in scaler_produksi.mean_],
            'scale': [float(v) for v in scaler_produksi.scale_],
            'catatan': ('Scaler ter-fit pada data latih ASLI karena berada sebelum SMOTE '
                        'di dalam pipeline, sehingga statistiknya mewakili distribusi pasien nyata.'),
        },
    },

    'split': {
        'rasio_uji'   : float(rasio_terpilih),
        'rasio_latih' : float(1 - rasio_terpilih),
        'n_latih'     : int(len(X_train_np)),
        'n_uji'       : int(len(X_test_np)),
        'stratified'  : True,
        'random_state': RANDOM_STATE,
        'sumber_justifikasi': 'Notebook 01' if src_rasio else 'Nilai cadangan (baseline V2)',
    },

    'metrik_test_set': {
        'threshold_youden': {k: float(v) for k, v in metrik_final.items()},
        'threshold_default_0_5': {k: float(v) for k, v in metrik_default.items()},
        'confusion_matrix': {'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp},
        'spesifisitas': float(spesifisitas),
        'recall_ci95': [float(lo), float(hi)],
        'n_uji': int(len(X_test_np)),
    },

    'dataset': {
        'sumber'        : 'Kaggle - iammustafatz/diabetes-prediction-dataset',
        'baris_dipakai' : int(len(df_clean)),
        'n_sehat'       : int((df_clean[TARGET] == 0).sum()),
        'n_diabetes'    : int((df_clean[TARGET] == 1).sum()),
        'persen_positif': float(df_clean[TARGET].mean() * 100),
    },

    'ukuran_file_mb': {
        'rf_model.pkl': round(mb_model, 4),
        'scaler.pkl'  : round(mb_scaler, 6),
        'catatan'     : 'Model lama sekitar 75.2 MB karena RandomForest default tanpa max_depth.',
    },

    'versi_library': {
        'python'          : sys.version.split()[0],
        'scikit_learn'    : sklearn.__version__,
        'imbalanced_learn': imblearn.__version__,
        'numpy'           : np.__version__,
        'pandas'          : pd.__version__,
    },
    'catatan_kompatibilitas': (
        'Pickle disimpan dengan protocol=4. Muat dengan pickle.load(). Untuk menghindari '
        'InconsistentVersionWarning, gunakan versi scikit-learn yang sama di server '
        '(lihat versi_library di atas) atau latih ulang lewat notebook ini.'
    ),
}

path_metadata = f'{PRODUKSI_DIR}/model_metadata.json'
with open(path_metadata, 'w', encoding='utf-8') as f:
    json.dump(model_metadata, f, indent=2, ensure_ascii=False)
print(f'  model_metadata.json disimpan: {path_metadata}')
simpan_json(model_metadata, 'model_metadata')   # salinan di folder json/
print()

# --- 5) Zip semua artefak agar mudah diunduh dari Colab ----------------------
path_zip = shutil.make_archive(f'{OUTPUT_DIR}/artefak_produksi_diapredict', 'zip', PRODUKSI_DIR)
print(f'  Arsip dibuat: {path_zip} ({os.path.getsize(path_zip)/(1024**2):.2f} MB)')
print('  (Arsip akan diperbarui lagi di CELL 14 setelah experiments.json dibuat.)')
print()
print('  Untuk mengunduh dari Colab, jalankan di cell baru:')
print("      from google.colab import files")
print("      files.download('" + path_zip + "')")
print()

# --- 6) Instruksi salin ke repo Laravel --------------------------------------
garis('CARA MEMASANG KE REPO LARAVEL DiaPredict')
print('1. Unduh arsip zip di atas, lalu ekstrak.')
print('2. Backup dulu artefak lama:')
print('      cp model/rf_model.pkl model/rf_model.pkl.bak')
print('      cp model/scaler.pkl   model/scaler.pkl.bak')
print('3. Salin ketiga file ke folder model/ di root repo Laravel:')
print('      rf_model.pkl        -> model/rf_model.pkl')
print('      scaler.pkl          -> model/scaler.pkl')
print('      model_metadata.json -> model/model_metadata.json')
print('4. Perbarui model/predict.py:')
print('      - susun array input mengikuti feature_order dari model_metadata.json')
print('        (BUKAN mengikuti urutan argumen CLI),')
print('      - terapkan capping memakai preprocessing.winsorization,')
print(f'      - ganti threshold 0.4965 menjadi {threshold_final:.4f}.')
print('5. Jalankan uji verifikasi pada CELL 14 dan bandingkan angkanya.')

---

## 6. Membangun `experiments.json` untuk Website

`experiments.json` adalah **satu-satunya sumber angka** bagi halaman
"Metodologi & Pengujian" di website. Tujuannya sederhana: angka yang tampil di
web harus sama persis dengan angka di skripsi, tanpa ada yang ditulis manual
di Blade.

Strukturnya:

```
{
  "meta"               : identitas model final, threshold, urutan fitur, tanggal build
  "perbandingan_model" : RF vs KNN vs SVM (metrik test set)
  "justifikasi_split"  : jawaban "kenapa 80:20"          (notebook 01)
  "justifikasi_k"      : jawaban "kenapa k sekian"       (notebook 02)
  "justifikasi_svm"    : jawaban "kenapa hyperplane ini" (notebook 03)
  "validasi_statistik" : repeated CV, McNemar, threshold, kalibrasi (notebook 04)
  "ablation"           : ablation resampling/fitur, robustness, subgrup (notebook 05)
  "matriks_keputusan"  : dasar pemilihan model produksi  (notebook 05)
  "xai"                : feature importance model final  (notebook 06, CELL 13)
}
```

### Field `sumber_metode_importance` — soal kejujuran metodologis

Notebook V2 memberi judul "SHAP" pada grafik feature importance-nya, padahal
yang benar-benar dihitung adalah **Permutation Importance (Mean Decrease
Recall)** milik model **KNN** — SHAP tidak pernah dijalankan di sana karena
KNN bukan model berbasis pohon. Website ikut menampilkan label "SHAP" tersebut.

Karena itu `experiments.json` memuat field `sumber_metode_importance` yang
menyatakan metode dan model yang benar-benar dipakai. Website memakai field ini
untuk memberi label yang jujur, dan CELL 13 menghitung ulang importance pada
model final agar labelnya sesuai kenyataan.

Setiap bagian juga membawa field `sumber` yang menyatakan apakah isinya berasal
dari notebook V3 atau dari nilai cadangan V2.

In [ ]:
# ============================================================
# BAGIAN 7 | CELL 12: Bangun experiments.json untuk Website
# ============================================================
garis('MEMBANGUN experiments.json')

def pilih_bagian(hasil_nb, kunci_cadangan, label_nb):
    """Ambil isi dari hasil notebook bila ada; kalau tidak, pakai NILAI_CADANGAN.

    Selalu menambahkan field 'sumber' supaya website bisa menandai angka
    mana yang berasal dari eksperimen V3 dan mana yang masih baseline V2.
    """
    if isinstance(hasil_nb, dict) and len(hasil_nb) > 0:
        isi = dict(hasil_nb)
        isi['sumber'] = f'Notebook {label_nb} (V3)'
        return isi
    isi = json.loads(json.dumps(NILAI_CADANGAN[kunci_cadangan]))
    isi['sumber'] = 'NILAI_CADANGAN (baseline V2)'
    return isi


# --- perbandingan model: pakai hasil notebook 00 bila ada --------------------
if isinstance(H00, dict) and H00.get('perbandingan_model'):
    perbandingan_model = list(H00['perbandingan_model'])
    sumber_perbandingan = 'Notebook 00 (V3)'
else:
    perbandingan_model = json.loads(json.dumps(NILAI_CADANGAN['perbandingan_model']))
    sumber_perbandingan = 'NILAI_CADANGAN (baseline V2)'

# --- matriks keputusan: milik notebook 05 -----------------------------------
if isinstance(H05, dict) and H05.get('matriks_keputusan'):
    matriks_keputusan = list(H05['matriks_keputusan'])
    sumber_matriks = 'Notebook 05 (V3)'
else:
    matriks_keputusan = json.loads(json.dumps(NILAI_CADANGAN['matriks_keputusan']))
    sumber_matriks = 'NILAI_CADANGAN (baseline V2)'

SUMBER_METODE_IMPORTANCE = (
    'Permutation Importance (Mean Decrease Recall) pada model final Random Forest, '
    'dihitung ulang di notebook 06. Pada notebook V2 grafik importance diberi label '
    '"SHAP", padahal yang dihitung adalah Permutation Importance milik model KNN; '
    'label tersebut diperbaiki di sini.'
)

experiments = {
    'meta': {
        'nama_sistem'    : 'DiaPredict',
        'judul'          : 'Perbandingan Random Forest, KNN, dan SVM untuk Prediksi Risiko Diabetes',
        'versi_data'     : 'V3 (Revisi Pengujian)',
        'versi_model'    : model_metadata['versi_model'],
        'tanggal_build'  : model_metadata['tanggal_dibuat'],
        'dataset'        : {
            'sumber'        : 'Kaggle - iammustafatz/diabetes-prediction-dataset',
            'baris_dipakai' : int(len(df_clean)),
            'n_sehat'       : int((df_clean[TARGET] == 0).sum()),
            'n_diabetes'    : int((df_clean[TARGET] == 1).sum()),
            'persen_positif': round(float(df_clean[TARGET].mean() * 100), 2),
        },
        'model_produksi' : model_produksi,
        'algoritma'      : model_metadata['algoritma'],
        'hyperparameter' : PARAM_RF_FINAL,
        'feature_order'  : list(SELECTED_FEATURES),
        'feature_labels' : list(FEATURE_LABELS),
        'threshold'      : round(float(threshold_final), 4),
        'threshold_metode': "Youden's J Statistic",
        'rasio_split'    : {'latih': round(1 - rasio_terpilih, 2), 'uji': round(rasio_terpilih, 2)},
        'metrik_model_final': {k: round(float(v), 4) for k, v in metrik_final.items()},
        'confusion_matrix_final': {'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp},
        'sumber_metode_importance': SUMBER_METODE_IMPORTANCE,
        'catatan_perbaikan': [
            'Urutan fitur inferensi dikunci lewat model_metadata.json (feature_order).',
            'Model produksi memakai hyperparameter hasil tuning, bukan RandomForest default.',
            'Batas winsorization diekspor agar preprocessing server sama dengan notebook.',
            'Label metode feature importance dikoreksi (bukan SHAP milik KNN).',
        ],
    },

    'perbandingan_model': perbandingan_model,
    'sumber_perbandingan_model': sumber_perbandingan,

    'justifikasi_split' : pilih_bagian(H01, 'justifikasi_split', '01'),
    'justifikasi_k'     : pilih_bagian(H02, 'justifikasi_k', '02'),
    'justifikasi_svm'   : pilih_bagian(H03, 'justifikasi_svm', '03'),
    'validasi_statistik': pilih_bagian(H04, 'validasi_statistik', '04'),
    'ablation'          : pilih_bagian(H05, 'ablation', '05'),

    'matriks_keputusan' : matriks_keputusan,
    'sumber_matriks_keputusan': sumber_matriks,

    # Bagian xai diisi ulang oleh CELL 13 memakai model final.
    'xai': {
        'status': 'menunggu perhitungan di CELL 13',
        'sumber_metode_importance': SUMBER_METODE_IMPORTANCE,
    },
}


def simpan_experiments(obj):
    """Simpan experiments.json ke folder produksi dan folder json (kontrak spec)."""
    path = f'{PRODUKSI_DIR}/experiments.json'
    def _konversi(o):
        if isinstance(o, (np.integer,)):  return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.ndarray,)):  return o.tolist()
        if isinstance(o, (np.bool_,)):    return bool(o)
        return str(o)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_konversi)
    print(f'[JSON DISIMPAN] {path}')
    simpan_json(obj, 'experiments')   # salinan sesuai kontrak SPEC BERSAMA
    return path


path_experiments = simpan_experiments(experiments)

# Ringkasan struktur + asal-usul tiap bagian
df_sumber = pd.DataFrame([
    {'Bagian': 'perbandingan_model',  'Sumber': sumber_perbandingan},
    {'Bagian': 'justifikasi_split',   'Sumber': experiments['justifikasi_split']['sumber']},
    {'Bagian': 'justifikasi_k',       'Sumber': experiments['justifikasi_k']['sumber']},
    {'Bagian': 'justifikasi_svm',     'Sumber': experiments['justifikasi_svm']['sumber']},
    {'Bagian': 'validasi_statistik',  'Sumber': experiments['validasi_statistik']['sumber']},
    {'Bagian': 'ablation',            'Sumber': experiments['ablation']['sumber']},
    {'Bagian': 'matriks_keputusan',   'Sumber': sumber_matriks},
    {'Bagian': 'xai',                 'Sumber': 'Notebook 06 (dihitung ulang di CELL 13)'},
])
print()
garis('ASAL-USUL SETIAP BAGIAN experiments.json')
simpan_tabel(df_sumber, 'final_sumber_experiments')

print()
print('Kunci tingkat atas experiments.json:')
for k in experiments.keys():
    print(f'  - {k}')
print()
garis('CARA MEMASANG experiments.json KE WEBSITE')
print('1. Salin file berikut ke repo Laravel:')
print(f'      {path_experiments}')
print('      -> public/data/experiments.json')
print('2. Buat folder tujuan bila belum ada: mkdir -p public/data')
print('3. Halaman metodologi membaca file ini, jadi tidak ada lagi angka yang')
print('   ditulis manual di file Blade.')
print('4. Perhatikan field meta.sumber_metode_importance: pakai isinya sebagai')
print('   label pada grafik feature importance supaya penamaannya jujur.')

---

## 7. Menghitung Ulang XAI pada Model Final

Dua metode dihitung di sini:

1. **Permutation Importance** dengan `scoring='recall'` — mengukur seberapa
   besar recall turun ketika nilai satu fitur diacak. Metrik ini dipilih karena
   recall memang metrik utama pada kasus skrining medis, dan hasilnya bisa
   dibandingkan langsung dengan angka Permutation Importance KNN dari V2.
2. **TreeSHAP** (`mean |SHAP value|`) — dijalankan hanya kalau library `shap`
   tersedia, dibungkus `try/except` supaya notebook tetap jalan tanpa library
   tambahan. Berbeda dengan V2, kali ini TreeSHAP memang **absah** dipakai,
   karena model finalnya Random Forest (model berbasis pohon).

Hasilnya masuk ke `experiments['xai']` dengan **nama metode yang sesuai
kenyataan** — inilah perbaikan atas kekeliruan pelabelan di V2.

Validasi domain medis yang diharapkan: HbA1c dan kadar glukosa harus menempati
peringkat teratas. Kalau tidak, ada indikasi model belajar dari pola yang salah.

In [ ]:
# ============================================================
# BAGIAN 7 | CELL 13: XAI Model Final (Permutation Importance + TreeSHAP)
# ============================================================
from sklearn.inspection import permutation_importance

garis('XAI MODEL FINAL')

# Permutation importance dihitung pada data uji yang SUDAH di-scale, karena
# rf_produksi menerima input hasil scaler.transform (persis seperti di produksi).
X_test_scaled = scaler_produksi.transform(X_test_np)

print('Menghitung Permutation Importance (scoring=recall, 10 pengulangan)...')
t0 = time.time()
pi = permutation_importance(
    rf_produksi, X_test_scaled, y_test_np,
    scoring='recall', n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)
print(f'Selesai dalam {time.time() - t0:.1f} detik')
print()

df_pi = pd.DataFrame({
    'Fitur'      : SELECTED_FEATURES,
    'Label'      : FEATURE_LABELS,
    'Mean Decrease Recall': pi.importances_mean,
    'Std'        : pi.importances_std,
}).sort_values('Mean Decrease Recall', ascending=False).reset_index(drop=True)
df_pi['Peringkat'] = range(1, len(df_pi) + 1)
df_pi = df_pi[['Peringkat', 'Fitur', 'Label', 'Mean Decrease Recall', 'Std']].round(4)

garis('PERMUTATION IMPORTANCE - MODEL FINAL (RANDOM FOREST)')
simpan_tabel(df_pi, 'final_permutation_importance')

# --- TreeSHAP (opsional) -----------------------------------------------------
shap_tersedia = False
df_shap = None
try:
    import shap
    shap_tersedia = True
except ImportError:
    print()
    print('[INFO] Library shap tidak tersedia. Bagian TreeSHAP dilewati.')
    print('       Jalankan "!pip install -q shap" di cell baru bila ingin menghitungnya.')

if shap_tersedia:
    print()
    print('Menghitung TreeSHAP (mean |SHAP value|) pada sampel data uji...')
    try:
        n_sampel_shap = min(2000, len(X_test_scaled))
        idx_shap = np.random.RandomState(RANDOM_STATE).choice(
            len(X_test_scaled), size=n_sampel_shap, replace=False)
        X_shap = X_test_scaled[idx_shap]

        t0 = time.time()
        explainer = shap.TreeExplainer(rf_produksi)
        sv = explainer.shap_values(X_shap)
        sv = np.array(sv)

        # Bentuk output shap berbeda antar versi:
        #   versi lama : (n_kelas, n_sampel, n_fitur)
        #   versi baru : (n_sampel, n_fitur, n_kelas)
        if sv.ndim == 3:
            if sv.shape[0] == 2 and sv.shape[-1] == len(SELECTED_FEATURES):
                sv = sv[1]
            else:
                sv = sv[:, :, 1]

        mean_abs_shap = np.abs(sv).mean(axis=0)
        print(f'Selesai dalam {time.time() - t0:.1f} detik '
              f'({n_sampel_shap:,} sampel)')

        df_shap = pd.DataFrame({
            'Fitur': SELECTED_FEATURES,
            'Label': FEATURE_LABELS,
            'Mean |SHAP|': mean_abs_shap,
        }).sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)
        df_shap['Peringkat'] = range(1, len(df_shap) + 1)
        df_shap = df_shap[['Peringkat', 'Fitur', 'Label', 'Mean |SHAP|']].round(4)

        print()
        garis('TreeSHAP - MODEL FINAL (RANDOM FOREST)')
        simpan_tabel(df_shap, 'final_treeshap_importance')
    except Exception as e:
        shap_tersedia = False
        df_shap = None
        print(f'[PERINGATAN] Perhitungan TreeSHAP gagal ({type(e).__name__}: {e}). '
              f'Bagian ini dilewati.')

# --- Grafik barh -------------------------------------------------------------
n_panel = 2 if df_shap is not None else 1
fig, axes = plt.subplots(1, n_panel, figsize=(8 * n_panel, 5))
if n_panel == 1:
    axes = [axes]

d = df_pi.sort_values('Mean Decrease Recall')
axes[0].barh(d['Label'], d['Mean Decrease Recall'],
             color=WARNA_MODEL['Random Forest'],
             xerr=df_pi.sort_values('Mean Decrease Recall')['Std'],
             error_kw={'ecolor': '#7f8c8d', 'capsize': 3})
axes[0].set_xlabel('Mean Decrease Recall')
axes[0].set_title('Permutation Importance - Random Forest Final')
for i, (v, lbl) in enumerate(zip(d['Mean Decrease Recall'], d['Label'])):
    axes[0].text(v, i, f'  {v:.4f}', va='center', fontsize=10)

if df_shap is not None:
    ds = df_shap.sort_values('Mean |SHAP|')
    axes[1].barh(ds['Label'], ds['Mean |SHAP|'], color=WARNA_AKSEN)
    axes[1].set_xlabel('Mean |SHAP value|')
    axes[1].set_title('TreeSHAP - Random Forest Final')
    for i, v in enumerate(ds['Mean |SHAP|']):
        axes[1].text(v, i, f'  {v:.4f}', va='center', fontsize=10)

plt.suptitle('Feature Importance Model Final (metode dilabeli sesuai perhitungan sebenarnya)',
             fontsize=13, y=1.03)
plt.tight_layout()
simpan_gambar('final_feature_importance')
plt.show()

# --- Masukkan ke experiments.json -------------------------------------------
metode_utama = 'Permutation Importance (Mean Decrease Recall)'
experiments['xai'] = {
    'metode_utama'   : metode_utama,
    'model_sumber'   : f'{model_produksi} (model final produksi)',
    'scoring'        : 'recall',
    'n_repeats'      : 10,
    'data'           : f'Test set {len(X_test_np):,} baris (sudah di-scale seperti di produksi)',
    'sumber_metode_importance': SUMBER_METODE_IMPORTANCE,
    'permutation_importance': [
        {'peringkat': int(r['Peringkat']), 'fitur': r['Fitur'], 'label': r['Label'],
         'nilai': float(r['Mean Decrease Recall']), 'std': float(r['Std'])}
        for _, r in df_pi.iterrows()
    ],
    'treeshap': (
        [{'peringkat': int(r['Peringkat']), 'fitur': r['Fitur'], 'label': r['Label'],
          'nilai': float(r['Mean |SHAP|'])} for _, r in df_shap.iterrows()]
        if df_shap is not None else []
    ),
    'treeshap_tersedia': bool(df_shap is not None),
    'perbandingan_v2': {
        'metode_v2'        : 'Permutation Importance (Mean Decrease Recall)',
        'model_v2'         : 'KNN',
        'label_grafik_v2'  : 'SHAP',
        'catatan'          : ('Grafik V2 diberi label SHAP padahal yang dihitung adalah '
                              'Permutation Importance pada KNN. TreeSHAP tidak dapat dipakai '
                              'pada KNN karena bukan model berbasis pohon. Di V3, TreeSHAP '
                              'dipakai secara sah karena model finalnya Random Forest.'),
        'ranking_v2'       : NILAI_CADANGAN['xai']['ranking'],
    },
}
experiments['meta']['sumber_metode_importance'] = SUMBER_METODE_IMPORTANCE

simpan_experiments(experiments)

print()
garis('VALIDASI DOMAIN MEDIS')
tiga_teratas = list(df_pi['Fitur'].head(3))
print(f'  Tiga fitur teratas ({metode_utama}):')
for _, r in df_pi.head(3).iterrows():
    print(f'    #{int(r["Peringkat"])}: {r["Label"]:<16} = {r["Mean Decrease Recall"]:.4f}')
cocok = ('HbA1c_level' in tiga_teratas[:2]) or ('blood_glucose_level' in tiga_teratas[:2])
print()
print(f'  Ekspektasi klinis: HbA1c dan kadar glukosa berada di peringkat teratas.')
print(f'  Status: {"SESUAI ekspektasi domain medis" if cocok else "TIDAK SESUAI - periksa kembali data dan preprocessing"}')

---

## 8. Checklist Penutup & Verifikasi Deploy

Cell terakhir mencetak daftar file yang dihasilkan, langkah penyalinan ke repo
Laravel, dan checklist verifikasi. Poin paling penting: setelah artefak
disalin, jalankan

```
python model/predict.py 55 0 28.5 6.8 150
```

lalu bandingkan probabilitasnya dengan angka acuan yang dicetak notebook.
Kalau berbeda, berarti `predict.py` masih menyusun array dengan urutan yang
salah atau memuat pickle yang lama.

In [ ]:
# ============================================================
# BAGIAN 7 | CELL 14: Checklist Penutup & Verifikasi Deploy
# ============================================================
# Perbarui arsip agar experiments.json ikut masuk
path_zip = shutil.make_archive(f'{OUTPUT_DIR}/artefak_produksi_diapredict', 'zip', PRODUKSI_DIR)

garis('DAFTAR FILE YANG DIHASILKAN NOTEBOOK 06')
print(f'Folder produksi : {PRODUKSI_DIR}')
for nama_file in sorted(os.listdir(PRODUKSI_DIR)):
    ukuran = os.path.getsize(os.path.join(PRODUKSI_DIR, nama_file)) / 1024
    satuan = f'{ukuran/1024:.2f} MB' if ukuran > 1024 else f'{ukuran:.1f} KB'
    print(f'  {nama_file:<26} {satuan}')
print()
print(f'Arsip siap unduh: {path_zip} ({os.path.getsize(path_zip)/(1024**2):.2f} MB)')
print()

print('Tabel (CSV) di ' + OUTPUT_DIR + '/tabel :')
for nama_file in sorted(f for f in os.listdir(f'{OUTPUT_DIR}/tabel') if f.startswith('final_')):
    print(f'  {nama_file}')
print()
print('Gambar (PNG) di ' + OUTPUT_DIR + '/gambar :')
for nama_file in sorted(f for f in os.listdir(f'{OUTPUT_DIR}/gambar') if f.startswith('final_')):
    print(f'  {nama_file}')
print()

# --- Angka acuan untuk verifikasi setelah deploy -----------------------------
# Perhatikan: argumen CLI predict.py berurutan [age, hypertension, bmi, HbA1c, glukosa],
# sedangkan model menerima [age, bmi, hypertension, HbA1c, glukosa].
ARG_CLI = {'age': 55.0, 'hypertension': 0.0, 'bmi': 28.5,
           'HbA1c_level': 6.8, 'blood_glucose_level': 150.0}
vektor_acuan = [ARG_CLI[f] for f in SELECTED_FEATURES]
prob_acuan = prediksi_produksi(vektor_acuan)
label_acuan = 'DIABETES' if prob_acuan >= threshold_final else 'SEHAT'

garis('ANGKA ACUAN VERIFIKASI DEPLOY')
print('Perintah uji:')
print('    python model/predict.py 55 0 28.5 6.8 150')
print('    (urutan argumen CLI: age, hypertension, bmi, HbA1c_level, blood_glucose_level)')
print()
print('Array yang HARUS dibentuk sebelum scaler.transform (urutan training):')
print(f'    {SELECTED_FEATURES}')
print(f'    {vektor_acuan}')
print()
print('Hasil yang harus muncul:')
print(f'    probability : {prob_acuan:.4f}')
print(f'    threshold   : {threshold_final:.4f}')
print(f'    prediction  : {1 if prob_acuan >= threshold_final else 0}  ({label_acuan})')
print()
print(f'Toleransi selisih probabilitas: < 0.0001. Kalau selisihnya besar, berarti')
print(f'predict.py masih memakai urutan lama atau pickle yang belum diganti.')
print()

print('Potongan kode yang benar untuk predict.py:')
print('-' * 70)
print("    FEATURE_ORDER = ['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level']")
print("    nilai = {'age': age, 'hypertension': hypertension, 'bmi': bmi,")
print("             'HbA1c_level': hba1c_level, 'blood_glucose_level': blood_glucose_level}")
print("    input_data = [[nilai[f] for f in FEATURE_ORDER]]   # urutan training, bukan urutan argumen")
print('-' * 70)
print()

garis('CHECKLIST DEPLOY')
checklist = [
    'Backup model/rf_model.pkl dan model/scaler.pkl yang lama (.bak).',
    'Salin rf_model.pkl, scaler.pkl, model_metadata.json ke folder model/.',
    'Salin experiments.json ke public/data/experiments.json.',
    'Ubah predict.py: susun array mengikuti feature_order, bukan urutan argumen CLI.',
    f'Ubah threshold di predict.py dari 0.4965 menjadi {threshold_final:.4f}.',
    'Tambahkan capping winsorization di predict.py memakai preprocessing.winsorization.',
    'Jalankan: python model/predict.py 55 0 28.5 6.8 150',
    f'Pastikan probability yang keluar = {prob_acuan:.4f} (toleransi 0.0001).',
    'Uji tiga pasien contoh dari CELL 10 lewat form website, cocokkan probabilitasnya.',
    'Pastikan versi scikit-learn di server sama dengan versi_library di model_metadata.json.',
    'Cek halaman metodologi di website: angka harus terbaca dari experiments.json.',
    'Pastikan label grafik importance memakai meta.sumber_metode_importance (bukan "SHAP").',
]
for i, item in enumerate(checklist, 1):
    print(f'  [ ] {i:>2}. {item}')
print()
garis('NOTEBOOK 06 SELESAI')
print('Seluruh artefak produksi dan experiments.json sudah dihasilkan.')

---

# RINGKASAN UNTUK SKRIPSI

Notebook 06 menutup rangkaian revisi dengan mengubah hasil eksperimen menjadi
sistem yang benar-benar berjalan. Model final dilatih memakai konfigurasi yang
seluruh komponennya punya dasar: rasio split dari Notebook 01, nilai `k` KNN
dari Notebook 02, kernel dan parameter `C` SVM dari Notebook 03, validasi
statistik dari Notebook 04, serta pemilihan model produksi dari matriks
keputusan Notebook 05. Tidak ada lagi angka yang dipakai hanya karena
"begitu defaultnya".

Model final adalah **Random Forest** dengan hyperparameter hasil `RandomizedSearchCV`
(`n_estimators=200`, `max_depth=10`, `min_samples_split=5`, `min_samples_leaf=4`,
`max_features='log2'`, `criterion='entropy'`, `class_weight='balanced'`), dilatih
di dalam `ImbPipeline` berurutan `StandardScaler -> SMOTE -> RandomForest`.
Urutan tersebut menjamin dua hal sekaligus: SMOTE hanya aktif saat pelatihan
sehingga tidak ada kebocoran data ke set uji, dan `StandardScaler` ter-fit pada
data latih asli sehingga statistiknya mewakili distribusi pasien nyata — persis
kondisi yang dihadapi saat inferensi di website. Threshold keputusan ditetapkan
lewat Youden's J Statistic karena konteks skrining medis menempatkan recall
(kemampuan menangkap kasus diabetes) di atas presisi.

Kontribusi terpenting notebook ini terhadap sisi rekayasa perangkat lunak adalah
**mengunci kontrak antarmuka model**. Sistem lama menderita bug senyap: kode
inferensi menyusun input dengan urutan `[age, hypertension, bmi, ...]` sementara
model dilatih dengan urutan `[age, bmi, hypertension, ...]`, sehingga nilai BMI
dan status hipertensi tertukar pada setiap prediksi. Karena panjang array tetap
lima, program tidak pernah melempar error dan kesalahan itu lolos ke produksi.
CELL 10 mendokumentasikan dampaknya secara kuantitatif, dan `model_metadata.json`
kini memuat field `feature_order` yang eksplisit sebagai satu-satunya rujukan
urutan fitur. Notebook juga mengekspor batas winsorization per fitur, agar
preprocessing di server identik dengan preprocessing saat pelatihan.

Perbaikan berikutnya menyangkut ukuran dan kejujuran pelaporan. Model produksi
lama berukuran sekitar 75 MB karena dilatih dengan `RandomForestClassifier`
default tanpa `max_depth`, sehingga setiap pohon tumbuh sampai daunnya murni.
Model final dengan `max_depth=10` menghasilkan artefak yang jauh lebih ringan
tanpa kehilangan recall, sekaligus konsisten dengan hyperparameter yang
dilaporkan di skripsi. Terakhir, notebook mengoreksi pelabelan XAI: grafik
importance di V2 diberi judul "SHAP" padahal yang dihitung adalah Permutation
Importance pada model KNN. Notebook 06 menghitung ulang importance pada model
final Random Forest — dengan Permutation Importance dan, bila tersedia, TreeSHAP
yang kali ini memang absah — lalu menuliskan metode sebenarnya pada field
`sumber_metode_importance` di `experiments.json` sehingga label yang tampil di
website sesuai dengan perhitungan yang benar-benar dilakukan.

---

# LANGKAH DEPLOY

### 1. Unduh artefak dari Colab

```python
from google.colab import files
files.download(f'{OUTPUT_DIR}/artefak_produksi_diapredict.zip')
```

Isi arsip: `rf_model.pkl`, `scaler.pkl`, `model_metadata.json`, `experiments.json`.

### 2. Backup artefak lama di repo Laravel

```bash
cp model/rf_model.pkl model/rf_model.pkl.bak
cp model/scaler.pkl   model/scaler.pkl.bak
```

### 3. Salin artefak baru

| Dari arsip | Ke repo Laravel |
|---|---|
| `rf_model.pkl` | `model/rf_model.pkl` |
| `scaler.pkl` | `model/scaler.pkl` |
| `model_metadata.json` | `model/model_metadata.json` |
| `experiments.json` | `public/data/experiments.json` |

### 4. Perbaiki `model/predict.py`

Tiga perubahan wajib:

1. **Urutan fitur** — susun array mengikuti `feature_order` dari
   `model_metadata.json`, bukan mengikuti urutan argumen CLI:

   ```python
   FEATURE_ORDER = ['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level']
   nilai = {'age': age, 'hypertension': hypertension, 'bmi': bmi,
            'HbA1c_level': hba1c_level, 'blood_glucose_level': blood_glucose_level}
   input_data = [[nilai[f] for f in FEATURE_ORDER]]
   ```

2. **Capping winsorization** — terapkan batas dari
   `model_metadata.json -> preprocessing.winsorization` sebelum `scaler.transform`.

3. **Threshold** — ganti nilai lama `0.4965` dengan threshold hasil notebook ini
   (dicetak di CELL 9 dan CELL 14).

### 5. Verifikasi

```bash
python model/predict.py 55 0 28.5 6.8 150
```

Bandingkan `probability` yang keluar dengan angka acuan pada CELL 14
(toleransi 0.0001). Lanjutkan dengan menguji tiga pasien contoh CELL 10 melalui
form website. Kalau ketiganya cocok, artinya model, scaler, urutan fitur, dan
threshold sudah tersinkronisasi antara notebook dan produksi.

### 6. Periksa halaman metodologi

Pastikan halaman "Metodologi & Pengujian" membaca `public/data/experiments.json`,
dan label grafik feature importance memakai `meta.sumber_metode_importance`
sehingga tidak lagi menyebut "SHAP" untuk perhitungan yang bukan SHAP.

---

*Notebook 06 — Model Final & Export untuk Produksi. Bagian dari Revisi Pengujian V3, DiaPredict.*

---
---

# PENUTUP

Seluruh bagian selesai. Yang perlu dilakukan berikutnya:

1. Unduh arsip `artefak_produksi_diapredict.zip` yang dibuat BAGIAN 7.
2. Salin `rf_model.pkl`, `scaler.pkl`, dan `model_metadata.json` ke folder `model/` pada repo Laravel.
3. Salin `experiments.json` ke `public/data/` pada repo Laravel.
4. Jalankan `php artisan cache:clear`, lalu buka halaman **Comparison** dan **Metodologi**.
5. Unduh isi folder `tabel/` dan `gambar/` sebagai lampiran skripsi.
6. Salin blok **RINGKASAN UNTUK SKRIPSI** dari tiap bagian ke Bab 3 dan Bab 4.

Bila angka di atas berasal dari `MODE_CEPAT = True`, ulangi seluruh notebook dengan
`MODE_CEPAT = False` sebelum melaporkannya.